# P4_worker_NSR: optional retention preview

This copy evaluates the first **one PCM realization per fold** by default, at all
configured retention times and compensation modes. It uses completed training and
existing P0 validation-selected B. It does not retrain or run full ablation/diagnostics.

Stop the original worker cell before running this copy; use the same Drive, run ID,
packages and GPU type. Config, scientific functions, fingerprints, seeds and the
planned 30-realization experiment are unchanged. Existing compatible shards are
reused, and newly calculated shards remain reusable by the full retention run.

The last cell controls preview size: `PREVIEW_REALIZATIONS = 1` (or 3).
It visits every fold, then exports clearly labeled PREVIEW CSVs and PNG/PDF charts
in a separate `retention_preview/first_N/` directory. Preview statistics use only
those N realizations; they do not satisfy the full experiment's completeness criteria.
Fewer device samples give less reliable estimates of programming/drift variability.
Historical cell outputs below are not results of this preview.


# DriftSlice / NSR-HWA: complete IEEE ISCAS experiment (Google Colab, GPU)

**Two experiments, selected with `MODEL_ARCHITECTURE` (Section 0):**
* `"resnet32"` (**default**): **ResNet-32 on CIFAR-10**, with 34 PCM-mapped conv/linear layers and digital BatchNorm.
* `"mlp"`: the original **784-250-10 MLP on MNIST**, unchanged.

Each architecture is a separate experiment, with its own run folder, fingerprints, smoke-test record and paper outputs.

**Pipeline (trained from scratch):** dataset download → digital FP32 pretraining (30 epochs) → weight + bias (+ BatchNorm) transfer → AnalogAdam hardware-aware fine-tuning (154 epochs) → best-validation checkpoint → PCM programming → retention (20 s … 5 y) → DriftSlice reference compensation → IEEE figures and Tables I–II.

**Protocol:** 5 independent training repetitions (folds) × 30 PCM programming realizations × retention timestamps. The MLP uses 75 timestamps; the ResNet uses the 12 explicit anchors from 20 s to 5 y. The models are Standard HWA, DAR-HWA (Drift Only), Drift-Attention-HWA, and NSR-HWA, plus an FP32-Direct baseline with zero HWA steps.

**Source of truth.** The scientific code in this notebook is consolidated from the project files `driftslice_core.py` (code revision `2026-10-15-v3-fp32-direct-hwa-selection`), `unified_experiment.py` and `analytic_error_budget.py`. Equations (NSR, DAR, Drift-Attention, PCM noise, DriftSlice) are copied without modification. Every deviation from the original code is listed in the **Change log** section below, and every addition is labelled `ADDED` in the code.

**Scientific-integrity rules enforced by this notebook**
* No previously trained weights are reused. Every checkpoint carries an experiment fingerprint and is refused if the fingerprint does not match.
* Checkpoints and DriftSlice B are selected on **validation data only**. The test split is used only for reporting.
* All statistics are two-stage: first average the 30 programming realizations within a training repetition, then take the mean ± SD (ddof = 1) across the 5 repetitions.
* DriftSlice per-device, class-dependent conductance correction is an **idealized numerical weight-reconstruction experiment** in simulation. It is **not** a demonstrated fabricated circuit. No area, energy, or silicon claims are made.
* Missing results are never filled in. Tables and figures are generated only from the result files on disk, and every number can be traced to a CSV.

**Before the first cell (every new Colab session):** Runtime → Change runtime type → **GPU** (G4, T4, L4 or A100) and **Runtime version 2026.04** (Python 3.12, torch 2.10, CUDA 12.8). IBM ships no CUDA build of AIHWKit 1.1.0 for Colab's current default runtime (Python 3.13, CUDA 13), and Section 1 stops with an explanation there. **On a G4 (Blackwell)**, the first session builds AIHWKit from IBM's official source for that GPU and caches it on Drive (Section 1). Later sessions reuse it.

**How to use.** Run the cells from the top. Leave the expensive flags at `False` until the smoke test reports `SMOKE TEST: PASS`. Then enable `RUN_FULL_TRAINING`, followed by `RUN_FULL_RETENTION`, `RUN_REFERENCE_ABLATION` and `RUN_MECHANISM_DIAGNOSTICS`. If Colab disconnects, re-run all cells: completed epochs, checkpoints and result shards are resumed from Google Drive and are never recomputed or overwritten.

## 0. Execution control

These flags control what actually runs. None of the expensive stages start automatically.

| Flag | What it runs | MLP / MNIST, measured on a reference GPU (RTX 4060, sm_89)* |
|---|---|---|
| `RUN_ENVIRONMENT_SETUP` | AIHWKit 1.1.0 GPU install and verification | T4/L4/A100: a few minutes (wheel ≈ 0.6 GB). **G4: first session builds AIHWKit for Blackwell (≈ 15–60 min), cached on Drive; later sessions ≈ 1 min** |
| `RUN_SMOKE_TEST` | the whole pipeline at reduced size, including the interrupt-and-resume equivalence test | ≈ 1 min |
| `RUN_FULL_TRAINING` | 5 × (30 FP32 + 4 × 154 HWA) epochs | ≈ 0.9 s per FP32 epoch, ≈ 2.6–4.6 s per HWA epoch, so **≈ 3 h**; resumable per epoch |
| `RUN_FULL_RETENTION` | 5 folds × 5 models × 30 realizations × 75 timestamps × 5 compensation modes | ≈ 15–25 s per shard × 750 shards, so **≈ 3–5 h**; resumable per shard |
| `RUN_REFERENCE_ABLATION` | B ∈ {1,2,4,8} × R_tot ∈ {16,64,256}, K = 4, at 5 years | ≈ 5 min |
| `RUN_MECHANISM_DIAGNOSTICS` | Table II diagnostics and Figure 5 device states | ≈ 2 min |
| `GENERATE_PAPER_RESULTS` | Figures 1–5, Tables I–II, source CSVs, reviewer audit | ≈ 1–2 min (reads all shards from Drive) |

**ResNet-32 / CIFAR-10 (`MODEL_ARCHITECTURE = "resnet32"`), measured on the same RTX 4060:**
* FP32 epoch: 13 s.
* HWA epoch: about 206 s (4.6 ms per image).
* One 10k test pass: 14.4 s.
* Smoke test: 28 min (with `SMOKE_BATCH_SIZES = (32, 64)`).

On that GPU, full training would take about 180 h, retention (12 anchors × 4 compensations) about 140 h, the ablation about 60 h and the diagnostics about 3 h. **Use a large Colab GPU (G4 / A100 / L4).** Full ResNet HWA training at batch 256 needs about 13 GB of GPU memory, so a T4 (16 GB) is marginal. Every stage is resumable, so it can span many Colab sessions.

\*Colab GPUs (T4, L4, A100) and Drive I/O differ. Each stage prints its own **benchmark-based estimate** on your runtime before it starts.

In [ ]:
# ============================ EXECUTION CONTROL ============================
# Network / dataset of this run (each architecture is a SEPARATE experiment with its own run folder,
# fingerprints, smoke-test record and paper outputs; switching never mixes or overwrites results):
#   "resnet32" - ResNet-32 (34 analog conv/linear PCM layers, digital BatchNorm) on CIFAR-10
#   "mlp"      - the original 784-250-10 MLP on MNIST (unchanged; identical fingerprints as before)
MODEL_ARCHITECTURE = "resnet32"

# Retention-only execution: training, full ablation, diagnostics and full paper generation disabled.
RUN_ENVIRONMENT_SETUP     = True   # install / verify IBM AIHWKit 1.1.0 (GPU) and CUDA
RUN_SMOKE_TEST            = True   # short end-to-end run; must PASS before full runs (skipped
                                   # automatically when a PASS exists for the identical code + env)
FORCE_SMOKE_TEST          = False  # True: repeat the smoke test even if a current PASS exists
RUN_FULL_TRAINING         = False  # coordinator protocol: 3 folds, 30 FP32 + 100 HWA epochs per assigned method
RUN_FULL_RETENTION        = False  # 30 PCM realizations x 75 timestamps (MLP) / 12 anchors (ResNet) per fold/model
RUN_REFERENCE_ABLATION    = False  # B in {1,2,4,8} x R_tot in {16,64,256}, K = 4, at 5 y
RUN_MECHANISM_DIAGNOSTICS = False  # Table II + Figure 5 device states (matched B = 1)
GENERATE_PAPER_RESULTS    = False   # figures / tables from COMPLETE full-run results only

# Optional training ablations. They are never silently mixed into the primary
# four-method comparison: when enabled they are trained, evaluated and reported
# in separate supplementary rows.
INCLUDE_ATTENTION_ONLY_AND_GAIN_ONLY = False

# ------------------------------- storage ----------------------------------
BASE_DIR_STR = "/content/drive/MyDrive/ISCAS_DriftSlice"   # persistent (Drive)
LOCAL_CACHE_STR = "/content/driftslice_local_cache"        # disposable (Colab VM disk)
FULL_RUN_ID = "driftslice_iscas_colab_fullrun_v1"          # MLP / MNIST experiment
FULL_RUN_ID_RESNET = "driftslice_iscas_colab_resnet32_cifar10_v2"   # ResNet-32 / CIFAR-10 experiment
SMOKE_RUN_ID = "driftslice_iscas_colab_smoke"
SMOKE_BATCH_SIZES = None   # ResNet smoke test only: e.g. (32, 64) on a GPU with < 16 GB memory (the full
                           # protocol uses batch 64 / eval 128, which needs ~13 GB for ResNet-32 HWA)

# ------------------------------- safety -----------------------------------
DRIVE_CHECKPOINT_EVERY_EPOCHS = 1       # resumable state -> Drive after every epoch (requested); a local
                                        # VM mirror is always written every epoch. Larger values (e.g. 5) cut
                                        # Drive write volume; at most N-1 epochs are recomputed after a disconnect.
RESUME_POLICY = "error"                 # a changed TRAINING fingerprint stops with an explanation
REQUIRE_SMOKE_PASS_BEFORE_FULL = True   # full stages refuse to start without a PASS
                                        # recorded for the *current* code + environment
ALLOW_PROVISIONAL_PAPER_OUTPUTS = True  # if results are incomplete, write clearly
                                        # watermarked figures to paper_PROVISIONAL/
                                        # (Tables I/II are never filled with missing data)

# AIHWKit installation policy (see Section 1):
#   "official_gpu"   – install IBM's CUDA-enabled AIHWKit 1.1.0; STOP if impossible
#   "verify_only"    – do not install anything, only verify the current runtime
AIHWKIT_INSTALL_MODE = "official_gpu"
AIHWKIT_ALLOW_SOURCE_BUILD = True  # GPUs without code in IBM's prebuilt wheel (e.g. Colab G4 = Blackwell
                                   # sm_120, H100 = sm_90): build AIHWKit 1.1.0 from IBM's official source
                                   # for this GPU ONCE and cache the wheel on Drive (IBM developer_install.rst:
                                   # set -DRPU_CUDA_ARCHITECTURES to your GPU's architecture)
AIHWKIT_EXACT_TORCH_PIN = False   # True: also replace Colab's torch 2.10.0+cu12x by the exact wheel pin
                                  # 2.10.0+cu126 (~3 GB download + restart; same torch ABI either way)

# Retention option (MLP only): the ORIGINAL all-ones-probe GDC is kept as a documented diagnostic (it
# saturates the fc1 ADC). Setting False skips its test evaluations (~20 % faster retention); the
# saturation-safe GDC, DriftSlice B=1 and validation-selected B are always evaluated. The ResNet protocol
# never evaluates the original GDC (its saturation fraction is still recorded).
RUN_ORIGINAL_SATURATING_GDC = True

# ------------------- parallel Colab runtimes (optional; see Section 13b) -------------------
#   "all"         : this one notebook runs everything (default)
#   "coordinator" : smoke test; FP32 pretraining + FP32-Direct of all folds; after the workers finished
#                   training: validation B selection + pre-PCM table; finally the paper
#   "worker"      : ONLY the models in WORKER_MODELS: their HWA training, then their retention /
#                   ablation / diagnostics shards. One notebook copy + one Colab runtime per worker.
PARALLEL_ROLE = "worker"
WORKER_MODELS = ("PCM-NSR-HWA",)        # worker only, e.g. ("PCM-NSR-HWA",) or ("PCM-HWA", "FP32-Direct")

if MODEL_ARCHITECTURE not in ("mlp", "resnet32"):
    raise ValueError(f"MODEL_ARCHITECTURE must be 'mlp' or 'resnet32', not {MODEL_ARCHITECTURE!r}")

## 1. Google Colab GPU runtime + IBM AIHWKit 1.1.0 (CUDA) installation

**Runtime.** Colab's *default* GPU image currently ships Python 3.13, torch 2.11.0+cu130 and CUDA 13.0 (checked 2026-10-17). IBM provides **no AIHWKit 1.1.0 GPU build** for it. In every new session, open **Runtime → Change runtime type**, choose a GPU and set **Runtime version = 2026.04** (Python 3.12, torch 2.10.0+cu128, CUDA 12.8). On any other runtime this cell **stops and explains**; it never silently falls back to a CPU build or to a different experiment.

**Which AIHWKit build your GPU gets.** IBM's prebuilt GPU wheel (`advanced_install.rst`) contains GPU machine code for **sm_75, sm_80 and sm_89 only, and no PTX**. This was verified with `cuobjdump` on 2026-10-18. A GPU can only run code compiled for its own major architecture, so:

| Colab GPU | Compute capability | AIHWKit route taken by this cell |
|---|---|---|
| T4 | 7.5 | IBM prebuilt GPU wheel (download ≈ 0.6 GB, about 3 min) |
| A100 | 8.0 | IBM prebuilt GPU wheel |
| L4 | 8.9 | IBM prebuilt GPU wheel |
| **G4 (RTX PRO 6000 Blackwell)** | **12.0** | **build from IBM's official source** for sm_120, **once**; the wheel is cached on Drive |
| H100 | 9.0 | build from IBM's official source for sm_90, once; cached on Drive |

**Source build (for G4 / H100).** IBM's developer installation guide (`docs/source/developer_install.rst`) states: "You may need to set `-DRPU_CUDA_ARCHITECTURES` to include the architecture of the GPU in your environment." The cell passes the same CMake flags (`-DUSE_CUDA=ON -DRPU_CUDA_ARCHITECTURES=<cc>`) through scikit-build's `CMAKE_ARGS` with `pip wheel`, rather than IBM's `python setup.py build_ext` command. It:
1. downloads the official PyPI source `aihwkit-1.1.0.tar.gz` and checks its SHA-256;
2. builds it with the runtime's CUDA 12.8 `nvcc` for exactly your GPU (`USE_CUDA=ON`, `RPU_CUDA_ARCHITECTURES=120` on a G4), against the runtime's torch 2.10 and OpenBLAS (installed with `apt` if missing);
3. stores the wheel, its build log and its metadata under `BASE_DIR/aihwkit_source_builds/<python-torch-nvcc-arch-sha>/`.

The first G4 session needs this build, which typically takes 15–60 min depending on the VM's CPU cores. **Later sessions reuse the cached wheel**, which takes about a minute.

**torch.** Every route needs torch from the **2.10 release**, because AIHWKit's C++ extension is ABI-coupled to it. The 2026.04 runtime's `torch 2.10.0+cu128` runs on all Colab GPUs including Blackwell, and is kept. (IBM's wheel metadata pins `+cu126`; that build has **no Blackwell kernels**, so `AIHWKIT_EXACT_TORCH_PIN=True` is refused on Blackwell GPUs such as the G4.) If the runtime's torch is not a CUDA-12 2.10 build, the cell installs the official one and asks for a session restart when necessary.

**Readiness is tested, not assumed.** In a fresh interpreter the cell launches a real AIHWKit CUDA analog-tile kernel and a torch kernel on your GPU. Section 2 repeats the checks in detail.

In [ ]:
# NOTE: this cell must not import torch (it may have to replace torch first).
import os, sys, json, subprocess, shutil, hashlib, platform, time, urllib.request
import importlib.metadata as _md

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
AIHWKIT_VERSION = "1.1.0"
AIHWKIT_WHEEL_BASE = "https://aihwkit-gpu-demo.s3.us-east.cloud-object-storage.appdomain.cloud/"
AIHWKIT_WHEELS = {  # official IBM GPU wheels (advanced_install.rst, master); sizes from HTTP HEAD
    (3, 10): ("aihwkit-1.1.0-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl", 592095416),
    (3, 11): ("aihwkit-1.1.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl", 592106344),
    (3, 12): ("aihwkit-1.1.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl", 592128160),
}
WHEEL_TORCH, WHEEL_TORCHVISION = "2.10.0+cu126", "0.25.0+cu126"   # Requires-Dist of the IBM GPU wheels
WHEEL_TORCH_RELEASE = "2.10."           # the C++ extension is ABI-coupled to the torch 2.10 release
KNOWN_WHEEL_SHA256 = {(3, 12): "16d2f34587c948cbdfaaebf5ec327c6a3da916dd8a3f5f4d0574f8d218c6af72"}  # observed 2026-10-17
TORCH_INDEX = "https://download.pytorch.org/whl/cu126"
# IBM's wheel contains GPU machine code (cubins) for sm_75, sm_80 and sm_89 ONLY -- no PTX (verified with
# cuobjdump, 2026-10-18). A cubin runs on GPUs of the same major version with an equal or higher minor version.
WHEEL_CUBINS = ((7, 5), (8, 0), (8, 9))
# Official IBM source distribution (PyPI) for every other GPU (e.g. Colab G4 = Blackwell sm_120, H100 = sm_90)
AIHWKIT_SDIST = {"file": "aihwkit-1.1.0.tar.gz", "bytes": 644865,
                 "sha256": "10315a8bfde7e35772a3ae0082fc0340f69b8ab61854dee7b81b1f0e82da0544",
                 "url": "https://files.pythonhosted.org/packages/a5/e7/1f1ba5d9317e418d51c90f75ea3d1e1c1003ee87f95b31d2885c07799bb5/aihwkit-1.1.0.tar.gz"}
MIN_NVCC_FOR_CC = {9: (11, 8), 10: (12, 8), 12: (12, 8)}     # nvcc needed to emit native code for that GPU major
SETUP_REPORT = {"in_colab": IN_COLAB, "python": sys.version.split()[0], "platform": platform.platform()}


class EnvironmentIncompatible(RuntimeError):
    pass


def _ver(pkg):
    try:
        return _md.version(pkg)
    except _md.PackageNotFoundError:
        return None


def _run(cmd):
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)


def _nvidia_smi():
    if shutil.which("nvidia-smi") is None:
        return None
    r = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,compute_cap,memory.total",
                        "--format=csv,noheader"], capture_output=True, text=True)
    if r.returncode != 0 or not r.stdout.strip():
        return None
    name, driver, cc, mem = [s.strip() for s in r.stdout.strip().splitlines()[0].split(",")]
    return {"name": name, "driver": driver, "compute_capability": cc, "memory": mem}


def _aihwkit_status():
    """Probe torch/AIHWKit in a *fresh* interpreter (never imports torch into this kernel)."""
    code = ("import json, platform, torch\n"
            "d = {'torch': torch.__version__, 'torch_cuda': torch.version.cuda, 'cuda_available': torch.cuda.is_available(),"
            " 'cxx11_abi': bool(torch._C._GLIBCXX_USE_CXX11_ABI), 'glibc': platform.libc_ver()[1],"
            " 'torch_arch_list': torch.cuda.get_arch_list() if torch.cuda.is_available() else []}\n"
            "try:\n"
            "    d['torch_gpu_ok'] = bool((torch.ones(64, 64, device='cuda') @ torch.ones(64, 64, device='cuda')).sum().item() == 64**3)\n"
            "except Exception as e:\n    d['torch_gpu_ok'] = False; d['torch_gpu_error'] = repr(e)[-300:]\n"
            "try:\n"
            "    import aihwkit\n    from aihwkit.simulator.rpu_base import cuda\n"
            "    d.update(aihwkit=aihwkit.__version__, aihwkit_cuda_compiled=bool(cuda.is_compiled()))\n"
            "    from aihwkit.nn import AnalogLinear\n    from aihwkit.simulator.configs import InferenceRPUConfig\n"
            "    y = AnalogLinear(16, 8, rpu_config=InferenceRPUConfig()).cuda().eval()(torch.ones(4, 16, device='cuda'))\n"
            "    torch.cuda.synchronize(); d['aihwkit_cuda_kernel_ok'] = bool(torch.isfinite(y).all().item())\n"
            "except Exception as e:\n    d.setdefault('aihwkit', None); d['aihwkit_cuda_kernel_ok'] = False; d['aihwkit_error'] = repr(e)[-500:]\n"
            "print(json.dumps(d))")
    r = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True, timeout=1800)
    if r.returncode != 0:
        return {"error": (r.stderr or r.stdout)[-1500:]}
    return json.loads(r.stdout.strip().splitlines()[-1])


def _download(url, dest, expected_size):
    if dest.exists() and dest.stat().st_size == expected_size:
        print("wheel already downloaded:", dest.name)
    else:
        print(f"downloading {url} ({expected_size/1e6:.0f} MB) ...", flush=True)
        tmp = dest.with_suffix(".part")
        with urllib.request.urlopen(url, timeout=120) as r, open(tmp, "wb") as f:
            shutil.copyfileobj(r, f, length=16 << 20)
        tmp.replace(dest)
    size = dest.stat().st_size
    if size != expected_size:
        raise EnvironmentIncompatible(f"Downloaded wheel size {size} != expected {expected_size}; refusing to install.")
    h = hashlib.sha256()
    with open(dest, "rb") as f:
        for chunk in iter(lambda: f.read(16 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(16 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def _install_ibm_prebuilt_wheel(py):
    wheel, size = AIHWKIT_WHEELS[py]
    cache = Path(LOCAL_CACHE_STR) / "wheels"
    cache.mkdir(parents=True, exist_ok=True)
    sha = _download(AIHWKIT_WHEEL_BASE + wheel, cache / wheel, size)
    SETUP_REPORT["aihwkit_wheel"] = {"url": AIHWKIT_WHEEL_BASE + wheel, "bytes": size, "sha256": sha}
    if py in KNOWN_WHEEL_SHA256 and sha != KNOWN_WHEEL_SHA256[py]:
        raise EnvironmentIncompatible(
            f"SHA-256 of {wheel} is {sha}, not the value observed when this notebook was validated "
            f"({KNOWN_WHEEL_SHA256[py]}). IBM may have replaced the file; inspect it before updating KNOWN_WHEEL_SHA256.")
    return cache / wheel


def _find_nvcc():
    for cand in (shutil.which("nvcc"), "/usr/local/cuda/bin/nvcc", *sorted(Path("/usr/local").glob("cuda-*/bin/nvcc"))[::-1]):
        if cand and Path(cand).exists():
            out = subprocess.run([str(cand), "--version"], capture_output=True, text=True).stdout
            for tok in out.replace(",", " ").split():
                if tok.startswith("V") and tok[1:2].isdigit():
                    major, minor = (int(x) for x in tok[1:].split(".")[:2])
                    return str(cand), (major, minor)
    return None, None


def _gxx_major(exe):
    try:
        out = subprocess.run([exe, "-dumpfullversion", "-dumpversion"], capture_output=True, text=True).stdout.strip()
        return int(out.split(".")[0])
    except Exception:
        return None


def _cuda_host_compiler(nv):
    """nvcc rejects too-new GCC host compilers (CUDA 12.8/12.9: GCC <= 14; older 12.x: <= 13).
    Colab 2026.04 (Ubuntu 22.04) ships GCC 11, which is fine; otherwise a supported g++-N is searched for."""
    max_gcc = 14 if nv >= (12, 8) else 13
    default = shutil.which("g++")
    if default and (_gxx_major(default) or 99) <= max_gcc:
        return default, _gxx_major(default)
    for v in range(max_gcc, 9, -1):
        cand = shutil.which(f"g++-{v}")
        if cand:
            return cand, v
    raise EnvironmentIncompatible(
        f"nvcc {nv[0]}.{nv[1]} supports GCC <= {max_gcc} as CUDA host compiler, but the default g++ is "
        f"GCC {_gxx_major(default) if default else 'missing'} and no g++-{max_gcc}..g++-10 was found. "
        f"Install one (e.g. apt-get install g++-{max_gcc}) or use Colab runtime version 2026.04.")


def _openblas_paths():
    """(include dir holding openblas_config.h + cblas.h, libopenblas.so) from ONE installation. They are passed to
    CMake explicitly: Ubuntu keeps the header in /usr/include/<arch>/openblas-pthread, which the sdist's FindOpenBLAS
    does not search, and mixing header/library sets from different prefixes can break compilation."""
    for pre in filter(None, [os.environ.get("CONDA_PREFIX", ""), "/usr"]):
        incs = sorted(Path(pre, "include").rglob("openblas_config.h"))
        libs = sorted(Path(pre).glob("lib*/**/libopenblas.so"))
        if incs and libs:
            return str(incs[0].parent), str(libs[0])
    return None, None


def _ensure_openblas():
    runtime = subprocess.run("ldconfig -p | grep -q libopenblas", shell=True).returncode == 0
    inc, lib = _openblas_paths()
    if not (runtime and inc):
        print("installing OpenBLAS (headers + runtime) with apt ...", flush=True)
        _run(["apt-get", "-qq", "update"])
        _run(["apt-get", "-qq", "install", "-y", "libopenblas-dev"])
        inc, lib = _openblas_paths()
    if inc is None:
        raise EnvironmentIncompatible("OpenBLAS headers/library not found after installing libopenblas-dev.")
    return inc, lib


def _source_built_wheel(py, cc):
    """Build AIHWKit 1.1.0 from IBM's official sdist for THIS GPU once; cache the wheel on Google Drive."""
    arch = f"{cc[0]}{cc[1]}"
    tc = _check_source_build_toolchain(cc)
    nvcc, nv = tc["nvcc"], tuple(int(x) for x in tc["nvcc_version"].split("."))
    torch_ver = _ver("torch")
    key = (f"aihwkit-{AIHWKIT_VERSION}-py{py[0]}{py[1]}-torch{torch_ver}-nvcc{nv[0]}.{nv[1]}-sm{arch.replace(';', '_')}-"
           f"{AIHWKIT_SDIST['sha256'][:12]}")
    if IN_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
    cache = Path(BASE_DIR_STR) / "aihwkit_source_builds" / key
    meta_file = cache / "build_meta.json"
    openblas_inc, openblas_lib = _ensure_openblas()   # runtime dependency of a source-built rpu_base, too
    if meta_file.exists():
        meta = json.loads(meta_file.read_text())
        whl = cache / meta["wheel"]
        if whl.exists() and _sha256(whl) == meta["wheel_sha256"]:
            print(f"Reusing the AIHWKit wheel built earlier for this GPU/toolchain: {whl}")
            SETUP_REPORT["aihwkit_wheel"] = {**meta, "path": str(whl), "reused": True}
            return whl
    work = Path(LOCAL_CACHE_STR) / "aihwkit_build" / key
    shutil.rmtree(work, ignore_errors=True)
    work.mkdir(parents=True)
    sdist = work / AIHWKIT_SDIST["file"]
    _download(AIHWKIT_SDIST["url"], sdist, AIHWKIT_SDIST["bytes"])
    if _sha256(sdist) != AIHWKIT_SDIST["sha256"]:
        raise EnvironmentIncompatible("AIHWKit source archive hash mismatch; refusing to build.")
    # Build tools only (pure Python; torch/CUDA untouched). mypy provides `stubgen`, which the sdist's CMake runs as a
    # post-build step; setuptools>=77 is needed for the sdist's PEP 639 license metadata.
    _run([sys.executable, "-m", "pip", "install", "-q", "scikit-build>=0.18.1", "pybind11>=3.0.1", "ninja", "cmake>=3.18",
          "wheel", "setuptools>=77,<81", "mypy"])      # <81 keeps pkg_resources available to other packages
    if shutil.which("stubgen", path=os.environ.get("PATH", "") + os.pathsep + str(Path(sys.executable).parent)) is None:
        raise EnvironmentIncompatible("`stubgen` (mypy) is required by the AIHWKit build but was not found.")
    torch_prefix = subprocess.run([sys.executable, "-c", "import torch; print(torch.utils.cmake_prefix_path)"],
                                  capture_output=True, text=True, check=True).stdout.strip()
    try:
        import psutil
        ram_gb = psutil.virtual_memory().available / 1e9
    except Exception:
        ram_gb = 8.0
    jobs = max(1, min(os.cpu_count() or 2, int(ram_gb // 6)))     # nvcc needs up to ~6 GB for rpucuda.cu
    host_cxx, host_gcc = tc["host_cxx"], tc["host_gcc"]
    host_cc = shutil.which(Path(host_cxx).name.replace("g++", "gcc")) or str(Path(host_cxx).with_name(
        Path(host_cxx).name.replace("g++", "gcc")))
    env = dict(os.environ, CXX=host_cxx, CC=host_cc,   # ONE toolchain for C, C++ and the CUDA host side
               USE_CUDA="ON", CUDACXX=nvcc, CUDAHOSTCXX=host_cxx, CMAKE_BUILD_PARALLEL_LEVEL=str(jobs),
               OpenBLAS_HOME=openblas_inc,
               CMAKE_ARGS=(f"-DUSE_CUDA=ON -DRPU_BLAS=OpenBLAS -DRPU_CUDA_ARCHITECTURES={arch} "
                           f"-DCMAKE_CUDA_COMPILER={nvcc} -DCMAKE_PREFIX_PATH={torch_prefix} -DBUILD_TEST=OFF "
                           f"-DOpenBLAS_INCLUDE_DIR={openblas_inc} -DOpenBLAS_LIB={openblas_lib}"))
    log = work / "build.log"
    print(f"building (nvcc {nv[0]}.{nv[1]}, {jobs} parallel jobs; typically 15-60 min, ONCE per GPU type -- the "
          f"wheel is then cached on Drive). Log: {log}", flush=True)
    t0 = time.time()
    with open(log, "w") as lf:
        proc = subprocess.Popen([sys.executable, "-m", "pip", "wheel", "--no-build-isolation", "--no-deps", "-v",
                                 "-w", str(work / "dist"), str(sdist)], cwd=work, env=env,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        last = time.time()
        for line in proc.stdout:
            lf.write(line)
            if time.time() - last > 60:
                print(f"  ... still building ({(time.time() - t0) / 60:.0f} min): {line.strip()[:120]}", flush=True)
                last = time.time()
        proc.wait()
    if proc.returncode != 0:
        tail = log.read_text()[-3000:]
        raise EnvironmentIncompatible(f"AIHWKit source build failed (see {log}). Last lines:\n{tail}")
    whl = next((work / "dist").glob("aihwkit-*.whl"))
    cache.mkdir(parents=True, exist_ok=True)
    shutil.copy2(whl, cache / whl.name)
    meta = {"wheel": whl.name, "wheel_sha256": _sha256(cache / whl.name), "sdist_sha256": AIHWKIT_SDIST["sha256"],
            "cuda_arch": arch, "nvcc": f"{nv[0]}.{nv[1]}", "torch": torch_ver, "python": f"{py[0]}.{py[1]}",
            "cuda_host_compiler": f"{host_cxx} (GCC {host_gcc})",
            "build_minutes": round((time.time() - t0) / 60, 1), "built": time.strftime("%Y-%m-%d %H:%M:%S")}
    meta_file.write_text(json.dumps(meta, indent=2))
    shutil.copy2(log, cache / "build.log")
    print(f"built in {meta['build_minutes']} min; cached at {cache}")
    SETUP_REPORT["aihwkit_wheel"] = {**meta, "path": str(cache / whl.name), "reused": False}
    return cache / whl.name


def _install_record_path():
    return Path(LOCAL_CACHE_STR) / "aihwkit_install_record.json"


CUDA_DEV_HEADERS = ("cuda_runtime_api.h", "cublas_v2.h", "cublasLt.h", "curand.h", "cusparse.h", "cusolverDn.h")


def _missing_cuda_headers(nvcc):
    """Headers needed by AIHWKit itself (cuBLAS/cuRAND) and by torch's ATen/cuda headers it includes."""
    root = Path(nvcc).resolve().parent.parent
    dirs = [root / "include", root / "targets" / "x86_64-linux" / "include"]
    return [h for h in CUDA_DEV_HEADERS if not any((d / h).exists() for d in dirs)]


def _check_source_build_toolchain(cc):
    """Everything a source build needs, checked BEFORE anything in the runtime is modified."""
    nvcc, nv = _find_nvcc()
    need = MIN_NVCC_FOR_CC.get(cc[0], (12, 8))
    if nvcc is None or nv < need or nv[0] != 12:
        raise EnvironmentIncompatible(
            f"Building AIHWKit for sm_{cc[0]}{cc[1]} needs a CUDA 12 nvcc >= {need[0]}.{need[1]}; found {nvcc} {nv}. "
            "Use Colab runtime version 2026.04 (CUDA 12.8). Nothing was installed or changed.")
    missing = _missing_cuda_headers(nvcc)
    if missing and IN_COLAB:
        print(f"CUDA development headers missing ({missing}); installing cuda-libraries-dev-{nv[0]}-{nv[1]} with apt ...")
        _run(["apt-get", "-qq", "update"])
        _run(["apt-get", "-qq", "install", "-y", f"cuda-libraries-dev-{nv[0]}-{nv[1]}"])
        missing = _missing_cuda_headers(nvcc)
    if missing:
        raise EnvironmentIncompatible(f"The CUDA toolkit next to {nvcc} lacks development headers {missing} needed to "
                                      "build AIHWKit. Nothing was installed or changed.")
    host_cxx, host_gcc = _cuda_host_compiler(nv)
    return {"nvcc": nvcc, "nvcc_version": f"{nv[0]}.{nv[1]}", "host_cxx": host_cxx, "host_gcc": host_gcc}


def setup_aihwkit_gpu(mode: str):
    gpu = _nvidia_smi()
    SETUP_REPORT["nvidia_smi"] = gpu
    print("Python", sys.version.split()[0], "| GPU:", gpu)
    if gpu is None:
        raise EnvironmentIncompatible(
            "No NVIDIA GPU visible. In Colab: Runtime -> Change runtime type -> select a GPU (G4, T4, L4 or A100).")
    cc = tuple(int(v) for v in gpu["compute_capability"].split(".")[:2])
    if cc < (7, 5):
        raise EnvironmentIncompatible(f"GPU {gpu['name']} (compute capability {gpu['compute_capability']}) is too old.")
    prebuilt_ok = any(cc[0] == c[0] and cc[1] >= c[1] for c in WHEEL_CUBINS)
    SETUP_REPORT["compute_capability"] = gpu["compute_capability"]
    SETUP_REPORT["aihwkit_route"] = "ibm_prebuilt_gpu_wheel" if prebuilt_ok else "ibm_source_build_for_this_gpu"
    status = _aihwkit_status()
    SETUP_REPORT["status_before"] = status
    ready = (status.get("aihwkit") == AIHWKIT_VERSION and status.get("aihwkit_cuda_compiled")
             and status.get("cuda_available") and status.get("aihwkit_cuda_kernel_ok"))
    if ready and (not IN_COLAB or str(status.get("torch", "")).startswith(WHEEL_TORCH_RELEASE)):
        rec = _install_record_path()
        if rec.exists():            # provenance of the install done earlier in this VM session
            SETUP_REPORT["aihwkit_wheel"] = json.loads(rec.read_text()).get("aihwkit_wheel")
        print("AIHWKit", status["aihwkit"], "with working CUDA kernels on this GPU is already installed:", status)
        return status
    if mode == "verify_only" or not IN_COLAB:
        if not IN_COLAB:
            print("Not running in Google Colab: nothing is installed; the current environment is only verified.")
        if not ready:
            raise EnvironmentIncompatible(f"AIHWKit {AIHWKIT_VERSION} with working CUDA kernels is not available here: {status}")
        return status
    py = sys.version_info[:2]
    if py not in AIHWKIT_WHEELS:
        raise EnvironmentIncompatible(
            f"This Colab runtime runs Python {py[0]}.{py[1]} (Colab's current default image: torch 2.11 / CUDA 13). "
            f"AIHWKit {AIHWKIT_VERSION} needs torch 2.10 with CUDA 12: IBM's GPU wheels exist only for CPython 3.10-3.12, "
            "and the official source build used for other GPUs needs the same toolchain. To proceed: Runtime -> Change "
            f"runtime type -> Runtime version '2026.04' (Python 3.12, torch 2.10, CUDA 12.8), keeping your GPU "
            f"({gpu['name']}), then re-run from the top. The experiment was NOT started and nothing was installed.")
    libc = platform.libc_ver()[1] or "0.0"
    if tuple(int(v) for v in libc.split(".")[:2]) < (2, 28):
        raise EnvironmentIncompatible(f"glibc {libc} < 2.28 required by AIHWKit's manylinux_2_28 wheels.")
    toolchain = None
    if not prebuilt_ok:
        if not AIHWKIT_ALLOW_SOURCE_BUILD:
            raise EnvironmentIncompatible(
                f"GPU {gpu['name']} (sm_{cc[0]}{cc[1]}) cannot run IBM's prebuilt AIHWKit CUDA kernels (sm_75/80/89 only) "
                "and AIHWKIT_ALLOW_SOURCE_BUILD=False. Enable the source build or choose a T4/L4/A100 runtime.")
        toolchain = _check_source_build_toolchain(cc)
    if AIHWKIT_EXACT_TORCH_PIN and cc[0] >= 10:
        raise EnvironmentIncompatible(f"AIHWKIT_EXACT_TORCH_PIN=True requests torch {WHEEL_TORCH}, whose CUDA 12.6 build "
                                      f"has no kernels for this Blackwell GPU (sm_{cc[0]}{cc[1]}). Set it to False.")
    # ---- torch: must be the 2.10 release (the C++ extension is ABI-coupled) AND must run on this GPU ----
    torch_now = _ver("torch") or ""
    torch_cuda_major = str(status.get("torch_cuda") or "").split(".")[0]
    needs_restart = "aihwkit" in sys.modules
    keep_torch = (torch_now.startswith(WHEEL_TORCH_RELEASE) and torch_cuda_major == "12" and status.get("torch_gpu_ok")
                  and not AIHWKIT_EXACT_TORCH_PIN)
    if keep_torch:
        print(f"Keeping the runtime's torch {torch_now} (2.10 release, CUDA {status.get('torch_cuda')}, runs on this GPU; "
              f"arch list {status.get('torch_arch_list')}).")
    else:
        want = (("2.10.0+cu128", "0.25.0+cu128", "https://download.pytorch.org/whl/cu128")
                if (cc[0] >= 10 or (cc[0] == 9 and not AIHWKIT_EXACT_TORCH_PIN)) else
                (WHEEL_TORCH, WHEEL_TORCHVISION, TORCH_INDEX))
        print(f"torch {torch_now} -> {want[0]} (official PyTorch index)")
        needs_restart = needs_restart or "torch" in sys.modules
        _run([sys.executable, "-m", "pip", "install", "-q", f"torch=={want[0]}", f"torchvision=={want[1]}",
              "--index-url", want[2]])
    if prebuilt_ok:
        wheel_path = _install_ibm_prebuilt_wheel(py)
    else:
        print(f"GPU {gpu['name']} is sm_{cc[0]}{cc[1]}: IBM's prebuilt wheel only contains sm_75/80/89 GPU code (no PTX), "
              "so its AIHWKit CUDA kernels cannot run here. Following IBM's documented route for other GPUs: building "
              f"AIHWKit {AIHWKIT_VERSION} from the official source with RPU_CUDA_ARCHITECTURES={cc[0]}{cc[1]} "
              f"(nvcc {toolchain['nvcc_version']}, host compiler GCC {toolchain['host_gcc']}).")
        wheel_path = _source_built_wheel(py, cc)
    if _ver("aihwkit") is not None:
        _run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "aihwkit"])   # may be the CPU-only PyPI build
    # --no-deps: the wheel metadata pins torch==2.10.0+cu126 (not on PyPI) and lists build tools as dependencies.
    _run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--force-reinstall", "--no-cache-dir", str(wheel_path)])
    for pkg, mod in (("scipy", "scipy"), ("requests", "requests"), ("tqdm", "tqdm"), ("protobuf", "google.protobuf")):
        if _ver(pkg) is None:
            _run([sys.executable, "-m", "pip", "install", "-q", pkg])
    status = _aihwkit_status()
    SETUP_REPORT["status_after"] = status
    if not (status.get("aihwkit") == AIHWKIT_VERSION and status.get("aihwkit_cuda_compiled")
            and status.get("cuda_available") and status.get("aihwkit_cuda_kernel_ok")
            and str(status.get("torch", "")).startswith(WHEEL_TORCH_RELEASE) and status.get("cxx11_abi")):
        raise EnvironmentIncompatible(f"AIHWKit GPU installation could not be verified on this GPU: {status}")
    rec = _install_record_path()
    rec.parent.mkdir(parents=True, exist_ok=True)
    rec.write_text(json.dumps({"aihwkit_wheel": SETUP_REPORT.get("aihwkit_wheel"),
                               "route": SETUP_REPORT["aihwkit_route"], "status": status}, indent=2, default=str))
    print("Installed and verified in a fresh interpreter (real AIHWKit CUDA kernel launched on this GPU):", status)
    if needs_restart:
        print("\n*** torch/aihwkit were already imported in this kernel. Runtime -> Restart session, then run "
              "all cells again (installation will be skipped). ***")
        raise EnvironmentIncompatible("Session restart required after replacing torch.")
    return status


from pathlib import Path
if RUN_ENVIRONMENT_SETUP:
    setup_aihwkit_gpu(AIHWKIT_INSTALL_MODE)
else:
    print("RUN_ENVIRONMENT_SETUP=False: skipping installation (verification in the next cell still runs).")

## 2. GPU runtime verification (versions + real AIHWKit execution on the GPU)

This cell checks the Python, PyTorch, CUDA runtime, GPU model and memory, and the AIHWKit version and its CUDA backend. It then runs two real analog tests on the GPU:
1. An `InferenceRPUConfig` `AnalogLinear`, which uses the compiled AIHWKit **CUDA** tile (`rpu_base` CUDA kernels).
2. A `TorchInferenceRPUConfig` `AnalogLinear` (the tile type this experiment uses): forward, backward and one `AnalogAdam` step on CUDA, with a CPU vs GPU numerical comparison.

The experiment stops if either test fails.

In [ ]:
import torch
import aihwkit
from aihwkit.simulator.rpu_base import cuda as rpu_cuda


def verify_gpu_runtime() -> dict:
    info = {
        "python": sys.version.split()[0], "torch": torch.__version__, "torch_cuda_runtime": torch.version.cuda,
        "cudnn": torch.backends.cudnn.version(), "cuda_available": torch.cuda.is_available(),
        "aihwkit": aihwkit.__version__, "aihwkit_cuda_compiled": bool(rpu_cuda.is_compiled()),
    }
    if not info["cuda_available"]:
        raise EnvironmentIncompatible("PyTorch CUDA is unavailable; the experiment requires a GPU and will not run on CPU.")
    p = torch.cuda.get_device_properties(0)
    info.update({"gpu": p.name, "gpu_memory_GB": round(p.total_memory / 1e9, 2),
                 "compute_capability": f"{p.major}.{p.minor}", "driver": (SETUP_REPORT.get("nvidia_smi") or {}).get("driver")})
    if info["aihwkit"] != AIHWKIT_VERSION:
        raise EnvironmentIncompatible(f"AIHWKit {info['aihwkit']} found; this notebook requires {AIHWKIT_VERSION}.")
    if not info["aihwkit_cuda_compiled"]:
        raise EnvironmentIncompatible("AIHWKit is CPU-only (rpu_base.cuda.is_compiled() is False). Re-run Section 1.")
    from aihwkit.nn import AnalogLinear
    from aihwkit.simulator.configs import InferenceRPUConfig, TorchInferenceRPUConfig, FloatingPointRPUConfig
    from aihwkit.optim import AnalogAdam
    from torch.profiler import profile, ProfilerActivity
    dev = torch.device("cuda")
    # (0) torch CUDA/NVRTC sanity (Colab issue #6111)
    if torch.tensor([2, 3, 4], device=dev).prod().item() != 24:
        raise EnvironmentIncompatible("torch CUDA reduction sanity check failed (NVRTC); runtime is broken.")
    # (1a) compiled AIHWKit CUDA tile: floating-point tile CPU vs GPU
    torch.manual_seed(0)
    fp_cpu = AnalogLinear(64, 32, bias=False, rpu_config=FloatingPointRPUConfig())
    fp_gpu = AnalogLinear(64, 32, bias=False, rpu_config=FloatingPointRPUConfig())
    fp_gpu.set_weights(fp_cpu.get_weights()[0])
    fp_gpu = fp_gpu.to(dev)
    xf = torch.randn(8, 64)
    with torch.no_grad():
        fp_diff = float((fp_cpu(xf) - fp_gpu(xf.to(dev)).cpu()).abs().max())
    fp_cls = type(next(fp_gpu.analog_tiles()).tile).__name__
    # (1b) compiled AIHWKit CUDA inference tile, and proof that aihwkit's own RPU:: kernels execute
    layer = AnalogLinear(256, 128, bias=True, rpu_config=InferenceRPUConfig()).to(dev).eval()
    tile_cls = type(next(layer.analog_tiles()).tile).__name__
    xx = torch.rand(64, 256, device=dev)
    with torch.no_grad():
        y = layer(xx)
        torch.cuda.synchronize()
        try:   # kernel-level evidence; the profiler itself is optional (CUPTI may be unavailable)
            import warnings
            warnings.filterwarnings("ignore", message=".*Profiler clears events.*")
            with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
                layer(xx)
                torch.cuda.synchronize()
            kernels = {e.name for e in prof.events() if getattr(e, "device_type", None) is not None}
        except Exception as exc:
            kernels = set()
            print("NOTE: torch profiler unavailable:", type(exc).__name__, exc)
    rpu_kernels = sorted(k for k in kernels if "RPU::" in k)
    info["rpu_cuda_kernels_seen"] = len(rpu_kernels) if rpu_kernels else "not observable (profiler/CUPTI)"
    ok1 = (y.is_cuda and torch.isfinite(y).all().item() and "Cuda" in tile_cls and "Cuda" in fp_cls
           and fp_diff < 1e-4)
    # (2) torch tile used by the experiment: CPU vs GPU, then backward + AnalogAdam on GPU
    rc = TorchInferenceRPUConfig()
    rc.forward.is_perfect = True
    torch.manual_seed(1)
    cpu_layer = AnalogLinear(784, 250, bias=True, rpu_config=rc).eval()
    gpu_layer = copy.deepcopy(cpu_layer).to(dev).eval()
    x = torch.rand(64, 784)
    with torch.no_grad():
        diff = float((cpu_layer(x) - gpu_layer(x.to(dev)).cpu()).abs().max())
    gpu_layer.train()
    opt = AnalogAdam(gpu_layer.parameters(), lr=1e-2)
    opt.regroup_param_groups(gpu_layer)
    w0 = next(gpu_layer.analog_tiles()).tile.weight.detach().clone()
    loss = gpu_layer(x.to(dev)).square().mean()
    loss.backward()
    opt.step()
    w1 = next(gpu_layer.analog_tiles()).tile.weight.detach()
    ok2 = diff < 1e-4 and w1.is_cuda and float((w1 - w0).abs().max()) > 0
    info.update({"cuda_tile_class": tile_cls, "torch_tile_cpu_gpu_max_diff": diff,
                 "torch_tile_on_cuda": bool(w1.is_cuda), "analogadam_step_changed_weights": bool((w1 != w0).any())})
    record_check_early("AIHWKit GPU availability", bool(ok1 and ok2),
                       f"AIHWKit {info['aihwkit']} CUDA-compiled; {fp_cls} CPU/GPU max|Δ|={fp_diff:.1e}; {tile_cls} "
                       f"inference on {info['gpu']} (aihwkit RPU:: CUDA kernels seen by profiler: {info['rpu_cuda_kernels_seen']}); "
                       f"TorchInferenceTile CPU/GPU max|Δ|={diff:.1e}; AnalogAdam step on CUDA updated weights")
    return info


_EARLY_CHECKS = []


def record_check_early(name, ok, detail):
    _EARLY_CHECKS.append((name, ok, detail))
    print(f"CHECK {'PASS' if ok else 'FAIL'} | {name} | {detail}")
    if not ok:
        raise EnvironmentIncompatible(f"{name} failed: {detail}")


import copy
RUNTIME_INFO = verify_gpu_runtime()
DEVICE = torch.device("cuda")
for k, v in RUNTIME_INFO.items():
    print(f"  {k:<34s} {v}")

## 2b. Google Drive: persistent storage

Checkpoints, result shards, logs and figures are stored under `BASE_DIR` on Google Drive. MNIST lives in `BASE_DIR/data/MNIST/raw`. Disposable files (for example the 592-MB wheel) go to the Colab VM disk.

In [ ]:
from pathlib import Path

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path(BASE_DIR_STR)
else:
    # Outside Colab (developer testing only) an explicit location must be given.
    if "DRIFTSLICE_BASE_DIR" not in os.environ:
        raise EnvironmentIncompatible("This notebook targets Google Colab. Outside Colab set DRIFTSLICE_BASE_DIR.")
    BASE_DIR = Path(os.environ["DRIFTSLICE_BASE_DIR"])
    LOCAL_CACHE_STR = os.environ.get("DRIFTSLICE_LOCAL_CACHE", str(BASE_DIR / "_local_cache"))

DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "results"
LOCAL_CACHE = Path(LOCAL_CACHE_STR)
for d in (BASE_DIR, DATA_DIR, OUTPUT_DIR, LOCAL_CACHE):
    d.mkdir(parents=True, exist_ok=True)

# Provenance of the original project files this notebook consolidates (SHA-256 of the files as provided).
ORIGINAL_SOURCE_PROVENANCE = {
    "driftslice_core.py": "f1736469cacc706124ce51b2a3f698fefbf3f092810fac76dcf7130b07a59e78",
    "unified_experiment.py": "d3f895f4a69ccc5758bf071d8c2785a99e3ca60c78c6bb567f67691f08f0a65b",
    "analytic_error_budget.py": "ed7d5a6acc7096dc79622f1a6da0eb69ce98b7f3a46918e40f3689e6322c9299",
    "Unified_DriftSlice_FP32_HWA_Retention.ipynb": "57cc8d21fa615312c93d97be96fc9e854a0d66cfbf68c256fcb54be08bba2c37",
    "original_code_revision": "2026-10-15-v3-fp32-direct-hwa-selection",
}
atomic_setup = BASE_DIR / "environment_setup_report.json"
atomic_setup.write_text(json.dumps({**SETUP_REPORT, "runtime": RUNTIME_INFO}, indent=2, default=str))
print("BASE_DIR  :", BASE_DIR)
print("DATA_DIR  :", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("LOCAL     :", LOCAL_CACHE)

## Change log: every difference from the original source code

**Scientific equations (NSR, DAR, Drift-Attention, PCM noise, DriftSlice, GDC): unchanged.** Hyperparameters, architecture, AIHWKit configuration, seeds, preprocessing and the data split are also unchanged. The list below is exhaustive.

| # | Original behaviour | Change in this notebook | Why / effect on results |
|---|---|---|---|
| C1 | `train_hwa_all` resume: `model.load_state_dict(cpu_state)` on a CUDA model | `model.to(device)` is called after every `load_state_dict` | **Correction.** In AIHWKit 1.1.0, loading a CPU-mapped state resets the tile's device bookkeeping to CPU, while the tensors stay on CUDA (verified by source audit). The values and the gradient hook (`set_scales` is not re-run by `.cuda()`) are unchanged. The smoke test checks that interrupted-then-resumed training is identical to uninterrupted training. |
| C2 | FP32 pretraining saved only at the end | Resumable checkpoint after **every** epoch, plus a separate final-epoch model | Resumability only; the optimization is identical. |
| C3 | HWA checkpoint written every epoch (single file, rename-over) | Two alternating slots (`*.slotA.pt` / `*.slotB.pt`) **overwritten in place**, plus a local VM mirror every epoch; Drive every `DRIVE_CHECKPOINT_EVERY_EPOCHS` (default 1). The DAR reference weights (`initial_refs`) are recomputed on resume and checked by hash instead of being stored | Survives a slot that was only partly synced at a disconnect (the other slot is intact). No rename/delete, so superseded checkpoints do not fill Google Drive's Trash. |
| C4 | `Subset(TensorDataset)` DataLoader | `FastGPULoader`: the same sampler and the same `torch.Generator` object, iterated by an index-only DataLoader, with batches gathered from GPU tensors. It reproduces the DataLoader's per-iteration base-seed draw (from the loader generator, or the **global CPU RNG** for val/test) | Performance only (roughly 20× faster evaluation). For every fold the check compares, against the original DataLoader, **bit-identical batches over 2 epochs and identical global-RNG / generator states afterwards**. The global-RNG state matters because, for example, the Drift-Attention gate is initialized right after two validation passes. |
| C5 | HWA history without training accuracy | Training accuracy, optimizer-step count, first-step max\|Δw\|, digital-checkpoint identity, and an exact re-evaluation of the restored best checkpoint | Logging and verification only. |
| C6 | Retention shard = one realization containing all methods | Shard = one (fold, model, realization) | Same seeds (`root+10` etc. depend only on fold/realization), so the programming noise is still paired across models. It also allows adding a model without recomputing the others. |
| C7 | Compensations `none`, `GDC`, `DriftSlice(selected B)` | Adds a **`DS-B1`** control (same reference seed and reads as the selected B) and a **saturation-safe GDC**. The original GDC is kept and its ADC saturation fraction is reported | Requested matched-compensation comparisons. The saturation-safe GDC's K calibration reads deliberately reuse the original GDC seeds (`root+850000+31i`, paired with the original GDC). Only the probe-amplitude search and the saturation diagnostic draw new randomness, from a separate namespace (`5·10⁷ + root`). The original streams are unchanged. |
| C8 | Fast mode: 1 training repetition | Fast mode: 2 repetitions | The smoke test exercises the across-fold SD; the full protocol is unaffected. |
| C9 | No split guard | `require_split()` ledger: checkpoint and B selection may read only validation; reporting may read only test | Leakage audit; no numerical effect. |
| C10 | Single-run fingerprint over all config fields and module bytes | Hierarchical fingerprints (train → retention → ablation/diagnostics) over the **registered** result-determining functions and classes (AST-normalized), the **values** of the module constants (T_REF, YEAR, DAY, ADDED_SEED_BASE, COMPENSATIONS), the stage-relevant configuration, the data hash and the versions. Results go into fingerprint-keyed directories. The smoke-test gate hashes **all** notebook code | Editing plotting or reporting code never forces retraining. Editing a registered scientific function or constant moves results into a new directory, so they cannot mix. Residual limitation: an edit to an *unregistered* helper is caught only by the smoke gate. Keep result-determining code inside `@scientific` functions. |
| C11 | `B` sweep at fixed R_tot = 64 (validation) | Also a test-split report for R_tot ∈ {16, 64, 256}, K = 4, on the same programmed arrays | Requested Fig. 4. Deployment still uses the original validation-only selection. |
| C12 | Original GDC always evaluated | `RUN_ORIGINAL_SATURATING_GDC` (default `True`) can skip the evaluation of the original all-ones-probe GDC | Run time only. Its saturation fraction is still recorded. The setting is part of the **retention** fingerprint only, so toggling it after training never forces retraining. |
| C13 | `_read_idx()` re-reads the IDX files on every call | `_read_idx()` is a thin in-process memo around the unchanged original reader (`_read_idx_original`) | Same bytes give the same tensor; this avoids re-reading MNIST from Drive for every fold and stage. |
| C14 | One global result list | Scientific checks are recorded per **check session** (smoke / full_training / full_retention / full_ablation / full_diagnostics / paper) and saved under `run_dir/checks/`. Smoke scopes carry a `smoke:` prefix. The paper report merges the latest report of every full stage | Smoke-test results can never appear as full-run results. |
| C15 | (no smoke test) | The smoke test writes its outputs to the VM disk and only its PASS record to Drive. It is skipped automatically when a PASS exists for the identical notebook code and environment (`FORCE_SMOKE_TEST` repeats it) | Re-running the notebook after a disconnect does not repeat the smoke test or fill Drive. |
| C17 | (installation not part of the original code) | GPUs without code in IBM's prebuilt wheel (Colab **G4** = Blackwell sm_120, H100 = sm_90) get AIHWKit 1.1.0 **built from IBM's official PyPI source** for exactly that GPU (`RPU_CUDA_ARCHITECTURES=<cc>`, CUDA 12.8 nvcc, torch 2.10, OpenBLAS), once per toolchain, with the wheel cached on Drive | Same AIHWKit 1.1.0 source, so the same simulator code. Readiness is proven by launching a real AIHWKit CUDA kernel. Build log, sdist SHA-256, nvcc, host compiler, torch and arch are recorded. |
| C18 | (implicit PyTorch defaults) | `allow_tf32=False` for matmul and cuDNN, `float32_matmul_precision="highest"` | Guarantees true FP32 arithmetic on Ampere/Ada/Hopper/Blackwell GPUs, identical to the original FP32 intent. Matmul TF32 is already off by default. |
| C16 | Paper outputs overwritten in place | FINAL `paper/` requires **complete** training, retention, ablation and diagnostics. Otherwise outputs go to `paper_PROVISIONAL/` (watermarked; each artifact uses only its complete repetitions; captions state the actual F and R). A previous output folder is renamed `*_superseded_<time>`, never mixed or deleted | No partial data in final figures or tables, and no stale files. |


### ResNet-32 / CIFAR-10 extension (`MODEL_ARCHITECTURE = "resnet32"`; ADDED, the original code is MLP/MNIST only)

The MLP experiment is **unchanged**: its registered functions, configuration and fingerprints are byte-identical to the previous version of this notebook. The ResNet experiment is a separate experiment, with its own run id, fingerprint components (`rn_*`), smoke-test record and outputs. It reuses **every** scientific function that does not depend on the network: `make_hwa_rpu`, `dar_penalty`, `nsr_penalty`, `AttentionGate`, `validation_score`, `PCMPhysicalMatrix`, `PCMClassReferenceBank`, `apply_driftslice`, the saturation-safe GDC, `exact_retention_times` and the statistics. **All equations, hyperparameters, seeds and the protocol (5 folds × 30 FP32 + 4 × 154 HWA epochs, 30 realizations) are the same as for the MLP**, except for the items below.

| # | Item | ResNet experiment | Why |
|---|---|---|---|
| R1 | Network | ResNet-32 (He et al. CIFAR variant, 3 stages × 5 basic blocks, widths 16/32/64, 1×1-conv shortcuts where the shape changes, global average pooling). The **34 conv/linear layers are AIHWKit `AnalogConv2d`/`AnalogLinear` tiles** with the identical `make_hwa_rpu()` configuration. BatchNorm, ReLU, pooling and residual additions are **digital** | The standard AIMC-HWA ResNet benchmark. BatchNorm is not mapped to PCM, as in the AIHWKit examples. |
| R2 | Data | CIFAR-10, normalized per channel; same 90/10 `seed+fold` split rule (45k/5k/10k) | Requested dataset. |
| R3 | Augmentation | Training batches only: pad-4 random crop + horizontal flip, from a dedicated generator seeded by (seed, fold, phase, epoch, batch). HWA methods share the identical augmented batches | Without it, a 30-epoch ResNet overfits CIFAR-10. Paired across methods; the global RNG is untouched, so resume stays bit-identical. |
| R4 | Drift-Attention gate | Same gate equation on the flattened normalized 3×32×32 image (dim 3072, MLP 784) | The gate acts on the network input, as in the MLP. |
| R5 | Guards | FP32 underfit guard: 80 % validation accuracy (MLP 90 %). Transfer guard: the transferred network evaluated with **ideal periphery** (no quantization, noise, bound or clipping) must match FP32 within 0.25 pp (MLP: realistic analog model within 5 pp). The realistic pre-HWA accuracy is still printed | Measured on fold 0, 30 FP32 epochs: FP32 90.04 %, ideal-periphery analog 90.04 %, realistic analog before HWA 50.4 %. Layer inputs reach \|x\| ≈ 15, while the initial input range is 3.0. HWA learns the range from data (`init_from_data = 100`); with the range set from data the realistic analog model gives 88.8 %. A realistic-model guard would therefore test the untrained input range, not the transfer. Guards only stop a broken run; they never select anything. |
| R5b | FP32-Direct (ResNet) | Unchanged definition: transferred weights, **zero** HWA steps, so the input ranges keep their initial value 3.0 | Expect FP32-Direct to be poor for ResNet even at 20 s (about 50 % on validation), mostly from input clipping rather than PCM drift. Read it as "no HWA at all", not as "an FP32 model with calibrated periphery". |
| R6 | Retention grid | The **12 explicit anchors** (20 s, 60 s, 30 min, 1 h, 1 d, 5 d, 30 d, 180 d, 1 y, 2 y, 3 y, 5 y), MLP 75 | Run time: one ResNet-32 pass over the 10k test images takes 14.4 s on an RTX 4060 (measured), and the protocol needs 4 compensations × 750 shards per timestamp. `exact_retention_times(cfg)` with 12 points returns exactly the anchors. |
| R7 | Original GDC | Not evaluated (its 20-s ADC saturation fraction is still recorded for every layer). The saturation-safe GDC, DS-B1 and DS-val are evaluated | Requested. The original all-ones probe saturates, so its value is not a valid calibration. |
| R8 | Deployment | The PCM effective weights are **substituted into the AIHWKit tiles** (eval mode) and the network runs through AIHWKit's own conv/linear forward: the same input range, DAC/ADC, `AnalogMVM` and output scales as `shared_mvm`. A GDC gain multiplies the learned output scale (before the digital bias, as in `shared_mvm`). Weights are restored afterwards | A conv layer is an MVM over unfolded patches. A check verifies, per fold, (a) bit-identical outputs for self-substitution, (b) exact restoration, (c) agreement of the fc layer with `shared_mvm` under the MLP's criterion, and (d) the GDC-gain algebra. |
| R9 | Per-layer seeds | Programming `seed_base+101·(i+1)`, read `seed+1009·i`, GDC-safe calibration `+8.5e5+31·t+i`, probe `5e7+root+100·i` for layer i | The MLP's rules (+101/+202, +0/+1009, +0/+1, +0/+100) generalized to 34 layers. One reference bank per realization is shared by all layers, as in the MLP. |
| R10 | Table II / Fig. 5 | Layer groups `conv` (33 conv layers pooled) and `fc`. Per-weight means are pooled over all weights; NSR is the mean of the per-layer NSR, as in `nsr_penalty`; weight errors are normalized by each layer's own max\|W\| | 34 per-layer columns would not fit in a paper table. The per-layer data are fully determined by the stored seeds. |
| R10b | Inactive FP32 filters | At transfer, FP32 conv filters whose max\|w\| is below **1 % of the layer's largest filter** are mapped as **exact zeros** (`flush_dead_filters`). The count per layer is printed and recorded in `weight_bias_transfer_errors.csv` | FP32 training (Adam + coupled L2) shrinks inactive channels towards 0, down to about 1e-11. With columnwise ω-scaling, every filter becomes a full-scale tile row, and AIHWKit's gradient hook (grad / mapping_scale²) amplifies the gradient of a small row by (layer max / row max)². Collapsed rows reach about 1e21. DAR and NSR act on tile weights, so they overflowed float32: run `…_v1` stopped with a non-finite gradient in DAR-HWA epoch 2. Below the overflow, they dominated the 0.8 gradient-norm clipping and froze all other weights. AIHWKit maps an exactly-zero row with scale 1. **Measured on fold 0:** 56 of 1,232 filters flushed; FP32 accuracy unchanged on validation (90.04 → 90.04 %) and on test (89.64 → 89.64 %); tile-weight gradient norms CE 15.1, DAR 7.4 (8.8·10⁷ before), NSR 0.09 (7.6·10⁴ before). The ideal-periphery guard (0.25 pp) stops the run if flushing ever costs accuracy, and a preflight regression test checks that the CE+NSR+DAR gradient is finite. |
| R11 | cuDNN | `cudnn.deterministic=True`, `benchmark=False` (global; the MLP has no convolutions, so it is unaffected) | Bit-identical resume of FP32 ResNet pretraining. |
| R12 | Smoke test | `SMOKE_BATCH_SIZES` (default `None`) can shrink only the smoke-test batches on a small GPU | The full protocol always uses 256 / 512. |

**Original code that is not executed.**
* `run_legacy_experiment()`, `train_analog_method_legacy()`, `retention_realization_legacy()` and `driftslice_bin_ablation_legacy()` are not used. `run_legacy_experiment()` calls `train_analog_method`, `retention_realization` and `driftslice_bin_ablation`, which do not exist after the rename, so it would raise `NameError`. The authoritative path is `unified_experiment.run()`, which this notebook reproduces.
* `analytic_error_budget.py` imports `legacy_reference/headroom_original.py`, which is **not in the project**. That auxiliary analysis therefore cannot be reproduced, and no substitute formulas are invented.
* The `hwa_early_stop_*` fields are unused, as in the original unified path: all 154 epochs always run.

**Facts verified in AIHWKit 1.1.0 (no code change needed).**
* `PCMLikeNoiseModel` uses μ_ν = clip(−0.0155·ln g + 0.0244, 0.049, 0.1). This matches DAR's `_sample_pcm_nu`.
* Output noise is also applied in eval mode, which is why the original uses 3 validation reads.
* Post-update weight clipping is active because `model.parameters()` puts the `AnalogContext` parameters into the AnalogAdam groups.
* `set_weights()` registers a gradient hook that divides the tile-weight gradient by mapping_scales² (AIHWKit design, unchanged).
* The original seed offsets (`+850000+31i`, `+700000+31i`, …) can coincide across realizations that are 15 or more apart. These are different physical quantities, so the effect is negligible, but the original convention is kept as is.

## 3. Scientific core: configuration, provenance, reproducibility utilities

The `Config` dataclass keeps **every original field and default** from `driftslice_core.Config`. The fields added for this notebook are grouped and labelled `ADDED`. Fast mode (the smoke test) shrinks the run in the same way as the original `Config.resolved()`. The one difference is that it uses 2 training repetitions instead of 1, so that the across-repetition statistics code is also exercised.

**Provenance.** Each scientific function or class is registered with `@scientific(<component>)`. A component hash is the SHA-256 of the AST-normalized source with comments and docstrings removed. These hashes feed hierarchical fingerprints:

* `TRAIN_FP` = data + model + training code, training hyperparameters, the MNIST hash, and the torch/AIHWKit versions.
* `RET_FP` = `TRAIN_FP` + PCM + retention code and settings.
* `ABL_FP`, `DIAG_FP` are built the same way for the ablation and diagnostics stages.

Changing only the plotting code therefore never invalidates trained checkpoints. Changing a training equation always does.

In [ ]:
from __future__ import annotations

import ast
import copy
import gzip
import hashlib
import inspect
import io as _io
import json
import math
import os
import platform
import random
import shutil
import sys
import textwrap
import time
import traceback
from collections import OrderedDict
from contextlib import contextmanager
from dataclasses import dataclass, asdict, replace, fields
from pathlib import Path
from typing import Dict, Tuple, Optional, List

import numpy as np
import pandas as pd
import torch
from torch import nn, Tensor
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

# ADDED: pin full-FP32 matrix arithmetic on every GPU generation (Ampere/Ada/Hopper/Blackwell support TF32).
# Matmul TF32 is already off by default in PyTorch; it is set explicitly so that a G4 run and a T4 run use the same
# arithmetic as the original FP32 code. cuDNN TF32 (on by default) only affects convolutions -- none in the MLP;
# the FP32 ResNet-32 pretraining uses cuDNN convolutions -- and is disabled. Deterministic cuDNN algorithms make
# resumed ResNet training bit-identical to uninterrupted training (verified by the smoke test).
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.set_float32_matmul_precision("highest")

# --------------------------------------------------------------------------
# Scientific-source registry (ADDED): every function/class that can change a scientific
# result is registered, so that stage fingerprints follow code changes. Module constants
# that enter results (T_REF, YEAR, DAY, ADDED_SEED_BASE, COMPENSATIONS) are hashed BY VALUE
# in compute_fingerprints(); the smoke gate additionally hashes ALL notebook code.
# --------------------------------------------------------------------------
_SCIENTIFIC_REGISTRY: "OrderedDict[str, tuple]" = OrderedDict()


def scientific(component: str):
    def deco(obj):
        _SCIENTIFIC_REGISTRY[obj.__qualname__] = (component, obj)
        return obj
    return deco


def _strip_docstrings(tree):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef, ast.Module)):
            body = node.body
            if body and isinstance(body[0], ast.Expr) and isinstance(getattr(body[0], "value", None), ast.Constant) \
                    and isinstance(body[0].value.value, str):
                node.body = body[1:] or [ast.Pass()]
    return tree


def _normalized_function_source(fn) -> str:
    """AST-normalized source (comments/docstrings/formatting removed)."""
    try:
        src = textwrap.dedent(inspect.getsource(fn))
        return ast.unparse(_strip_docstrings(ast.parse(src)))
    except (OSError, TypeError, SyntaxError):
        code = getattr(fn, "__code__", None)
        if code is None or code.co_filename.startswith("<"):
            return ""          # e.g. dataclass-generated __init__: derived from fields
        import marshal
        return "marshal:" + hashlib.sha256(marshal.dumps(code.replace(co_filename="<nb>", co_firstlineno=1))).hexdigest()


def _object_source(obj) -> str:
    if inspect.isclass(obj):
        parts = [f"class {obj.__qualname__}({','.join(b.__qualname__ for b in obj.__bases__)})"]
        dataclass_fields = set(getattr(obj, "__dataclass_fields__", {}))  # values hashed per stage
        for name, value in sorted(vars(obj).items()):
            if name in dataclass_fields:
                continue
            fn = value.__func__ if isinstance(value, (staticmethod, classmethod)) else value
            fn = fn.fget if isinstance(fn, property) else fn
            if inspect.isfunction(fn):
                parts.append(f"def {name}: " + _normalized_function_source(fn))
            elif not name.startswith("__"):
                parts.append(f"{name} = {value!r}")
        return "\n".join(parts)
    return _normalized_function_source(obj)


def component_source_hash(components) -> str:
    h = hashlib.sha256()
    for qualname, (component, obj) in sorted(_SCIENTIFIC_REGISTRY.items()):
        if component in components:
            h.update(f"[{component}] {qualname}\n".encode())
            h.update(_object_source(obj).encode())
    return h.hexdigest()


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def stable_hash(obj) -> str:
    return sha256_bytes(json.dumps(obj, sort_keys=True, default=str).encode())


def state_dict_hash(state) -> str:
    """Content hash of a (possibly nested) state dict; tensors hashed by value."""
    h = hashlib.sha256()

    def walk(prefix, value):
        if isinstance(value, torch.Tensor):
            t = value.detach().cpu().contiguous()
            h.update(prefix.encode()); h.update(str(t.dtype).encode()); h.update(str(tuple(t.shape)).encode())
            h.update(t.numpy().tobytes() if t.dtype != torch.bfloat16 else t.float().numpy().tobytes())
        elif isinstance(value, dict):
            for k in sorted(value.keys(), key=str):
                walk(f"{prefix}/{k}", value[k])
        elif isinstance(value, (list, tuple)):
            for i, v in enumerate(value):
                walk(f"{prefix}[{i}]", v)
        elif isinstance(value, (int, float, str, bool)) or value is None:
            h.update(f"{prefix}={value!r}".encode())
    walk("", state)
    return h.hexdigest()


# --------------------------------------------------------------------------
# Original constants (driftslice_core.py / unified_experiment.py)
# --------------------------------------------------------------------------
DAY = 86400.0
YEAR = 365.25 * DAY
T_REF = 20.0

METHOD_HWA = "PCM-HWA"
METHOD_DAR = "PCM-DAR-HWA"
METHOD_DS = "DriftSlice-PCM"
DIRECT = "FP32-Direct"
HWA = METHOD_HWA
DAR = METHOD_DAR
ATT = "PCM-DriftAttention-HWA"
NSR = "PCM-NSR-HWA"
ATT_ONLY = "PCM-AttentionOnly-HWA"
GAIN_ONLY = "PCM-GainOnly-HWA"
ALL_TRAIN_METHODS = (HWA, DAR, ATT, NSR, ATT_ONLY, GAIN_ONLY)
PRIMARY_METHODS = (HWA, DAR, ATT, NSR)
EXTRA_METHODS = (ATT_ONLY, GAIN_ONLY)

PAPER_LABEL = {
    DIRECT: "FP32-Direct",
    HWA: "Standard HWA",
    DAR: "DAR-HWA (Drift Only)",
    ATT: "Drift-Attention-HWA",
    NSR: "NSR-HWA (proposed)",
    ATT_ONLY: "AttentionOnly-HWA (ablation)",
    GAIN_ONLY: "GainOnly-HWA (ablation)",
}

SCHEMA = "driftslice-colab-iscas-v4"
CODE_REVISION = "2026-10-17-colab-v4 (from driftslice_core 2026-10-15-v3-fp32-direct-hwa-selection)"


@scientific("config")
@dataclass
class Config:
    # Reproducibility / storage
    seed: int = 42
    fast_mode: bool = False
    require_cuda: bool = True
    run_id: str = "driftslice_smoke_001"
    output_root: str = "driftslice_verified_runs"
    resume_policy: str = "auto_new"   # "auto_new" or "error"
    data_root: str = "data/MNIST/raw"
    allow_download: bool = True

    # Requested training semantics
    training_semantics: str = "fp32_pretrain_then_analog_hwa"
    digital_epochs: int = 30  # reconstruction assumption: paper does not specify digital epochs
    digital_lr: float = 1e-3
    digital_max_lr: float = 0.004
    digital_min_val_accuracy: float = 90.0  # validation only; full MNIST
    transfer_max_val_drop_pp: float = 5.0
    hwa_early_stop_drop_pp: float = 8.0
    hwa_early_stop_patience: int = 2
    batch_size: int = 64
    test_batch_size: int = 128
    epochs: int = 100
    train_repeats: int = 3
    lr: float = 1e-3
    max_lr: float = 0.004
    weight_decay: float = 1e-4
    grad_clip: float = 0.8

    # Paper-aligned HWA
    weight_scaling_omega: float = 0.8
    bits: int = 8
    weight_noise_std: float = 0.02
    output_noise_std: float = 0.04  # standard-HWA preset value
    output_bound: float = 10.0
    input_range_init: float = 3.0
    gmax_us: float = 25.0
    ir_drop: float = 0.0

    # DAR: paper values + explicitly undisclosed conventions
    dar_lambda_init: float = 0.5
    dar_fraction: float = 0.80
    dar_tc: float = 1e-2
    dar_eta: float = 1.0                     # NOT disclosed by paper
    dar_delta_mode: str = "signed_initial"   # NOT disclosed
    dar_g_units: str = "siemens"             # NOT disclosed: "siemens" or "normalized"
    dar_time_mode: str = "literal_seconds"   # source-literal interpretation
    dar_times_s: Tuple[float, ...] = (
        20.0, 60.0, 1800.0, 3600.0, DAY,
        30.0*DAY, 180.0*DAY, YEAR, 2.0*YEAR, 3.0*YEAR, 5.0*YEAR
    )

    # Retention evaluation
    realizations: int = 30
    retention_points: int = 75
    programming_noise_scale: float = 1.0
    read_noise_scale: float = 1.0
    drift_scale: float = 1.0

    # Proposed DriftSlice
    driftslice_bins: int = 4
    reference_cell_budget: int = 64
    refs_per_bin: int = 16
    reference_read_repeats: int = 4
    gain_min: float = 0.50
    gain_max: float = 3.00
    reference_floor_us: float = 0.10

    # Unified method selection and scientific safeguards
    methods: Tuple[str, ...] = (
        "PCM-HWA", "PCM-DAR-HWA", "PCM-DriftAttention-HWA",
        "PCM-NSR-HWA", "PCM-AttentionOnly-HWA", "PCM-GainOnly-HWA",
    )
    attention_lr: float = 5e-5
    attention_lambda: float = 0.5
    attention_input_gain: float = 0.9
    nsr_lambda: float = 0.1
    validation_reads: int = 3
    driftslice_selection_realizations: int = 3
    strict_154_epochs: bool = False

    # Diagnostics / ablations
    run_gdc_baselines: bool = True
    run_bin_ablation: bool = True
    ablation_bins: Tuple[int, ...] = (1, 2, 4, 8)
    cdf_times_s: Tuple[float, ...] = (20.0, 5.0*DAY, 5.0*YEAR)

    # ---------------- ADDED for this notebook (not in the original) ----------------
    smoke_train_repeats: int = 2            # fast mode: 2 repetitions to exercise fold statistics
    ablation_reference_budgets: Tuple[int, ...] = (16, 64, 256)   # R_tot sweep (K fixed)
    low_state_threshold_us: float = 2.5     # PRESPECIFIED low-state threshold = 0.1 * Gmax
    gdc_safe_max_fraction: float = 0.5      # saturation-safe GDC: probe keeps |ADC out| <= 0.5*out_bound
    gdc_reads: int = 4                      # repeated calibration reads (same as original GDC)
    bin_selection_method: str = "PCM-HWA"   # original: B selected on PCM-HWA validation, shared per fold
    fast_gpu_loader: bool = True            # performance only; bit-identical batches are VERIFIED
    include_original_gdc: bool = True       # evaluate the original (saturating) GDC as a diagnostic

    def resolved(self) -> "Config":
        c = copy.deepcopy(self)
        if c.fast_mode:
            c.epochs = 2
            c.digital_epochs = 2
            c.train_repeats = c.smoke_train_repeats   # original: 1 (see change log)
            c.realizations = 2
            c.retention_points = 8
            c.refs_per_bin = min(c.refs_per_bin, 4)
            c.reference_read_repeats = 2
        if c.training_semantics != "fp32_pretrain_then_analog_hwa":
            raise ValueError("This notebook requires digital FP32 pretraining followed by analog HWA.")
        if c.digital_epochs < 1 or c.epochs < 1:
            raise ValueError("Both digital_epochs and epochs must be positive")
        if c.hwa_early_stop_patience < 1:
            raise ValueError("hwa_early_stop_patience must be positive")
        if c.bits < 2:
            raise ValueError("bits must be >= 2")
        if not 0 < c.dar_fraction <= 1:
            raise ValueError("dar_fraction must be in (0,1]")
        if c.dar_delta_mode not in ("signed_initial", "absolute_initial"):
            raise ValueError("Unknown dar_delta_mode")
        if c.dar_g_units not in ("siemens", "normalized"):
            raise ValueError("Unknown dar_g_units")
        if c.dar_time_mode not in ("literal_seconds", "schedule_index"):
            raise ValueError("Unknown dar_time_mode")
        if c.driftslice_bins < 1 or c.refs_per_bin < 1:
            raise ValueError("Invalid DriftSlice bank size")
        if not (0 < c.gain_min <= 1 <= c.gain_max):
            raise ValueError("gain bounds must include 1")
        if not c.fast_mode and c.strict_154_epochs:
            if c.digital_epochs != 30 or c.epochs != 154:
                raise ValueError("Full protocol requires 30 FP32 + 154 analog epochs")
        if not c.methods or len(set(c.methods)) != len(c.methods):
            raise ValueError("Training methods must be nonempty and unique")
        if c.validation_reads < 1 or c.driftslice_selection_realizations < 1:
            raise ValueError("Validation read/realization counts must be >= 1")
        if c.resume_policy not in ("auto_new", "error"):
            raise ValueError("resume_policy must be auto_new or error")
        if any(c.reference_cell_budget % b for b in (*c.ablation_bins, c.driftslice_bins)):
            raise ValueError("Reference budget must be divisible by every bin count")
        # ADDED: every ablation budget must split evenly over every B
        if any(r % b for r in c.ablation_reference_budgets for b in c.ablation_bins):
            raise ValueError("Every ablation reference budget must be divisible by every bin count")
        if 1 not in c.ablation_bins:
            raise ValueError("B = 1 control must be part of ablation_bins")
        return c


@scientific("train")
def seed_all(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


@scientific("train")
@contextmanager
def random_stream(seed: int, device: torch.device):
    dev = torch.device(device)
    devices = []
    if dev.type == "cuda":
        idx = dev.index if dev.index is not None else torch.cuda.current_device()
        devices = [idx]
    with torch.random.fork_rng(devices=devices):
        torch.manual_seed(int(seed))
        if devices:
            torch.cuda.manual_seed(int(seed))
        yield


# --------------------------------------------------------------------------
# Atomic, Drive-friendly persistence (ADDED; replaces original atomic_torch_save
# with an equivalent single-write + os.replace that is faster on Drive FUSE)
# --------------------------------------------------------------------------
def atomic_torch_save(obj, path: Path) -> str:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    buf = _io.BytesIO()
    torch.save(obj, buf)
    data = buf.getvalue()
    tmp = path.with_name(path.name + f".tmp{os.getpid()}")
    with open(tmp, "wb") as f:
        f.write(data)
        f.flush()
        try:
            os.fsync(f.fileno())
        except OSError:
            pass
    os.replace(tmp, path)
    return sha256_bytes(data)


def atomic_write_text(path: Path, text: str) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + f".tmp{os.getpid()}")
    tmp.write_text(text, encoding="utf-8")
    os.replace(tmp, path)


def atomic_write_csv(df: pd.DataFrame, path: Path) -> None:
    atomic_write_text(path, df.to_csv(index=False))


def torch_load(path: Path):
    return torch.load(Path(path), map_location="cpu", weights_only=False)


def _write_inplace(path: Path, data: bytes) -> None:
    """Overwrite a file IN PLACE (no rename/delete -> nothing lands in Google Drive's Trash)."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "r+b" if path.exists() else "wb") as f:
        f.seek(0)
        f.write(data)
        f.truncate()
        f.flush()
        try:
            os.fsync(f.fileno())
        except OSError:
            pass


def write_text_inplace(path: Path, text: str) -> None:
    _write_inplace(Path(path), text.encode("utf-8"))


_RUN_NONCE_CACHE: Dict[str, str] = {}


def _run_nonce_for(path: Path) -> str:
    """Identity of the run folder that owns `path` (random nonce written into manifest.json when the run is
    created). Ties the local VM mirror to that folder: a run deleted and re-created on Drive gets a new nonce."""
    for parent in Path(path).parents:
        m = parent / "manifest.json"
        if m.exists():
            key = f"{m}|{m.stat().st_mtime_ns}"
            if key not in _RUN_NONCE_CACHE:
                _RUN_NONCE_CACHE[key] = json.loads(m.read_text()).get("run_nonce", "legacy")
            return _RUN_NONCE_CACHE[key]
    return "no-manifest"


def _resume_slots(path: Path):
    path = Path(path)
    local_dir = Path(globals().get("LOCAL_CACHE_STR", "/tmp")) / "resume_mirror"
    tag = hashlib.sha1(f"{path}|{_run_nonce_for(path)}".encode()).hexdigest()[:16]
    drive = [path.with_name(path.stem + ".slotA.pt"), path.with_name(path.stem + ".slotB.pt")]
    local = [local_dir / f"{tag}_{p.name}" for p in drive]
    return drive, local


def save_rotating(obj, path: Path, epoch_completed: int, final: bool = False) -> None:
    """Resumable state in two alternating slots written in place (one complete copy always survives a partial
    write; nothing goes to Drive's Trash). Local VM mirror every epoch; Google Drive every
    DRIVE_CHECKPOINT_EVERY_EPOCHS epochs and at the end. Drive slots alternate per DRIVE WRITE (any interval)."""
    buf = _io.BytesIO()
    torch.save(obj, buf)
    data = buf.getvalue()
    drive, local = _resume_slots(path)
    e = int(epoch_completed)
    _write_inplace(local[e % 2], data)
    every = max(1, int(globals().get("DRIVE_CHECKPOINT_EVERY_EPOCHS", 1)))
    if e % every == 0:
        _write_inplace(drive[(e // every) % 2], data)
    elif final:      # the slot NOT holding the newest periodic Drive copy
        _write_inplace(drive[1 - (e // every) % 2], data)


def load_rotating(path: Path, retries: int = 3):
    """Newest readable resumable state among the Drive slots and the local mirror (largest next_epoch).
    FAILS CLOSED: if slot files exist but none can be read (e.g. a transient Drive I/O error), it raises
    instead of reporting 'no state', which would restart training and overwrite good checkpoints."""
    drive, local = _resume_slots(path)
    best, src, failed = None, None, []
    for candidate in (*drive, *local):
        if not candidate.exists():
            continue
        obj = None
        for attempt in range(retries):
            try:
                obj = torch_load(candidate)
                break
            except Exception as exc:
                err = repr(exc)[:200]
                time.sleep(2.0 * (attempt + 1))
        if obj is None:
            failed.append((str(candidate), err))
            print(f"WARNING: could not read {candidate.name} ({err}); trying the remaining slots")
            continue
        if best is None or obj.get("next_epoch", -1) > best.get("next_epoch", -1):
            best, src = obj, candidate
    if best is None and failed:
        raise ScientificCheckError(
            f"Resumable state exists but no slot is readable: {failed}. Check that Google Drive is mounted and "
            "re-run; delete slot files by hand only if they are genuinely corrupt. Training was NOT restarted.")
    return best, src


# --------------------------------------------------------------------------
# Scientific-check registry (ADDED): PASS / FAIL / NOT EXECUTED
# --------------------------------------------------------------------------
class ScientificCheckError(RuntimeError):
    pass


CHECK_NAMES = [
    "AIHWKit GPU availability",
    "MNIST download and dataset sizes",
    "Train/validation/test separation",
    "Fast GPU loader bit-identical to original DataLoader",
    "Exactly 30 digital epochs",
    "Weight and bias transfer correctness",
    "No post-transfer reinitialization",
    "Actual AnalogAdam updates",
    "Exactly 154 HWA epochs",
    "Best-validation checkpoint restoration",
    "FP32-Direct has zero HWA steps",
    "Snapshot MVM reproduces AIHWKit forward",
    "NSR diagnostic surrogate matches nsr_penalty()",
    "PCM programming and drift behaviour",
    "Persistent programmed PCM device states",
    "Exact retention grid",
    "Reference-cell budget correctness",
    "DriftSlice B class assignment",
    "Consistent gain application",
    "No validation/test leakage",
    "Five-fold statistical aggregation",
    "No missing or duplicated data records",
    "Checkpoint resumption",
    "Ablation/diagnostics traceable to retention shards",
    "All table values traceable to CSV source data",
]


def check_names_for(arch: Optional[str] = None) -> List[str]:
    """Required checks of an architecture: the ResNet experiment checks CIFAR-10 instead of MNIST."""
    arch = arch or globals().get("MODEL_ARCHITECTURE", "mlp")
    if arch == "mlp":
        return list(CHECK_NAMES)
    return ["CIFAR-10 download and dataset sizes" if n == "MNIST download and dataset sizes" else n for n in CHECK_NAMES]


CHECKS: "OrderedDict[str, dict]" = OrderedDict((n, {"status": "NOT EXECUTED", "detail": "", "scope": ""})
                                               for n in check_names_for())


CHECK_SESSION = {"label": "session", "prefix": ""}   # e.g. prefix "smoke:" so smoke scopes are unambiguous


def record_check(name: str, ok: bool, detail: str = "", scope: str = "", essential: bool = True):
    scope = (CHECK_SESSION["prefix"] + scope) if scope and scope != "runtime" else scope
    if name not in CHECKS:
        CHECKS[name] = {"status": "NOT EXECUTED", "detail": "", "scope": ""}
    entry = CHECKS[name]
    status = "PASS" if ok else "FAIL"
    # A FAIL is sticky within a session; a PASS never overwrites an earlier FAIL.
    if entry["status"] != "FAIL":
        entry["status"] = status
    tag = f"[{scope}] " if scope else ""
    entry["detail"] = (entry["detail"] + " | " if entry["detail"] else "") + tag + detail
    entry["scope"] = ",".join(sorted(set(filter(None, (entry["scope"].split(",") + [scope])))))
    print(f"CHECK {status:4s} | {name} | {tag}{detail}")
    if not ok and essential:
        raise ScientificCheckError(f"Essential scientific check FAILED: {name}: {detail}")
    return ok


def checks_dataframe() -> pd.DataFrame:
    return pd.DataFrame([{"check": k, **v} for k, v in CHECKS.items()])


def reset_checks():
    CHECKS.clear()
    for k in check_names_for():
        CHECKS[k] = {"status": "NOT EXECUTED", "detail": "", "scope": ""}
    for name, ok, detail in globals().get("_EARLY_CHECKS", []):   # GPU check from Section 2
        record_check(name, ok, detail, "runtime")


def begin_check_session(label: str, prefix: str = ""):
    """Start an isolated check record (smoke / full_training / full_retention / ... / paper)."""
    CHECK_SESSION.update(label=label, prefix=prefix)
    reset_checks()
    if "SPLIT_LEDGER" in globals():
        SPLIT_LEDGER.clear()


reset_checks()


print("Core infrastructure loaded. torch", torch.__version__)

## 4. MNIST: the original loader (unchanged) plus a verified GPU-resident batch iterator

`_read_idx()`, `resolve_mnist_raw_dir()` and `build_loaders()` are the original functions, unchanged. When MNIST is missing they download it with `torchvision.datasets.MNIST(root=BASE_DIR/'data', train=..., download=True)`, which creates `BASE_DIR/data/MNIST/raw/`. Existing files (raw or `.gz`) are reused.

The original protocol is preserved:
* pixels scaled to [0,1], 28×28 images (flattened to 784 inputs inside the models);
* a 90/10 split of the official 60,000 training images using `torch.Generator().manual_seed(seed + fold)`, giving 54,000 train and 6,000 validation images;
* the 10,000 official test images, used for reporting only.

**ADDED: `FastGPULoader` (performance only).** In the benchmark, the original `Subset(TensorDataset)` DataLoader spent most of its time on per-sample Python indexing. It took about 0.6 s to evaluate the 10k test images once, which would make full retention take roughly 2 days. `FastGPULoader` keeps the original DataLoader's own sampler and the **same `torch.Generator` object**. It iterates an index-only DataLoader built with identical arguments, so each epoch's generator consumption (base-seed draw followed by `randperm`) is identical, and then gathers the batches from GPU-resident tensors. A check compares every batch against the original DataLoader over two consecutive epochs and stops the run unless they are **bit-identical**.

In [ ]:
# ===================== ORIGINAL (driftslice_core.py, CELL 4) =====================
@scientific("data")
def _read_idx_original(path: Path, kind: str):
    """Read an MNIST IDX file, transparently accepting either raw or .gz form."""
    path = Path(path)
    if not path.exists():
        gz = Path(str(path) + ".gz")
        if gz.exists():
            path = gz
        else:
            raise FileNotFoundError(f"Missing IDX file: {path} (or {gz})")

    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rb") as f:
        data = f.read()

    expected_magic = 2051 if kind == "image" else 2049
    magic = int.from_bytes(data[:4], "big")
    if magic != expected_magic:
        raise ValueError(
            f"Invalid IDX file {path}: magic={magic}, expected={expected_magic}"
        )

    n = int.from_bytes(data[4:8], "big")
    if kind == "image":
        h = int.from_bytes(data[8:12], "big")
        w = int.from_bytes(data[12:16], "big")
        arr = np.frombuffer(data, np.uint8, offset=16).copy().reshape(n, 1, h, w)
        return torch.from_numpy(arr).float() / 255.0

    arr = np.frombuffer(data, np.uint8, offset=8).copy()
    if arr.shape[0] != n:
        raise ValueError(f"Label count mismatch in {path}: header={n}, data={arr.shape[0]}")
    return torch.from_numpy(arr).long()


_IDX_CACHE: Dict[tuple, Tensor] = {}


@scientific("data")
def _read_idx(path: Path, kind: str):
    """ADDED: in-process memo of the ORIGINAL reader (same bytes -> same tensor; avoids re-reading
    and re-decoding MNIST from Google Drive for every fold/stage). Consumers never mutate it."""
    p = Path(path)
    actual = p if p.exists() else Path(str(p) + ".gz")
    st = actual.stat() if actual.exists() else None
    key = (str(actual), kind, st.st_size if st else None, st.st_mtime_ns if st else None)
    if key not in _IDX_CACHE:
        _IDX_CACHE[key] = _read_idx_original(path, kind)
    return _IDX_CACHE[key]


_MNIST_FILES = (
    "train-images-idx3-ubyte",
    "train-labels-idx1-ubyte",
    "t10k-images-idx3-ubyte",
    "t10k-labels-idx1-ubyte",
)


def _idx_exists(raw_dir: Path, name: str) -> bool:
    p = raw_dir / name
    return p.exists() or Path(str(p) + ".gz").exists()


def _mnist_complete(raw_dir: Path) -> bool:
    return all(_idx_exists(raw_dir, name) for name in _MNIST_FILES)


@scientific("data")
def resolve_mnist_raw_dir(cfg: Config, base_dir: Path) -> Path:
    """Find or create the MNIST/raw directory.

    Search order:
      1) cfg.data_root relative to BASE_DIR
      2) BASE_DIR/data/MNIST/raw
      3) current-working-directory/data/MNIST/raw

    If absent and cfg.allow_download=True, torchvision downloads MNIST into
    BASE_DIR/data, yielding BASE_DIR/data/MNIST/raw.
    """
    base_dir = Path(base_dir).expanduser().resolve()

    requested = Path(cfg.data_root).expanduser()
    candidates = []
    if requested.is_absolute():
        candidates.append(requested)
    else:
        candidates.append(base_dir / requested)

    candidates.extend([
        base_dir / "data" / "MNIST" / "raw",
        Path.cwd().resolve() / "data" / "MNIST" / "raw",
    ])

    # De-duplicate without requiring paths to exist.
    unique = []
    seen = set()
    for p in candidates:
        key = str(p)
        if key not in seen:
            seen.add(key)
            unique.append(p)

    for raw_dir in unique:
        if _mnist_complete(raw_dir):
            print(f"Using MNIST IDX files from: {raw_dir}")
            return raw_dir

    if not cfg.allow_download:
        searched = "\n  - ".join(str(p) for p in unique)
        raise FileNotFoundError(
            "MNIST was not found. Searched:\n"
            f"  - {searched}\n"
            "Either copy the four MNIST IDX files into one of those directories "
            "or set Config(allow_download=True)."
        )

    # Canonical torchvision destination: BASE_DIR/data/MNIST/raw
    dataset_root = base_dir / "data"
    dataset_root.mkdir(parents=True, exist_ok=True)

    print(f"MNIST not found locally. Downloading with torchvision into: {dataset_root}")
    try:
        from torchvision.datasets import MNIST
        MNIST(root=str(dataset_root), train=True, download=True)
        MNIST(root=str(dataset_root), train=False, download=True)
    except Exception as exc:
        raise RuntimeError(
            "Automatic MNIST download failed. Check internet access / proxy settings, "
            f"or manually place the IDX files under {dataset_root / 'MNIST' / 'raw'}."
        ) from exc

    raw_dir = dataset_root / "MNIST" / "raw"
    if not _mnist_complete(raw_dir):
        missing = [name for name in _MNIST_FILES if not _idx_exists(raw_dir, name)]
        raise FileNotFoundError(
            f"torchvision returned without a complete MNIST dataset at {raw_dir}. "
            f"Missing: {missing}"
        )

    print(f"MNIST download complete: {raw_dir}")
    return raw_dir


@scientific("data")
def build_loaders(cfg: Config, base_dir: Path, fold: int):
    root = resolve_mnist_raw_dir(cfg, base_dir)

    train_ds = TensorDataset(
        _read_idx(root / "train-images-idx3-ubyte", "image"),
        _read_idx(root / "train-labels-idx1-ubyte", "label"),
    )
    test_ds = TensorDataset(
        _read_idx(root / "t10k-images-idx3-ubyte", "image"),
        _read_idx(root / "t10k-labels-idx1-ubyte", "label"),
    )

    if len(train_ds) != 60000:
        raise RuntimeError(f"Expected 60,000 MNIST training samples, found {len(train_ds)}")
    if len(test_ds) != 10000:
        raise RuntimeError(f"Expected 10,000 MNIST test samples, found {len(test_ds)}")

    split_gen = torch.Generator().manual_seed(cfg.seed + fold)
    perm = torch.randperm(len(train_ds), generator=split_gen)
    n_train = int(0.9 * len(train_ds))
    train_idx = perm[:n_train]
    val_idx = perm[n_train:]

    if cfg.fast_mode:
        train_idx = train_idx[:1024]
        val_idx = val_idx[:256]
        test_idx = torch.arange(min(512, len(test_ds)))
    else:
        test_idx = torch.arange(len(test_ds))

    train_sub = torch.utils.data.Subset(train_ds, train_idx.tolist())
    val_sub = torch.utils.data.Subset(train_ds, val_idx.tolist())
    test_sub = torch.utils.data.Subset(test_ds, test_idx.tolist())

    train_gen = torch.Generator().manual_seed(cfg.seed + fold + 1000)
    loader_kwargs = dict(
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    train_loader = DataLoader(
        train_sub,
        batch_size=cfg.batch_size,
        shuffle=True,
        generator=train_gen,
        **loader_kwargs,
    )
    val_loader = DataLoader(
        val_sub,
        batch_size=cfg.test_batch_size,
        shuffle=False,
        **loader_kwargs,
    )
    test_loader = DataLoader(
        test_sub,
        batch_size=cfg.test_batch_size,
        shuffle=False,
        **loader_kwargs,
    )

    split_info = {
        "fold": fold,
        "mnist_raw_dir": str(root),
        "train_indices": train_idx,
        "val_indices": val_idx,
        "test_indices": test_idx,
    }
    return train_loader, val_loader, test_loader, split_info


# ============================ ADDED: fast loader ============================
@scientific("data")
class _IndexOnlyDataset(torch.utils.data.Dataset):
    def __init__(self, n: int):
        self.n = int(n)

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        return i


@scientific("data")
class FastGPULoader:
    """GPU-resident iterator producing batches bit-identical to `loader`, with the SAME RNG side effects.

    Shares `loader.generator` (same object), so manual_seed/get_state/set_state calls in the training
    code act exactly as on the original DataLoader. Every `iter(DataLoader)` (num_workers=0) draws one
    int64 base seed from `loader.generator` -- the GLOBAL CPU RNG when the generator is None (val/test).
    The shuffled path reproduces this by iterating an index-only DataLoader built with identical
    arguments; the sequential path performs the identical draw explicitly. Both are verified.
    """

    def __init__(self, loader: DataLoader, device: torch.device, split_name: str):
        subset = loader.dataset
        base = subset.dataset
        idx = torch.as_tensor(list(subset.indices), dtype=torch.long)
        self.x = base.tensors[0][idx].to(device)
        self.y = base.tensors[1][idx].to(device)
        self.n = int(idx.numel())
        self.batch_size = loader.batch_size
        self.drop_last = loader.drop_last
        self.generator = loader.generator
        self.dataset = subset
        self.split_name = split_name
        self.original = loader
        self.shuffle = isinstance(loader.sampler, torch.utils.data.RandomSampler)
        if self.shuffle:
            self._index_loader = DataLoader(
                _IndexOnlyDataset(self.n), batch_size=loader.batch_size, shuffle=True,
                generator=loader.generator, num_workers=0, drop_last=loader.drop_last)

    def __len__(self):
        return self.n // self.batch_size if self.drop_last else math.ceil(self.n / self.batch_size)

    def __iter__(self):
        if self.shuffle:
            for batch in self._index_loader:
                b = batch.to(self.x.device, non_blocking=True)
                yield self.x[b], self.y[b]
        else:
            # identical RNG side effect to torch.utils.data._BaseDataLoaderIter.__init__ (base-seed draw)
            torch.empty((), dtype=torch.int64).random_(generator=self.generator)
            stop = (self.n // self.batch_size) * self.batch_size if self.drop_last else self.n
            for s in range(0, stop, self.batch_size):
                yield self.x[s:s + self.batch_size], self.y[s:s + self.batch_size]


def _batch_digest(x: Tensor, y: Tensor) -> str:
    h = hashlib.sha1()
    h.update(x.detach().cpu().contiguous().numpy().tobytes())
    h.update(y.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()


def verify_fast_loader_equivalence(original: DataLoader, fast: FastGPULoader, epochs: int = 2,
                                   probe_seed: int = 987654321) -> Tuple[bool, str]:
    """Compare every batch over `epochs` consecutive epochs from identical generator and global-RNG states,
    AND the generator / global CPU RNG states left behind (the DataLoader's base-seed side effect)."""
    g = original.generator
    saved_g = g.get_state() if g is not None else None
    saved_global = torch.get_rng_state()
    n_ep = epochs if fast.shuffle else 1

    def run(loader):
        torch.manual_seed(probe_seed + 1)
        if g is not None:
            g.manual_seed(probe_seed)
        digests = [[_batch_digest(x, y) for x, y in loader] for _ in range(n_ep)]
        return digests, torch.get_rng_state(), (g.get_state() if g is not None else None)

    ref, ref_global, ref_g = run(original)
    new, new_global, new_g = run(fast)
    torch.set_rng_state(saved_global)
    if g is not None:
        g.set_state(saved_g)
    same_rng = torch.equal(ref_global, new_global) and (ref_g is None or torch.equal(ref_g, new_g))
    n = sum(len(e) for e in ref)
    ok = ref == new and same_rng
    return ok, (f"{fast.split_name}: {n} batches over {len(ref)} epoch(s) {'identical' if ref == new else 'DIFFER'}; "
                f"global-RNG/generator side effects {'identical' if same_rng else 'DIFFER'}")


def mnist_file_hashes(raw_dir: Path) -> Dict[str, str]:
    """SHA-256 of the *decompressed* IDX content (independent of .gz packaging)."""
    out = {}
    for name in _MNIST_FILES:
        p = Path(raw_dir) / name
        if not p.exists():
            p = Path(str(p) + ".gz")
        opener = gzip.open if p.suffix == ".gz" else open
        with opener(p, "rb") as f:
            out[name] = sha256_bytes(f.read())
    return out


def split_digest(split_info) -> Dict[str, str]:
    return {k: sha256_bytes(split_info[k].numpy().astype(np.int64).tobytes())
            for k in ("train_indices", "val_indices", "test_indices")}


def check_split_integrity(cfg: Config, split_info, scope: str):
    tr = set(split_info["train_indices"].tolist())
    va = set(split_info["val_indices"].tolist())
    n_tr, n_va, n_te = len(tr), len(va), len(split_info["test_indices"])
    expected = (1024, 256, 512) if cfg.fast_mode else (54000, 6000, 10000)
    disjoint = not (tr & va)
    ok = disjoint and (n_tr, n_va, n_te) == expected and len(split_info["train_indices"]) == n_tr
    record_check("Train/validation/test separation", ok,
                 f"train={n_tr}, val={n_va}, test={n_te} (expected {expected}); "
                 f"train∩val={'∅' if disjoint else 'NONEMPTY'}; test = official t10k file (separate)", scope)


@scientific("data")
def get_fold_loaders(cfg: Config, base_dir: Path, fold: int, device: torch.device,
                     verify_fast: bool = True):
    """Original build_loaders() + split tags + (optionally) verified fast GPU loaders."""
    train, val, test, split = build_loaders(cfg, base_dir, fold)
    for loader, name in ((train, "train"), (val, "val"), (test, "test")):
        loader.split_name = name
    base_tr, base_te = train.dataset.dataset, test.dataset.dataset
    x, y = base_tr.tensors
    mnist_ok = (len(base_tr) == 60000 and len(base_te) == 10000 and tuple(x.shape[1:]) == (1, 28, 28)
                and x.dtype == torch.float32 and float(x.min()) >= 0 and float(x.max()) <= 1
                and y.dtype == torch.int64 and set(y.unique().tolist()) == set(range(10)))
    record_check("MNIST download and dataset sizes", mnist_ok,
                 f"{split['mnist_raw_dir']}: 60000 train + 10000 test IDX images, 1x28x28 float32 in [0,1], "
                 f"labels 0-9 int64 (original _read_idx)", f"fold{fold}")
    check_split_integrity(cfg, split, scope=f"fold{fold}")
    if cfg.fast_gpu_loader:
        fast = {name: FastGPULoader(l, device, name) for name, l in (("train", train), ("val", val), ("test", test))}
        if verify_fast:
            details = []
            for name, l in (("train", train), ("val", val), ("test", test)):
                ok, msg = verify_fast_loader_equivalence(l, fast[name])
                details.append(msg)
                if not ok:
                    record_check("Fast GPU loader bit-identical to original DataLoader", False, msg, f"fold{fold}")
            record_check("Fast GPU loader bit-identical to original DataLoader", True, "; ".join(details), f"fold{fold}")
        return fast["train"], fast["val"], fast["test"], split
    return train, val, test, split


SPLIT_LEDGER: List[tuple] = []   # (stage, split) pairs actually consumed (ADDED leakage audit)


def require_split(loader, name: str, stage: str = "unspecified"):
    """Leakage guard (ADDED): each stage declares which split it is allowed to read."""
    actual = getattr(loader, "split_name", None)
    SPLIT_LEDGER.append((stage, actual))
    if actual != name:
        record_check("No validation/test leakage", False, f"stage '{stage}' expected split '{name}', got '{actual}'")
    return loader


print("Data functions loaded.")

### 4b. CIFAR-10 for the ResNet-32 experiment (`MODEL_ARCHITECTURE = "resnet32"`)

This section is **added**; the original code is MNIST-only. It mirrors the original MNIST protocol as closely as the dataset allows:
* **Download.** `torchvision.datasets.CIFAR10(root=BASE_DIR/'data', download=True)` creates `BASE_DIR/data/cifar-10-batches-py/`, and existing files are reused. The pickled batches are read directly; torchvision is only the downloader.
* **Preprocessing.** Pixels are scaled to [0, 1] and then normalized per channel with the standard CIFAR-10 statistics, mean (0.4914, 0.4822, 0.4465) and std (0.2470, 0.2435, 0.2616).
* **Split.** The official 50,000 training images are split 90/10 with `torch.Generator().manual_seed(seed + fold)`, the same rule as for MNIST, giving 45,000 train and 5,000 validation images. The 10,000 official test images are used for reporting only. Fast mode uses the same 1024/256/512 subsets.
* **Loaders.** The DataLoader arguments and generators are the same as in `build_loaders()`. `FastGPULoader` and its bit-identity check are reused unchanged.
* **Augmentation, training only.** The standard ResNet/CIFAR augmentation is used: a 32×32 random crop from the image zero-padded by 4 pixels (black, i.e. padded *before* normalization), and a random horizontal flip. It runs on the GPU with a dedicated `torch.Generator` seeded by (seed, fold, phase, epoch, batch). Phase 0 is FP32 pretraining and phase 1 is HWA, where **all HWA methods see the identical augmented batches** (paired). The global RNG is never touched, so resumed training stays bit-identical. Validation and test images are never augmented.

In [ ]:
# ============================ ADDED: CIFAR-10 (ResNet-32 experiment) ============================
import pickle

CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)
_CIFAR10_FILES = ("data_batch_1", "data_batch_2", "data_batch_3", "data_batch_4", "data_batch_5", "test_batch")
_CIFAR_CACHE: Dict[tuple, tuple] = {}


def _cifar10_complete(d: Path) -> bool:
    return all((Path(d) / n).exists() for n in _CIFAR10_FILES)


@scientific("rn_data")
def resolve_cifar10_dir(cfg: Config, base_dir: Path) -> Path:
    """Find or create BASE_DIR/data/cifar-10-batches-py (same search/download policy as resolve_mnist_raw_dir)."""
    base_dir = Path(base_dir).expanduser().resolve()
    requested = Path(cfg.data_root).expanduser()
    root = requested if requested.is_absolute() else base_dir / requested
    candidates = []
    for p in (root / "cifar-10-batches-py", base_dir / "data" / "cifar-10-batches-py",
              Path.cwd().resolve() / "data" / "cifar-10-batches-py"):
        if str(p) not in map(str, candidates):
            candidates.append(p)
    for d in candidates:
        if _cifar10_complete(d):
            print(f"Using CIFAR-10 batches from: {d}")
            return d
    if not cfg.allow_download:
        raise FileNotFoundError("CIFAR-10 was not found. Searched:\n  - " + "\n  - ".join(map(str, candidates)))
    dataset_root = base_dir / "data"
    dataset_root.mkdir(parents=True, exist_ok=True)
    print(f"CIFAR-10 not found locally. Downloading with torchvision into: {dataset_root}")
    try:
        from torchvision.datasets import CIFAR10
        CIFAR10(root=str(dataset_root), train=True, download=True)
        CIFAR10(root=str(dataset_root), train=False, download=True)
    except Exception as exc:
        raise RuntimeError("Automatic CIFAR-10 download failed. Check internet access, or place the extracted "
                           f"'cifar-10-batches-py' folder under {dataset_root}.") from exc
    d = dataset_root / "cifar-10-batches-py"
    if not _cifar10_complete(d):
        raise FileNotFoundError(f"torchvision returned without a complete CIFAR-10 dataset at {d}")
    print(f"CIFAR-10 download complete: {d}")
    return d


@scientific("rn_data")
def _read_cifar_batch(path: Path):
    """One CIFAR-10 python batch -> (uint8 [N,3,32,32], int64 [N])."""
    with open(path, "rb") as f:
        d = pickle.load(f, encoding="bytes")
    x = np.asarray(d[b"data"], dtype=np.uint8).reshape(-1, 3, 32, 32)
    y = np.asarray(d[b"labels"], dtype=np.int64)
    if x.shape[0] != y.shape[0]:
        raise ValueError(f"Label count mismatch in {path}")
    return torch.from_numpy(x.copy()), torch.from_numpy(y.copy())


@scientific("rn_data")
def cifar10_normalize(x_uint8: Tensor) -> Tensor:
    mean = torch.tensor(CIFAR10_MEAN, dtype=torch.float32).view(1, 3, 1, 1)
    std = torch.tensor(CIFAR10_STD, dtype=torch.float32).view(1, 3, 1, 1)
    return (x_uint8.float() / 255.0 - mean) / std


@scientific("rn_data")
def load_cifar10_tensors(root: Path):
    """Normalized float32 train/test tensors (in-process memo; consumers never mutate them)."""
    root = Path(root)
    key = tuple((n, (root / n).stat().st_size, (root / n).stat().st_mtime_ns) for n in _CIFAR10_FILES) + (str(root),)
    if key not in _CIFAR_CACHE:
        parts = [_read_cifar_batch(root / n) for n in _CIFAR10_FILES[:5]]
        x_tr = torch.cat([p[0] for p in parts])
        y_tr = torch.cat([p[1] for p in parts])
        x_te, y_te = _read_cifar_batch(root / "test_batch")
        _CIFAR_CACHE.clear()
        _CIFAR_CACHE[key] = (cifar10_normalize(x_tr), y_tr, cifar10_normalize(x_te), y_te)
    return _CIFAR_CACHE[key]


@scientific("rn_data")
def build_cifar10_loaders(cfg: Config, base_dir: Path, fold: int):
    """build_loaders() with CIFAR-10: identical split rule, generators and DataLoader arguments."""
    root = resolve_cifar10_dir(cfg, base_dir)
    x_tr, y_tr, x_te, y_te = load_cifar10_tensors(root)
    train_ds = TensorDataset(x_tr, y_tr)
    test_ds = TensorDataset(x_te, y_te)
    if len(train_ds) != 50000:
        raise RuntimeError(f"Expected 50,000 CIFAR-10 training samples, found {len(train_ds)}")
    if len(test_ds) != 10000:
        raise RuntimeError(f"Expected 10,000 CIFAR-10 test samples, found {len(test_ds)}")

    split_gen = torch.Generator().manual_seed(cfg.seed + fold)
    perm = torch.randperm(len(train_ds), generator=split_gen)
    n_train = int(0.9 * len(train_ds))
    train_idx = perm[:n_train]
    val_idx = perm[n_train:]
    if cfg.fast_mode:
        train_idx = train_idx[:1024]
        val_idx = val_idx[:256]
        test_idx = torch.arange(min(512, len(test_ds)))
    else:
        test_idx = torch.arange(len(test_ds))

    train_sub = torch.utils.data.Subset(train_ds, train_idx.tolist())
    val_sub = torch.utils.data.Subset(train_ds, val_idx.tolist())
    test_sub = torch.utils.data.Subset(test_ds, test_idx.tolist())
    train_gen = torch.Generator().manual_seed(cfg.seed + fold + 1000)
    loader_kwargs = dict(num_workers=0, pin_memory=torch.cuda.is_available())
    train_loader = DataLoader(train_sub, batch_size=cfg.batch_size, shuffle=True, generator=train_gen, **loader_kwargs)
    val_loader = DataLoader(val_sub, batch_size=cfg.test_batch_size, shuffle=False, **loader_kwargs)
    test_loader = DataLoader(test_sub, batch_size=cfg.test_batch_size, shuffle=False, **loader_kwargs)
    split_info = {"fold": fold, "cifar10_dir": str(root), "train_indices": train_idx,
                  "val_indices": val_idx, "test_indices": test_idx}
    return train_loader, val_loader, test_loader, split_info


def cifar10_file_hashes(root: Path) -> Dict[str, str]:
    return {n: sha256_bytes((Path(root) / n).read_bytes()) for n in _CIFAR10_FILES}


def check_split_integrity_cifar(cfg: Config, split_info, scope: str):
    tr = set(split_info["train_indices"].tolist())
    va = set(split_info["val_indices"].tolist())
    n_tr, n_va, n_te = len(tr), len(va), len(split_info["test_indices"])
    expected = (1024, 256, 512) if cfg.fast_mode else (45000, 5000, 10000)
    disjoint = not (tr & va)
    ok = disjoint and (n_tr, n_va, n_te) == expected and len(split_info["train_indices"]) == n_tr
    record_check("Train/validation/test separation", ok,
                 f"train={n_tr}, val={n_va}, test={n_te} (expected {expected}); "
                 f"train∩val={'∅' if disjoint else 'NONEMPTY'}; test = official test_batch file (separate)", scope)


@scientific("rn_data")
def get_fold_loaders_cifar(cfg: Config, base_dir: Path, fold: int, device: torch.device, verify_fast: bool = True):
    """build_cifar10_loaders() + split tags + (optionally) verified fast GPU loaders (same as get_fold_loaders)."""
    train, val, test, split = build_cifar10_loaders(cfg, base_dir, fold)
    for loader, name in ((train, "train"), (val, "val"), (test, "test")):
        loader.split_name = name
    base_tr, base_te = train.dataset.dataset, test.dataset.dataset
    x, y = base_tr.tensors
    ref = cifar10_normalize(torch.tensor([[0], [255]], dtype=torch.uint8).view(2, 1, 1, 1).expand(2, 3, 1, 1))
    lo, hi = ref[0].flatten(), ref[1].flatten()
    in_range = bool(((x.amin(dim=(0, 2, 3)) >= lo - 1e-5) & (x.amax(dim=(0, 2, 3)) <= hi + 1e-5)).all())
    ok = (len(base_tr) == 50000 and len(base_te) == 10000 and tuple(x.shape[1:]) == (3, 32, 32)
          and x.dtype == torch.float32 and in_range and y.dtype == torch.int64
          and set(y.unique().tolist()) == set(range(10)) and bool((torch.bincount(y) == 5000).all()))
    record_check("CIFAR-10 download and dataset sizes", ok,
                 f"{split['cifar10_dir']}: 50000 train + 10000 test images, 3x32x32 float32 normalized with "
                 f"mean {CIFAR10_MEAN} / std {CIFAR10_STD}; labels 0-9 int64, 5000 per class", f"fold{fold}")
    check_split_integrity_cifar(cfg, split, scope=f"fold{fold}")
    if cfg.fast_gpu_loader:
        fast = {name: FastGPULoader(l, device, name) for name, l in (("train", train), ("val", val), ("test", test))}
        if verify_fast:
            details = []
            for name, l in (("train", train), ("val", val), ("test", test)):
                ok, msg = verify_fast_loader_equivalence(l, fast[name])
                details.append(msg)
                if not ok:
                    record_check("Fast GPU loader bit-identical to original DataLoader", False, msg, f"fold{fold}")
            record_check("Fast GPU loader bit-identical to original DataLoader", True, "; ".join(details), f"fold{fold}")
        return fast["train"], fast["val"], fast["test"], split
    return train, val, test, split


# ----------------------------- training-only augmentation -----------------------------
AUGMENT_SEED_BASE = 70_000_000      # separate namespace: augmentation randomness does not exist in the original code


@scientific("rn_data")
def augment_seed(cfg: Config, fold: int, phase: int, epoch: int, batch: int) -> int:
    """phase 0 = FP32 pretraining, 1 = HWA (shared by every HWA method -> paired augmented batches)."""
    return AUGMENT_SEED_BASE + cfg.seed + fold * 1_000_000 + phase * 500_000 + epoch * 1_000 + batch


@scientific("rn_data")
def cifar_augment(x: Tensor, seed: int, pad: int = 4) -> Tensor:
    """Random 32x32 crop of the zero(black)-padded image + random horizontal flip, on the batch's device.
    Uses its own CPU generator (never the global RNG). Input/output: normalized [N,3,32,32] (or flattened)."""
    x = x.view(-1, 3, 32, 32)
    n, dev = x.shape[0], x.device
    g = torch.Generator().manual_seed(int(seed))
    oy = torch.randint(0, 2 * pad + 1, (n,), generator=g)
    ox = torch.randint(0, 2 * pad + 1, (n,), generator=g)
    flip = torch.randint(0, 2, (n,), generator=g).bool()
    mean = torch.tensor(CIFAR10_MEAN, device=dev).view(1, 3, 1, 1)
    std = torch.tensor(CIFAR10_STD, device=dev).view(1, 3, 1, 1)
    black = (0.0 - mean) / std                                   # zero padding BEFORE normalization
    xp = black.expand(n, 3, 32 + 2 * pad, 32 + 2 * pad).clone()
    xp[:, :, pad:pad + 32, pad:pad + 32] = x
    ar = torch.arange(32)
    rows = oy[:, None] + ar[None, :]
    cols = torch.where(flip[:, None], ox[:, None] + 31 - ar[None, :], ox[:, None] + ar[None, :])
    bi = torch.arange(n)[:, None, None, None].to(dev)
    ci = torch.arange(3)[None, :, None, None].to(dev)
    return xp[bi, ci, rows[:, None, :, None].to(dev), cols[:, None, None, :].to(dev)]


print("CIFAR-10 functions loaded.")

## 5. Models, AIHWKit configuration and training regularizers (original equations)

The following are copied without modification from `driftslice_core.py` / `unified_experiment.py`. The only edit is that the `ds.` module prefix was removed:
* `DigitalMLP` (784 → 250 → 10, tanh) and `AnalogMLP` (two `AnalogLinear` layers with `TorchInferenceRPUConfig`);
* `make_hwa_rpu()`: the AIHWKit HWA configuration (8-bit DAC/ADC, out-noise 0.04, out-bound 10, ADD_NORMAL weight noise 0.02, columnwise weight scaling ω = 0.8, learned out-scaling, learned input range, digital bias);
* `transfer_pretrained_digital_to_analog()`: copies the FP32 weights **and biases** with `set_weights` and checks them with `assert_close`;
* `dar_penalty()`, the DAR drift-aware regularizer: top-80 % weights, λ-schedule `λ0/(epoch+1)^1.5/(1+t_c·t)`, a sampled PCM ν, and `|expm1(−ζ ln(t/t0))|·G`;
* `AttentionGate` and the attention regularizer (training-only input transform; validation and deployment use **raw** inputs, as in the original);
* `nsr_penalty()`: the Noise-to-Signal regularizer (programming, drift and read-noise variance divided by normalized signal energy, averaged over layers, times λ_NSR = 0.1).

**ADDED for diagnostics only:** `nsr_components()` re-expresses the per-layer terms of `nsr_penalty()` for Table II. A check verifies that `λ·mean(NSR_layer)` from these components reproduces `nsr_penalty(model)` to floating-point precision.

In [ ]:
# ===================== ORIGINAL (driftslice_core.py, CELL 7/8) =====================
@scientific("model")
class DigitalMLP(nn.Module):
    """Paper architecture, digital FP32 pretraining (no analog simulation)."""
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 250)
        self.fc2 = nn.Linear(250, 10)

    def forward(self, x: Tensor) -> Tensor:
        x = x.flatten(1)
        return self.fc2(torch.tanh(self.fc1(x)))


@scientific("train")
def cpu_cloned_state(model: nn.Module):
    """Snapshot a PyTorch/AIHWKit state_dict without assuming all entries are Tensors.

    Analog tiles include non-Tensor state (e.g., nested dictionaries containing
    RPU configuration and metadata).  deepcopy preserves OrderedDict._metadata
    and non-Tensor structure needed by load_state_dict(); only Tensor leaves are
    detached, cloned, and transferred to CPU for checkpoint isolation.
    """
    state = copy.deepcopy(model.state_dict())

    def move_tensor_leaves(value):
        if isinstance(value, torch.Tensor):
            return value.detach().cpu().clone()
        if isinstance(value, dict):
            # Update in place so OrderedDict._metadata and mapping class survive.
            for key in value:
                value[key] = move_tensor_leaves(value[key])
            return value
        if isinstance(value, list):
            for index, item in enumerate(value):
                value[index] = move_tensor_leaves(item)
            return value
        if isinstance(value, tuple):
            if hasattr(value, '_fields'):  # namedtuple
                return type(value)(*(move_tensor_leaves(x) for x in value))
            return tuple(move_tensor_leaves(x) for x in value)
        return value  # scalars, strings, config objects, and other metadata

    return move_tensor_leaves(state)


@scientific("model")
def make_hwa_rpu(cfg: Config):
    from aihwkit.simulator.configs import TorchInferenceRPUConfig
    from aihwkit.simulator.parameters.enums import (
        BoundManagementType,
        NoiseManagementType,
        WeightModifierType,
        WeightClipType,
        WeightRemapType,
    )

    r = TorchInferenceRPUConfig()

    r.mapping.max_input_size = 0
    r.mapping.max_output_size = 0
    r.mapping.digital_bias = True
    r.mapping.weight_scaling_omega = cfg.weight_scaling_omega
    r.mapping.weight_scaling_columnwise = True
    r.mapping.learn_out_scaling = True
    r.mapping.out_scaling_columnwise = True

    levels = float(2**cfg.bits - 2)
    r.forward.inp_res = levels
    r.forward.out_res = levels
    r.forward.bound_management = BoundManagementType.NONE
    r.forward.noise_management = NoiseManagementType.CONSTANT
    r.forward.nm_thres = 1.0
    r.forward.out_noise = cfg.output_noise_std
    r.forward.out_bound = cfg.output_bound
    r.forward.ir_drop = cfg.ir_drop

    r.modifier.type = WeightModifierType.ADD_NORMAL
    r.modifier.std_dev = cfg.weight_noise_std
    r.modifier.rel_to_actual_wmax = True

    r.clip.type = WeightClipType.FIXED_VALUE
    r.clip.fixed_value = 1.0
    r.remap.type = WeightRemapType.NONE

    ir = r.pre_post.input_range
    ir.enable = True
    ir.init_value = cfg.input_range_init
    ir.init_from_data = 100
    ir.init_std_alpha = 3.0
    ir.decay = 0.001
    ir.input_min_percentage = 0.95
    ir.output_min_percentage = 0.95
    ir.manage_output_clipping = False

    r.drift_compensation = None
    return r


@scientific("model")
class AnalogMLP(nn.Module):
    def __init__(self, cfg: Config, seed: int, device: torch.device,
                 pretrained_state: Optional[dict] = None):
        super().__init__()
        from aihwkit.nn import AnalogLinear

        self.fc1 = AnalogLinear(784, 250, bias=True, rpu_config=make_hwa_rpu(cfg))
        self.fc2 = AnalogLinear(250, 10, bias=True, rpu_config=make_hwa_rpu(cfg))
        if pretrained_state is None:
            # Only used for synthetic preflight; production requires a digital checkpoint.
            gen = torch.Generator().manual_seed(seed)
            for layer in (self.fc1, self.fc2):
                w = torch.empty(layer.out_features, layer.in_features)
                nn.init.xavier_uniform_(w, generator=gen)
                layer.set_weights(w, torch.zeros(layer.out_features))
        else:
            for name, layer in self.layers().items():
                w = pretrained_state[f"{name}.weight"].detach().to("cpu", dtype=torch.float32)
                b = pretrained_state[f"{name}.bias"].detach().to("cpu", dtype=torch.float32)
                if tuple(w.shape) != (layer.out_features, layer.in_features):
                    raise RuntimeError(f"Pretrained {name}: shape mismatch: {tuple(w.shape)}")
                layer.set_weights(w, b)
        self.to(device)

    def forward(self, x: Tensor) -> Tensor:
        x = x.flatten(1)
        x = torch.tanh(self.fc1(x))
        return self.fc2(x)

    def layers(self):
        return {"fc1": self.fc1, "fc2": self.fc2}


@scientific("model")
def transfer_pretrained_digital_to_analog(cfg: Config, digital_model: DigitalMLP,
                                          seed: int, device: torch.device,
                                          val_loader=None):
    """Transfer *trained* FP32 weights into HWA layers without reinitialization."""
    state = cpu_cloned_state(digital_model)
    analog_model = AnalogMLP(cfg, seed, device, pretrained_state=state)
    for name, layer in analog_model.layers().items():
        logical_w, logical_b = layer.get_weights(apply_weight_scaling=True)
        torch.testing.assert_close(logical_w.detach().cpu(), state[f"{name}.weight"],
                                   rtol=2e-3, atol=2e-4,
                                   msg=lambda msg: f"Digital→analog transfer failed for {name} weights: {msg}")
        torch.testing.assert_close(logical_b.detach().cpu(), state[f"{name}.bias"],
                                   rtol=2e-3, atol=2e-4,
                                   msg=lambda msg: f"Digital→analog transfer failed for {name} bias: {msg}")
    if val_loader is not None:
        _, digital_acc = evaluate_model(digital_model, val_loader, device)
        _, analog_acc = evaluate_model(analog_model, val_loader, device)
        print(f"TRANSFER check: digital FP32={digital_acc:.2f}% → "
              f"analog HWA (before optimizer)={analog_acc:.2f}%")
        if (not cfg.fast_mode and
                digital_acc - analog_acc > cfg.transfer_max_val_drop_pp):
            raise RuntimeError("Weight transfer changed accuracy excessively; "
                               "the analog training has NOT begun")
    return analog_model


@scientific("train")
def physical_weight_parameter(layer) -> Tensor:
    tile, = list(layer.analog_tiles())
    w = tile.tile.weight
    if not w.requires_grad:
        raise RuntimeError("DAR requires a differentiable TorchInferenceTile weight tensor.")
    return w


@scientific("train")
def _sample_pcm_nu(g_rel: Tensor) -> Tensor:
    g_rel = g_rel.detach().clamp_min(1e-7)
    mu = (-0.0155 * g_rel.log() + 0.0244).clamp(0.049, 0.1)
    sigma = (-0.0125 * g_rel.log() - 0.0059).clamp(0.008, 0.045)
    return (mu + sigma * torch.randn_like(g_rel)).abs()


@scientific("train")
def dar_penalty(w: Tensor, initial_w: Tensor, epoch: int, t_inf_s: float, cfg: Config) -> Tensor:
    if t_inf_s < T_REF:
        raise ValueError("Absolute retention time must be >=20 seconds")
    flat = w.flatten()
    ref = initial_w.flatten().to(w)

    k = max(1, math.ceil(cfg.dar_fraction * flat.numel()))
    idx = flat.abs().topk(k, sorted=False).indices
    selected = flat[idx]

    w_inf = flat.abs().max().clamp_min(1e-8)
    g_rel = (selected.abs() / w_inf).clamp(1e-7, 1.0)
    nu = _sample_pcm_nu(g_rel)

    delta = selected - ref[idx]
    if cfg.dar_delta_mode == "absolute_initial":
        delta = delta.abs()

    if cfg.dar_time_mode == "literal_seconds":
        time_term = float(t_inf_s)
    else:
        time_term = float(cfg.dar_times_s.index(t_inf_s) + 1)

    lam = (
        cfg.dar_lambda_init
        / (epoch + 1.0) ** 1.5
        / (1.0 + cfg.dar_tc * time_term)
    )
    zeta = lam * (nu + cfg.dar_eta * delta / w_inf)

    if cfg.dar_g_units == "siemens":
        g = g_rel * cfg.gmax_us * 1e-6
    else:
        g = g_rel

    ratio = max(float(t_inf_s) / T_REF, 1.0)
    temporal = torch.expm1(-zeta * math.log(ratio)).abs()
    return (g * temporal).sum()


@scientific("train")
@torch.inference_mode()
def evaluate_model(model: nn.Module, loader, device: torch.device):
    model.eval()
    loss_sum = 0.0
    correct = 0
    n = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        if not torch.isfinite(logits).all():
            raise FloatingPointError("Nonfinite evaluation logits")
        loss_sum += F.cross_entropy(logits, y, reduction="sum").item()
        correct += (logits.argmax(1) == y).sum().item()
        n += y.numel()
    if not n:
        raise ValueError("Empty evaluation loader")
    return loss_sum / n, 100.0 * correct / n


@scientific("pcm")
def snapshot_analog_model(model: AnalogMLP) -> Dict[str, Dict[str, Tensor]]:
    result = {}
    for name, layer in model.layers().items():
        tile, = list(layer.analog_tiles())
        device = physical_weight_parameter(layer).device

        w, b = layer.get_weights(apply_weight_scaling=False)
        result[name] = {
            "weight": w.detach().to(device).clone(),
            "bias": b.detach().to(device).clone(),
            "scale": tile.get_scales().detach().to(device).clone(),
            "input_range": tile.input_range.detach().to(device).clone(),
        }
    return result


# ===================== ORIGINAL (unified_experiment.py) =====================
@scientific("train")
class AttentionGate(nn.Module):
    """AICAS-inspired training-only input gate, as in v2_lib's AttentionGate."""
    def __init__(self, dim=784, cg=0.9):
        super().__init__()
        self.proj = nn.Linear(dim, dim, bias=False)
        self.cg = cg

    def forward(self, x):
        alpha = torch.softmax(self.proj(x), dim=-1)
        return self.cg * x + (1.0 - self.cg) * alpha, alpha


@scientific("train")
def nsr_penalty(model: AnalogMLP, cfg: Config):
    """Preserve nsr_run.py noise-variance regularizer; not the AICAS DAR loss."""
    L = math.log(5.0 * YEAR / T_REF)
    L2_mean = L ** 2 / 3.0    # logarithmically uniform retention schedule
    t_read = 250e-9
    lt = np.linspace(math.log(T_REF), math.log(5.0 * YEAR), 200)
    R_mean = float(np.mean(lt + math.log(1.0 / (2.0 * t_read))))
    sp0 = (0.26348 / 25.0) ** 2
    losses = []
    for layer in model.layers().values():
        w = physical_weight_parameter(layer)
        g = w.abs() / w.detach().abs().max().clamp_min(1e-12)
        gc = g.clamp_min(1e-7)
        sigma_p = (0.26348 + 1.9650 * g - 1.1731 * g.square()) / 25.0
        sigma_nu = (-0.0125 * gc.log() - 0.0059).clamp(0.008, 0.045)
        q_s = (0.0088 / gc.pow(0.65).clamp_min(1e-3)).clamp(max=0.2)
        variance = sigma_p.square() + g.square() * (
            sigma_nu.square() * L2_mean + q_s.square() * R_mean
        )
        losses.append((variance.sum() + sp0 * g.numel()) / (g.square().sum() + 1e-12))
    return cfg.nsr_lambda * torch.stack(losses).mean()


@scientific("train")
def validation_score(model: nn.Module, val_loader, device, n_reads: int, seed: int):
    """Independent of held-out test; repeated evaluations reduce noisy selection."""
    scores = []
    for j in range(n_reads):
        with random_stream(seed + j * 173, device):
            scores.append(evaluate_model(model, val_loader, device))
    losses, accuracies = np.asarray(scores, dtype=float).T
    return float(accuracies.mean()), float(losses.mean())


def ensure_same_pretrained_weights(model: AnalogMLP, digital: DigitalMLP):
    """Assert copying both logical analog weights and digital bias before updates."""
    d_state = digital.state_dict()
    for name, layer in model.layers().items():
        w, b = layer.get_weights(apply_weight_scaling=True)
        torch.testing.assert_close(w.cpu(), d_state[name + '.weight'].cpu(), atol=2e-4, rtol=2e-3)
        torch.testing.assert_close(b.cpu(), d_state[name + '.bias'].cpu(), atol=2e-4, rtol=2e-3)


def training_metadata(cfg, fold, method, selected_epoch, trained_epochs):
    return {
        'fold': fold, 'method': method,
        'digital_epochs': cfg.digital_epochs,
        'analog_epochs_completed': trained_epochs,
        'selected_best_validation_epoch': selected_epoch,
        'optimizer': 'aihwkit.optim.AnalogAdam',
        'wd_mode': 'manual_decoupled_analog_weights_only',
        'selection_split': 'validation_only',
        'analog_bias_storage': 'digital_bias=True',
        'retention_model': 'AIHWKit PCMLikeNoiseModel; snapshot backend',
    }


# ============================ ADDED (diagnostics) ============================
def tile_mapping_error_rows(name, tile, logical_target: Tensor, omega: float) -> List[dict]:
    """AIHWKit 1.1.0 periphery.apply_weight_scaling: zero rows stay zero, alpha=1.

    Derive expected rows from the intended digital target, never from the actual tile.
    This is a transfer diagnostic only; it does not remap or modify any weights.
    """
    if not math.isfinite(omega) or omega <= 0:
        raise ScientificCheckError("Columnwise transfer check requires positive finite omega")
    target = logical_target.detach().cpu().float().reshape(logical_target.shape[0], -1)
    target_max = target.abs().amax(dim=1)
    active = target_max != 0
    expected_max = torch.where(active, torch.full_like(target_max, omega), torch.zeros_like(target_max))
    expected_scale = torch.where(active, target_max / omega, torch.ones_like(target_max))
    tile_w = tile.tile.weight.detach().cpu().float()
    if tile_w.shape != target.shape:
        raise ScientificCheckError(f"{name}: tile shape {tile_w.shape} != transfer target {target.shape}")
    actual_max = tile_w.abs().amax(dim=1)
    scales = tile.get_mapping_scales().detach().cpu().float().reshape(-1)
    if scales.shape != expected_scale.shape:
        raise ScientificCheckError(f"{name}: columnwise mapping scale shape mismatch")
    errors = (
        ("tile_weight_row_absmax_minus_expected", (actual_max - expected_max).abs(), actual_max),
        ("tile_mapping_scale_relative_error", (scales - expected_scale).abs() / expected_scale, scales),
    )
    return [{"layer": name, "tensor": kind, "max_abs_error": float(error.max()),
             "rms_error": float(error.square().mean().sqrt()), "max_abs_value": float(values.abs().max()),
             "zero_rows": int((~active).sum()), "active_rows": int(active.sum())}
            for kind, error, values in errors]


def transfer_error_report(analog: AnalogMLP, digital: DigitalMLP, omega: float) -> List[dict]:
    """Max-abs and RMS transfer errors of logical weights and biases (record only)."""
    d_state = digital.state_dict()
    rows = []
    for name, layer in analog.layers().items():
        w, b = layer.get_weights(apply_weight_scaling=True)
        tile, = list(layer.analog_tiles())
        tile_w = tile.tile.weight.detach().cpu()
        for kind, a, d in (("weight", w, d_state[name + ".weight"]), ("bias", b, d_state[name + ".bias"])):
            e = (a.detach().cpu().float() - d.detach().cpu().float())
            rows.append({"layer": name, "tensor": kind, "max_abs_error": float(e.abs().max()),
                         "rms_error": float(e.square().mean().sqrt()),
                         "max_abs_value": float(d.abs().max())})
        rows.extend(tile_mapping_error_rows(name, tile, d_state[name + ".weight"], omega))
    return rows


@scientific("diagnostics")
def nsr_components(w: Tensor) -> Dict[str, float]:
    """Per-layer terms of nsr_penalty(), same formulas, without lambda (Table II)."""
    w = w.detach().float()
    L = math.log(5.0 * YEAR / T_REF)
    L2_mean = L ** 2 / 3.0
    t_read = 250e-9
    lt = np.linspace(math.log(T_REF), math.log(5.0 * YEAR), 200)
    R_mean = float(np.mean(lt + math.log(1.0 / (2.0 * t_read))))
    sp0 = (0.26348 / 25.0) ** 2
    g = w.abs() / w.abs().max().clamp_min(1e-12)
    gc = g.clamp_min(1e-7)
    sigma_p = (0.26348 + 1.9650 * g - 1.1731 * g.square()) / 25.0
    sigma_nu = (-0.0125 * gc.log() - 0.0059).clamp(0.008, 0.045)
    q_s = (0.0088 / gc.pow(0.65).clamp_min(1e-3)).clamp(max=0.2)
    prog = sigma_p.square()
    drift = g.square() * sigma_nu.square() * L2_mean
    read = g.square() * q_s.square() * R_mean
    n = g.numel()
    signal = g.square().sum()
    nsr = ((prog + drift + read).sum() + sp0 * n) / (signal + 1e-12)
    return {
        "n_weights": int(n),
        "signal_energy_mean_g2": float(signal / n),
        "prog_noise_surrogate_mean": float(prog.mean()),
        "drift_noise_surrogate_mean": float(drift.mean()),
        "read_noise_surrogate_mean": float(read.mean()),
        "inactive_partner_floor_sp0": float(sp0),
        "nsr_layer": float(nsr),
    }


print("Models and regularizers loaded.")

### 5b. ResNet-32 for CIFAR-10 (ADDED; used when `MODEL_ARCHITECTURE = "resnet32"`)

* **`ResNet32`** is the He et al. CIFAR ResNet (6n+2 with n = 5): a 3×3 stem, then 3 stages × 5 basic blocks with widths 16/32/64, a 1×1-conv + BatchNorm shortcut where the shape changes, global average pooling, and a 64 → 10 fc layer. It has 466,906 parameters. The digital and analog networks are built by the **same class** from different conv/linear factories, so their module names match one-to-one.
* **`AnalogResNet32`.** All **34 conv/linear layers** (33 convolutions and the fc layer) are AIHWKit `AnalogConv2d` / `AnalogLinear` tiles with the **identical** `make_hwa_rpu()` configuration as the MLP: 8-bit DAC/ADC, out-noise 0.04, out-bound 10, ω = 0.8, learned out-scaling and input range, and a digital bias (only the fc layer has a bias). BatchNorm, ReLU, pooling and residual additions stay digital. Each layer is one tile: `max_input_size = 0`, as in the MLP. A convolution is AIHWKit's MVM over unfolded patches; the tile weight is `[out, in·kh·kw]`.
* **`transfer_pretrained_resnet_to_analog()`** copies the trained FP32 conv/fc weights (via `set_weights`), the fc bias and the **BatchNorm state**. `digital_resnet_state_matches()` checks the logical analog weights against the FP32 weights with the MLP's tolerances, and checks BatchNorm for exact equality.
* **Deployment (`deployed_weights`).** For PCM evaluation, the effective weights of every layer are written into the tiles. The AIHWKit forward then performs the identical DAC → `AnalogMVM` → ADC → scale → bias path that `shared_mvm` performs for the MLP, and the original weights are restored afterwards. `verify_resnet_deployment()` proves this for every checkpoint (Section 9b).

In [ ]:
# ============================ ADDED: ResNet-32 (CIFAR-10) ============================
@scientific("rn_model")
@dataclass(frozen=True)
class ResNetSpec:
    """Architecture/dataset identity of the ResNet experiment (hashed BY VALUE into the ResNet fingerprints)."""
    arch: str = "resnet32"
    dataset: str = "CIFAR-10"
    blocks_per_stage: int = 5                     # 6n+2 = 32 layers
    widths: Tuple[int, ...] = (16, 32, 64)
    shortcut: str = "B: 1x1 conv + BN where shape changes (analog), identity otherwise"
    batchnorm: str = "digital BatchNorm2d after every conv (not mapped to PCM)"
    pooling: str = "global average pooling (mean over H,W)"
    init: str = "kaiming_normal_(fan_out, relu) convs; BN weight 1 / bias 0; nn.Linear default"
    augmentation: str = "train only: pad-4 (black) random 32x32 crop + horizontal flip; seed (seed,fold,phase,epoch,batch)"
    normalization_mean: Tuple[float, ...] = CIFAR10_MEAN
    normalization_std: Tuple[float, ...] = CIFAR10_STD
    attention_gate_dim: int = 3 * 32 * 32         # Drift-Attention gate on the flattened normalized image
    inactive_filter_rel_threshold: float = 1e-2   # FP32 filters with max|w| < 1% of the layer's largest -> exact zeros


RESNET_SPEC = ResNetSpec()
# Config values that differ from the MLP experiment (all other hyperparameters are IDENTICAL; see Section 0 log)
RESNET_CONFIG_OVERRIDES = dict(
    data_root="data",
    digital_min_val_accuracy=80.0,      # validation-only underfit guard for 30-epoch FP32 ResNet-32 (MNIST MLP: 90)
    transfer_max_val_drop_pp=0.25,      # guard: |digital - analog with IDEAL periphery| (transfer + inactive-filter flush; MLP: 5 pp
                                        # on the realistic analog model, whose [0,1] inputs never clip at range 3)
    retention_points=12,                # the 12 explicit anchors (20 s ... 5 y) instead of 75 timestamps
    include_original_gdc=False,         # original all-ones-probe GDC skipped (saturating); safe GDC is kept
)


@scientific("rn_model")
class ResNetBasicBlock(nn.Module):
    def __init__(self, conv, cin: int, cout: int, stride: int):
        super().__init__()
        self.conv1 = conv(cin, cout, 3, stride, 1)
        self.bn1 = nn.BatchNorm2d(cout)
        self.conv2 = conv(cout, cout, 3, 1, 1)
        self.bn2 = nn.BatchNorm2d(cout)
        self.shortcut_conv = self.shortcut_bn = None
        if stride != 1 or cin != cout:
            self.shortcut_conv = conv(cin, cout, 1, stride, 0)
            self.shortcut_bn = nn.BatchNorm2d(cout)

    def forward(self, x: Tensor) -> Tensor:
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        sc = x if self.shortcut_conv is None else self.shortcut_bn(self.shortcut_conv(x))
        return F.relu(out + sc)


@scientific("rn_model")
class ResNet32(nn.Module):
    """He et al. CIFAR ResNet (6n+2, n=5). `conv`/`linear` factories make the digital and the analog network
    structurally identical (same module names), so weight transfer is a 1:1 name mapping."""
    def __init__(self, conv, linear, spec: ResNetSpec = RESNET_SPEC):
        super().__init__()
        w0 = spec.widths[0]
        self.stem = conv(3, w0, 3, 1, 1)
        self.stem_bn = nn.BatchNorm2d(w0)
        cin, stages = w0, []
        for s, cout in enumerate(spec.widths):
            blocks = []
            for b in range(spec.blocks_per_stage):
                blocks.append(ResNetBasicBlock(conv, cin, cout, 2 if (s > 0 and b == 0) else 1))
                cin = cout
            stages.append(nn.Sequential(*blocks))
        self.stage1, self.stage2, self.stage3 = stages
        self.fc = linear(cin, 10)

    def forward(self, x: Tensor) -> Tensor:
        x = x.view(-1, 3, 32, 32)                         # accepts flattened (gated) inputs
        x = F.relu(self.stem_bn(self.stem(x)))
        x = self.stage3(self.stage2(self.stage1(x)))
        return self.fc(x.mean(dim=(2, 3)))                # deterministic backward (no atomics)

    def mapped_layers(self) -> "OrderedDict[str, nn.Module]":
        """Every conv / linear layer (the layers that become PCM tiles), in forward order."""
        kinds = (nn.Conv2d, nn.Linear)
        try:
            from aihwkit.nn import AnalogConv2d, AnalogLinear
            kinds = kinds + (AnalogConv2d, AnalogLinear)
        except ImportError:
            pass
        return OrderedDict((n, m) for n, m in self.named_modules() if isinstance(m, kinds))


@scientific("rn_model")
def make_digital_resnet32(seed: int) -> ResNet32:
    """FP32 ResNet-32; deterministic initialization from `seed` (own generator, global RNG untouched)."""
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(int(seed))
        model = ResNet32(lambda ci, co, k, s, p: nn.Conv2d(ci, co, k, s, p, bias=False), nn.Linear)
        for m in model.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
    return model


@scientific("rn_model")
def batchnorm_state_keys(model: nn.Module) -> List[str]:
    bn = {n for n, m in model.named_modules() if isinstance(m, nn.BatchNorm2d)}
    return [k for k in model.state_dict() if k.rsplit(".", 1)[0] in bn]


@scientific("rn_model")
def flush_dead_filters(w: Tensor, rel_threshold: float = RESNET_SPEC.inactive_filter_rel_threshold):
    """Set FP32 filters (output rows) whose max|w| < rel_threshold * (largest row max|w| of the layer) to EXACT zero.
    Why: FP32 training (Adam + coupled L2) shrinks the filters of inactive channels towards 0 (down to ~1e-11).
    With columnwise weight_scaling_omega every row is stored as a full-scale (0.8) tile row, and AIHWKit's
    set_scales() gradient hook divides its gradient by mapping_scale^2, i.e. multiplies it by
    (layer max / row max)^2 relative to the largest row (> 1e4 below the threshold, up to 1e21 for collapsed rows).
    Regularizers acting on tile weights (DAR, NSR) then dominate -- or overflow -- the global gradient norm, and
    the 0.8 grad-norm clipping freezes every other weight. An exactly-zero row is mapped by AIHWKit with scale 1.
    Fold 0: 77 of 1,232 filters, validation accuracy unchanged (90.04 % -> 90.04 %); checked by the guard."""
    rows = w.reshape(w.shape[0], -1)
    rmax = rows.abs().amax(dim=1)
    dead = rmax < rel_threshold * rmax.max()
    if bool(dead.any()):
        w = w.clone()
        w[dead] = 0.0
    return w, int(dead.sum())


@scientific("rn_model")
class AnalogResNet32(ResNet32):
    """ResNet-32 whose 34 conv/linear layers are AIHWKit AnalogConv2d / AnalogLinear tiles with the SAME
    TorchInferenceRPUConfig as the MLP (make_hwa_rpu); BatchNorm, ReLU, pooling and residual adds are digital."""
    def __init__(self, cfg: Config, device: torch.device, pretrained_state: Optional[dict] = None):
        from aihwkit.nn import AnalogConv2d, AnalogLinear
        conv = lambda ci, co, k, s, p: AnalogConv2d(ci, co, k, s, p, bias=False, rpu_config=make_hwa_rpu(cfg))
        linear = lambda ci, co: AnalogLinear(ci, co, bias=True, rpu_config=make_hwa_rpu(cfg))
        with torch.random.fork_rng(devices=[]):          # construction-time inits never touch the global RNG
            torch.manual_seed(0)
            super().__init__(conv, linear)
        self.dead_filters_flushed = {}
        if pretrained_state is not None:
            bn_state = {k: v for k, v in pretrained_state.items() if k in batchnorm_state_keys(self)}
            result = self.load_state_dict(bn_state, strict=False)
            if result.unexpected_keys or set(batchnorm_state_keys(self)) - set(bn_state):
                raise RuntimeError(f"BatchNorm transfer incomplete: {result.unexpected_keys}")
            for name, layer in self.layers().items():
                w = pretrained_state[f"{name}.weight"].detach().to("cpu", dtype=torch.float32)
                b = pretrained_state.get(f"{name}.bias")
                b = None if b is None else b.detach().to("cpu", dtype=torch.float32)
                if w.shape[0] != layer.out_features or w[0].numel() != layer.in_features:
                    raise RuntimeError(f"Pretrained {name}: shape mismatch: {tuple(w.shape)}")
                w, n_dead = flush_dead_filters(w)
                self.dead_filters_flushed[name] = n_dead
                layer.set_weights(w, b)
        self.to(device)

    def layers(self):
        return self.mapped_layers()


@scientific("rn_model")
def digital_resnet_state_matches(model: AnalogResNet32, d_state: dict, atol=2e-4, rtol=2e-3):
    """Logical analog weights/biases == FP32 weights/biases, and BatchNorm state copied exactly."""
    for name, layer in model.layers().items():
        w, b = layer.get_weights(apply_weight_scaling=True)
        dw, _ = flush_dead_filters(d_state[name + ".weight"].detach().cpu().float())   # the intended mapping target
        torch.testing.assert_close(w.cpu(), dw.reshape(dw.shape[0], -1), atol=atol, rtol=rtol)
        if name + ".bias" in d_state:
            torch.testing.assert_close(b.cpu(), d_state[name + ".bias"].cpu(), atol=atol, rtol=rtol)
        elif b is not None:
            raise RuntimeError(f"{name}: unexpected analog bias")
    a_state = model.state_dict()
    for k in batchnorm_state_keys(model):
        if not torch.equal(a_state[k].cpu(), d_state[k].cpu()):
            raise RuntimeError(f"BatchNorm state {k} differs after transfer")


@scientific("rn_model")
def transfer_pretrained_resnet_to_analog(cfg: Config, digital_model: ResNet32, device: torch.device, val_loader=None):
    """Transfer *trained* FP32 conv/linear weights (+ fc bias) and BatchNorm state; no reinitialization."""
    state = cpu_cloned_state(digital_model)
    analog_model = AnalogResNet32(cfg, device, pretrained_state=state)
    digital_resnet_state_matches(analog_model, state)
    if val_loader is not None:
        _, digital_acc = evaluate_model(digital_model, val_loader, device)
        _, analog_acc = evaluate_model(analog_model, val_loader, device)
        _, ideal_acc = evaluate_model(ideal_periphery_copy(analog_model), val_loader, device)
        n_dead = sum(analog_model.dead_filters_flushed.values())
        print(f"TRANSFER: {n_dead} inactive FP32 filter(s) (max|w| < {RESNET_SPEC.inactive_filter_rel_threshold:g} x layer max) mapped as exact "
              f"zeros: {({k: v for k, v in analog_model.dead_filters_flushed.items() if v})}")
        print(f"TRANSFER check: digital FP32={digital_acc:.2f}% → analog, ideal periphery={ideal_acc:.2f}% "
              f"(guard: within {cfg.transfer_max_val_drop_pp} pp); analog HWA before optimizer={analog_acc:.2f}% "
              f"(8-bit I/O with the untrained input range {cfg.input_range_init}; learned during HWA)")
        if not cfg.fast_mode and abs(digital_acc - ideal_acc) > cfg.transfer_max_val_drop_pp:
            raise RuntimeError("Weight transfer changed accuracy excessively; the analog training has NOT begun")
    return analog_model


@scientific("rn_model")
def ideal_periphery_copy(analog_model: "AnalogResNet32") -> "AnalogResNet32":
    """Copy of the analog network with ideal periphery (no DAC/ADC quantization, no out-noise, no out-bound, no
    input clipping): its accuracy equals the FP32 network iff weights, biases, scales and BatchNorm transferred
    correctly. Used ONLY by the transfer guard; the experiment itself never uses it."""
    ideal = copy.deepcopy(analog_model).eval()
    for layer in ideal.layers().values():
        tile, = list(layer.analog_tiles())
        f = tile.rpu_config.forward
        f.inp_res, f.out_res, f.out_noise, f.out_bound = -1, -1, 0.0, 0.0
        with torch.no_grad():
            tile.input_range.fill_(1e6)
    return ideal


def ensure_same_pretrained_resnet(model: AnalogResNet32, digital: ResNet32):
    digital_resnet_state_matches(model, digital.state_dict())


def transfer_error_report_resnet(analog: AnalogResNet32, digital: ResNet32, omega: float) -> List[dict]:
    """Same rows as transfer_error_report() for every mapped layer (record only)."""
    d_state = digital.state_dict()
    rows = []
    for name, layer in analog.layers().items():
        w, b = layer.get_weights(apply_weight_scaling=True)
        tile, = list(layer.analog_tiles())
        tile_w = tile.tile.weight.detach().cpu()
        d_raw = d_state[name + ".weight"].detach().cpu().float()
        d_fl, n_dead = flush_dead_filters(d_raw)
        dw = d_fl.reshape(d_fl.shape[0], -1)
        rows.append({"layer": name, "tensor": "inactive_filters_flushed_to_zero", "max_abs_error": float(
                     (d_raw - d_fl).abs().max()), "rms_error": float(n_dead), "max_abs_value": float(d_raw.abs().max())})
        pairs = [("weight", w, dw)] + ([("bias", b, d_state[name + ".bias"])] if name + ".bias" in d_state else [])
        for kind, a, d in pairs:
            e = a.detach().cpu().float() - d.detach().cpu().float()
            rows.append({"layer": name, "tensor": kind, "max_abs_error": float(e.abs().max()),
                         "rms_error": float(e.square().mean().sqrt()), "max_abs_value": float(d.abs().max())})
        rows.extend(tile_mapping_error_rows(name, tile, dw, omega))
    return rows


@scientific("rn_pcm")
def snapshot_analog_resnet(model: AnalogResNet32) -> Dict[str, Dict[str, Tensor]]:
    """Per mapped layer: the 2-D PCM tile weight, digital bias (zeros if none), output scales, input range."""
    result = OrderedDict()
    for name, layer in model.layers().items():
        tile, = list(layer.analog_tiles())
        w = physical_weight_parameter(layer).detach()
        _, b = layer.get_weights(apply_weight_scaling=False)
        result[name] = {
            "weight": w.clone(),
            "bias": (b.detach().to(w.device).clone() if b is not None else torch.zeros(w.shape[0], device=w.device)),
            "scale": tile.get_scales().detach().to(w.device).clone(),
            "input_range": tile.input_range.detach().to(w.device).clone(),
        }
    return result


@scientific("rn_pcm")
@contextmanager
def deployed_weights(model: AnalogResNet32, weights: Dict[str, Tensor], layer_gains: Optional[Dict[str, float]] = None):
    """Deployment of PCM effective weights by SUBSTITUTION into the AIHWKit tiles (eval mode): every conv/linear
    MVM then runs through AIHWKit's own forward (input range, 8-bit DAC, AnalogMVM out-noise/ADC/bound, output
    scales, digital bias), exactly as SnapshotMLP/shared_mvm does for the MLP. A per-layer GDC gain multiplies the
    learned output scale (i.e. the MVM output before the digital bias, as in shared_mvm). Restored afterwards."""
    saved_w, saved_a = {}, {}
    layers = model.layers()
    with torch.no_grad():
        for name, layer in layers.items():
            w = physical_weight_parameter(layer)
            saved_w[name] = w.detach().clone()
            w.copy_(weights[name])
            g = 1.0 if layer_gains is None else float(layer_gains.get(name, 1.0))
            if g != 1.0:
                tile, = list(layer.analog_tiles())
                saved_a[name] = tile.out_scaling_alpha.detach().clone()
                tile.out_scaling_alpha.mul_(g)
    try:
        yield model.eval()
    finally:
        with torch.no_grad():
            for name, layer in layers.items():
                physical_weight_parameter(layer).copy_(saved_w[name])
                if name in saved_a:
                    tile, = list(layer.analog_tiles())
                    tile.out_scaling_alpha.copy_(saved_a[name])


@scientific("rn_pcm")
def load_analog_resnet_from_pack(cfg: Config, pack: dict, device: torch.device) -> AnalogResNet32:
    model = AnalogResNet32(cfg, device)
    load_model_state(model, pack['model'], device)
    return model.eval()


print("ResNet-32 model functions loaded.")

## 6. Training: FP32 pretraining → transfer → AnalogAdam HWA (original optimization, resumable per epoch)

**Digital FP32 pretraining** follows the original `train_digital_pretrained()`: Adam(lr = 1e-3, wd = 1e-4), OneCycleLR(max_lr = 4e-3) over exactly 30 epochs, gradient clipping at 0.8, loader generator seed `seed+fold+3000`, and best-validation selection (strictly greater accuracy). **ADDED:** a resumable checkpoint after every epoch (model, optimizer, scheduler, history, loader/torch/CUDA/NumPy/Python RNG states), written to two alternating slots in place, and a separately saved final-epoch model.

**HWA fine-tuning** follows the original `train_hwa_all()` for every method:
`AnalogAdam` with explicit parameter groups, then `regroup_param_groups(model)`; OneCycleLR(1e-3 → 4e-3); manual decoupled weight decay on the PCM tile weights only; input-range clamp; gradient clipping. All **154 epochs always run**, with no early stopping. Selection uses the mean validation accuracy over 3 noisy reads (ties broken by lower loss) at epochs ≥ 1. Both the final-epoch and the best-validation checkpoints are saved, and retention uses the best-validation one.

**ADDED (logging and verification only; the optimization is unchanged):** training accuracy per epoch, an optimizer-step counter, the max |Δw| of the first AnalogAdam step, the digital-checkpoint identity stored in every HWA checkpoint, and an exact re-evaluation check after the best checkpoint is restored.

**Correction (documented in the change log):** AIHWKit 1.1.0 resets the tile's device bookkeeping to CPU when a CPU-mapped `state_dict` is loaded into a CUDA model. The original resume path did exactly that. After every `load_state_dict` we now call `model.to(device)`. This only moves bookkeeping; it does not touch the values or the gradient hook. The smoke test verifies that an interrupted-and-resumed training is **bit-identical** to an uninterrupted one.

In [ ]:
# ============================ TRAINING ============================
class SimulatedInterruption(RuntimeError):
    """Raised only by the smoke test's resume test (simulated Colab disconnect)."""


_INTERRUPT_HOOK = {"stage": None, "after_epoch": None}
_RESUME_LOG: List[tuple] = []          # (stage, fold, resumed_at_epoch, slot file) -- verified by the smoke test


def _maybe_interrupt(stage: str, epochs_completed: int):
    if _INTERRUPT_HOOK["stage"] == stage and _INTERRUPT_HOOK["after_epoch"] == epochs_completed:
        _INTERRUPT_HOOK["stage"] = None
        raise SimulatedInterruption(f"simulated disconnect after {stage} epoch {epochs_completed}")


@scientific("train")
def rng_snapshot(device: torch.device) -> dict:
    return {
        "torch": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state(device) if device.type == "cuda" else None,
        "numpy": np.random.get_state(),
        "python": random.getstate(),
    }


@scientific("train")
def rng_restore(snap: dict, device: torch.device) -> None:
    torch.set_rng_state(snap["torch"].cpu())
    if device.type == "cuda" and snap.get("cuda") is not None:
        torch.cuda.set_rng_state(snap["cuda"].cpu(), device)
    np.random.set_state(snap["numpy"])
    random.setstate(snap["python"])


def _gpu_name() -> str:
    return torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"


def verify_pack(pack: dict, train_fp: str, kind: str, fold: int, path: Path):
    problems = []
    if pack.get("train_fp") != train_fp:
        problems.append(f"train_fp {str(pack.get('train_fp'))[:12]} != {train_fp[:12]}")
    if pack.get("kind") != kind:
        problems.append(f"kind {pack.get('kind')} != {kind}")
    if pack.get("fold") != fold:
        problems.append(f"fold {pack.get('fold')} != {fold}")
    if problems:
        raise ScientificCheckError(
            f"Refusing incompatible checkpoint {path}: " + "; ".join(problems) +
            ". It belongs to a different experiment; it will NOT be reused or overwritten.")


@scientific("train")
def load_model_state(model: nn.Module, state: dict, device: torch.device) -> nn.Module:
    """load_state_dict + model.to(device) (ADDED correction: AIHWKit 1.1.0 resets tile
    device bookkeeping to CPU when a CPU-mapped state_dict is loaded)."""
    model.load_state_dict(state)
    return model.to(device)


@scientific("train")
def train_digital_pretrained(cfg: Config, fold: int, train_loader, val_loader,
                             device: torch.device, fold_dir: Path, train_fp: str):
    """Original digital FP32 recipe + ADDED per-epoch resumable checkpoints and final-epoch model."""
    require_split(val_loader, "val", "FP32 best-epoch selection")
    fold_dir = Path(fold_dir)
    seed_all(cfg.seed + fold)
    model = DigitalMLP().to(device=device, dtype=torch.float32)
    assert all(p.dtype == torch.float32 for p in model.parameters())
    best_path = fold_dir / "digital_best_validation.pt"
    final_path = fold_dir / "digital_final.pt"
    state_path = fold_dir / "digital_training.pt"
    history_file = fold_dir / "digital_history.csv"

    if best_path.exists() and final_path.exists():
        pack = torch_load(best_path)
        verify_pack(pack, train_fp, "digital_best_validation", fold, best_path)
        model.load_state_dict(pack["model"])
        _, acc = evaluate_model(model, val_loader, device)
        if abs(acc - pack["best_val_accuracy"]) > 1e-6:
            raise ScientificCheckError(f"Digital checkpoint re-evaluation mismatch: {acc} vs {pack['best_val_accuracy']}")
        print(f"Digital fold={fold}: completed checkpoint loaded (best epoch {pack['best_epoch']}, val={acc:.2f}%)")
        return model.eval(), pd.DataFrame(pack["history"]), pack

    train_loader.generator.manual_seed(cfg.seed + fold + 3000)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.digital_lr,
                                 weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=cfg.digital_max_lr,
        epochs=cfg.digital_epochs, steps_per_epoch=len(train_loader),
        div_factor=cfg.digital_max_lr / cfg.digital_lr,
    )
    best_acc = -1.0
    best_state = None
    best_epoch = 0
    rows = []
    start_epoch, steps = 0, 0
    saved, src = load_rotating(state_path)
    if saved is not None:
        verify_pack(saved, train_fp, "digital_training", fold, src)
        model.load_state_dict(saved["model"])
        optimizer.load_state_dict(saved["optimizer"])
        scheduler.load_state_dict(saved["scheduler"])
        best_acc, best_state, best_epoch = saved["best_val_accuracy"], saved["best_model"], saved["best_epoch"]
        rows, start_epoch, steps = saved["history"], saved["next_epoch"], saved["optimizer_steps"]
        train_loader.generator.set_state(saved["loader_rng"].cpu())
        rng_restore(saved["rng"], device)
        _RESUME_LOG.append(("digital", fold, start_epoch, src.name))
        print(f"RESUME digital fold={fold}: continuing at epoch {start_epoch+1}/{cfg.digital_epochs} (from {src.name})")

    for epoch in range(start_epoch, cfg.digital_epochs):
        model.train()
        total_ce = 0.0
        correct = 0
        n = 0
        for x, y in train_loader:
            x = x.to(device, dtype=torch.float32)
            y = y.to(device)
            optimizer.zero_grad(set_to_none=True)
            pred = model(x)
            loss = F.cross_entropy(pred, y)
            if not torch.isfinite(loss):
                raise RuntimeError(f"Digital loss nonfinite at epoch {epoch+1}")
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip,
                                           error_if_nonfinite=True)
            optimizer.step()
            scheduler.step()
            steps += 1
            bs = y.numel()
            n += bs
            correct += (pred.argmax(1) == y).sum().item()
            total_ce += loss.item() * bs
        val_ce, val_acc = evaluate_model(model, val_loader, device)
        if val_acc > best_acc:
            best_acc = val_acc
            best_state = cpu_cloned_state(model)
            best_epoch = epoch + 1
        rows.append({"fold": fold, "epoch": epoch + 1,
                     "train_ce": total_ce / n, "train_accuracy": 100 * correct / n,
                     "val_ce": val_ce, "val_accuracy": val_acc,
                     "best_epoch_so_far": best_epoch, "optimizer_steps": steps,
                     "lr_end": scheduler.get_last_lr()[0], "gpu": _gpu_name()})
        print(f"DIGITAL FP32 fold={fold} epoch={epoch+1}/{cfg.digital_epochs} "
              f"train={100*correct/n:.2f}% val={val_acc:.2f}%", flush=True)
        save_rotating({
            "kind": "digital_training", "train_fp": train_fp, "fold": fold,
            "model": cpu_cloned_state(model), "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(), "history": rows, "next_epoch": epoch + 1,
            "optimizer_steps": steps, "best_val_accuracy": best_acc, "best_model": best_state,
            "best_epoch": best_epoch, "loader_rng": train_loader.generator.get_state(),
            "rng": rng_snapshot(device),
        }, state_path, epoch + 1, final=(epoch + 1 == cfg.digital_epochs))
        write_text_inplace(history_file, pd.DataFrame(rows).to_csv(index=False))
        _maybe_interrupt("digital", epoch + 1)

    if best_state is None:
        raise RuntimeError("Digital training produced no valid model")
    if len(rows) != cfg.digital_epochs or [r["epoch"] for r in rows] != list(range(1, cfg.digital_epochs + 1)):
        raise ScientificCheckError(f"Digital history has {len(rows)} epochs, expected {cfg.digital_epochs}")
    final_state = cpu_cloned_state(model)
    model.load_state_dict(best_state)
    _, best_val_acc = evaluate_model(model, val_loader, device)
    if not cfg.fast_mode and best_val_acc < cfg.digital_min_val_accuracy:
        raise RuntimeError(f"Digital pretraining underfit: val={best_val_acc:.2f}%; "
                           "increase digital_epochs or inspect preprocessing")
    common = {"train_fp": train_fp, "fold": fold, "history": rows, "dtype": "torch.float32",
              "digital_epochs_completed": len(rows), "optimizer_steps": steps,
              "steps_per_epoch": len(train_loader), "best_epoch": best_epoch,
              "best_val_accuracy": best_val_acc}
    atomic_torch_save({"kind": "digital_final", "model": final_state, **common}, final_path)
    atomic_torch_save({"kind": "digital_best_validation", "model": best_state,
                       "model_identity": state_dict_hash(best_state), **common}, best_path)
    print(f"FP32 PRETRAIN PASS fold={fold}: best epoch {best_epoch}, val={best_val_acc:.2f}%")
    return model.eval(), pd.DataFrame(rows), torch_load(best_path)


@scientific("train")
def train_hwa_all(method, cfg, fold, digital, digital_identity, train_loader, val_loader,
                  device, fold_dir, train_fp):
    """Original train_hwa_all(): full `cfg.epochs` epochs, best-VAL checkpoint (epoch>=1),
    FINAL kept separately.  Optimizer type is exactly AnalogAdam; manually decoupled
    weight decay on physical analog weights only.  ADDED: per-epoch resumable
    rotating checkpoints, RNG states, training accuracy, verification metadata."""
    from aihwkit.optim import AnalogAdam
    require_split(val_loader, "val", "HWA best-epoch selection")
    if method not in ALL_TRAIN_METHODS:
        raise ValueError(f'Unknown HWA method {method}')
    fold_dir = Path(fold_dir)
    tag = method
    path = fold_dir / (tag + '_training.pt')
    history_path = fold_dir / (tag + '_history.csv')
    best_path = fold_dir / (tag + '_best_validation.pt')
    final_path = fold_dir / (tag + '_final.pt')

    if best_path.exists() and final_path.exists():
        pack = torch_load(best_path)
        verify_pack(pack, train_fp, "hwa_best_validation", fold, best_path)
        if pack.get("method") != method or pack.get("digital_identity") != digital_identity:
            raise ScientificCheckError(f"{best_path}: method/digital-identity mismatch")
        print(f"[{method}] fold {fold}: completed ({pack['analog_epochs_completed']} epochs, "
              f"best epoch {pack['selected_best_validation_epoch']}); not retrained")
        return pack

    ds_seed = cfg.seed + fold
    seed_all(ds_seed)
    train_loader.generator.manual_seed(cfg.seed + fold + 1000)
    model = transfer_pretrained_digital_to_analog(
        cfg, digital, cfg.seed + fold, device, val_loader=val_loader
    )
    ensure_same_pretrained_weights(model, digital)
    gate = AttentionGate(cg=cfg.attention_input_gain).to(device) if method in (ATT, ATT_ONLY) else None
    weights = [physical_weight_parameter(layer) for layer in model.layers().values()]
    ids = {id(p) for p in weights}
    nonweights = [p for p in model.parameters() if id(p) not in ids]
    # Set optimizer weight_decay=0; explicit *decoupled* weight decay applied post-step.
    opt = AnalogAdam([
        {'params': weights, 'weight_decay': 0.0},
        {'params': nonweights, 'weight_decay': 0.0},
    ], lr=cfg.lr, weight_decay=0.0)
    opt.regroup_param_groups(model)
    assert isinstance(opt, AnalogAdam), 'Wrong optimizer class'
    opt_gate = torch.optim.Adam(gate.parameters(), lr=cfg.attention_lr) if gate else None
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=cfg.max_lr, epochs=cfg.epochs,
        steps_per_epoch=len(train_loader), div_factor=cfg.max_lr / cfg.lr,
    )
    initial_refs = {
        name: physical_weight_parameter(layer).detach().clone()
        for name, layer in model.layers().items()
    }
    initial_refs_id = state_dict_hash(initial_refs)   # ADDED: refs are recomputed (deterministic) on resume
    # ADDED verification: no reinitialization between transfer and the first step
    ensure_same_pretrained_weights(model, digital)
    from aihwkit.optim.context import AnalogContext
    n_ctx = sum(isinstance(p, AnalogContext) for g in opt.param_groups for p in g['params'])

    history, start_epoch, steps = [], 0, 0
    best_val_acc, best_val_loss, best_state, best_epoch = -1.0, math.inf, None, -1
    first_step_change = None
    transfer_rows = transfer_error_report(model, digital, cfg.weight_scaling_omega)
    saved, src = load_rotating(path)
    if saved is not None:
        verify_pack(saved, train_fp, "hwa_training", fold, src)
        if saved.get('method') != method or saved.get('digital_identity') != digital_identity:
            raise ScientificCheckError('Incompatible training checkpoint (method/digital identity)')
        load_model_state(model, saved['model'], device)
        opt.load_state_dict(saved['optimizer'])
        scheduler.load_state_dict(saved['scheduler'])
        if gate is not None:
            gate.load_state_dict(saved['gate'])
            opt_gate.load_state_dict(saved['gate_optimizer'])
        history, start_epoch, steps = saved['history'], saved['next_epoch'], saved['optimizer_steps']
        best_val_acc, best_val_loss = saved['best_val_accuracy'], saved['best_val_loss']
        best_state, best_epoch = saved['best_model'], saved['best_epoch']
        first_step_change = saved.get('first_step_max_weight_change')
        if saved.get('initial_refs_id') != initial_refs_id:   # identical transfer -> identical DAR reference
            raise ScientificCheckError('Recomputed DAR reference weights differ from the interrupted run')
        train_loader.generator.set_state(saved['loader_rng'].cpu())
        rng_restore(saved['rng'], device)
        _RESUME_LOG.append((f"hwa:{method}", fold, start_epoch, src.name))
        print(f"RESUME [{method}] fold {fold}: continuing at epoch {start_epoch+1}/{cfg.epochs} (from {src.name})")

    t_start = time.time()
    for epoch in range(start_epoch, cfg.epochs):
        model.train()
        if gate is not None:
            gate.train()
        sum_ce = sum_extra = sum_loss = 0.0
        n_samples = 0
        n_correct = 0
        for batch, (x, y) in enumerate(train_loader):
            x, y = x.to(device).flatten(1), y.to(device)
            opt.zero_grad(set_to_none=True)
            if opt_gate: opt_gate.zero_grad(set_to_none=True)
            if gate is not None:
                x, alpha = gate(x)
            elif method == GAIN_ONLY:
                x = cfg.attention_input_gain * x
                alpha = None
            else:
                alpha = None
            logits = model(x)
            ce = F.cross_entropy(logits, y)
            extra = torch.zeros((), device=device)
            if method in (DAR, ATT):
                td = cfg.dar_times_s[epoch % len(cfg.dar_times_s)]
                with random_stream(cfg.seed + fold * 1_000_000 + epoch * 10_000 + batch, device):
                    extra = sum(dar_penalty(physical_weight_parameter(layer),
                                            initial_refs[name], epoch, td, cfg)
                                for name, layer in model.layers().items())
            if method == NSR:
                extra = nsr_penalty(model, cfg)
            if alpha is not None:
                # v2_lib uses scalar mean; keep equivalent regularizer for parity.
                extra = extra + cfg.attention_lambda * ((alpha - alpha.mean()) ** 2).sum(1).mean()
            loss = ce + extra
            if not torch.isfinite(loss):
                raise FloatingPointError(f'{method} nonfinite loss at epoch={epoch+1}, batch={batch}')
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip, error_if_nonfinite=True)
            if gate is not None:
                torch.nn.utils.clip_grad_norm_(gate.parameters(), cfg.grad_clip, error_if_nonfinite=True)
            if first_step_change is None:                       # ADDED (verification only)
                before = [w.detach().clone() for w in weights]
            opt.step()
            if first_step_change is None:
                first_step_change = max(float((w.detach() - b).abs().max()) for w, b in zip(weights, before))
                del before
            # Decoupled weight decay only on PCM weight tensors, not biases or scales.
            with torch.no_grad():
                group_lr = {id(p): float(g['lr']) for g in opt.param_groups for p in g['params']}
                for p in weights:
                    p.mul_(1.0 - group_lr.get(id(p), cfg.lr) * cfg.weight_decay)
            scheduler.step()
            if opt_gate is not None: opt_gate.step()
            with torch.no_grad():
                for layer in model.layers().values():
                    tile, = list(layer.analog_tiles())
                    tile.input_range.clamp_(min=1e-3)
            steps += 1
            bs = y.numel()
            n_samples += bs
            n_correct += int((logits.detach().argmax(1) == y).sum())   # ADDED (logging only)
            sum_ce += float(ce.detach()) * bs
            sum_extra += float(extra.detach()) * bs
            sum_loss += float(loss.detach()) * bs
        # Selection strictly on validation, using RAW input / no training-only gate.
        model.eval()
        val_acc, val_loss = validation_score(
            model, val_loader, device, cfg.validation_reads,
            cfg.seed + fold * 100_000 + epoch * 41,
        )
        if val_acc > best_val_acc or (val_acc == best_val_acc and val_loss < best_val_loss):
            best_val_acc, best_val_loss = val_acc, val_loss
            best_epoch = epoch + 1
            best_state = cpu_cloned_state(model)
        history.append({
            'fold': fold, 'method': method, 'epoch': epoch + 1,
            'train_ce': sum_ce / n_samples, 'train_extra': sum_extra / n_samples,
            'train_loss': sum_loss / n_samples,
            'train_accuracy': 100.0 * n_correct / n_samples,
            'val_accuracy': val_acc, 'val_loss': val_loss,
            'best_val_accuracy': best_val_acc, 'best_epoch': best_epoch,
            'optimizer': type(opt).__name__, 'optimizer_steps': steps,
            'lr_end': scheduler.get_last_lr()[0], 'gpu': _gpu_name(),
        })
        pack = {
            'kind': 'hwa_training', 'train_fp': train_fp, 'fold': fold, 'method': method,
            'digital_identity': digital_identity,
            'model': cpu_cloned_state(model), 'optimizer': opt.state_dict(),
            'scheduler': scheduler.state_dict(), 'initial_refs_id': initial_refs_id,
            'gate': cpu_cloned_state(gate) if gate else None,
            'gate_optimizer': opt_gate.state_dict() if opt_gate else None,
            'next_epoch': epoch + 1, 'history': history, 'optimizer_steps': steps,
            'best_model': best_state, 'best_epoch': best_epoch,
            'best_val_accuracy': best_val_acc, 'best_val_loss': best_val_loss,
            'first_step_max_weight_change': first_step_change,
            'loader_rng': train_loader.generator.get_state(),
            'rng': rng_snapshot(device),
        }
        save_rotating(pack, path, epoch + 1, final=(epoch + 1 == cfg.epochs))
        write_text_inplace(history_path, pd.DataFrame(history).to_csv(index=False))
        if (epoch + 1) % 10 == 0 or epoch == start_epoch or epoch == cfg.epochs - 1:
            print(f'[{method}] fold {fold} epoch {epoch+1}/{cfg.epochs}'
                  f' CE={sum_ce/n_samples:.4f} extra={sum_extra/n_samples:.3g}'
                  f' train={100.0*n_correct/n_samples:.2f}% val={val_acc:.2f}% best={best_epoch}'
                  f' time={(time.time()-t_start)/60:.1f}m', flush=True)
        _maybe_interrupt(f"hwa:{method}", epoch + 1)

    if len(history) != cfg.epochs or best_state is None or best_epoch < 1:
        raise RuntimeError(f'Incomplete training: {method}, epochs={len(history)}/{cfg.epochs}')
    final_state = cpu_cloned_state(model)
    meta = {
        'train_fp': train_fp, 'fold': fold, 'method': method,
        'digital_identity': digital_identity, 'optimizer_steps': steps,
        'steps_per_epoch': len(train_loader), 'history': history,
        'first_step_max_weight_change': first_step_change,
        'analog_context_groups': n_ctx, 'transfer_errors': transfer_rows,
    }
    ds_meta_final = training_metadata(cfg, fold, method, cfg.epochs, len(history))
    ds_meta_best = training_metadata(cfg, fold, method, best_epoch, len(history))
    atomic_torch_save({'kind': 'hwa_final', 'model': final_state,
                       'model_identity': state_dict_hash(final_state), **meta, **ds_meta_final}, final_path)
    # Retention uses best checkpoint, but full 154e optimization was completed.
    load_model_state(model, best_state, device)
    model.eval()
    # ADDED: exact re-evaluation of the restored best checkpoint (same validation seed)
    re_acc, re_loss = validation_score(model, val_loader, device, cfg.validation_reads,
                                       cfg.seed + fold * 100_000 + (best_epoch - 1) * 41)
    restore_diff = abs(re_acc - best_val_acc)
    atomic_torch_save({'kind': 'hwa_best_validation', 'model': best_state,
                       'model_identity': state_dict_hash(best_state),
                       'val_accuracy': best_val_acc, 'val_loss': best_val_loss,
                       'restore_reeval_val_accuracy': re_acc, 'restore_abs_diff_pp': restore_diff,
                       **meta, **ds_meta_best}, best_path)
    print(f'[{method}] fold {fold}: DONE {len(history)} epochs, {steps} AnalogAdam steps; '
          f'best-validation epoch {best_epoch} (val {best_val_acc:.3f}%, re-eval diff {restore_diff:.2e} pp)')
    return torch_load(best_path)


@scientific("train")
def make_fp32_direct(cfg, fold, digital, digital_identity, val_loader, device, fold_dir, train_fp):
    """FP32-Direct (original run()): transfer the pretrained weights+biases, ZERO HWA steps."""
    path = Path(fold_dir) / f'{DIRECT}_analog_checkpoint.pt'
    if path.exists():
        pack = torch_load(path)
        verify_pack(pack, train_fp, "fp32_direct", fold, path)
        if pack.get("digital_identity") != digital_identity:
            raise ScientificCheckError("FP32-Direct checkpoint was made from a different digital model")
        return pack
    direct = transfer_pretrained_digital_to_analog(cfg, digital, cfg.seed + fold, device, val_loader=val_loader)
    ensure_same_pretrained_weights(direct, digital)
    state = cpu_cloned_state(direct)
    pack = {
        'kind': 'fp32_direct', 'train_fp': train_fp, 'fold': fold, 'method': DIRECT,
        'model': state, 'model_identity': state_dict_hash(state),
        'digital_identity': digital_identity, 'digital_epochs': cfg.digital_epochs,
        'analog_epochs': 0, 'analog_epochs_completed': 0, 'optimizer_steps': 0,
        'selection_split': 'validation', 'transfer_errors': transfer_error_report(direct, digital, cfg.weight_scaling_omega),
    }
    atomic_torch_save(pack, path)
    return pack


@scientific("pcm")
def load_analog_from_pack(cfg: Config, pack: dict, device: torch.device) -> AnalogMLP:
    model = AnalogMLP(cfg, 0, device)
    load_model_state(model, pack['model'], device)
    return model.eval()


print("Training functions loaded.")

### 6b. ResNet-32 training (ADDED)

`train_digital_resnet`, `train_hwa_resnet` and `make_fp32_direct_resnet` are line-for-line copies of the MLP trainers above. They have the **same** optimizers, schedules, losses (CE, DAR, NSR, attention penalty), gradient clipping, decoupled weight decay, input-range clamp, validation selection (3 noisy reads), seeds, checkpoint files and resume logic. The only differences:
1. the ResNet networks, with BatchNorm in training mode while training and in eval mode for validation;
2. the training-only CIFAR augmentation of every training batch, which is identical for all HWA methods within a fold;
3. the Drift-Attention gate on the flattened normalized image (dim 3072);
4. layer-generic transfer checks.

Memory: the full protocol (batch 256) needs about 13 GB of GPU memory for ResNet-32 HWA training, which fits on L4, A100 and G4.

In [ ]:
# ============================ ADDED: ResNet-32 TRAINING ============================
# Line-for-line copies of train_digital_pretrained / train_hwa_all / make_fp32_direct. The ONLY differences:
#   (1) ResNet32 / AnalogResNet32 instead of DigitalMLP / AnalogMLP (BatchNorm in train mode while training),
#   (2) training-only CIFAR augmentation of every training batch (cifar_augment, paired across HWA methods),
#   (3) Drift-Attention gate on the flattened 3x32x32 image (dim 3072; MLP: 784),
#   (4) layer-generic transfer checks (conv weights reshaped to the tile's [out, in*kh*kw]).
# Optimizers, schedules, losses (CE, DAR, NSR, gate penalty), clipping, decoupled weight decay, input-range clamp,
# validation selection (3 noisy reads), seeds, checkpoint files and resume logic are unchanged.

@scientific("rn_train")
def train_digital_resnet(cfg: Config, fold: int, train_loader, val_loader,
                         device: torch.device, fold_dir: Path, train_fp: str):
    require_split(val_loader, "val", "FP32 best-epoch selection")
    fold_dir = Path(fold_dir)
    seed_all(cfg.seed + fold)
    model = make_digital_resnet32(cfg.seed + fold).to(device=device, dtype=torch.float32)
    assert all(p.dtype == torch.float32 for p in model.parameters())
    best_path = fold_dir / "digital_best_validation.pt"
    final_path = fold_dir / "digital_final.pt"
    state_path = fold_dir / "digital_training.pt"
    history_file = fold_dir / "digital_history.csv"

    if best_path.exists() and final_path.exists():
        pack = torch_load(best_path)
        verify_pack(pack, train_fp, "digital_best_validation", fold, best_path)
        model.load_state_dict(pack["model"])
        _, acc = evaluate_model(model, val_loader, device)
        if abs(acc - pack["best_val_accuracy"]) > 1e-6:
            raise ScientificCheckError(f"Digital checkpoint re-evaluation mismatch: {acc} vs {pack['best_val_accuracy']}")
        print(f"Digital fold={fold}: completed checkpoint loaded (best epoch {pack['best_epoch']}, val={acc:.2f}%)")
        return model.eval(), pd.DataFrame(pack["history"]), pack

    train_loader.generator.manual_seed(cfg.seed + fold + 3000)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.digital_lr, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=cfg.digital_max_lr,
        epochs=cfg.digital_epochs, steps_per_epoch=len(train_loader),
        div_factor=cfg.digital_max_lr / cfg.digital_lr,
    )
    best_acc = -1.0
    best_state = None
    best_epoch = 0
    rows = []
    start_epoch, steps = 0, 0
    saved, src = load_rotating(state_path)
    if saved is not None:
        verify_pack(saved, train_fp, "digital_training", fold, src)
        model.load_state_dict(saved["model"])
        optimizer.load_state_dict(saved["optimizer"])
        scheduler.load_state_dict(saved["scheduler"])
        best_acc, best_state, best_epoch = saved["best_val_accuracy"], saved["best_model"], saved["best_epoch"]
        rows, start_epoch, steps = saved["history"], saved["next_epoch"], saved["optimizer_steps"]
        train_loader.generator.set_state(saved["loader_rng"].cpu())
        rng_restore(saved["rng"], device)
        _RESUME_LOG.append(("digital", fold, start_epoch, src.name))
        print(f"RESUME digital fold={fold}: continuing at epoch {start_epoch+1}/{cfg.digital_epochs} (from {src.name})")

    t_start = time.time()
    for epoch in range(start_epoch, cfg.digital_epochs):
        model.train()
        total_ce = 0.0
        correct = 0
        n = 0
        for batch, (x, y) in enumerate(train_loader):
            x = cifar_augment(x.to(device, dtype=torch.float32), augment_seed(cfg, fold, 0, epoch, batch))
            y = y.to(device)
            optimizer.zero_grad(set_to_none=True)
            pred = model(x)
            loss = F.cross_entropy(pred, y)
            if not torch.isfinite(loss):
                raise RuntimeError(f"Digital loss nonfinite at epoch {epoch+1}")
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip, error_if_nonfinite=True)
            optimizer.step()
            scheduler.step()
            steps += 1
            bs = y.numel()
            n += bs
            correct += (pred.argmax(1) == y).sum().item()
            total_ce += loss.item() * bs
        val_ce, val_acc = evaluate_model(model, val_loader, device)
        if val_acc > best_acc:
            best_acc = val_acc
            best_state = cpu_cloned_state(model)
            best_epoch = epoch + 1
        rows.append({"fold": fold, "epoch": epoch + 1,
                     "train_ce": total_ce / n, "train_accuracy": 100 * correct / n,
                     "val_ce": val_ce, "val_accuracy": val_acc,
                     "best_epoch_so_far": best_epoch, "optimizer_steps": steps,
                     "lr_end": scheduler.get_last_lr()[0], "gpu": _gpu_name()})
        print(f"DIGITAL FP32 ResNet-32 fold={fold} epoch={epoch+1}/{cfg.digital_epochs} "
              f"train={100*correct/n:.2f}% val={val_acc:.2f}% time={(time.time()-t_start)/60:.1f}m", flush=True)
        save_rotating({
            "kind": "digital_training", "train_fp": train_fp, "fold": fold,
            "model": cpu_cloned_state(model), "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(), "history": rows, "next_epoch": epoch + 1,
            "optimizer_steps": steps, "best_val_accuracy": best_acc, "best_model": best_state,
            "best_epoch": best_epoch, "loader_rng": train_loader.generator.get_state(),
            "rng": rng_snapshot(device),
        }, state_path, epoch + 1, final=(epoch + 1 == cfg.digital_epochs))
        write_text_inplace(history_file, pd.DataFrame(rows).to_csv(index=False))
        _maybe_interrupt("digital", epoch + 1)

    if best_state is None:
        raise RuntimeError("Digital training produced no valid model")
    if len(rows) != cfg.digital_epochs or [r["epoch"] for r in rows] != list(range(1, cfg.digital_epochs + 1)):
        raise ScientificCheckError(f"Digital history has {len(rows)} epochs, expected {cfg.digital_epochs}")
    final_state = cpu_cloned_state(model)
    model.load_state_dict(best_state)
    _, best_val_acc = evaluate_model(model, val_loader, device)
    if not cfg.fast_mode and best_val_acc < cfg.digital_min_val_accuracy:
        raise RuntimeError(f"Digital pretraining underfit: val={best_val_acc:.2f}% < {cfg.digital_min_val_accuracy}%")
    common = {"train_fp": train_fp, "fold": fold, "history": rows, "dtype": "torch.float32",
              "digital_epochs_completed": len(rows), "optimizer_steps": steps,
              "steps_per_epoch": len(train_loader), "best_epoch": best_epoch,
              "best_val_accuracy": best_val_acc, "arch": "resnet32"}
    atomic_torch_save({"kind": "digital_final", "model": final_state, **common}, final_path)
    atomic_torch_save({"kind": "digital_best_validation", "model": best_state,
                       "model_identity": state_dict_hash(best_state), **common}, best_path)
    print(f"FP32 PRETRAIN PASS fold={fold}: best epoch {best_epoch}, val={best_val_acc:.2f}%")
    return model.eval(), pd.DataFrame(rows), torch_load(best_path)


@scientific("rn_train")
def train_hwa_resnet(method, cfg, fold, digital, digital_identity, train_loader, val_loader,
                     device, fold_dir, train_fp):
    """train_hwa_all() for ResNet-32 (see the header of this cell for the only differences)."""
    from aihwkit.optim import AnalogAdam
    require_split(val_loader, "val", "HWA best-epoch selection")
    if method not in ALL_TRAIN_METHODS:
        raise ValueError(f'Unknown HWA method {method}')
    fold_dir = Path(fold_dir)
    tag = method
    path = fold_dir / (tag + '_training.pt')
    history_path = fold_dir / (tag + '_history.csv')
    best_path = fold_dir / (tag + '_best_validation.pt')
    final_path = fold_dir / (tag + '_final.pt')

    if best_path.exists() and final_path.exists():
        pack = torch_load(best_path)
        verify_pack(pack, train_fp, "hwa_best_validation", fold, best_path)
        if pack.get("method") != method or pack.get("digital_identity") != digital_identity:
            raise ScientificCheckError(f"{best_path}: method/digital-identity mismatch")
        print(f"[{method}] fold {fold}: completed ({pack['analog_epochs_completed']} epochs, "
              f"best epoch {pack['selected_best_validation_epoch']}); not retrained")
        return pack

    ds_seed = cfg.seed + fold
    seed_all(ds_seed)
    train_loader.generator.manual_seed(cfg.seed + fold + 1000)
    model = transfer_pretrained_resnet_to_analog(cfg, digital, device, val_loader=val_loader)
    ensure_same_pretrained_resnet(model, digital)
    gate = (AttentionGate(dim=RESNET_SPEC.attention_gate_dim, cg=cfg.attention_input_gain).to(device)
            if method in (ATT, ATT_ONLY) else None)
    weights = [physical_weight_parameter(layer) for layer in model.layers().values()]
    ids = {id(p) for p in weights}
    nonweights = [p for p in model.parameters() if id(p) not in ids]
    opt = AnalogAdam([
        {'params': weights, 'weight_decay': 0.0},
        {'params': nonweights, 'weight_decay': 0.0},
    ], lr=cfg.lr, weight_decay=0.0)
    opt.regroup_param_groups(model)
    assert isinstance(opt, AnalogAdam), 'Wrong optimizer class'
    opt_gate = torch.optim.Adam(gate.parameters(), lr=cfg.attention_lr) if gate else None
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=cfg.max_lr, epochs=cfg.epochs,
        steps_per_epoch=len(train_loader), div_factor=cfg.max_lr / cfg.lr,
    )
    initial_refs = {
        name: physical_weight_parameter(layer).detach().clone()
        for name, layer in model.layers().items()
    }
    initial_refs_id = state_dict_hash(initial_refs)
    ensure_same_pretrained_resnet(model, digital)
    from aihwkit.optim.context import AnalogContext
    n_ctx = sum(isinstance(p, AnalogContext) for g in opt.param_groups for p in g['params'])

    history, start_epoch, steps = [], 0, 0
    best_val_acc, best_val_loss, best_state, best_epoch = -1.0, math.inf, None, -1
    first_step_change = None
    transfer_rows = transfer_error_report_resnet(model, digital, cfg.weight_scaling_omega)
    saved, src = load_rotating(path)
    if saved is not None:
        verify_pack(saved, train_fp, "hwa_training", fold, src)
        if saved.get('method') != method or saved.get('digital_identity') != digital_identity:
            raise ScientificCheckError('Incompatible training checkpoint (method/digital identity)')
        load_model_state(model, saved['model'], device)
        opt.load_state_dict(saved['optimizer'])
        scheduler.load_state_dict(saved['scheduler'])
        if gate is not None:
            gate.load_state_dict(saved['gate'])
            opt_gate.load_state_dict(saved['gate_optimizer'])
        history, start_epoch, steps = saved['history'], saved['next_epoch'], saved['optimizer_steps']
        best_val_acc, best_val_loss = saved['best_val_accuracy'], saved['best_val_loss']
        best_state, best_epoch = saved['best_model'], saved['best_epoch']
        first_step_change = saved.get('first_step_max_weight_change')
        if saved.get('initial_refs_id') != initial_refs_id:
            raise ScientificCheckError('Recomputed DAR reference weights differ from the interrupted run')
        train_loader.generator.set_state(saved['loader_rng'].cpu())
        rng_restore(saved['rng'], device)
        _RESUME_LOG.append((f"hwa:{method}", fold, start_epoch, src.name))
        print(f"RESUME [{method}] fold {fold}: continuing at epoch {start_epoch+1}/{cfg.epochs} (from {src.name})")

    t_start = time.time()
    for epoch in range(start_epoch, cfg.epochs):
        model.train()
        if gate is not None:
            gate.train()
        sum_ce = sum_extra = sum_loss = 0.0
        n_samples = 0
        n_correct = 0
        for batch, (x, y) in enumerate(train_loader):
            x = cifar_augment(x.to(device), augment_seed(cfg, fold, 1, epoch, batch)).flatten(1)
            y = y.to(device)
            opt.zero_grad(set_to_none=True)
            if opt_gate: opt_gate.zero_grad(set_to_none=True)
            if gate is not None:
                x, alpha = gate(x)
            elif method == GAIN_ONLY:
                x = cfg.attention_input_gain * x
                alpha = None
            else:
                alpha = None
            logits = model(x)
            ce = F.cross_entropy(logits, y)
            extra = torch.zeros((), device=device)
            if method in (DAR, ATT):
                td = cfg.dar_times_s[epoch % len(cfg.dar_times_s)]
                with random_stream(cfg.seed + fold * 1_000_000 + epoch * 10_000 + batch, device):
                    extra = sum(dar_penalty(physical_weight_parameter(layer),
                                            initial_refs[name], epoch, td, cfg)
                                for name, layer in model.layers().items())
            if method == NSR:
                extra = nsr_penalty(model, cfg)
            if alpha is not None:
                extra = extra + cfg.attention_lambda * ((alpha - alpha.mean()) ** 2).sum(1).mean()
            loss = ce + extra
            if not torch.isfinite(loss):
                raise FloatingPointError(f'{method} nonfinite loss at epoch={epoch+1}, batch={batch}')
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip, error_if_nonfinite=True)
            if gate is not None:
                torch.nn.utils.clip_grad_norm_(gate.parameters(), cfg.grad_clip, error_if_nonfinite=True)
            if first_step_change is None:
                before = [w.detach().clone() for w in weights]
            opt.step()
            if first_step_change is None:
                first_step_change = max(float((w.detach() - b).abs().max()) for w, b in zip(weights, before))
                del before
            with torch.no_grad():
                group_lr = {id(p): float(g['lr']) for g in opt.param_groups for p in g['params']}
                for p in weights:
                    p.mul_(1.0 - group_lr.get(id(p), cfg.lr) * cfg.weight_decay)
            scheduler.step()
            if opt_gate is not None: opt_gate.step()
            with torch.no_grad():
                for layer in model.layers().values():
                    tile, = list(layer.analog_tiles())
                    tile.input_range.clamp_(min=1e-3)
            steps += 1
            bs = y.numel()
            n_samples += bs
            n_correct += int((logits.detach().argmax(1) == y).sum())
            sum_ce += float(ce.detach()) * bs
            sum_extra += float(extra.detach()) * bs
            sum_loss += float(loss.detach()) * bs
        model.eval()
        val_acc, val_loss = validation_score(
            model, val_loader, device, cfg.validation_reads,
            cfg.seed + fold * 100_000 + epoch * 41,
        )
        if val_acc > best_val_acc or (val_acc == best_val_acc and val_loss < best_val_loss):
            best_val_acc, best_val_loss = val_acc, val_loss
            best_epoch = epoch + 1
            best_state = cpu_cloned_state(model)
        history.append({
            'fold': fold, 'method': method, 'epoch': epoch + 1,
            'train_ce': sum_ce / n_samples, 'train_extra': sum_extra / n_samples,
            'train_loss': sum_loss / n_samples,
            'train_accuracy': 100.0 * n_correct / n_samples,
            'val_accuracy': val_acc, 'val_loss': val_loss,
            'best_val_accuracy': best_val_acc, 'best_epoch': best_epoch,
            'optimizer': type(opt).__name__, 'optimizer_steps': steps,
            'lr_end': scheduler.get_last_lr()[0], 'gpu': _gpu_name(),
        })
        pack = {
            'kind': 'hwa_training', 'train_fp': train_fp, 'fold': fold, 'method': method,
            'digital_identity': digital_identity,
            'model': cpu_cloned_state(model), 'optimizer': opt.state_dict(),
            'scheduler': scheduler.state_dict(), 'initial_refs_id': initial_refs_id,
            'gate': cpu_cloned_state(gate) if gate else None,
            'gate_optimizer': opt_gate.state_dict() if opt_gate else None,
            'next_epoch': epoch + 1, 'history': history, 'optimizer_steps': steps,
            'best_model': best_state, 'best_epoch': best_epoch,
            'best_val_accuracy': best_val_acc, 'best_val_loss': best_val_loss,
            'first_step_max_weight_change': first_step_change,
            'loader_rng': train_loader.generator.get_state(),
            'rng': rng_snapshot(device),
        }
        save_rotating(pack, path, epoch + 1, final=(epoch + 1 == cfg.epochs))
        write_text_inplace(history_path, pd.DataFrame(history).to_csv(index=False))
        if (epoch + 1) % 10 == 0 or epoch == start_epoch or epoch == cfg.epochs - 1:
            print(f'[{method}] fold {fold} epoch {epoch+1}/{cfg.epochs}'
                  f' CE={sum_ce/n_samples:.4f} extra={sum_extra/n_samples:.3g}'
                  f' train={100.0*n_correct/n_samples:.2f}% val={val_acc:.2f}% best={best_epoch}'
                  f' time={(time.time()-t_start)/60:.1f}m', flush=True)
        _maybe_interrupt(f"hwa:{method}", epoch + 1)

    if len(history) != cfg.epochs or best_state is None or best_epoch < 1:
        raise RuntimeError(f'Incomplete training: {method}, epochs={len(history)}/{cfg.epochs}')
    final_state = cpu_cloned_state(model)
    meta = {
        'train_fp': train_fp, 'fold': fold, 'method': method,
        'digital_identity': digital_identity, 'optimizer_steps': steps,
        'steps_per_epoch': len(train_loader), 'history': history,
        'first_step_max_weight_change': first_step_change,
        'analog_context_groups': n_ctx, 'transfer_errors': transfer_rows, 'arch': 'resnet32',
    }
    ds_meta_final = training_metadata(cfg, fold, method, cfg.epochs, len(history))
    ds_meta_best = training_metadata(cfg, fold, method, best_epoch, len(history))
    atomic_torch_save({'kind': 'hwa_final', 'model': final_state,
                       'model_identity': state_dict_hash(final_state), **meta, **ds_meta_final}, final_path)
    load_model_state(model, best_state, device)
    model.eval()
    re_acc, re_loss = validation_score(model, val_loader, device, cfg.validation_reads,
                                       cfg.seed + fold * 100_000 + (best_epoch - 1) * 41)
    restore_diff = abs(re_acc - best_val_acc)
    atomic_torch_save({'kind': 'hwa_best_validation', 'model': best_state,
                       'model_identity': state_dict_hash(best_state),
                       'val_accuracy': best_val_acc, 'val_loss': best_val_loss,
                       'restore_reeval_val_accuracy': re_acc, 'restore_abs_diff_pp': restore_diff,
                       **meta, **ds_meta_best}, best_path)
    print(f'[{method}] fold {fold}: DONE {len(history)} epochs, {steps} AnalogAdam steps; '
          f'best-validation epoch {best_epoch} (val {best_val_acc:.3f}%, re-eval diff {restore_diff:.2e} pp)')
    return torch_load(best_path)


@scientific("rn_train")
def make_fp32_direct_resnet(cfg, fold, digital, digital_identity, val_loader, device, fold_dir, train_fp):
    """FP32-Direct: transfer the pretrained ResNet (weights, fc bias, BatchNorm), ZERO HWA steps."""
    path = Path(fold_dir) / f'{DIRECT}_analog_checkpoint.pt'
    if path.exists():
        pack = torch_load(path)
        verify_pack(pack, train_fp, "fp32_direct", fold, path)
        if pack.get("digital_identity") != digital_identity:
            raise ScientificCheckError("FP32-Direct checkpoint was made from a different digital model")
        return pack
    direct = transfer_pretrained_resnet_to_analog(cfg, digital, device, val_loader=val_loader)
    ensure_same_pretrained_resnet(direct, digital)
    state = cpu_cloned_state(direct)
    pack = {
        'kind': 'fp32_direct', 'train_fp': train_fp, 'fold': fold, 'method': DIRECT,
        'model': state, 'model_identity': state_dict_hash(state),
        'digital_identity': digital_identity, 'digital_epochs': cfg.digital_epochs,
        'analog_epochs': 0, 'analog_epochs_completed': 0, 'optimizer_steps': 0,
        'selection_split': 'validation', 'arch': 'resnet32',
        'transfer_errors': transfer_error_report_resnet(direct, digital, cfg.weight_scaling_omega),
    }
    atomic_torch_save(pack, path)
    return pack


print("ResNet-32 training functions loaded.")

## 7. PCM programming, drift, read noise and DriftSlice (original equations)

These are copied without modification from `driftslice_core.py` (CELL 10/12) and `unified_experiment.py`:
* `make_pcm_noise_model()`: AIHWKit `PCMLikeNoiseModel` with `SinglePairConductanceConverter(g_max = 25 µS, g_min = 0)`.
* `PCMPhysicalMatrix`: differential G⁺/G⁻ targets, programming noise clipped to [0, G_max], and drift exponents ν drawn **once** per realization. That state persists, and every later read applies drift and read noise to the same programmed state. Absolute 20 s maps to AIHWKit elapsed time 0.
* `PCMClassReferenceBank`: R_tot reference cells spread uniformly over B equal-width conductance classes, programmed and drifted with the same PCM model, read K = 4 times. Its gain is γ_b = clip( mean G_ref,b(20 s) / mean G_ref,b(t), 0.5, 3 ).
* `apply_driftslice()`: each G⁺ and G⁻ device is multiplied by the gain of its class, where the class comes from its **target** conductance.
* `shared_mvm()` / `SnapshotMLP`: inference through AIHWKit `AnalogMVM` (8-bit DAC/ADC, out-noise, out-bound) with the learned input range, output scales and digital bias.
* `global_gdc_gain()`: the original per-layer measured GDC (all-ones probe).
* `exact_retention_times()`: exactly 75 log-spaced times between the explicit anchors 20 s … 5 y.
* `select_driftslice_bins_on_validation()`: B is chosen on **validation** data at 5 years (3 programming realizations, PCM-HWA model, smallest B on ties) and shared by all methods in that fold.

**ADDED: saturation-safe GDC.** The original all-ones probe drives `fc1` far past the ADC bound. With a trained network, about 77 % of the 250 calibration outputs clip at ±10 (measured in a pre-build benchmark), so the measured ratio is biased. The saturation-safe variant keeps the same measurement (same DAC/ADC path, bias excluded, K = 4 repeated reads, the mean |y| ratio, the per-layer gain applied at the layer output). The only change is that the probe amplitude u ∈ {1, ½, ¼, …} is chosen **once at programming time** from ADC readouts of the 20-s reference, so that max |y_ADC| ≤ 0.5·out_bound. A measurement is **valid** only if no ADC output saturates in either the reference or the current reads. Invalid measurements are reported as missing (NaN) and are never filled in. The original GDC is still computed, together with its saturation fraction, as a documented diagnostic.

**ADDED: DriftSlice B = 1 control.** For every model and timestamp, a B = 1 reference bank is evaluated next to the validation-selected B. It uses the same reference seed, main-array read, read-noise seed and evaluation-noise seed, so the two are paired.

In [ ]:
# ===================== ORIGINAL (driftslice_core.py, CELL 10) =====================
@scientific("pcm")
def elapsed_pcm_time(t_absolute_s):
    """Paper-reference convention: t=20 s maps to AIHWKit elapsed time zero."""
    if float(t_absolute_s) < T_REF:
        raise ValueError("Absolute retention time must be >=20 seconds")
    return float(t_absolute_s) - T_REF


@scientific("pcm")
def make_pcm_noise_model(cfg: Config):
    from aihwkit.inference import PCMLikeNoiseModel
    from aihwkit.inference.converter.conductance import SinglePairConductanceConverter

    converter = SinglePairConductanceConverter(g_max=cfg.gmax_us, g_min=0.0)
    noise = PCMLikeNoiseModel(
        g_converter=converter,
        prog_noise_scale=cfg.programming_noise_scale,
        read_noise_scale=cfg.read_noise_scale,
        drift_scale=cfg.drift_scale,
    )
    return noise, converter


@scientific("pcm")
class PCMPhysicalMatrix:
    """One differential PCM matrix realization."""
    def __init__(self, logical_weight: Tensor, cfg: Config, seed: int):
        self.cfg = cfg
        self.device = logical_weight.device
        self.noise, self.converter = make_pcm_noise_model(cfg)

        g_target, params = self.converter.convert_to_conductances(logical_weight.detach())
        if len(g_target) != 2:
            raise RuntimeError("Expected one positive and one negative conductance tensor.")

        self.g_target = tuple(g.detach().clone() for g in g_target)
        self.params = params

        with random_stream(seed, self.device):
            self.g_programmed = tuple(
                self.noise.apply_programming_noise_to_conductance(g).clamp(0, cfg.gmax_us)
                for g in self.g_target
            )
            self.nu = tuple(
                self.noise.generate_drift_coefficients(g)
                for g in self.g_target
            )

    def read_conductances(self, t_inf_s: float, seed: int, noiseless: bool = False):
        elapsed = elapsed_pcm_time(t_inf_s)

        if noiseless:
            t = elapsed + self.noise.t_0
            return tuple(
                g * ((t / self.noise.t_0) ** (-nu))
                for g, nu in zip(self.g_programmed, self.nu)
            )

        with random_stream(seed, self.device):
            return tuple(
                self.noise.apply_drift_noise_to_conductance(g, nu, elapsed)
                for g, nu in zip(self.g_programmed, self.nu)
            )

    def logical_weight(self, conductances):
        return self.converter.convert_back_to_weights(list(conductances), self.params)


@scientific("pcm")
def conductance_edges(cfg: Config, bins: int, device: torch.device):
    return torch.linspace(0.0, cfg.gmax_us, int(bins) + 1, device=device)


@scientific("pcm")
def class_indices(g_target: Tensor, edges: Tensor):
    return torch.bucketize(g_target, edges[1:-1], right=False).long()


@scientific("pcm")
class PCMClassReferenceBank:
    """Small class reference bank shared by all layers of one realization."""
    def __init__(self, cfg: Config, bins: int, seed: int, device: torch.device):
        self.cfg = cfg
        self.bins = int(bins)
        self.seed = int(seed)
        self.device = device
        self.noise, _ = make_pcm_noise_model(cfg)
        self.edges = conductance_edges(cfg, self.bins, device)

        refs_per_class = cfg.reference_cell_budget // self.bins
        if cfg.reference_cell_budget % self.bins or refs_per_class < 1:
            raise ValueError("Invalid fixed reference budget")
        refs = []
        for b in range(self.bins):
            lo = float(self.edges[b].item())
            hi = float(self.edges[b+1].item())
            hi_eff = hi - min(1e-4, (hi-lo)/10)
            lo_eff = min(max(lo + 1e-4, cfg.reference_floor_us), hi_eff)
            refs.append(torch.linspace(lo_eff, hi_eff, refs_per_class, device=device))
        self.target = torch.stack(refs, dim=0)  # [B,R]

        with random_stream(seed, device):
            self.programmed = self.noise.apply_programming_noise_to_conductance(
                self.target
            ).clamp(0, cfg.gmax_us)
            self.nu = self.noise.generate_drift_coefficients(self.target)

        # One stored reference measurement at the 20-s reference point.
        self.ref20 = self._read(T_REF, seed + 333)

    def _read(self, t_inf_s: float, seed: int):
        reads = []
        for rep in range(self.cfg.reference_read_repeats):
            with random_stream(seed + rep * 7919, self.device):
                reads.append(
                    self.noise.apply_drift_noise_to_conductance(
                        self.programmed, self.nu, elapsed_pcm_time(t_inf_s)
                    )
                )
        return torch.stack(reads, dim=0).mean(dim=0)

    def gains(self, t_inf_s: float, seed: int):
        now = self._read(t_inf_s, seed)
        numerator = self.ref20.mean(dim=1)
        denominator = now.mean(dim=1).clamp_min(1e-6)
        gain = numerator / denominator
        return gain.clamp(self.cfg.gain_min, self.cfg.gain_max)


@scientific("pcm")
def apply_driftslice(
    matrix: PCMPhysicalMatrix,
    current_g: Tuple[Tensor, Tensor],
    gains: Tensor,
    edges: Tensor,
):
    corrected = []
    for target, current in zip(matrix.g_target, current_g):
        idx = class_indices(target, edges)
        corrected.append(current * gains[idx])
    return tuple(corrected)


@scientific("pcm")
def global_gdc_gain(w_ref, w_now, snap, io, seed, device):
    """Measured all-ones calibration through the same DAC/ADC path.

    Per-layer functional calibration, not native mapped-tile GDC.
    Bias is excluded from the measurement. Independent repeated read noise.
    """
    calibration = dict(snap)
    calibration["bias"] = torch.zeros_like(snap["bias"])
    ones = torch.ones(1, w_ref.shape[1], device=device, dtype=w_ref.dtype)
    with torch.inference_mode(), random_stream(seed, device):
        baseline = torch.stack([shared_mvm(ones, w_ref, calibration, io).abs().mean()
                                for _ in range(4)]).mean()
        current = torch.stack([shared_mvm(ones, w_now, calibration, io).abs().mean()
                               for _ in range(4)]).mean()
    if not torch.isfinite(current) or current <= 1e-8:
        raise FloatingPointError("Invalid GDC calibration readout")
    return float(baseline/current)


@scientific("pcm")
def shared_mvm(
    x: Tensor,
    weight: Tensor,
    snap: Dict[str, Tensor],
    io,
    gain: float = 1.0,
):
    from aihwkit.simulator.tiles.analog_mvm import AnalogMVM

    r = snap["input_range"].abs().clamp_min(1e-8)
    normalized = x.clamp(-r.item(), r.item()) / r
    y = AnalogMVM.matmul(weight, normalized, io, trans=False, is_test=True)
    y = y * r
    y = y * snap["scale"].view(1, -1) * float(gain)
    return y + snap["bias"]


@scientific("pcm")
class SnapshotMLP(nn.Module):
    def __init__(self, snapshots, effective_weights, cfg: Config, layer_gains=None):
        super().__init__()
        self.snapshots = snapshots
        self.effective_weights = effective_weights
        self.layer_gains = layer_gains or {"fc1": 1.0, "fc2": 1.0}
        self.io = copy.deepcopy(make_hwa_rpu(cfg).forward)

    def forward(self, x):
        x = shared_mvm(
            x.flatten(1),
            self.effective_weights["fc1"],
            self.snapshots["fc1"],
            self.io,
            self.layer_gains["fc1"],
        )
        x = torch.tanh(x)
        return shared_mvm(
            x,
            self.effective_weights["fc2"],
            self.snapshots["fc2"],
            self.io,
            self.layer_gains["fc2"],
        )


@scientific("pcm")
@torch.inference_mode()
def evaluate_snapshot(model: SnapshotMLP, loader, device: torch.device):
    model.eval()
    loss_sum = 0.0
    correct = 0
    n = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss_sum += F.cross_entropy(logits, y, reduction="sum").item()
        correct += (logits.argmax(1) == y).sum().item()
        n += y.numel()
    return loss_sum / n, 100.0 * correct / n


@scientific("pcm")
def build_main_pcm(snapshot, cfg: Config, seed_base: int):
    return {
        "fc1": PCMPhysicalMatrix(snapshot["fc1"]["weight"], cfg, seed_base + 101),
        "fc2": PCMPhysicalMatrix(snapshot["fc2"]["weight"], cfg, seed_base + 202),
    }


@scientific("pcm")
def plain_effective_weights(main_pcm, t_s: float, seed: int):
    out = {}
    conductances = {}
    for i, name in enumerate(("fc1", "fc2")):
        g = main_pcm[name].read_conductances(t_s, seed + i*1009)
        conductances[name] = g
        out[name] = main_pcm[name].logical_weight(g)
    return out, conductances


@scientific("pcm")
def driftslice_effective_weights(main_pcm, conductances, ref_bank, t_s: float, seed: int):
    gains = ref_bank.gains(t_s, seed)
    out = {}
    for name in ("fc1", "fc2"):
        g_corr = apply_driftslice(
            main_pcm[name], conductances[name], gains, ref_bank.edges
        )
        out[name] = main_pcm[name].logical_weight(g_corr)
    return out, gains


# ===================== ORIGINAL (unified_experiment.py) =====================
@scientific("pcm")
def exact_retention_times(cfg):
    """Exactly N points, including all notebook anchors (original produced >N)."""
    anchors = np.array([20., 60., 1800., 3600., DAY, 5*DAY,
                        30*DAY, 180*DAY, YEAR, 2*YEAR, 3*YEAR, 5*YEAR])
    anchors = np.unique(anchors)
    N = cfg.retention_points
    if N < len(anchors):
        return np.geomspace(T_REF, 5*YEAR, N)  # smoke only
    need = N - len(anchors)
    gaps = np.diff(np.log(anchors))
    shares = need * gaps / gaps.sum()
    allocations = np.floor(shares).astype(int)
    for k in np.argsort(-(shares - allocations))[:need - allocations.sum()]:
        allocations[k] += 1
    sections = [anchors]
    for left, right, n in zip(anchors[:-1], anchors[1:], allocations):
        if n:
            sections.append(np.geomspace(left, right, n + 2)[1:-1])
    result = np.unique(np.concatenate(sections))
    assert len(result) == N and result[0] == T_REF and result[-1] == 5*YEAR
    return result


@scientific("pcm")
def select_driftslice_bins_on_validation(hwa_snapshot, cfg, fold, val_loader, device):
    """Select B ONLY using validation samples, never MNIST test labels."""
    require_split(val_loader, "val", "DriftSlice B selection")   # ADDED leakage guard
    candidates = cfg.ablation_bins
    rows = []
    n_realizations = 1 if cfg.fast_mode else cfg.driftslice_selection_realizations
    for r in range(n_realizations):
        root = cfg.seed + fold * 1_000_000 + r * 100_000 + 9_000_000
        main = build_main_pcm(hwa_snapshot, cfg, root)
        weights, conductances = plain_effective_weights(main, 5*YEAR, root+55)
        for b in candidates:
            bank = PCMClassReferenceBank(cfg, int(b), root + 100, device)
            corr, gains = driftslice_effective_weights(main, conductances, bank, 5*YEAR, root+333)
            with random_stream(root + 123_456, device):
                _, acc = evaluate_snapshot(SnapshotMLP(hwa_snapshot, corr, cfg).to(device), val_loader, device)
            rows.append({'fold': fold, 'realization': r, 'bins': int(b),
                         'validation_accuracy': acc, 'validation_error': 100-acc})
    df = pd.DataFrame(rows)
    tab = df.groupby('bins').validation_error.mean().sort_values()
    best_value = float(tab.iloc[0])
    # Deterministic smallest B on a numerical tie.
    best_bins = min(int(b) for b, v in tab.items() if abs(float(v) - best_value) < 1e-10)
    return best_bins, df


# ============================ ADDED: saturation-safe GDC ============================
@scientific("retention")
def _adc_readout(weight: Tensor, u: float, io, n_reads: int):
    """Raw ADC outputs (normalized units) for a uniform DAC probe of amplitude u."""
    from aihwkit.simulator.tiles.analog_mvm import AnalogMVM
    x = torch.full((1, weight.shape[1]), float(u), device=weight.device, dtype=weight.dtype)
    return torch.cat([AnalogMVM.matmul(weight, x, io, trans=False, is_test=True) for _ in range(n_reads)], 0)


@scientific("retention")
def gdc_safe_probe_amplitude(w_ref: Tensor, io, seed: int, device, cfg: Config) -> dict:
    """Pick, at programming time (20-s reference read), the largest u in {1, 1/2, ...}
    such that the ADC calibration readout satisfies max|y| <= gdc_safe_max_fraction*out_bound."""
    limit = cfg.gdc_safe_max_fraction * float(io.out_bound)
    u = 1.0
    with torch.inference_mode():
        for k in range(16):
            with random_stream(seed + k, device):
                y = _adc_readout(w_ref, u, io, cfg.gdc_reads)
            ymax = float(y.abs().max())
            if ymax <= limit:
                return {"u": u, "ref_max_abs_adc": ymax, "halvings": k, "converged": True}
            u *= 0.5
    return {"u": u, "ref_max_abs_adc": ymax, "halvings": 16, "converged": False}


@scientific("retention")
def saturation_safe_gdc_gain(w_ref, w_now, snap, io, probe: dict, seed, device, cfg: Config) -> dict:
    """Same measurement as global_gdc_gain (bias excluded, K repeated reads, mean|y| ratio
    through r*scale), with the pre-selected non-saturating probe and an explicit validity flag."""
    bound = float(io.out_bound)
    r = snap["input_range"].abs().clamp_min(1e-8)
    col = snap["scale"].view(1, -1) * r
    with torch.inference_mode(), random_stream(seed, device):
        y_ref = _adc_readout(w_ref, probe["u"], io, cfg.gdc_reads)
        y_now = _adc_readout(w_now, probe["u"], io, cfg.gdc_reads)
    sat_ref = float((y_ref.abs() >= bound * (1 - 1e-4)).float().mean())
    sat_now = float((y_now.abs() >= bound * (1 - 1e-4)).float().mean())
    baseline = float((y_ref * col).abs().mean())
    current = float((y_now * col).abs().mean())
    valid = bool(probe["converged"] and sat_ref == 0.0 and sat_now == 0.0
                 and math.isfinite(current) and current > 1e-8)
    return {"gain": baseline / current if valid else float("nan"), "valid": valid,
            "sat_ref": sat_ref, "sat_now": sat_now, "u": probe["u"]}


@scientific("retention")
def original_gdc_saturation_fraction(w, snap, io, seed, device) -> float:
    """Diagnostic: fraction of ADC outputs clipped by the ORIGINAL all-ones probe."""
    from aihwkit.simulator.tiles.analog_mvm import AnalogMVM
    r = snap["input_range"].abs().clamp_min(1e-8)
    x = torch.ones(1, w.shape[1], device=device, dtype=w.dtype).clamp(-r.item(), r.item()) / r
    with torch.inference_mode(), random_stream(seed, device):
        y = AnalogMVM.matmul(w, x, io, trans=False, is_test=True)
    return float((y.abs() >= float(io.out_bound) * (1 - 1e-4)).float().mean())


def pcm_state_hash(main_pcm) -> str:
    h = hashlib.sha256()
    for name in ("fc1", "fc2"):
        m = main_pcm[name]
        for t in (*m.g_target, *m.g_programmed, *m.nu):
            h.update(t.detach().float().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()


COMPENSATIONS = ("none", "GDC", "GDC-safe", "DS-B1", "DS-val")


@scientific("retention")
def compensations_for(cfg: Config) -> Tuple[str, ...]:
    out = ["none"]
    if cfg.run_gdc_baselines:
        out += (["GDC"] if cfg.include_original_gdc else []) + ["GDC-safe"]
    return tuple(out + ["DS-B1", "DS-val"])
COMP_LABEL = {
    "none": "No compensation",
    "GDC": "GDC, original all-ones probe (saturating; diagnostic)",
    "GDC-safe": "Saturation-safe GDC",
    "DS-B1": "DriftSlice, B = 1",
    "DS-val": "DriftSlice, validation-selected B",
}
print("PCM / DriftSlice functions loaded.")

### 7b. PCM deployment of ResNet-32 (ADDED)

The MLP's PCM functions are applied to all 34 layers, using a generalization of the MLP's seed rule:
* programming seed `seed_base + 101·(i+1)`; the MLP's fc1 / fc2 use +101 / +202;
* read seed `seed + 1009·i`.

One DriftSlice reference bank (B classes, R_tot = 64 cells, K = 4 reads) per programming realization is shared by **all** layers, exactly as for the MLP's two layers. B is selected on **validation** data at 5 years with the Standard-HWA checkpoint (3 realizations, smallest B on ties), with the same seeds as the MLP.

In [ ]:
# ============================ ADDED: PCM deployment of ResNet-32 ============================
# Same PCMPhysicalMatrix / PCMClassReferenceBank / apply_driftslice / GDC-safe functions as the MLP, applied to every
# one of the 34 mapped layers. The per-layer seed rule generalizes the MLP's exactly: programming seed_base+101*(i+1)
# (MLP: +101 fc1, +202 fc2) and read seed+1009*i (MLP: fc1 +0, fc2 +1009). One DriftSlice reference bank (one gain
# per conductance class) is shared by all layers of a realization, as in the MLP.

@scientific("rn_pcm")
def build_main_pcm_all(snapshot, cfg: Config, seed_base: int):
    return OrderedDict((name, PCMPhysicalMatrix(s["weight"], cfg, seed_base + 101 * (i + 1)))
                       for i, (name, s) in enumerate(snapshot.items()))


@scientific("rn_pcm")
def plain_effective_weights_all(main_pcm, t_s: float, seed: int):
    out, conductances = OrderedDict(), OrderedDict()
    for i, name in enumerate(main_pcm):
        g = main_pcm[name].read_conductances(t_s, seed + i * 1009)
        conductances[name] = g
        out[name] = main_pcm[name].logical_weight(g)
    return out, conductances


@scientific("rn_pcm")
def driftslice_effective_weights_all(main_pcm, conductances, ref_bank, t_s: float, seed: int):
    gains = ref_bank.gains(t_s, seed)
    out = OrderedDict()
    for name in main_pcm:
        g_corr = apply_driftslice(main_pcm[name], conductances[name], gains, ref_bank.edges)
        out[name] = main_pcm[name].logical_weight(g_corr)
    return out, gains


def pcm_state_hash_all(main_pcm) -> str:
    h = hashlib.sha256()
    for name in main_pcm:
        m = main_pcm[name]
        for t in (*m.g_target, *m.g_programmed, *m.nu):
            h.update(t.detach().float().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()


@scientific("rn_pcm")
def evaluate_deployed(model: AnalogResNet32, weights, loader, device, layer_gains=None):
    """evaluate_snapshot() on the AIHWKit network carrying the PCM effective weights."""
    with deployed_weights(model, weights, layer_gains):
        return evaluate_snapshot(model, loader, device)


@scientific("rn_pcm")
def select_driftslice_bins_on_validation_resnet(hwa_model, hwa_snapshot, cfg, fold, val_loader, device):
    """select_driftslice_bins_on_validation() for ResNet-32 (same seeds, validation only, smallest B on ties)."""
    require_split(val_loader, "val", "DriftSlice B selection")
    candidates = cfg.ablation_bins
    rows = []
    n_realizations = 1 if cfg.fast_mode else cfg.driftslice_selection_realizations
    for r in range(n_realizations):
        root = cfg.seed + fold * 1_000_000 + r * 100_000 + 9_000_000
        main = build_main_pcm_all(hwa_snapshot, cfg, root)
        weights, conductances = plain_effective_weights_all(main, 5*YEAR, root+55)
        for b in candidates:
            bank = PCMClassReferenceBank(cfg, int(b), root + 100, device)
            corr, gains = driftslice_effective_weights_all(main, conductances, bank, 5*YEAR, root+333)
            with random_stream(root + 123_456, device):
                _, acc = evaluate_deployed(hwa_model, corr, val_loader, device)
            rows.append({'fold': fold, 'realization': r, 'bins': int(b),
                         'validation_accuracy': acc, 'validation_error': 100-acc})
        del main, weights, conductances
    df = pd.DataFrame(rows)
    tab = df.groupby('bins').validation_error.mean().sort_values()
    best_value = float(tab.iloc[0])
    best_bins = min(int(b) for b, v in tab.items() if abs(float(v) - best_value) < 1e-10)
    return best_bins, df


def verify_resnet_deployment(model: AnalogResNet32, snap, cfg: Config, loader, device, seed: int = 4242):
    """Evidence that substitution deployment == AIHWKit forward, and that it is the SAME MVM path as shared_mvm.
    (a) substituting the tile weights by themselves reproduces the forward bit-identically (same noise stream);
    (b) the model is restored exactly afterwards (state hash);
    (c) the fc layer through shared_mvm (the MLP deployment path) matches AIHWKit's fc forward on the same inputs
        (same criterion as the MLP check: isolated 1-LSB DAC flips allowed);
    (d) a GDC gain applied through the learned output scale equals (y - bias)*gain + bias on the fc layer."""
    model.eval()
    h0 = state_dict_hash(model.state_dict())
    eff = OrderedDict((k, v["weight"]) for k, v in snap.items())
    feats, same_all, diffs, agree, n_batches = [], True, [], [], 0
    hook = model.fc.register_forward_hook(lambda m, i, o: feats.append(i[0].detach()))
    try:
        for bi, (x, _) in enumerate(loader):
            x = x.to(device)
            with torch.inference_mode(), random_stream(seed + bi, device):
                y0 = model(x)
            with torch.inference_mode(), random_stream(seed + bi, device), deployed_weights(model, eff):
                y1 = model(x)
            same_all &= torch.equal(y0, y1)
            n_batches += 1
    finally:
        hook.remove()
    fc_snap, io = snap["fc"], make_hwa_rpu(cfg).forward
    for bi, f in enumerate(feats[::2]):
        with torch.inference_mode(), random_stream(seed + 10_000 + bi, device):
            ya = model.fc(f)
        with torch.inference_mode(), random_stream(seed + 10_000 + bi, device):
            ys = shared_mvm(f, fc_snap["weight"], fc_snap, io)
        d = (ya - ys).abs()
        diffs.append(d.flatten())
        agree.append((ya.argmax(1) == ys.argmax(1)).float())
    d = torch.cat(diffs)
    med, frac, arg = float(d.median()), float((d > 3e-4).float().mean()), float(torch.cat(agree).mean())
    f0 = feats[0][:64]
    with torch.inference_mode(), random_stream(seed + 99, device):
        y_plain = model.fc(f0)
    with torch.inference_mode(), random_stream(seed + 99, device), deployed_weights(model, eff, {"fc": 1.37}):
        y_gain = model.fc(f0)
    b = fc_snap["bias"].view(1, -1)
    gain_err = float(((y_gain - b) - 1.37 * (y_plain - b)).abs().max())
    restored = state_dict_hash(model.state_dict()) == h0
    ok = (same_all and restored and med <= 1e-6 and frac <= 0.005 and arg >= 0.995
          and gain_err <= 1e-4 * max(1.0, float(y_plain.abs().max())))
    detail = (f"substitution forward bit-identical over {n_batches} batches: {same_all}; model restored: {restored}; "
              f"fc via shared_mvm vs AIHWKit: median |Δ|={med:.1e}, |Δ|>3e-4: {100*frac:.3f}%, argmax agreement "
              f"{100*arg:.2f}%; GDC gain via output scale max error {gain_err:.1e}")
    return ok, detail


print("ResNet-32 PCM deployment functions loaded.")

## 8. Experiment identity, fingerprints, storage layout, workload estimates

```
BASE_DIR/results/<full|smoke>/<run_id>/
  manifest.json                         # config, fingerprints, environment, history of sessions
  training/<TRAIN_FP[:12]>/fold{f}/     # digital_*.pt, <method>_{best_validation,final}.pt, resumable
                                        #   <name>_training.slotA/B.pt (in-place), histories
  retention/<RET_FP[:12]>/fold{f}/      # selected_bins.json, <method>/real{r}.pt shards
  ablation/<ABL_FP[:12]>/fold{f}/<method>/real{r}.pt
  diagnostics/<DIAG_FP[:12]>/fold{f}/<method>/real{r}.pt (+ distribution states)
  checks/                               # scientific-check reports of every full-run stage
  paper/ | paper_PROVISIONAL/           # figures (PDF/PNG), tables (LaTeX/CSV), source CSVs, audit
```
Each stage writes into a directory keyed by its fingerprint. That fingerprint covers the registered scientific functions, the constant values, the stage configuration, the data hash and the versions. If any of these change, the new results go to a new directory, so earlier results are never overwritten and never mixed. A run whose *training* fingerprint differs from the manifest is refused by default (`RESUME_POLICY = "error"`). The refusal lists exactly what changed.

In [ ]:
# ============================ EXPERIMENT MANAGEMENT ============================
BOOKKEEPING_FIELDS = {"run_id", "output_root", "resume_policy", "data_root", "allow_download", "methods",
                      "fast_gpu_loader", "smoke_train_repeats"}
RETENTION_ONLY_FIELDS = {
    "train_repeats", "realizations", "retention_points", "programming_noise_scale", "read_noise_scale",
    "drift_scale", "driftslice_bins", "reference_cell_budget", "refs_per_bin", "reference_read_repeats",
    "gain_min", "gain_max", "reference_floor_us", "driftslice_selection_realizations", "run_gdc_baselines",
    "run_bin_ablation", "ablation_bins", "cdf_times_s", "ablation_reference_budgets", "low_state_threshold_us",
    "gdc_safe_max_fraction", "gdc_reads", "bin_selection_method", "include_original_gdc",
}


def config_subset(cfg: Config, exclude) -> dict:
    return {k: v for k, v in asdict(cfg).items() if k not in exclude}


def environment_identity() -> dict:
    import aihwkit
    return {"torch": torch.__version__, "aihwkit": aihwkit.__version__,
            "python": ".".join(platform.python_version().split(".")[:2])}


def notebook_code_hash() -> str:
    """Hash of ALL functions/classes defined in this notebook (smoke gate: any code edit -> re-run smoke)."""
    h = hashlib.sha256()
    for name, obj in sorted(globals().items()):
        if ((inspect.isfunction(obj) or inspect.isclass(obj)) and getattr(obj, "__module__", None) == "__main__"
                and getattr(obj, "__name__", None) == name):          # ignore IPython cache aliases (_, _4, ...)
            h.update(name.encode())
            h.update(_object_source(obj).encode())
    return h.hexdigest()


def compute_fingerprints(cfg: Config, data_hashes: dict) -> dict:
    env = environment_identity()
    consts_train = {"DAY": DAY, "YEAR": YEAR, "T_REF": T_REF}
    consts_ret = {**consts_train, "ADDED_SEED_BASE": ADDED_SEED_BASE, "COMPENSATIONS": list(COMPENSATIONS)}
    code_train = component_source_hash({"config", "data", "model", "train"})
    code_pcm = component_source_hash({"pcm", "retention"})
    train = stable_hash({"schema": SCHEMA, "code": code_train, "env": env, "data": data_hashes, "const": consts_train,
                         "cfg": config_subset(cfg, BOOKKEEPING_FIELDS | RETENTION_ONLY_FIELDS)})
    ret = stable_hash({"train": train, "code": code_pcm, "const": consts_ret,
                       "cfg": config_subset(cfg, BOOKKEEPING_FIELDS)})
    abl = stable_hash({"ret": ret, "code": component_source_hash({"ablation"})})
    diag = stable_hash({"ret": ret, "code": component_source_hash({"diagnostics"})})
    return {"train": train, "retention": ret, "ablation": abl, "diagnostics": diag,
            "code_hashes": {"train": code_train, "pcm_retention": code_pcm,
                            "ablation": component_source_hash({"ablation"}),
                            "diagnostics": component_source_hash({"diagnostics"}),
                            "all_scientific": component_source_hash({"config", "data", "model", "train", "pcm",
                                                                     "retention", "ablation", "diagnostics"}),
                            "notebook_all_code": notebook_code_hash()},
            "env": env}


def compute_fingerprints_resnet(cfg: Config, data_hashes: dict, spec: "ResNetSpec") -> dict:
    """ResNet-32 / CIFAR-10 fingerprints: the MLP scheme + the rn_* code components + ResNetSpec BY VALUE.
    (The MLP fingerprints do not contain any rn_* component, so adding the ResNet leaves them unchanged.)"""
    env = environment_identity()
    consts_train = {"DAY": DAY, "YEAR": YEAR, "T_REF": T_REF, "AUGMENT_SEED_BASE": AUGMENT_SEED_BASE}
    consts_ret = {**consts_train, "ADDED_SEED_BASE": ADDED_SEED_BASE, "COMPENSATIONS": list(COMPENSATIONS)}
    train_comps = {"config", "data", "model", "train", "rn_data", "rn_model", "rn_train"}
    pcm_comps = {"pcm", "retention", "rn_pcm", "rn_retention"}
    code_train = component_source_hash(train_comps)
    code_pcm = component_source_hash(pcm_comps)
    spec_v = json.loads(json.dumps(asdict(spec), default=str))
    train = stable_hash({"schema": SCHEMA, "arch": spec_v, "code": code_train, "env": env, "data": data_hashes,
                         "const": consts_train, "cfg": config_subset(cfg, BOOKKEEPING_FIELDS | RETENTION_ONLY_FIELDS)})
    ret = stable_hash({"train": train, "code": code_pcm, "const": consts_ret,
                       "cfg": config_subset(cfg, BOOKKEEPING_FIELDS)})
    abl_code = component_source_hash({"ablation", "rn_ablation"})
    diag_code = component_source_hash({"diagnostics", "rn_diagnostics"})
    abl = stable_hash({"ret": ret, "code": abl_code})
    diag = stable_hash({"ret": ret, "code": diag_code})
    return {"train": train, "retention": ret, "ablation": abl, "diagnostics": diag,
            "code_hashes": {"train": code_train, "pcm_retention": code_pcm, "ablation": abl_code,
                            "diagnostics": diag_code,
                            "all_scientific": component_source_hash(train_comps | pcm_comps | {
                                "ablation", "rn_ablation", "diagnostics", "rn_diagnostics"}),
                            "notebook_all_code": notebook_code_hash()},
            "env": env, "arch": "resnet32"}


ARCHITECTURES = ("mlp", "resnet32")


class Experiment:
    """Holds config, directories, fingerprints and device for one run (smoke or full)."""

    def __init__(self, cfg: Config, base_dir: Path, output_dir: Path, mode: str, run_id: str,
                 device: torch.device, methods, resume_policy: str = "error", arch: str = "mlp",
                 include_direct: bool = True, write_manifest: bool = True, role_tag: str = ""):
        if arch not in ARCHITECTURES:
            raise ValueError(f"Unknown architecture {arch!r}; choose one of {ARCHITECTURES}")
        self.arch = arch
        self.cfg = cfg.resolved()
        self.base_dir = Path(base_dir)
        self.mode = mode
        self.run_id = run_id
        self.device = device
        self.methods = tuple(methods)
        self.include_direct = include_direct     # parallel workers: only the worker that owns FP32-Direct
        self.write_manifest = write_manifest     # parallel workers never rewrite the shared manifest
        self.role_tag = role_tag
        if arch == "mlp":
            raw_dir = resolve_mnist_raw_dir(self.cfg, self.base_dir)
            self.mnist_raw_dir = raw_dir
            self.data_hashes = mnist_file_hashes(raw_dir)
            self.fp = compute_fingerprints(self.cfg, self.data_hashes)
        else:
            if self.cfg.include_original_gdc:
                raise ValueError("The ResNet protocol does not evaluate the original saturating GDC")
            raw_dir = resolve_cifar10_dir(self.cfg, self.base_dir)
            self.mnist_raw_dir = raw_dir          # (dataset directory; name kept for the manifest schema)
            self.data_hashes = cifar10_file_hashes(raw_dir)
            self.fp = compute_fingerprints_resnet(self.cfg, self.data_hashes, RESNET_SPEC)
        self.run_dir = Path(output_dir) / mode / run_id
        self.run_dir.mkdir(parents=True, exist_ok=True)
        self._open_manifest(resume_policy)
        self.train_dir = self.run_dir / "training" / self.fp["train"][:12]
        self.ret_dir = self.run_dir / "retention" / self.fp["retention"][:12]
        self.abl_dir = self.run_dir / "ablation" / self.fp["ablation"][:12]
        self.diag_dir = self.run_dir / "diagnostics" / self.fp["diagnostics"][:12]
        self.paper_dir = self.run_dir / "paper"

    @property
    def models(self):
        return ((DIRECT,) if self.include_direct else ()) + self.methods

    def _open_manifest(self, policy: str):
        path = self.run_dir / "manifest.json"
        now = time.strftime("%Y-%m-%d %H:%M:%S")
        session = {"time": now, "gpu": _gpu_name(), "role": self.role_tag or "all", **self.fp["env"]}
        if not self.write_manifest:
            # parallel worker: read-only check against the coordinator's manifest; own session record only
            if not path.exists():
                raise ScientificCheckError(f"Run '{self.run_id}' does not exist yet. Run the COORDINATOR notebook "
                                           "first (smoke test + FP32 pretraining), then start the workers.")
            old = json.loads(path.read_text())
            if old.get("arch", "mlp") != self.arch or old["fingerprints"]["train"] != self.fp["train"]:
                raise ScientificCheckError(
                    f"This worker's code/config/environment differs from the coordinator's run '{self.run_id}' "
                    f"(train fingerprint {old['fingerprints']['train'][:12]} vs {self.fp['train'][:12]}). Use the SAME "
                    "notebook version and Colab runtime version in every parallel notebook.")
            self.manifest = old
            atomic_write_text(self.run_dir / "sessions" / f"{now.replace(' ', '_').replace(':', '')}_{self.role_tag}.json",
                              json.dumps(session, indent=2))
            print(f"Worker '{self.role_tag}' attached to run '{self.run_id}' (train fp {self.fp['train'][:12]}); "
                  f"models: {', '.join(self.models)}")
            return
        if path.exists():
            old = json.loads(path.read_text())
            if old.get("arch", "mlp") != self.arch:
                raise ScientificCheckError(f"Run '{self.run_id}' holds a {old.get('arch', 'mlp')} experiment, not "
                                           f"{self.arch}. Use the run id configured for this architecture.")
            if old["fingerprints"]["train"] != self.fp["train"]:
                new_cfg = json.loads(json.dumps(asdict(self.cfg), default=str))
                diff = {k: (old["config"].get(k), v) for k, v in new_cfg.items() if old["config"].get(k) != v}
                msg = ("TRAINING FINGERPRINT CHANGED for run '%s'.\n  config differences: %s\n"
                       "  code hash (train): %s -> %s\n  environment: %s -> %s\n"
                       "Existing checkpoints were NOT touched. Restore the original code/config/"
                       "environment to resume, or choose a new run id." % (
                           self.run_id, diff or "none", old["fingerprints"]["code_hashes"]["train"][:12],
                           self.fp["code_hashes"]["train"][:12], old["fingerprints"]["env"], self.fp["env"]))
                if policy == "error":
                    raise ScientificCheckError(msg)
                print("WARNING:", msg)
            old.setdefault("sessions", []).append(session)
            old.setdefault("fingerprint_history", [])
            if old["fingerprints"] != self.fp:
                old["fingerprint_history"].append({"time": now, "previous": old["fingerprints"]})
            old["fingerprints"] = self.fp
            old["config"] = json.loads(json.dumps(asdict(self.cfg), default=str))
            old["methods"] = list(self.methods)
            manifest = old
            print(f"Resuming run '{self.run_id}' (train fp {self.fp['train'][:12]}).")
        else:
            import uuid
            manifest = {"schema": SCHEMA, "code_revision": CODE_REVISION, "run_id": self.run_id, "arch": self.arch,
                        "dataset": ARCH_OPS[self.arch].dataset if "ARCH_OPS" in globals() else None,
                        "run_nonce": uuid.uuid4().hex,
                        "mode": self.mode, "created": now, "fingerprints": self.fp,
                        "config": json.loads(json.dumps(asdict(self.cfg), default=str)),
                        "methods": list(self.methods), "mnist_raw_dir": str(self.mnist_raw_dir),
                        "mnist_sha256_decompressed": self.data_hashes, "sessions": [session]}
            print(f"Starting NEW run '{self.run_id}' in {self.run_dir}")
        atomic_write_text(path, json.dumps(manifest, indent=2, default=str))
        self.manifest = manifest

    def fold_train_dir(self, fold):
        d = self.train_dir / f"fold{fold}"
        d.mkdir(parents=True, exist_ok=True)
        return d


# ---------------------- smoke-test gate for expensive stages ----------------------
def smoke_pass_path(output_dir: Path, arch: Optional[str] = None) -> Path:
    arch = arch or MODEL_ARCHITECTURE
    return Path(output_dir) / "smoke" / ("SMOKE_PASS_RECORD.json" if arch == "mlp" else f"SMOKE_PASS_RECORD_{arch}.json")


def require_smoke_pass(output_dir: Path):
    if not REQUIRE_SMOKE_PASS_BEFORE_FULL:
        print("WARNING: REQUIRE_SMOKE_PASS_BEFORE_FULL=False (smoke gate disabled by user)")
        return
    p = smoke_pass_path(output_dir)
    if not p.exists():
        raise ScientificCheckError(f"No smoke-test PASS recorded for MODEL_ARCHITECTURE={MODEL_ARCHITECTURE!r}. "
                                   "Run the smoke test (RUN_SMOKE_TEST=True) first.")
    rec = json.loads(p.read_text())
    if not smoke_record_is_current(rec):
        raise ScientificCheckError(
            "The recorded smoke PASS was for different notebook code or environment "
            f"({rec.get('notebook_all_code','')[:12]} / {rec.get('env')}); re-run the smoke test.")
    print(f"Smoke-test PASS on record ({rec['time']}, notebook code {rec['notebook_all_code'][:12]}).")


def smoke_scope_identity() -> str:
    """Everything the smoke test validates besides code: configuration defaults (fast and full), methods,
    constant values, run-time options and the GPU model."""
    drop = {"run_id", "output_root", "data_root", "resume_policy"}
    return stable_hash({
        "arch": MODEL_ARCHITECTURE,
        "resnet_spec": asdict(RESNET_SPEC) if MODEL_ARCHITECTURE != "mlp" else None,
        "smoke_cfg": config_subset(make_config(MODEL_ARCHITECTURE, True, "smoke").resolved(), drop),
        "full_cfg": config_subset(make_config(MODEL_ARCHITECTURE, False, "full").resolved(), drop),
        "methods": list(SELECTED_METHODS),
        "consts": {"DAY": DAY, "YEAR": YEAR, "T_REF": T_REF, "ADDED_SEED_BASE": ADDED_SEED_BASE,
                   "COMPENSATIONS": list(COMPENSATIONS)},
        "drive_every": DRIVE_CHECKPOINT_EVERY_EPOCHS, "gpu": _gpu_name()})


def smoke_record_is_current(rec: dict) -> bool:
    return (rec.get("notebook_all_code") == notebook_code_hash() and rec.get("env") == environment_identity()
            and rec.get("scope") == smoke_scope_identity())


# ---------------------- workload and capacity ----------------------
def capacity_report(paths) -> dict:
    out = {}
    if torch.cuda.is_available():
        free, total = torch.cuda.mem_get_info()
        out["gpu"] = _gpu_name()
        out["gpu_mem_free_GB"] = round(free / 1e9, 2)
        out["gpu_mem_total_GB"] = round(total / 1e9, 2)
    for label, p in paths.items():
        try:
            du = shutil.disk_usage(str(p))
            out[f"{label}_free_GB"] = round(du.free / 1e9, 1)
        except Exception as exc:
            out[f"{label}_free_GB"] = f"unknown ({type(exc).__name__})"
    return out


def print_workload(title: str, rows: dict, seconds: float, capacity: dict, need_gb: float):
    print("=" * 78)
    print(title)
    for k, v in rows.items():
        print(f"  {k:<46s} {v}")
    print(f"  {'ESTIMATED wall time on this GPU':<46s} {seconds/3600:.1f} h  (micro-benchmark based)")
    print(f"  {'Estimated new Drive storage':<46s} {need_gb:.2f} GB")
    for k, v in capacity.items():
        print(f"  {k:<46s} {v}")
    print("=" * 78, flush=True)


print("Experiment management loaded.")

## 9. Stage runners: retention, reference-budget ablation, mechanism diagnostics

**Seed conventions.** All of these are the original `retention_all()` conventions, with `root = seed + fold·10⁶ + realization·10⁴`:
* The main array is programmed with `root+10`, which keeps the programming seeds **paired** across models and across compensation modes.
* The reference bank uses `root+30` and the 20-s GDC reference read uses `root+40`.
* The read at time index i uses `root+100000+100i`, and the reference-gain read uses `root+500000+101i`.
* The evaluation noise uses `root+700000+31i`, so it is identical for all compensation modes.

The ablation and diagnostics stages rebuild the **same** device states from the same seeds. They do not retrain, and they do not program anything new. A check confirms that their 5-year results equal the corresponding retention-shard values.

A retention shard is one (fold, model, realization) evaluated at 75 timestamps × 5 compensation modes. Existing shards are verified (fingerprint, checkpoint identity, selected B) and skipped. They are never overwritten.

In [ ]:
# ============================ PER-SHARD SCIENTIFIC FUNCTIONS ============================
ADDED_SEED_BASE = 50_000_000   # randomness that does not exist in the original code lives here
@scientific("retention")
def retention_one(method, snap, cfg, fold, realization, selected_bins, test_loader, device):
    """Original retention_all() for ONE trained model (paired seeds) + ADDED DS-B1, GDC-safe."""
    require_split(test_loader, "test", "retention")
    root = cfg.seed + fold * 1_000_000 + realization * 10_000
    all_times = exact_retention_times(cfg)
    io = make_hwa_rpu(cfg).forward
    main = build_main_pcm(snap, cfg, root + 10)
    state_hash_start = pcm_state_hash(main)
    bank = PCMClassReferenceBank(cfg, selected_bins, root + 30, device)
    bank1 = PCMClassReferenceBank(cfg, 1, root + 30, device)                 # ADDED B=1 control
    reference, _ = plain_effective_weights(main, T_REF, root + 40)
    added_root = ADDED_SEED_BASE + root      # separate seed namespace for ADDED randomness
    probe = {name: gdc_safe_probe_amplitude(reference[name], io, added_root + 100 * (name == 'fc2'), device, cfg)
             for name in ('fc1', 'fc2')}                                      # ADDED, fixed at t = 20 s
    orig_sat = {name: original_gdc_saturation_fraction(reference[name], snap[name], io,
                                                       added_root + 5_000 + (name == 'fc2'), device)
                for name in ('fc1', 'fc2')}
    data = []
    for ti, ts in enumerate(all_times):
        ts = float(ts)
        base, conductances = plain_effective_weights(main, ts, root + 100_000 + ti*100)
        variants = OrderedDict()
        variants['none'] = (base, {'fc1': 1., 'fc2': 1.}, {})
        if cfg.run_gdc_baselines:
            if cfg.include_original_gdc:
                gains = {name: global_gdc_gain(
                    reference[name], base[name], snap[name], make_hwa_rpu(cfg).forward,
                    root + 850_000 + ti*31 + (name == 'fc2'), device,
                ) for name in ('fc1', 'fc2')}
                variants['GDC'] = (base, gains, {'gdc_gain_fc1': gains['fc1'], 'gdc_gain_fc2': gains['fc2']})
            safe = {name: saturation_safe_gdc_gain(reference[name], base[name], snap[name], io, probe[name],
                                                   root + 850_000 + ti*31 + (name == 'fc2'), device, cfg)
                    for name in ('fc1', 'fc2')}
            extra = {'gdc_valid': bool(safe['fc1']['valid'] and safe['fc2']['valid'])}
            for name in ('fc1', 'fc2'):
                extra.update({f'gdc_gain_{name}': safe[name]['gain'], f'gdc_u_{name}': safe[name]['u'],
                              f'gdc_sat_now_{name}': safe[name]['sat_now']})
            variants['GDC-safe'] = (base, {n: safe[n]['gain'] for n in ('fc1', 'fc2')}, extra)
        ds1_w, gain_1 = driftslice_effective_weights(main, conductances, bank1, ts, root + 500_000 + ti*101)
        variants['DS-B1'] = (ds1_w, {'fc1': 1., 'fc2': 1.}, {'class_gain_0': float(gain_1[0])})
        ds_w, gain_b = driftslice_effective_weights(main, conductances, bank, ts, root + 500_000 + ti*101)
        variants['DS-val'] = (ds_w, {'fc1': 1., 'fc2': 1.},
                              {f'class_gain_{i}': float(v) for i, v in enumerate(gain_b.tolist())})
        for comp, (weights, gains, extra) in variants.items():
            if comp == 'GDC-safe' and not extra['gdc_valid']:
                loss = accuracy = float('nan')          # never invent a value for an invalid calibration
            else:
                with random_stream(root + 700_000 + ti*31, device):
                    model = SnapshotMLP(snap, weights, cfg, layer_gains=gains).to(device)
                    loss, accuracy = evaluate_snapshot(model, test_loader, device)
            data.append({
                'method': method, 'compensation': comp, 'fold': fold, 'realization': realization,
                't_index': ti, 't_s': ts, 'selected_bins': selected_bins, 'split': 'test',
                'loss': loss, 'accuracy': accuracy, 'error': 100. - accuracy,
                **extra,
            })
    state_hash_end = pcm_state_hash(main)
    meta = {'root_seed': root, 'state_hash_start': state_hash_start, 'state_hash_end': state_hash_end,
            'gdc_safe_probe': probe, 'orig_gdc_ref_saturation_fraction': orig_sat,
            'bank_target_numel': int(bank.target.numel()), 'bank_shape': list(bank.target.shape),
            'bank1_shape': list(bank1.target.shape), 'reference_read_repeats': cfg.reference_read_repeats,
            'times': [float(t) for t in all_times]}
    return pd.DataFrame(data), meta


@scientific("ablation")
def reference_ablation_one(method, snap, cfg, fold, realization, val_loader, test_loader, device):
    """5-year B x R_tot sweep on the SAME programmed main array / read / eval noise as retention."""
    require_split(val_loader, "val", "ablation(report-only)")
    require_split(test_loader, "test", "ablation(report-only)")
    root = cfg.seed + fold * 1_000_000 + realization * 10_000
    times = exact_retention_times(cfg)
    ti = len(times) - 1
    ts = float(times[ti])
    main = build_main_pcm(snap, cfg, root + 10)
    state_hash = pcm_state_hash(main)
    base, conductances = plain_effective_weights(main, ts, root + 100_000 + ti*100)
    rows, budget_ok = [], []

    def ev(weights, loader):
        with random_stream(root + 700_000 + ti*31, device):
            return evaluate_snapshot(SnapshotMLP(snap, weights, cfg).to(device), loader, device)

    for split, loader in (("val", val_loader), ("test", test_loader)):
        _, acc = ev(base, loader)
        rows.append({'method': method, 'fold': fold, 'realization': realization, 'split': split,
                     'compensation': 'none', 'bins': 0, 'R_tot': 0, 'R_b': 0, 'K': 0,
                     't_s': ts, 'accuracy': acc, 'error': 100. - acc})
    for R in cfg.ablation_reference_budgets:
        cfg_r = replace(cfg, reference_cell_budget=int(R))
        for B in cfg.ablation_bins:
            bank = PCMClassReferenceBank(cfg_r, int(B), root + 30, device)
            budget_ok.append(bank.target.numel() == R and tuple(bank.target.shape) == (B, R // B))
            corr, gains = driftslice_effective_weights(main, conductances, bank, ts, root + 500_000 + ti*101)
            for split, loader in (("val", val_loader), ("test", test_loader)):
                _, acc = ev(corr, loader)
                rows.append({'method': method, 'fold': fold, 'realization': realization, 'split': split,
                             'compensation': 'DriftSlice', 'bins': int(B), 'R_tot': int(R), 'R_b': int(R) // int(B),
                             'K': cfg.reference_read_repeats, 't_s': ts, 'accuracy': acc, 'error': 100. - acc,
                             **{f'class_gain_{i}': float(v) for i, v in enumerate(gains.tolist())}})
    return pd.DataFrame(rows), {'state_hash': state_hash, 'budget_shapes_ok': bool(all(budget_ok)), 'root_seed': root}


@scientific("diagnostics")
def _rms(t: Tensor) -> float:
    return float(t.double().square().mean().sqrt())


@scientific("diagnostics")
def diagnostics_one(method, snap, cfg, fold, realization, selected_bins, test_loader, device, keep_states=False):
    """Matched B = 1 mechanism diagnostics at 5 years (Table II) on the retention device state."""
    require_split(test_loader, "test", "diagnostics(report-only)")
    root = cfg.seed + fold * 1_000_000 + realization * 10_000
    times = exact_retention_times(cfg)
    ti = len(times) - 1
    ts = float(times[ti])
    main = build_main_pcm(snap, cfg, root + 10)
    state_hash = pcm_state_hash(main)
    base, conductances = plain_effective_weights(main, ts, root + 100_000 + ti*100)
    bank1 = PCMClassReferenceBank(cfg, 1, root + 30, device)
    corr, gains = driftslice_effective_weights(main, conductances, bank1, ts, root + 500_000 + ti*101)
    gamma = float(gains[0])
    with random_stream(root + 700_000 + ti*31, device):
        _, acc_b1 = evaluate_snapshot(SnapshotMLP(snap, corr, cfg).to(device), test_loader, device)
    thr = cfg.low_state_threshold_us
    rows, states = [], {}
    num_all = den_all = 0.0
    for name in ("fc1", "fc2"):
        m = main[name]
        w_t = snap[name]["weight"]
        wmax = float(w_t.abs().max())
        active = [gt > 0 for gt in m.g_target]                       # inactive differential partners excluded
        g_act = torch.cat([gt[a] for gt, a in zip(m.g_target, active)])
        nu_act = torch.cat([nu[a] for nu, a in zip(m.nu, active)])
        drift_noiseless = m.read_conductances(ts, 0, noiseless=True)
        num = sum(float(gp[a].double().sum()) for gp, a in zip(m.g_programmed, active))
        den = sum(float(gd[a].double().sum()) for gd, a in zip(drift_noiseless, active))
        num_all += num
        den_all += den
        oracle = num / den
        w_prog0 = m.logical_weight(m.g_programmed)
        comp = nsr_components(w_t)
        rows.append({
            'method': method, 'fold': fold, 'realization': realization, 'layer': name, 't_s': ts,
            **comp,
            'active_devices': int(g_act.numel()), 'inactive_partners_excluded': int(sum((~a).sum() for a in active)),
            'G_active_mean_uS': float(g_act.mean()), 'G_active_median_uS': float(g_act.median()),
            'G_active_p10_uS': float(torch.quantile(g_act.float().cpu(), 0.10)),
            'G_active_p90_uS': float(torch.quantile(g_act.float().cpu(), 0.90)),
            'low_state_occupancy_pct': 100.0 * float((g_act < thr).float().mean()),
            'nu_active_mean': float(nu_act.mean()),
            'ref_gain_B1': gamma, 'oracle_gain_layer': oracle,
            'ref_gain_rel_error_pct': 100.0 * (gamma - oracle) / oracle,
            'weight_nrmse_uncorrected_pct': 100.0 * _rms(base[name] - w_t) / wmax,
            'weight_nrmse_B1_corrected_pct': 100.0 * _rms(corr[name] - w_t) / wmax,
            'residual_correction_error_pct': 100.0 * _rms(corr[name] - w_prog0) / wmax,
            'test_error_B1_5y_pct': 100.0 - acc_b1,
        })
        if keep_states:
            states[name] = {
                'w_target': w_t, 'wmax': wmax, 'g_target': m.g_target, 'g_programmed': m.g_programmed,
                'g_5y': conductances[name], 'w_5y': base[name], 'w_5y_B1': corr[name], 'active': active,
            }
    oracle_all = num_all / den_all
    for r in rows:
        r['oracle_gain_pooled'] = oracle_all
        r['ref_gain_rel_error_pooled_pct'] = 100.0 * (gamma - oracle_all) / oracle_all
    meta = {'state_hash': state_hash, 'root_seed': root, 'gamma_B1': gamma, 'test_error_B1_5y_pct': 100.0 - acc_b1}
    if keep_states:
        # selected-B correction + 20-s reference read for Figure 5 (same seeds as retention)
        bank = PCMClassReferenceBank(cfg, selected_bins, root + 30, device)
        corr_sel, gains_sel = driftslice_effective_weights(main, conductances, bank, ts, root + 500_000 + ti*101)
        ref20, g20 = plain_effective_weights(main, T_REF, root + 40)
        edges = bank.edges
        for name in ("fc1", "fc2"):
            m = main[name]
            g_corr_sel = apply_driftslice(m, conductances[name], gains_sel, edges)
            st = states[name]
            act = st['active']
            pack = lambda gs: torch.cat([g[a] for g, a in zip(gs, act)]).half().cpu().numpy()
            states[name] = {
                'G_target_uS': pack(st['g_target']), 'G_programmed_uS': pack(st['g_programmed']),
                'G_read_20s_uS': pack(g20[name]), 'G_read_5y_uS': pack(st['g_5y']),
                'G_corrected_5y_selB_uS': pack(g_corr_sel),
                'W_target_norm': (st['w_target'] / st['wmax']).flatten().half().cpu().numpy(),
                'W_20s_norm': (ref20[name] / st['wmax']).flatten().half().cpu().numpy(),
                'W_5y_norm': (st['w_5y'] / st['wmax']).flatten().half().cpu().numpy(),
                'W_5y_B1_norm': (st['w_5y_B1'] / st['wmax']).flatten().half().cpu().numpy(),
                'W_5y_selB_norm': (corr_sel[name] / st['wmax']).flatten().half().cpu().numpy(),
            }
        meta['selected_bins'] = selected_bins
        meta['selected_gains'] = [float(v) for v in gains_sel.tolist()]
    return pd.DataFrame(rows), meta, states


# ============================ STAGE ORCHESTRATION ============================
def pre_pcm_test_accuracy(exp: Experiment, fold: int, digital, packs: dict, test_loader):
    """Test-only reporting AFTER all validation-based selection (original run())."""
    require_split(test_loader, "test", "pre-PCM reporting")
    cfg, dev = exp.cfg, exp.device
    rows = []
    with random_stream(cfg.seed + fold, dev):
        _, acc = evaluate_model(digital, test_loader, dev)
    rows.append({'fold': fold, 'method': 'FP32-Digital', 'test_accuracy': acc, 'trained_epochs': cfg.digital_epochs})
    for m, pack in packs.items():
        model = ops(exp).load_analog(cfg, pack, dev)
        with random_stream(cfg.seed + fold, dev):
            _, acc = evaluate_model(model, test_loader, dev)
        rows.append({'fold': fold, 'method': m, 'test_accuracy': acc,
                     'trained_epochs': pack.get('analog_epochs_completed', 0),
                     'selected_epoch': pack.get('selected_best_validation_epoch', 0)})
    return pd.DataFrame(rows)


def best_epoch_from_history(history) -> int:
    best_acc, best_loss, best = -1.0, math.inf, -1
    for row in history:
        if row['val_accuracy'] > best_acc or (row['val_accuracy'] == best_acc and row['val_loss'] < best_loss):
            best_acc, best_loss, best = row['val_accuracy'], row['val_loss'], row['epoch']
    return best


def verify_training_fold(exp: Experiment, fold: int, packs: dict, digital_identity: str, digital):
    cfg = exp.cfg
    scope = f"fold{fold}"
    d = packs[DIRECT]
    # Existing Direct checkpoints contain the old diagnostic. Recheck the saved model,
    # rather than trusting stale report rows or rewriting training checkpoints.
    if state_dict_hash(d["model"]) != d.get("model_identity") or state_dict_hash(digital.state_dict()) != digital_identity:
        raise ScientificCheckError("Transfer verification: checkpoint model identity mismatch")
    if d.get("digital_identity") != digital_identity:
        raise ScientificCheckError("Transfer verification: Direct checkpoint has a different FP32 parent")
    # AIHWKit constructors may initialize temporary CPU tensors. Keep verification
    # from consuming the training RNG stream (particularly for the MLP).
    with random_stream(cfg.seed + fold, torch.device("cpu")):
        direct = ops(exp).load_analog(cfg, d, torch.device("cpu"))
    if exp.arch == "resnet32":
        digital_resnet_state_matches(direct, digital.state_dict())  # includes BatchNorm
        report = transfer_error_report_resnet(direct, digital, cfg.weight_scaling_omega)
    else:
        ensure_same_pretrained_weights(direct, digital)
        report = transfer_error_report(direct, digital, cfg.weight_scaling_omega)
    n_layers = len(direct.layers())
    del direct
    te = pd.DataFrame(report)
    wb = te[te.tensor.isin(["weight", "bias"])]
    tol_ok = bool((wb.max_abs_error <= 2e-4 + 2e-3 * wb.max_abs_value).all())
    mapping = te[te.tensor.str.startswith("tile_")]
    omega_ok = len(mapping) == 2 * n_layers and bool((mapping.max_abs_error < 1e-5).all())
    row_stats = mapping[mapping.tensor == "tile_weight_row_absmax_minus_expected"]
    zero_rows = int(row_stats.zero_rows.sum())
    active_rows = int(row_stats.active_rows.sum())
    record_check("Weight and bias transfer correctness", tol_ok and omega_ok,
                 "; ".join(f"{r.layer}.{r.tensor}: max|e|={r.max_abs_error:.2e} rms={r.rms_error:.2e}"
                           for r in wb.itertuples()) + f"; tile mapping correct: {omega_ok} (zero rows={zero_rows}, active rows={active_rows}); digital_bias=True", scope)
    same = all(p.get("digital_identity") == digital_identity for p in packs.values())
    record_check("No post-transfer reinitialization", same,
                 f"all {len(packs)} models carry digital checkpoint identity {digital_identity[:12]} and passed "
                 "ensure_same_pretrained_weights() immediately before the first optimizer step", scope)
    d_ok = d["analog_epochs"] == 0 and d["optimizer_steps"] == 0
    record_check("FP32-Direct has zero HWA steps", d_ok, f"analog_epochs={d['analog_epochs']}, optimizer_steps={d['optimizer_steps']}", scope)
    for m in exp.methods:
        p = packs[m]
        hist = p["history"]
        epochs = [h["epoch"] for h in hist]
        n_ok = p["analog_epochs_completed"] == cfg.epochs and epochs == list(range(1, cfg.epochs + 1))
        record_check("Exactly 154 HWA epochs" if not cfg.fast_mode else "Exactly 154 HWA epochs", n_ok,
                     f"{m}: {p['analog_epochs_completed']} epochs completed (expected {cfg.epochs})", scope)
        steps_ok = p["optimizer_steps"] == cfg.epochs * p["steps_per_epoch"]
        upd_ok = (p["first_step_max_weight_change"] or 0) > 0 and all(h["optimizer"] == "AnalogAdam" for h in hist) \
            and p["analog_context_groups"] >= 2 and steps_ok
        record_check("Actual AnalogAdam updates", upd_ok,
                     f"{m}: {p['optimizer_steps']} AnalogAdam steps, first-step max|Δw|={p['first_step_max_weight_change']:.3e}, "
                     f"{p['analog_context_groups']} AnalogContext groups (post-update clipping active)", scope)
        sel = p["selected_best_validation_epoch"]
        recomputed = best_epoch_from_history(hist)
        rest_ok = sel == recomputed and sel >= 1 and p["restore_abs_diff_pp"] <= 0.05
        record_check("Best-validation checkpoint restoration", rest_ok,
                     f"{m}: selected epoch {sel} (recomputed {recomputed}) of {len(hist)}; "
                     f"re-eval |Δacc|={p['restore_abs_diff_pp']:.2e} pp", scope)


def run_training_stage(exp: Experiment, verify_fast_loader: bool = True):
    cfg, dev, o = exp.cfg, exp.device, ops(exp)
    for fold in range(cfg.train_repeats):
        fdir = exp.fold_train_dir(fold)
        train, val, test, split = o.loaders(cfg, exp.base_dir, fold, dev, verify_fast=verify_fast_loader)
        dig = split_digest(split)
        split_path = fdir / "split.pt"
        if split_path.exists():
            if torch_load(split_path)["digest"] != dig:
                raise ScientificCheckError(f"fold {fold}: data split differs from the stored split")
        else:
            atomic_torch_save({**split, "digest": dig}, split_path)
        digital, dhist, dpack = o.train_digital(cfg, fold, train, val, dev, fdir, exp.fp["train"])
        n_ep, n_steps = dpack["digital_epochs_completed"], dpack["optimizer_steps"]
        record_check("Exactly 30 digital epochs",
                     n_ep == cfg.digital_epochs and n_steps == cfg.digital_epochs * dpack["steps_per_epoch"]
                     and (cfg.fast_mode or cfg.digital_epochs == 30),
                     f"{n_ep} epochs, {n_steps} optimizer steps, best epoch {dpack['best_epoch']}", f"fold{fold}")
        digital_identity = dpack["model_identity"]
        packs = {DIRECT: o.make_direct(cfg, fold, digital, digital_identity, val, dev, fdir, exp.fp["train"])}
        for m in exp.methods:
            packs[m] = o.train_hwa(m, cfg, fold, digital, digital_identity, train, val, dev, fdir, exp.fp["train"])
        verify_training_fold(exp, fold, packs, digital_identity, digital)
        pre_path = fdir / "pre_pcm_test_accuracy.csv"
        expected = {"FP32-Digital", DIRECT, *exp.methods}
        if exp.write_manifest and (not pre_path.exists() or set(pd.read_csv(pre_path).method) != expected):
            # (parallel workers never write this shared file; the coordinator writes it for all models)
            atomic_write_csv(pre_pcm_test_accuracy(exp, fold, digital, packs, test), pre_path)
        del digital
        if dev.type == "cuda":
            torch.cuda.empty_cache()
    print(f"TRAINING STAGE COMPLETE ({o.label}, {o.dataset}): {cfg.train_repeats} folds x ({len(exp.methods)} HWA methods + FP32-Direct)")


def training_complete(exp: Experiment) -> Tuple[bool, list]:
    missing = []
    for fold in range(exp.cfg.train_repeats):
        fdir = exp.train_dir / f"fold{fold}"
        need = [fdir / "digital_best_validation.pt", fdir / f"{DIRECT}_analog_checkpoint.pt"] + \
               [fdir / f"{m}_best_validation.pt" for m in exp.methods]
        missing += [str(p.relative_to(exp.run_dir)) for p in need if not p.exists()]
    return not missing, missing


@scientific("retention")
def load_fold_snapshots(exp: Experiment, fold: int, val_loader) -> dict:
    cfg, dev = exp.cfg, exp.device
    fdir = exp.train_dir / f"fold{fold}"
    out, stats = {}, []
    x_probe = torch.cat([x for x, _ in val_loader])      # whole validation split
    for m in exp.models:
        path = fdir / (f"{DIRECT}_analog_checkpoint.pt" if m == DIRECT else f"{m}_best_validation.pt")
        pack = torch_load(path)
        kind = "fp32_direct" if m == DIRECT else "hwa_best_validation"
        verify_pack(pack, exp.fp["train"], kind, fold, path)
        if m != DIRECT and pack["analog_epochs_completed"] != cfg.epochs:
            raise ScientificCheckError(f"{path}: only {pack['analog_epochs_completed']} HWA epochs")
        model = load_analog_from_pack(cfg, pack, dev)
        snap = snapshot_analog_model(model)
        eff = {k: v["weight"] for k, v in snap.items()}
        with torch.inference_mode(), random_stream(4242, dev):
            y0 = model(x_probe)
        with torch.inference_mode(), random_stream(4242, dev):
            y1 = SnapshotMLP(snap, eff, cfg).to(dev)(x_probe)
        d = (y0 - y1).abs()
        stats.append({"model": m, "median": float(d.median()), "max": float(d.max()),
                      "frac_gt": float((d > 3e-4).float().mean()),
                      "argmax_agree": float((y0.argmax(1) == y1.argmax(1)).float().mean())})
        tile_ok = all(torch.equal(snap[n]["weight"], physical_weight_parameter(l).detach())
                      for n, l in model.layers().items())
        if not tile_ok:
            record_check("Snapshot MVM reproduces AIHWKit forward", False, f"{m}: snapshot weight != tile weight", f"fold{fold}")
        out[m] = {"snap": snap, "identity": pack["model_identity"], "model": model}
    # The original deployment path (shared_mvm) multiplies by the pre-combined output scale, AIHWKit by its two
    # factors in sequence: a 1-ulp difference can flip ONE 8-bit DAC level of fc2's input for a rare sample.
    # A real mismatch (scale, bias, input range) would affect almost every logit, so the criterion is:
    ok = all(s["median"] <= 1e-6 and s["frac_gt"] <= 0.005 and s["argmax_agree"] >= 0.995 for s in stats)
    worst = max(stats, key=lambda s: s["frac_gt"])
    record_check("Snapshot MVM reproduces AIHWKit forward", ok,
                 f"{len(out)} models on {len(x_probe)} validation images (same noise stream): median |Δlogit| <= "
                 f"{max(s['median'] for s in stats):.1e}; logits with |Δ|>3e-4: <= {100 * worst['frac_gt']:.3f}% "
                 f"({worst['model']}, isolated DAC-rounding flips); argmax agreement >= "
                 f"{100 * min(s['argmax_agree'] for s in stats):.2f}%; max |Δ| = {max(s['max'] for s in stats):.1e}",
                 f"fold{fold}")
    return out


@scientific("retention")
def get_selected_bins(exp: Experiment, fold: int, snaps: dict, val_loader) -> int:
    cfg = exp.cfg
    sel_method = cfg.bin_selection_method if cfg.bin_selection_method in snaps else exp.methods[0]
    d = exp.ret_dir / f"fold{fold}"
    path = d / "selected_bins.json"
    ident = snaps[sel_method]["identity"]
    if path.exists():
        rec = json.loads(path.read_text())
        if rec["selection_model_identity"] != ident:
            raise ScientificCheckError(f"{path}: selection made on a different checkpoint")
        return int(rec["bins"])
    bins, table = select_driftslice_bins_on_validation(snaps[sel_method]["snap"], cfg, fold, val_loader, exp.device)
    atomic_write_csv(table, d / "bin_selection_validation.csv")
    atomic_write_text(path, json.dumps({
        "fold": fold, "bins": bins, "selection_method": sel_method, "selection_model_identity": ident,
        "selection": "validation_only_5yr", "candidates": list(cfg.ablation_bins),
        "realizations": 1 if cfg.fast_mode else cfg.driftslice_selection_realizations,
        "validation_error_by_B": table.groupby("bins").validation_error.mean().to_dict()}, indent=2))
    print(f"fold {fold}: DriftSlice B selected on VALIDATION = {bins}  "
          f"({table.groupby('bins').validation_error.mean().round(3).to_dict()})")
    return bins


def fold_selected_bins(exp: Experiment, fold: int, snaps: dict, val_loader) -> int:
    """Validation-selected B of a fold. Unchanged path (select or verify) when this runtime holds the selection
    model; a parallel worker without it reads the coordinator's selection and verifies it against the
    selection model's checkpoint identity on disk."""
    sel = exp.cfg.bin_selection_method
    if sel in snaps or exp.write_manifest:
        return ops(exp).selected_bins(exp, fold, snaps, val_loader)
    path = exp.ret_dir / f"fold{fold}" / "selected_bins.json"
    if not path.exists():
        raise ScientificCheckError(f"fold {fold}: DriftSlice B has not been selected yet (coordinator step missing)")
    rec = json.loads(path.read_text())
    ident = torch_load(exp.train_dir / f"fold{fold}" / f"{sel}_best_validation.pt")["model_identity"]
    if rec.get("selection_method") != sel or rec.get("selection_model_identity") != ident:
        raise ScientificCheckError(f"{path}: selection does not belong to the {sel} checkpoint of fold {fold}")
    return int(rec["bins"])


def _shard_ok(path: Path, fp: str, identity: str, extra: dict) -> bool:
    if not path.exists():
        return False
    try:
        pack = torch_load(path)
    except Exception as exc:
        print(f"WARNING: unreadable shard {path.name} ({exc}); it will be recomputed")
        return False
    if pack.get("fp") != fp or pack.get("model_identity") != identity or any(pack.get(k) != v for k, v in extra.items()):
        raise ScientificCheckError(f"Shard {path} belongs to a different experiment state; refusing to overwrite or reuse it.")
    return True


def _eta(done, total, t0):
    if done == 0:
        return "?"
    s = (time.time() - t0) / done * (total - done)
    return f"{s/3600:.2f} h" if s > 5400 else f"{s/60:.1f} min"


def run_retention_stage(exp: Experiment):
    cfg, dev, o = exp.cfg, exp.device, ops(exp)
    ok, missing = training_complete(exp)
    if not ok:
        raise ScientificCheckError("Retention requires completed training. Missing: " + ", ".join(missing[:8]))
    total = cfg.train_repeats * cfg.realizations * len(exp.models)
    done = skipped = 0
    t0 = time.time()
    for fold in range(cfg.train_repeats):
        _, val, test, _ = o.loaders(cfg, exp.base_dir, fold, dev, verify_fast=False)
        snaps = o.load_snapshots(exp, fold, val)
        bins = fold_selected_bins(exp, fold, snaps, val)
        for r in range(cfg.realizations):
            for m in exp.models:
                path = exp.ret_dir / f"fold{fold}" / m / f"real{r:02d}.pt"
                if _shard_ok(path, exp.fp["retention"], snaps[m]["identity"], {"selected_bins": bins}):
                    skipped += 1
                    continue
                ts = time.time()
                df, meta = o.retention(m, snaps[m], cfg, fold, r, bins, test, dev)
                atomic_torch_save({"fp": exp.fp["retention"], "model_identity": snaps[m]["identity"],
                                   "selected_bins": bins, "results": df, "meta": meta, "gpu": _gpu_name(),
                                   "seconds": time.time() - ts}, path)
                done += 1
                if not meta["state_hash_start"] == meta["state_hash_end"]:
                    record_check("Persistent programmed PCM device states", False, f"{path.name}: state changed during time loop")
            print(f"retention fold {fold+1}/{cfg.train_repeats} realization {r+1}/{cfg.realizations}: "
                  f"{done} new, {skipped} verified-existing shards; ETA {_eta(done, total - skipped, t0)}", flush=True)
        for m in list(snaps):
            del snaps[m]["model"]
    print(f"RETENTION STAGE COMPLETE: {done} new shards, {skipped} existing shards verified.")


def run_ablation_stage(exp: Experiment):
    cfg, dev, o = exp.cfg, exp.device, ops(exp)
    ok, missing = training_complete(exp)
    if not ok:
        raise ScientificCheckError("Ablation requires completed training.")
    total = cfg.train_repeats * cfg.realizations * len(exp.models)
    done = skipped = 0
    t0 = time.time()
    all_shapes_ok = True
    for fold in range(cfg.train_repeats):
        _, val, test, _ = o.loaders(cfg, exp.base_dir, fold, dev, verify_fast=False)
        snaps = o.load_snapshots(exp, fold, val)
        for r in range(cfg.realizations):
            for m in exp.models:
                path = exp.abl_dir / f"fold{fold}" / m / f"real{r:02d}.pt"
                if _shard_ok(path, exp.fp["ablation"], snaps[m]["identity"], {}):
                    skipped += 1
                    all_shapes_ok &= bool(torch_load(path)["meta"]["budget_shapes_ok"])   # stored evidence
                    continue
                df, meta = o.ablation(m, snaps[m], cfg, fold, r, val, test, dev)
                all_shapes_ok &= meta["budget_shapes_ok"]
                atomic_torch_save({"fp": exp.fp["ablation"], "model_identity": snaps[m]["identity"],
                                   "results": df, "meta": meta, "gpu": _gpu_name()}, path)
                done += 1
            print(f"ablation fold {fold+1}/{cfg.train_repeats} realization {r+1}/{cfg.realizations}: "
                  f"{done} new, {skipped} existing; ETA {_eta(done, total - skipped, t0)}", flush=True)
    record_check("Reference-cell budget correctness", all_shapes_ok,
                 f"every bank had R_tot cells as [B, R_tot/B] for R_tot in {cfg.ablation_reference_budgets}, "
                 f"B in {cfg.ablation_bins}; K = {cfg.reference_read_repeats} reads"
                 + ("" if cfg.fast_mode else f" (full protocol requires K=4: {cfg.reference_read_repeats == 4})"),
                 "ablation", essential=True)
    if not cfg.fast_mode and cfg.reference_read_repeats != 4:
        record_check("Reference-cell budget correctness", False, "K must be 4 in the full protocol", "ablation")
    print(f"ABLATION STAGE COMPLETE: {done} new shards, {skipped} existing shards verified.")


def run_diagnostics_stage(exp: Experiment):
    cfg, dev, o = exp.cfg, exp.device, ops(exp)
    ok, missing = training_complete(exp)
    if not ok:
        raise ScientificCheckError("Diagnostics require completed training.")
    done = skipped = 0
    for fold in range(cfg.train_repeats):
        _, val, test, _ = o.loaders(cfg, exp.base_dir, fold, dev, verify_fast=False)
        snaps = o.load_snapshots(exp, fold, val)
        bins = fold_selected_bins(exp, fold, snaps, val)
        # NSR surrogate consistency check against the training regularizer itself
        for m in exp.models:
            model = snaps[m]["model"]
            with torch.no_grad():
                pen = float(nsr_penalty(model, cfg))
            comp = [nsr_components(physical_weight_parameter(l).detach())["nsr_layer"] for l in model.layers().values()]
            rebuilt = cfg.nsr_lambda * float(np.mean(comp))
            record_check("NSR diagnostic surrogate matches nsr_penalty()", math.isclose(pen, rebuilt, rel_tol=1e-5),
                         f"{m}: nsr_penalty={pen:.6g}, lambda*mean(components)={rebuilt:.6g}", f"fold{fold}")
        for r in range(cfg.realizations):
            for m in exp.models:
                path = exp.diag_dir / f"fold{fold}" / m / f"real{r:02d}.pt"
                if _shard_ok(path, exp.fp["diagnostics"], snaps[m]["identity"], {"selected_bins": bins}):
                    skipped += 1
                    continue
                keep = (fold == 0 and r == 0)
                df, meta, states = o.diagnostics(m, snaps[m], cfg, fold, r, bins, test, dev, keep_states=keep)
                if keep:
                    buf = _io.BytesIO()
                    np.savez_compressed(buf, **{f"{layer}__{k}": v for layer, d in states.items() for k, v in d.items()})
                    npz = exp.diag_dir / f"fold{fold}" / f"distribution_states_{m}.npz"
                    npz.parent.mkdir(parents=True, exist_ok=True)
                    tmp = npz.with_name(npz.name + ".tmp")
                    tmp.write_bytes(buf.getvalue())
                    os.replace(tmp, npz)
                atomic_torch_save({"fp": exp.fp["diagnostics"], "model_identity": snaps[m]["identity"],
                                   "selected_bins": bins, "results": df, "meta": meta, "gpu": _gpu_name()}, path)
                done += 1
        print(f"diagnostics fold {fold+1}/{cfg.train_repeats}: {done} new, {skipped} existing shards", flush=True)
    print(f"DIAGNOSTICS STAGE COMPLETE: {done} new shards, {skipped} existing shards verified.")


# ----------------------------- shard collection -----------------------------
def collect_shards(stage_dir: Path, fp: str, models=None) -> Tuple[pd.DataFrame, list]:
    frames, metas = [], []
    for p in sorted(Path(stage_dir).glob("fold*/*/real*.pt")):
        if models is not None and p.parent.name not in models:     # e.g. leftovers of a deselected ablation method
            continue
        pack = torch_load(p)
        if pack.get("fp") != fp:
            raise ScientificCheckError(f"{p} has a foreign fingerprint")
        frames.append(pack["results"])
        metas.append({"file": str(p.relative_to(stage_dir)), **{k: v for k, v in pack.items() if k not in ("results",)}})
    return (pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()), metas


print("Stage runners loaded.")

### 9b. ResNet-32 retention, ablation and diagnostics (ADDED)

These are the MLP stage functions with the **same seeds** (root, +10, +30, +40, +1e5+100i, +5e5+101i, +7e5+31i), applied to the 34 layers:
* **Retention:** the 12 anchor timestamps, with the compensations none, saturation-safe GDC (per layer), DriftSlice B = 1 and DriftSlice with the validation-selected B. The original all-ones-probe GDC is not evaluated, but its 20-s ADC saturation fraction is recorded for every layer.
* **Ablation:** B ∈ {1,2,4,8} × R_tot ∈ {16,64,256}, K = 4, at 5 years, on the same programmed arrays.
* **Diagnostics (Table II, Fig. 5):** the groups `conv` (33 conv layers pooled) and `fc`.

When the checkpoints are loaded, the scientific check *"Snapshot MVM reproduces AIHWKit forward"* verifies the deployment path of every model on the whole validation split:
* self-substitution is bit-identical;
* the model is restored exactly;
* the fc layer through `shared_mvm` matches AIHWKit;
* the GDC gain is applied correctly.

In [ ]:
# ============================ ADDED: ResNet-32 retention / ablation / diagnostics ============================
# Mirrors retention_one / reference_ablation_one / diagnostics_one with the SAME seed conventions
# (root = seed + fold*1e6 + realization*1e4; main +10, bank +30, ref read +40, read +1e5+100i, gain read +5e5+101i,
# eval +7e5+31i, GDC-safe calibration +8.5e5+31i(+layer index)), applied to all 34 mapped layers.
# Differences (ResNet only): 12 anchor timestamps; the ORIGINAL all-ones-probe GDC is not evaluated (its
# saturation fraction is still recorded); Table II / Fig. 5 pool all conv layers ("conv") and report "fc".

def resnet_layer_group(name: str) -> str:
    return "fc" if name == "fc" else "conv"


@scientific("rn_retention")
def retention_one_resnet(method, model, snap, cfg, fold, realization, selected_bins, test_loader, device):
    require_split(test_loader, "test", "retention")
    root = cfg.seed + fold * 1_000_000 + realization * 10_000
    all_times = exact_retention_times(cfg)
    io = make_hwa_rpu(cfg).forward
    names = list(snap)
    main = build_main_pcm_all(snap, cfg, root + 10)
    state_hash_start = pcm_state_hash_all(main)
    bank = PCMClassReferenceBank(cfg, selected_bins, root + 30, device)
    bank1 = PCMClassReferenceBank(cfg, 1, root + 30, device)
    reference, _ = plain_effective_weights_all(main, T_REF, root + 40)
    added_root = ADDED_SEED_BASE + root
    probe = {name: gdc_safe_probe_amplitude(reference[name], io, added_root + 100 * i, device, cfg)
             for i, name in enumerate(names)}
    orig_sat = {name: original_gdc_saturation_fraction(reference[name], snap[name], io, added_root + 5_000 + i, device)
                for i, name in enumerate(names)}
    unity = {name: 1.0 for name in names}
    data = []
    for ti, ts in enumerate(all_times):
        ts = float(ts)
        base, conductances = plain_effective_weights_all(main, ts, root + 100_000 + ti*100)
        variants = OrderedDict()
        variants['none'] = (base, unity, {})
        if cfg.run_gdc_baselines:
            if cfg.include_original_gdc:
                raise ScientificCheckError("The original saturating GDC is not part of the ResNet protocol")
            safe = {name: saturation_safe_gdc_gain(reference[name], base[name], snap[name], io, probe[name],
                                                   root + 850_000 + ti*31 + i, device, cfg)
                    for i, name in enumerate(names)}
            gains = [s['gain'] for s in safe.values() if s['valid']]
            extra = {'gdc_valid': bool(all(s['valid'] for s in safe.values())),
                     'gdc_invalid_layers': int(sum(not s['valid'] for s in safe.values())),
                     'gdc_gain_fc': safe['fc']['gain'],
                     'gdc_gain_conv_mean': float(np.mean([safe[n]['gain'] for n in names if n != 'fc'])),
                     'gdc_sat_now_max': float(max(s['sat_now'] for s in safe.values()))}
            variants['GDC-safe'] = (base, {n: safe[n]['gain'] for n in names}, extra)
        ds1_w, gain_1 = driftslice_effective_weights_all(main, conductances, bank1, ts, root + 500_000 + ti*101)
        variants['DS-B1'] = (ds1_w, unity, {'class_gain_0': float(gain_1[0])})
        ds_w, gain_b = driftslice_effective_weights_all(main, conductances, bank, ts, root + 500_000 + ti*101)
        variants['DS-val'] = (ds_w, unity, {f'class_gain_{i}': float(v) for i, v in enumerate(gain_b.tolist())})
        for comp, (weights, gains, extra) in variants.items():
            if comp == 'GDC-safe' and not extra['gdc_valid']:
                loss = accuracy = float('nan')          # never invent a value for an invalid calibration
            else:
                with random_stream(root + 700_000 + ti*31, device):
                    loss, accuracy = evaluate_deployed(model, weights, test_loader, device,
                                                       None if gains is unity else gains)
            data.append({
                'method': method, 'compensation': comp, 'fold': fold, 'realization': realization,
                't_index': ti, 't_s': ts, 'selected_bins': selected_bins, 'split': 'test',
                'loss': loss, 'accuracy': accuracy, 'error': 100. - accuracy,
                **extra,
            })
    state_hash_end = pcm_state_hash_all(main)
    meta = {'root_seed': root, 'state_hash_start': state_hash_start, 'state_hash_end': state_hash_end,
            'gdc_safe_probe': probe, 'orig_gdc_ref_saturation_fraction': orig_sat,
            'bank_target_numel': int(bank.target.numel()), 'bank_shape': list(bank.target.shape),
            'bank1_shape': list(bank1.target.shape), 'reference_read_repeats': cfg.reference_read_repeats,
            'times': [float(t) for t in all_times], 'arch': 'resnet32'}
    return pd.DataFrame(data), meta


@scientific("rn_ablation")
def reference_ablation_one_resnet(method, model, snap, cfg, fold, realization, val_loader, test_loader, device):
    require_split(val_loader, "val", "ablation(report-only)")
    require_split(test_loader, "test", "ablation(report-only)")
    root = cfg.seed + fold * 1_000_000 + realization * 10_000
    times = exact_retention_times(cfg)
    ti = len(times) - 1
    ts = float(times[ti])
    main = build_main_pcm_all(snap, cfg, root + 10)
    state_hash = pcm_state_hash_all(main)
    base, conductances = plain_effective_weights_all(main, ts, root + 100_000 + ti*100)
    rows, budget_ok = [], []

    def ev(weights, loader):
        with random_stream(root + 700_000 + ti*31, device):
            return evaluate_deployed(model, weights, loader, device)

    for split, loader in (("val", val_loader), ("test", test_loader)):
        _, acc = ev(base, loader)
        rows.append({'method': method, 'fold': fold, 'realization': realization, 'split': split,
                     'compensation': 'none', 'bins': 0, 'R_tot': 0, 'R_b': 0, 'K': 0,
                     't_s': ts, 'accuracy': acc, 'error': 100. - acc})
    for R in cfg.ablation_reference_budgets:
        cfg_r = replace(cfg, reference_cell_budget=int(R))
        for B in cfg.ablation_bins:
            bank = PCMClassReferenceBank(cfg_r, int(B), root + 30, device)
            budget_ok.append(bank.target.numel() == R and tuple(bank.target.shape) == (B, R // B))
            corr, gains = driftslice_effective_weights_all(main, conductances, bank, ts, root + 500_000 + ti*101)
            for split, loader in (("val", val_loader), ("test", test_loader)):
                _, acc = ev(corr, loader)
                rows.append({'method': method, 'fold': fold, 'realization': realization, 'split': split,
                             'compensation': 'DriftSlice', 'bins': int(B), 'R_tot': int(R), 'R_b': int(R) // int(B),
                             'K': cfg.reference_read_repeats, 't_s': ts, 'accuracy': acc, 'error': 100. - acc,
                             **{f'class_gain_{i}': float(v) for i, v in enumerate(gains.tolist())}})
    return pd.DataFrame(rows), {'state_hash': state_hash, 'budget_shapes_ok': bool(all(budget_ok)), 'root_seed': root}


@scientific("rn_diagnostics")
def pooled_nsr_components(ws: List[Tensor]) -> Dict[str, float]:
    """nsr_components() of several layers pooled into one group: per-weight means are weighted by layer size
    (= means over all pooled weights); nsr_layer is the UNWEIGHTED mean of the per-layer NSR (as in nsr_penalty)."""
    comps = [nsr_components(w) for w in ws]
    n = np.array([c["n_weights"] for c in comps], dtype=float)
    out = {"n_weights": int(n.sum()), "n_layers": len(comps)}
    for k in ("signal_energy_mean_g2", "prog_noise_surrogate_mean", "drift_noise_surrogate_mean",
              "read_noise_surrogate_mean"):
        out[k] = float(np.sum(n * np.array([c[k] for c in comps])) / n.sum())
    out["inactive_partner_floor_sp0"] = comps[0]["inactive_partner_floor_sp0"]
    out["nsr_layer"] = float(np.mean([c["nsr_layer"] for c in comps]))
    return out


@scientific("rn_diagnostics")
def diagnostics_one_resnet(method, model, snap, cfg, fold, realization, selected_bins, test_loader, device,
                           keep_states=False):
    """diagnostics_one() for ResNet-32; per-layer quantities pooled over the groups 'conv' (33 layers) and 'fc'.
    Weight errors are normalized by each layer's own max|W_target| before pooling."""
    require_split(test_loader, "test", "diagnostics(report-only)")
    root = cfg.seed + fold * 1_000_000 + realization * 10_000
    times = exact_retention_times(cfg)
    ti = len(times) - 1
    ts = float(times[ti])
    main = build_main_pcm_all(snap, cfg, root + 10)
    state_hash = pcm_state_hash_all(main)
    base, conductances = plain_effective_weights_all(main, ts, root + 100_000 + ti*100)
    bank1 = PCMClassReferenceBank(cfg, 1, root + 30, device)
    corr, gains = driftslice_effective_weights_all(main, conductances, bank1, ts, root + 500_000 + ti*101)
    gamma = float(gains[0])
    with random_stream(root + 700_000 + ti*31, device):
        _, acc_b1 = evaluate_deployed(model, corr, test_loader, device)
    thr = cfg.low_state_threshold_us
    groups = OrderedDict()
    for name in snap:
        groups.setdefault(resnet_layer_group(name), []).append(name)
    rows, states = [], {}
    num_all = den_all = 0.0
    for group, members in groups.items():
        g_act, nu_act, d_unc, d_cor, d_res = [], [], [], [], []
        num = den = 0.0
        n_inactive = 0
        for name in members:
            m = main[name]
            w_t = snap[name]["weight"]
            wmax = float(w_t.abs().max())
            active = [gt > 0 for gt in m.g_target]
            g_act.append(torch.cat([gt[a] for gt, a in zip(m.g_target, active)]))
            nu_act.append(torch.cat([nu[a] for nu, a in zip(m.nu, active)]))
            n_inactive += int(sum((~a).sum() for a in active))
            drift_noiseless = m.read_conductances(ts, 0, noiseless=True)
            num += sum(float(gp[a].double().sum()) for gp, a in zip(m.g_programmed, active))
            den += sum(float(gd[a].double().sum()) for gd, a in zip(drift_noiseless, active))
            w_prog0 = m.logical_weight(m.g_programmed)
            d_unc.append(((base[name] - w_t) / wmax).flatten())
            d_cor.append(((corr[name] - w_t) / wmax).flatten())
            d_res.append(((corr[name] - w_prog0) / wmax).flatten())
        num_all += num
        den_all += den
        oracle = num / den
        g_act, nu_act = torch.cat(g_act), torch.cat(nu_act)
        comp = pooled_nsr_components([snap[n]["weight"] for n in members])
        rows.append({
            'method': method, 'fold': fold, 'realization': realization, 'layer': group, 't_s': ts,
            **comp,
            'active_devices': int(g_act.numel()), 'inactive_partners_excluded': n_inactive,
            'G_active_mean_uS': float(g_act.mean()), 'G_active_median_uS': float(g_act.median()),
            'G_active_p10_uS': float(torch.quantile(g_act.float().cpu(), 0.10)),
            'G_active_p90_uS': float(torch.quantile(g_act.float().cpu(), 0.90)),
            'low_state_occupancy_pct': 100.0 * float((g_act < thr).float().mean()),
            'nu_active_mean': float(nu_act.mean()),
            'ref_gain_B1': gamma, 'oracle_gain_layer': oracle,
            'ref_gain_rel_error_pct': 100.0 * (gamma - oracle) / oracle,
            'weight_nrmse_uncorrected_pct': 100.0 * _rms(torch.cat(d_unc)),
            'weight_nrmse_B1_corrected_pct': 100.0 * _rms(torch.cat(d_cor)),
            'residual_correction_error_pct': 100.0 * _rms(torch.cat(d_res)),
            'test_error_B1_5y_pct': 100.0 - acc_b1,
        })
    oracle_all = num_all / den_all
    for r in rows:
        r['oracle_gain_pooled'] = oracle_all
        r['ref_gain_rel_error_pooled_pct'] = 100.0 * (gamma - oracle_all) / oracle_all
    meta = {'state_hash': state_hash, 'root_seed': root, 'gamma_B1': gamma, 'test_error_B1_5y_pct': 100.0 - acc_b1,
            'layer_groups': {g: list(v) for g, v in groups.items()}}
    if keep_states:
        bank = PCMClassReferenceBank(cfg, selected_bins, root + 30, device)
        corr_sel, gains_sel = driftslice_effective_weights_all(main, conductances, bank, ts, root + 500_000 + ti*101)
        ref20, g20 = plain_effective_weights_all(main, T_REF, root + 40)
        for group, members in groups.items():
            parts = {k: [] for k in ('G_target_uS', 'G_programmed_uS', 'G_read_20s_uS', 'G_read_5y_uS',
                                     'G_corrected_5y_selB_uS', 'W_target_norm', 'W_20s_norm', 'W_5y_norm',
                                     'W_5y_B1_norm', 'W_5y_selB_norm')}
            for name in members:
                m = main[name]
                act = [gt > 0 for gt in m.g_target]
                pack = lambda gs: torch.cat([g[a] for g, a in zip(gs, act)]).half().cpu()
                g_corr_sel = apply_driftslice(m, conductances[name], gains_sel, bank.edges)
                wmax = float(snap[name]["weight"].abs().max())
                nrm = lambda w: (w / wmax).flatten().half().cpu()
                parts['G_target_uS'].append(pack(m.g_target))
                parts['G_programmed_uS'].append(pack(m.g_programmed))
                parts['G_read_20s_uS'].append(pack(g20[name]))
                parts['G_read_5y_uS'].append(pack(conductances[name]))
                parts['G_corrected_5y_selB_uS'].append(pack(g_corr_sel))
                parts['W_target_norm'].append(nrm(snap[name]["weight"]))
                parts['W_20s_norm'].append(nrm(ref20[name]))
                parts['W_5y_norm'].append(nrm(base[name]))
                parts['W_5y_B1_norm'].append(nrm(corr[name]))
                parts['W_5y_selB_norm'].append(nrm(corr_sel[name]))
            states[group] = {k: torch.cat(v).numpy() for k, v in parts.items()}
        meta['selected_bins'] = selected_bins
        meta['selected_gains'] = [float(v) for v in gains_sel.tolist()]
    return pd.DataFrame(rows), meta, states


@scientific("rn_retention")
def load_fold_snapshots_resnet(exp: "Experiment", fold: int, val_loader) -> dict:
    cfg, dev = exp.cfg, exp.device
    fdir = exp.train_dir / f"fold{fold}"
    out, details = {}, []
    ok_all = True
    for m in exp.models:
        path = fdir / (f"{DIRECT}_analog_checkpoint.pt" if m == DIRECT else f"{m}_best_validation.pt")
        pack = torch_load(path)
        kind = "fp32_direct" if m == DIRECT else "hwa_best_validation"
        verify_pack(pack, exp.fp["train"], kind, fold, path)
        if m != DIRECT and pack["analog_epochs_completed"] != cfg.epochs:
            raise ScientificCheckError(f"{path}: only {pack['analog_epochs_completed']} HWA epochs")
        model = load_analog_resnet_from_pack(cfg, pack, dev)
        snap = snapshot_analog_resnet(model)
        ok, detail = verify_resnet_deployment(model, snap, cfg, val_loader, dev)
        ok_all &= ok
        details.append(f"{m}: {detail}")
        out[m] = {"snap": snap, "identity": pack["model_identity"], "model": model}
    record_check("Snapshot MVM reproduces AIHWKit forward", ok_all,
                 f"ResNet-32 weight-substitution deployment on {len(out)} models (whole validation split): "
                 + " || ".join(details[:1]) + (" (same for all models)" if ok_all else " || " + " || ".join(details[1:])),
                 f"fold{fold}")
    return out


@scientific("rn_retention")
def get_selected_bins_resnet(exp: "Experiment", fold: int, snaps: dict, val_loader) -> int:
    cfg = exp.cfg
    sel_method = cfg.bin_selection_method if cfg.bin_selection_method in snaps else exp.methods[0]
    d = exp.ret_dir / f"fold{fold}"
    path = d / "selected_bins.json"
    ident = snaps[sel_method]["identity"]
    if path.exists():
        rec = json.loads(path.read_text())
        if rec["selection_model_identity"] != ident:
            raise ScientificCheckError(f"{path}: selection made on a different checkpoint")
        return int(rec["bins"])
    bins, table = select_driftslice_bins_on_validation_resnet(snaps[sel_method]["model"], snaps[sel_method]["snap"],
                                                              cfg, fold, val_loader, exp.device)
    atomic_write_csv(table, d / "bin_selection_validation.csv")
    atomic_write_text(path, json.dumps({
        "fold": fold, "bins": bins, "selection_method": sel_method, "selection_model_identity": ident,
        "selection": "validation_only_5yr", "candidates": list(cfg.ablation_bins),
        "realizations": 1 if cfg.fast_mode else cfg.driftslice_selection_realizations,
        "validation_error_by_B": table.groupby("bins").validation_error.mean().to_dict()}, indent=2))
    print(f"fold {fold}: DriftSlice B selected on VALIDATION = {bins}  "
          f"({table.groupby('bins').validation_error.mean().round(3).to_dict()})")
    return bins


# ============================ ARCHITECTURE DISPATCH (orchestration only) ============================
class ArchOps:
    """Which scientific functions a stage runner calls for an architecture. The runners are shared; the
    registered functions they call are the MLP originals or their ResNet counterparts above."""
    def __init__(self, **kw):
        self.__dict__.update(kw)


ARCH_OPS = {
    "mlp": ArchOps(
        name="mlp", label="784-250-10 MLP", dataset="MNIST", n_test_full=10000,
        layer_groups=("fc1", "fc2"), loaders=get_fold_loaders,
        train_digital=train_digital_pretrained, make_direct=make_fp32_direct, train_hwa=train_hwa_all,
        load_analog=load_analog_from_pack, load_snapshots=load_fold_snapshots, selected_bins=get_selected_bins,
        retention=lambda m, e, *a, **k: retention_one(m, e["snap"], *a, **k),
        ablation=lambda m, e, *a, **k: reference_ablation_one(m, e["snap"], *a, **k),
        diagnostics=lambda m, e, *a, **k: diagnostics_one(m, e["snap"], *a, **k),
        pcm_state_hash=pcm_state_hash),
    "resnet32": ArchOps(
        name="resnet32", label="ResNet-32", dataset="CIFAR-10", n_test_full=10000,
        layer_groups=("conv", "fc"), loaders=get_fold_loaders_cifar,
        train_digital=train_digital_resnet, make_direct=make_fp32_direct_resnet, train_hwa=train_hwa_resnet,
        load_analog=load_analog_resnet_from_pack, load_snapshots=load_fold_snapshots_resnet,
        selected_bins=get_selected_bins_resnet,
        retention=lambda m, e, *a, **k: retention_one_resnet(m, e["model"], e["snap"], *a, **k),
        ablation=lambda m, e, *a, **k: reference_ablation_one_resnet(m, e["model"], e["snap"], *a, **k),
        diagnostics=lambda m, e, *a, **k: diagnostics_one_resnet(m, e["model"], e["snap"], *a, **k),
        pcm_state_hash=pcm_state_hash_all),
}


def ops(exp: "Experiment") -> ArchOps:
    return ARCH_OPS[getattr(exp, "arch", "mlp")]


print("ResNet-32 stage functions and architecture dispatch loaded.")

## 10. Statistics, completeness and traceability checks

All summaries are **two-stage**, matching the paper's definition:

$$\bar E_f=\frac1R\sum_{r=1}^{R}E_{f,r},\qquad \bar E=\frac1F\sum_{f=1}^{F}\bar E_f,\qquad s=\sqrt{\frac{1}{F-1}\sum_{f=1}^{F}(\bar E_f-\bar E)^2}.$$

A fold mean is computed only when **all** R realizations of that fold are present and valid. Otherwise the cell is reported as missing; a smaller sample is never averaged silently. Paired comparisons (method A vs. B, B vs. B = 1, and so on) first take the difference within each (fold, realization), where the device seeds are paired. They then average over realizations and report the mean ± SD across folds with a t-based 95 % CI (df = F − 1).

In [ ]:
# ============================ STATISTICS ============================
def _mean_strict(s: pd.Series) -> float:
    return float(s.mean(skipna=False)) if len(s) else float("nan")


def two_stage_summary(df: pd.DataFrame, keys: List[str], value: str = "error",
                      expected_folds: Optional[int] = None, expected_reals: Optional[int] = None) -> pd.DataFrame:
    """Stage 1: mean over realizations within a fold (NaN if any realization missing/invalid).
    Stage 2: mean and SD (ddof=1) across folds (NaN if any fold incomplete)."""
    g = df.groupby(keys + ["fold"], dropna=False)[value]
    per_fold = g.agg(fold_mean=_mean_strict, n_real="size",
                     n_invalid=lambda s: int(s.isna().sum())).reset_index()
    if expected_reals is not None:
        per_fold.loc[per_fold.n_real != expected_reals, "fold_mean"] = np.nan
    out = per_fold.groupby(keys, dropna=False).agg(
        mean=("fold_mean", _mean_strict),
        sd=("fold_mean", lambda s: float(s.std(ddof=1)) if s.notna().all() and len(s) > 1 else float("nan")),
        n_folds=("fold_mean", "size"),
        realizations_per_fold_min=("n_real", "min"),
        realizations_per_fold_max=("n_real", "max"),
        invalid_records=("n_invalid", "sum"),
    ).reset_index()
    if expected_folds is not None:
        out.loc[out.n_folds != expected_folds, ["mean", "sd"]] = np.nan
    return out


def fold_means(df: pd.DataFrame, keys: List[str], value: str = "error") -> pd.DataFrame:
    return df.groupby(keys + ["fold"], dropna=False)[value].agg(_mean_strict).reset_index()


def t_ci(values: np.ndarray, alpha: float = 0.05):
    from scipy import stats
    v = np.asarray(values, float)
    n = len(v)
    m = float(v.mean()) if n else float("nan")
    if n < 2 or not np.isfinite(v).all():
        return m, float("nan"), float("nan"), float("nan"), float("nan")
    sd = float(v.std(ddof=1))
    half = float(stats.t.ppf(1 - alpha / 2, n - 1) * sd / math.sqrt(n))
    p = float(stats.ttest_1samp(v, 0.0).pvalue) if sd > 0 else float("nan")   # undefined at zero variance
    return m, sd, m - half, m + half, p


def paired_difference(df: pd.DataFrame, sel_a: dict, sel_b: dict, value: str = "error") -> pd.DataFrame:
    """Per-fold mean of the within-(fold, realization) difference A - B."""
    def pick(sel):
        q = df
        for k, v in sel.items():
            q = q[q[k] == v]
        return q.set_index(["fold", "realization"])[value]
    a, b = pick(sel_a), pick(sel_b)
    if not a.index.equals(b.index):
        common = a.index.intersection(b.index)
        if len(common) != len(a) or len(common) != len(b):
            raise ScientificCheckError(f"Unpaired records for {sel_a} vs {sel_b}")
        a, b = a.loc[common], b.loc[common]
    d = (a - b).rename("diff").reset_index()
    return d.groupby("fold")["diff"].agg(_mean_strict).reset_index()


def verify_aggregation_unit_test():
    """Synthetic data with a known answer for the two-stage estimator."""
    rows = []
    rng = np.random.default_rng(0)
    truth = {}
    for f in range(5):
        vals = rng.normal(5 + f, 1.0, size=30)
        truth[f] = vals.mean()
        rows += [{"m": "A", "fold": f, "realization": r, "error": v} for r, v in enumerate(vals)]
    df = pd.DataFrame(rows)
    s = two_stage_summary(df, ["m"], expected_folds=5, expected_reals=30).iloc[0]
    fm = np.array(list(truth.values()))
    ok = math.isclose(s["mean"], fm.mean(), rel_tol=1e-12) and math.isclose(s["sd"], fm.std(ddof=1), rel_tol=1e-12)
    # pooled SD (wrong estimator) must differ from the two-stage SD
    pooled = df.error.std(ddof=1)
    df_missing = df.drop(index=3)
    s2 = two_stage_summary(df_missing, ["m"], expected_folds=5, expected_reals=30).iloc[0]
    ok2 = math.isnan(s2["mean"])            # an incomplete fold must not be averaged silently
    return ok and ok2 and not math.isclose(pooled, s["sd"]), (
        f"synthetic 5x30: mean {s['mean']:.6f} == {fm.mean():.6f}, SD across fold means {s['sd']:.6f} == "
        f"{fm.std(ddof=1):.6f} (pooled SD {pooled:.3f} not used); a missing realization yields NaN: {ok2}")


def completeness_report(df: pd.DataFrame, exp: Experiment, comps=None) -> Tuple[bool, str, pd.DataFrame]:
    cfg = exp.cfg
    comps = compensations_for(cfg) if comps is None else comps
    df = df[df.method.isin(exp.models)] if len(df) else df
    n_t = len(exact_retention_times(cfg))
    key = ["fold", "method", "compensation", "realization", "t_index"]
    dup = int(df.duplicated(key).sum()) if len(df) else 0
    expected = cfg.train_repeats * len(exp.models) * len(comps) * cfg.realizations * n_t
    present = len(df.drop_duplicates(key)) if len(df) else 0
    have = df.groupby(["fold", "method"]).realization.nunique().reset_index() if len(df) else pd.DataFrame()
    ok = dup == 0 and present == expected
    return ok, f"{present}/{expected} retention records, {dup} duplicates", have


print("Statistics loaded.")

In [ ]:
# ============================ PHYSICS / MECHANICS VERIFICATION ============================
def verify_retention_grid(cfg: Config, scope: str):
    t = exact_retention_times(cfg)
    anchors = [20., 60., 1800., 3600., DAY, 5*DAY, 30*DAY, 180*DAY, YEAR, 2*YEAR, 3*YEAR, 5*YEAR]
    n_expected = 8 if cfg.fast_mode else cfg.retention_points       # 75 (MLP) or the 12 anchors (ResNet)
    ok = (len(t) == n_expected == cfg.retention_points and (cfg.fast_mode or n_expected in (75, 12))
          and t[0] == 20.0 and math.isclose(t[-1], 5*YEAR)
          and bool(np.all(np.diff(t) > 0)) and (cfg.fast_mode or all(np.isclose(t, a).any() for a in anchors)))
    record_check("Exact retention grid", ok,
                 f"{len(t)} timestamps, {t[0]:.0f} s .. {t[-1]/YEAR:.3f} y, strictly increasing"
                 + ("" if cfg.fast_mode else ", all 12 explicit anchors present"), scope)


def verify_pcm_physics(cfg: Config, device: torch.device, scope: str):
    """Independent numerical tests of the PCM model used by the experiment."""
    details, ok = [], True
    noise, conv = make_pcm_noise_model(cfg)
    # (1) converter round-trip
    gen = torch.Generator(device="cpu").manual_seed(1)
    w = (torch.rand(250, 784, generator=gen) * 2 - 1).to(device)
    g, params = conv.convert_to_conductances(w)
    rt = float((conv.convert_back_to_weights(list(g), params) - w).abs().max())
    gmax_seen = float(max(x.max() for x in g))
    ok &= rt < 1e-5 and math.isclose(gmax_seen, cfg.gmax_us, rel_tol=1e-6)
    ok &= bool(((g[0] > 0) & (g[1] > 0)).sum() == 0)    # one active branch per device
    details.append(f"converter round-trip max|e|={rt:.1e}, max G={gmax_seen:.3f} uS, one active branch/device")
    # (2) programming noise std vs formula
    n = 400_000
    for gt in (2.5, 12.5, 25.0):
        tgt = torch.full((n,), gt, device=device)
        with random_stream(11, device):
            gp = noise.apply_programming_noise_to_conductance(tgt)
        rel = gt / cfg.gmax_us
        sig = (0.26348 + 1.9650 * rel - 1.1731 * rel**2) * cfg.gmax_us / 25.0
        emp = float(gp.std())
        ok &= abs(emp / sig - 1) < 0.02
        details.append(f"prog σ({gt}uS)={emp:.4f} vs model {sig:.4f}")
    # (3) drift coefficients and noiseless power law
    for gt, mu_exp in ((25.0, 0.049), (0.0, 0.1)):
        tgt = torch.full((n,), gt, device=device)
        with random_stream(12, device):
            nu = noise.generate_drift_coefficients(tgt)
        details.append(f"mean ν(G={gt})={float(nu.mean()):.4f}")
        ok &= abs(float(nu.mean()) - mu_exp) < 0.01
    m = PCMPhysicalMatrix(w, cfg, 99)
    g20 = m.read_conductances(20.0, 0, noiseless=True)
    ok &= all(torch.equal(a, b) for a, b in zip(g20, m.g_programmed))
    t5 = 5 * YEAR
    g5 = m.read_conductances(t5, 0, noiseless=True)
    expected = [gp * (t5 / 20.0) ** (-nu) for gp, nu in zip(m.g_programmed, m.nu)]
    pl = max(float((a - b).abs().max()) for a, b in zip(g5, expected))
    ok &= pl < 1e-4
    details.append(f"noiseless G(20 s)=G_prog exactly; G(5 y)=G_prog·(t/t0)^-ν (max|e|={pl:.1e})")
    # (4) read noise at 20 s (elapsed 0): relative std vs q_s*sqrt(log((t0+t_read)/(2 t_read)))
    tgt = torch.full((n,), 12.5, device=device)
    with random_stream(13, device):
        gr = noise.apply_drift_noise_to_conductance(tgt, torch.zeros_like(tgt), 0.0)
    qs = min(0.0088 / max((12.5 / cfg.gmax_us) ** 0.65, 1e-3), 0.2)
    sig_rel = qs * math.sqrt(math.log((20.0 + noise.t_read) / (2 * noise.t_read)))
    emp = float((gr / 12.5 - 1).std())
    ok &= abs(emp / sig_rel - 1) < 0.03
    details.append(f"read-noise rel σ(20 s)={emp:.4f} vs model {sig_rel:.4f}")
    # (5) monotone mean-conductance decay over the retention grid (noiseless)
    times = exact_retention_times(cfg)
    means = [float(sum(x.sum() for x in m.read_conductances(float(t), 0, noiseless=True))) for t in times]
    ok &= bool(np.all(np.diff(means) < 0))
    details.append(f"mean conductance decays monotonically over {len(times)} timestamps "
                   f"({means[-1]/means[0]*100:.1f}% of 20-s value at 5 y)")
    # (6) persistence: reading does not modify programmed state
    h0 = pcm_state_hash({"fc1": m, "fc2": m})
    for t in times:
        m.read_conductances(float(t), 5)
    ok_persist = h0 == pcm_state_hash({"fc1": m, "fc2": m})
    record_check("PCM programming and drift behaviour", bool(ok), "; ".join(details), scope)
    record_check("Persistent programmed PCM device states", ok_persist,
                 f"programmed G and ν unchanged after {len(times)} reads (state hash {h0[:10]})", scope)


def verify_driftslice_mechanics(cfg: Config, device: torch.device, scope: str):
    ok, details = True, []
    gvals = torch.linspace(0, cfg.gmax_us, 100_001, device=device)
    for B in cfg.ablation_bins:
        edges = conductance_edges(cfg, B, device)
        idx = class_indices(gvals, edges)
        width = cfg.gmax_us / B
        manual = (torch.ceil(gvals.double() / width) - 1).clamp(0, B - 1).long()   # right-closed (lo, hi]; float64 reference
        ok &= bool(torch.equal(idx, manual)) and int(idx.min()) == 0 and int(idx.max()) == B - 1
        for R in cfg.ablation_reference_budgets:
            bank = PCMClassReferenceBank(replace(cfg, reference_cell_budget=R), B, 7, device)
            in_class = all(bool((class_indices(bank.target[b], edges) == b).all()) for b in range(B))
            ok &= in_class and tuple(bank.target.shape) == (B, R // B)
    details.append(f"class index == ceil(G/(Gmax/B))-1 for B in {cfg.ablation_bins}; every reference target "
                   f"lies inside its own class for R_tot in {cfg.ablation_reference_budgets}")
    record_check("DriftSlice B class assignment", bool(ok), "; ".join(details), scope)
    # gain application: both branches, class from TARGET conductance, conductance domain
    w = (torch.rand(50, 60, device=device) * 2 - 1)
    m = PCMPhysicalMatrix(w, cfg, 3)
    cur = m.read_conductances(YEAR, 4)
    bank = PCMClassReferenceBank(cfg, 4, 5, device)
    gains = bank.gains(YEAR, 6)
    corr = apply_driftslice(m, cur, gains, bank.edges)
    manual = []
    for tgt, c in zip(m.g_target, cur):
        k = (torch.ceil(tgt.double() / (cfg.gmax_us / 4)) - 1).clamp(0, 3).long()
        manual.append(c * gains[k])
    w_corr = m.logical_weight(corr)
    w_manual = (manual[0] - manual[1]) / m.params["scale_ratio"]
    ok2 = all(torch.allclose(a, b) for a, b in zip(corr, manual)) and torch.allclose(w_corr, w_manual)
    ok2 &= bool(((gains >= cfg.gain_min) & (gains <= cfg.gain_max)).all())
    record_check("Consistent gain application", ok2,
                 "DriftSlice multiplies G+ and G- by the gain of their TARGET-conductance class before differential "
                 "weight reconstruction (same location for every B, R_tot and model); gains within "
                 f"[{cfg.gain_min}, {cfg.gain_max}]; GDC gains act at the layer output (original)", scope)


# ===================== ORIGINAL preflight() (driftslice_core.py, CELL 20) =====================
def preflight(cfg: Config, device: torch.device):
    cfg = cfg.resolved()

    # DAR: zero at t/t0=1 and nonzero finite gradient later.
    w = torch.tensor([[0.2, -0.8, 0.4, -0.1]], device=device, requires_grad=True)
    ref = w.detach().clone()

    with random_stream(123, device):
        p0 = dar_penalty(w, ref, 0, 20.0, cfg)
    assert abs(float(p0.item())) < 1e-12

    with random_stream(124, device):
        p1 = dar_penalty(w, ref, 0, 60.0, cfg)
    grad = torch.autograd.grad(p1, w)[0]
    assert torch.isfinite(grad).all() and grad.abs().sum() > 0

    # FP32 -> analog transfer must preserve weights AND bias, with no optimizer step.
    digital_fixture = DigitalMLP().to(device=device, dtype=torch.float32).eval()
    model = transfer_pretrained_digital_to_analog(
        cfg, digital_fixture, 999, device).eval()

    # Regression test for AIHWKit state_dict containing non-Tensor tile metadata.
    import io
    original_state = model.state_dict()
    saved_state = cpu_cloned_state(model)
    assert set(saved_state) == set(original_state), 'State clone lost entries'
    assert hasattr(saved_state, '_metadata') == hasattr(original_state, '_metadata'), (
        'State clone lost PyTorch OrderedDict metadata')
    for key, value in original_state.items():
        if isinstance(value, torch.Tensor):
            assert saved_state[key].device.type == 'cpu', (key, 'not CPU')
            assert saved_state[key].data_ptr() != value.data_ptr(), (
                key, 'checkpoint unexpectedly aliases live weights')
    test_input = torch.rand(8, 1, 28, 28, device=device)
    with torch.inference_mode(), random_stream(812, device):
        before_reload = model(test_input)
    checkpoint_buffer = io.BytesIO()
    torch.save({'model': saved_state}, checkpoint_buffer)
    checkpoint_buffer.seek(0)
    restored_state = torch.load(checkpoint_buffer, map_location='cpu', weights_only=False)
    model.load_state_dict(restored_state['model'], strict=True)
    model.to(device)                                     # ADDED (C1)
    with torch.inference_mode(), random_stream(812, device):
        after_reload = model(test_input)
    torch.testing.assert_close(before_reload, after_reload, rtol=3e-5, atol=3e-5)
    print('PASS: AIHWKit non-Tensor state_dict checkpoint clone/save/load')

    # Snapshot deployment should reproduce eval-mode forward with the same physical weights.
    snap = snapshot_analog_model(model)
    eff = {k: v["weight"] for k, v in snap.items()}
    dep = SnapshotMLP(snap, eff, cfg).to(device).eval()
    x = torch.rand(8, 1, 28, 28, device=device)

    with torch.inference_mode(), random_stream(77, device):
        y0 = model(x)
    with torch.inference_mode(), random_stream(77, device):
        y1 = dep(x)

    torch.testing.assert_close(y1, y0, atol=3e-5, rtol=3e-4)

    # Absolute 20 s maps to elapsed zero; absolute 40 s includes 20 s drift.
    pcm = PCMPhysicalMatrix(snap["fc1"]["weight"], cfg, 321)
    g0 = pcm.read_conductances(20.0, 555, noiseless=True)
    g20 = pcm.read_conductances(40.0, 555, noiseless=True)
    assert any((a-b).abs().max().item() > 0 for a, b in zip(g20, g0))

    # DriftSlice must remain finite.
    bank = PCMClassReferenceBank(cfg, 1, 777, device)
    gains = bank.gains(20.0, 888)
    assert torch.isfinite(gains).all()

    read = pcm.read_conductances(20.0, 999)
    corr = apply_driftslice(pcm, read, gains, bank.edges)
    w_corr = pcm.logical_weight(corr)
    assert torch.isfinite(w_corr).all()

    schedule = []
    for epoch, t in enumerate(cfg.dar_times_s):
        time_term = t if cfg.dar_time_mode == "literal_seconds" else epoch + 1
        lam = cfg.dar_lambda_init / (epoch+1.0)**1.5 / (1.0 + cfg.dar_tc*time_term)
        schedule.append((epoch+1, t, lam))

    print("PASS: FP32 digital-to-analog weight transfer, DAR gradient, snapshot forward, PCM timing and DriftSlice finite-state.")
    print("DAR reconstruction assumptions:", {
        "eta": cfg.dar_eta,
        "delta_mode": cfg.dar_delta_mode,
        "g_units": cfg.dar_g_units,
        "time_mode": cfg.dar_time_mode,
    })
    print("First DAR-cycle lambda schedule:")
    print(pd.DataFrame(schedule, columns=["epoch_in_cycle", "t_s", "lambda_drift"]).to_string(index=False))


def preflight_resnet(cfg: Config, device: torch.device):
    """preflight() for ResNet-32: DAR, FP32->analog transfer (weights, fc bias, BatchNorm), AIHWKit state clone/
    save/load, weight-substitution deployment == AIHWKit forward == shared_mvm path, PCM timing, DriftSlice and
    saturation-safe GDC finite on conv tiles."""
    cfg = cfg.resolved()
    w = torch.tensor([[0.2, -0.8, 0.4, -0.1]], device=device, requires_grad=True)
    with random_stream(123, device):
        assert abs(dar_penalty(w, w.detach().clone(), 0, 20.0, cfg).detach().item()) < 1e-12
    with random_stream(124, device):
        g = torch.autograd.grad(dar_penalty(w, w.detach().clone() * 0.9, 0, 60.0, cfg), w)[0]
    assert torch.isfinite(g).all() and g.abs().sum() > 0
    digital = make_digital_resnet32(999).to(device)
    with torch.no_grad():                       # non-trivial BatchNorm state, so its transfer is really tested
        for m in digital.modules():
            if isinstance(m, nn.BatchNorm2d):
                m.running_mean.uniform_(-0.1, 0.1); m.running_var.uniform_(0.5, 1.5)
                m.weight.uniform_(0.5, 1.5); m.bias.uniform_(-0.1, 0.1)
        digital.stage3[4].conv1.weight[:3] = 3e-11       # dead filters, as produced by real FP32 training
    digital.eval()
    model = transfer_pretrained_resnet_to_analog(cfg, digital, device).eval()
    n_layers = len(model.layers())
    # regression test: dead filters must not create exploding tile-weight gradients (DAR / NSR)
    dead_layer = model.stage3[4].conv1
    tile, = list(dead_layer.analog_tiles())
    assert model.dead_filters_flushed["stage3.4.conv1"] >= 3 and torch.all(tile.get_mapping_scales()[:3] == 1.0)
    model.train()
    refs = {n: physical_weight_parameter(l).detach().clone() for n, l in model.layers().items()}
    with random_stream(5, device):
        out = model(torch.randn(8, 3, 32, 32, device=device))
        loss = F.cross_entropy(out, torch.arange(8, device=device) % 10) + nsr_penalty(model, cfg) + sum(
            dar_penalty(physical_weight_parameter(l), refs[n] * 0.9, 1, 60.0, cfg) for n, l in model.layers().items())
    loss.backward()
    gn = torch.nn.utils.clip_grad_norm_(model.parameters(), 1e9)
    assert torch.isfinite(gn), f"non-finite gradient norm {gn}"
    model.zero_grad(set_to_none=True)
    model.eval()
    print(f"PASS: dead FP32 filters mapped as exact zeros; CE+NSR+DAR gradient norm finite ({float(gn):.3g})")
    assert n_layers == 34, n_layers
    import io
    saved_state = cpu_cloned_state(model)
    x = torch.randn(16, 3, 32, 32, generator=torch.Generator().manual_seed(5)).to(device)
    with torch.inference_mode(), random_stream(812, device):
        before = model(x)
    buf = io.BytesIO()
    torch.save({'model': saved_state}, buf)
    buf.seek(0)
    model.load_state_dict(torch.load(buf, map_location='cpu', weights_only=False)['model'], strict=True)
    model.to(device)
    with torch.inference_mode(), random_stream(812, device):
        after = model(x)
    torch.testing.assert_close(before, after, rtol=3e-5, atol=3e-5)
    snap = snapshot_analog_resnet(model)
    ok, detail = verify_resnet_deployment(model, snap, cfg, [(x[:8], None), (x[8:], None)], device)
    assert ok, detail
    print("PASS: ResNet-32 deployment:", detail)
    pcm = PCMPhysicalMatrix(snap["stage2.0.conv1"]["weight"], cfg, 321)
    g0 = pcm.read_conductances(20.0, 555, noiseless=True)
    g40 = pcm.read_conductances(40.0, 555, noiseless=True)
    assert any((a - b).abs().max().item() > 0 for a, b in zip(g40, g0))
    bank = PCMClassReferenceBank(cfg, 1, 777, device)
    gains = bank.gains(20.0, 888)
    assert torch.isfinite(gains).all()
    w_corr = pcm.logical_weight(apply_driftslice(pcm, pcm.read_conductances(20.0, 999), gains, bank.edges))
    assert torch.isfinite(w_corr).all()
    io_p = make_hwa_rpu(cfg).forward
    probe = gdc_safe_probe_amplitude(pcm.logical_weight(g0), io_p, 11, device, cfg)
    safe = saturation_safe_gdc_gain(pcm.logical_weight(g0), w_corr, snap["stage2.0.conv1"], io_p, probe, 12, device, cfg)
    assert probe["converged"] and safe["valid"] and math.isfinite(safe["gain"]), (probe, safe)
    print(f"PASS: ResNet-32 ({n_layers} analog layers) FP32->analog transfer incl. BatchNorm, checkpoint round trip, "
          "deployment equivalence, PCM timing, DriftSlice and saturation-safe GDC on a conv tile.")


def run_all_preflight_checks(cfg: Config, device: torch.device, scope: str, arch: str = "mlp"):
    if arch == "mlp":
        preflight(replace(cfg, fast_mode=True), device)
    else:
        preflight_resnet(replace(cfg, fast_mode=True), device)
    verify_retention_grid(cfg, scope)
    verify_pcm_physics(cfg, device, scope)
    verify_driftslice_mechanics(cfg, device, scope)
    ok, msg = verify_aggregation_unit_test()
    record_check("Five-fold statistical aggregation", ok, "unit test: " + msg, scope)


print("Verification functions loaded.")

## 11. IEEE figures and tables

Every figure is generated from a source CSV written next to it. Bands show **±1 SD across training repetitions**, computed after averaging the programming realizations within each repetition. No smoothing is applied. The time axis is logarithmic. Colour identifies the training method and stays the same across all figures; markers and line styles add a second encoding for grayscale print. Fonts are embedded as TrueType (`pdf.fonttype = 42`) for IEEE PDF eXpress.

In [ ]:
# ============================ FIGURES ============================
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import FixedLocator, NullFormatter, MaxNLocator

IEEE_COL, IEEE_DCOL = 3.5, 7.16
LAYER_LABEL = {"conv": "conv (33 layers)", "fc": "fc"}
# Categorical identity (validated: adjacent-pair CVD dE >= 9.1, normal-vision dE >= 19.6; low-contrast
# slots carry markers + line styles and every figure ships a CSV table view).
METHOD_STYLE = {
    NSR:       dict(color="#2a78d6", marker="o", ls="-"),
    HWA:       dict(color="#eb6834", marker="s", ls="-"),
    DAR:       dict(color="#1baf7a", marker="^", ls="-"),
    ATT:       dict(color="#eda100", marker="D", ls="-"),
    ATT_ONLY:  dict(color="#e87ba4", marker="v", ls="-"),
    GAIN_ONLY: dict(color="#008300", marker="P", ls="-"),
    DIRECT:    dict(color="#6b6a66", marker="x", ls="--"),
}
INK, INK2, GRID = "#0b0b0b", "#52514e", "#d9d8d4"
TIME_TICKS = [20, 3600, DAY, 30 * DAY, YEAR, 5 * YEAR]
TIME_LABELS = ["20 s", "1 h", "1 d", "1 mo", "1 y", "5 y"]


def ieee_style():
    plt.rcParams.update({
        "font.family": "STIXGeneral", "mathtext.fontset": "stix", "font.size": 8,
        "axes.titlesize": 8, "axes.labelsize": 8, "legend.fontsize": 6.5, "xtick.labelsize": 7,
        "ytick.labelsize": 7, "axes.linewidth": 0.6, "lines.linewidth": 1.2, "lines.markersize": 3.5,
        "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2,
        "grid.color": GRID, "grid.linewidth": 0.4, "axes.grid": True, "legend.frameon": False,
        "pdf.fonttype": 42, "ps.fonttype": 42, "savefig.dpi": 300, "figure.dpi": 110,
        "axes.spines.top": False, "axes.spines.right": False,
    })


def save_figure(fig, stem: Path, source: pd.DataFrame, caption: str, provisional: bool = False):
    stem = Path(stem)
    stem.parent.mkdir(parents=True, exist_ok=True)
    if provisional:
        fig.text(0.5, 0.5, "PROVISIONAL - INCOMPLETE DATA", ha="center", va="center", rotation=25,
                 fontsize=16, color="#c0392b", alpha=0.25, transform=fig.transFigure)
    fig.savefig(stem.with_suffix(".pdf"), bbox_inches="tight")
    fig.savefig(stem.with_suffix(".png"), bbox_inches="tight", dpi=300)
    plt.close(fig)
    atomic_write_csv(source, stem.with_name(stem.name + "_source.csv"))
    atomic_write_text(stem.with_name(stem.name + "_caption.txt"), caption)
    print("saved", stem.with_suffix(".pdf").name)


def _time_axis(ax, label=True):
    ax.set_xscale("log")
    ax.xaxis.set_major_locator(FixedLocator(TIME_TICKS))
    ax.set_xticklabels(TIME_LABELS)
    ax.xaxis.set_minor_formatter(NullFormatter())
    if label:
        ax.set_xlabel("Time after programming")


def _band(ax, x, mean, sd, style, label, every=8):
    ax.plot(x, mean, color=style["color"], ls=style["ls"], marker=style["marker"], markevery=every,
            markerfacecolor="white", markeredgewidth=0.8, label=label)
    if np.isfinite(sd).any():
        ax.fill_between(x, mean - sd, mean + sd, color=style["color"], alpha=0.15, lw=0)


def fig1_training(digital_hist: pd.DataFrame, hwa_hist: pd.DataFrame, methods, out: Path, provisional=False, n_folds=5):
    ieee_style()
    fig, axes = plt.subplots(1, 3, figsize=(IEEE_DCOL, 2.25))
    src = []
    g = digital_hist.groupby("epoch")[["train_accuracy", "val_accuracy"]].agg(["mean", "std"])
    x = g.index.to_numpy()
    for col, st, lab in (("train_accuracy", dict(color=INK2, marker="s", ls="--"), "train"),
                         ("val_accuracy", dict(color=INK, marker="o", ls="-"), "validation")):
        _band(axes[0], x, g[(col, "mean")].to_numpy(), g[(col, "std")].fillna(0).to_numpy(), st, lab, every=5)
        src.append(pd.DataFrame({"panel": "a_fp32", "series": lab, "epoch": x,
                                 "mean": g[(col, "mean")].to_numpy(), "sd": g[(col, "std")].to_numpy()}))
    best = digital_hist.loc[digital_hist.groupby("fold").val_accuracy.idxmax()]   # first max = selection rule
    axes[0].scatter(best["epoch"], best["val_accuracy"], marker="v", s=14, color=INK, zorder=5, label="selected (per fold)")
    axes[0].set_title("(a) FP32 pretraining", loc="left")
    axes[0].set_xlabel("FP32 epoch")
    axes[0].set_ylabel("Accuracy (%)")
    for k, (panel, col, title) in enumerate((("b_hwa_val", "val_accuracy", "(b) HWA validation (3 noisy reads)"),
                                             ("c_hwa_train", "train_accuracy", "(c) HWA training (noisy forward)"))):
        ax = axes[k + 1]
        for m in methods:
            h = hwa_hist[hwa_hist.method == m]
            gg = h.groupby("epoch")[col].agg(["mean", "std"])
            _band(ax, gg.index.to_numpy(), gg["mean"].to_numpy(), gg["std"].fillna(0).to_numpy(),
                  METHOD_STYLE[m], PAPER_LABEL[m], every=max(1, len(gg) // 8))
            src.append(pd.DataFrame({"panel": panel, "series": m, "epoch": gg.index, "mean": gg["mean"], "sd": gg["std"]}))
            if col == "val_accuracy":
                sel = h[h.epoch == h.groupby("fold").best_epoch.transform("last")]
                sel = sel.groupby("fold").tail(1)
                ax.scatter(sel.epoch, sel.val_accuracy, marker="v", s=14, color=METHOD_STYLE[m]["color"],
                           edgecolor=INK, linewidths=0.4, zorder=5)
                src.append(pd.DataFrame({"panel": "b_selected_epochs", "series": m, "epoch": sel.epoch.values,
                                         "mean": sel.val_accuracy.values, "sd": np.nan, "fold": sel.fold.values}))
        ax.set_title(title, loc="left")
        ax.set_xlabel("HWA epoch")
    for ax in axes:
        lo = min((float(np.nanmin(l.get_ydata())) for l in ax.get_lines() if len(l.get_ydata())), default=90.0)
        ax.set_ylim(max(0.0, lo - 0.5), 100.2)
    for ax in axes:
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    axes[0].legend(loc="lower right")
    h, l = axes[2].get_legend_handles_labels()
    fig.legend(h, l, loc="lower center", ncol=len(l), bbox_to_anchor=(0.5, -0.01), fontsize=6.5)
    fig.tight_layout(w_pad=0.6, rect=(0, 0.08, 1, 1))
    cap = (f"Fig. 1. (a) FP32 pretraining (30 epochs) and (b, c) AnalogAdam HWA fine-tuning (154 epochs, from the same "
           f"FP32 checkpoint) for each method; mean ± SD across {n_folds} training repetitions. Triangles mark the "
           "validation-selected checkpoint of every repetition. HWA validation accuracy is the mean of three noisy "
           "AIHWKit reads on raw inputs; HWA training accuracy is measured in training mode (weight noise, and the "
           "training-only input gate for Drift-Attention).")
    save_figure(fig, out / "fig1_training_validation", pd.concat(src, ignore_index=True), cap, provisional)


def _retention_series(summary, method, comp):
    z = summary[(summary.method == method) & (summary.compensation == comp)].sort_values("t_s")
    return z.t_s.to_numpy(), z["mean"].to_numpy(), z["sd"].to_numpy()


def fig2_main_retention(summary: pd.DataFrame, methods, out: Path, provisional=False, F=5, R=30,
                        dataset="MNIST", n_times=75):
    ieee_style()
    fig, ax = plt.subplots(figsize=(IEEE_COL, 2.9))
    curves = [(DIRECT, "none", "FP32-Direct, no comp."), (HWA, "none", "Standard HWA"),
              (DAR, "none", "DAR-HWA (Drift Only)"), (ATT, "none", "Drift-Attention-HWA"),
              (NSR, "DS-val", "NSR-HWA + DriftSlice (proposed)")]
    src = []
    for m, comp, lab in curves:
        if m != DIRECT and m not in methods:
            continue
        x, y, s = _retention_series(summary, m, comp)
        if not len(x):
            continue
        _band(ax, x, y, s, METHOD_STYLE[m], lab)
        src.append(summary[(summary.method == m) & (summary.compensation == comp)].assign(curve=lab))
    _time_axis(ax)
    ax.set_ylabel("Test error (%)")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=2, fontsize=6)
    fig.tight_layout()
    cap = (f"Fig. 2. {dataset} test error versus PCM retention time ({n_times} timestamps, 20 s to 5 y). Standard HWA, DAR-HWA and "
           "Drift-Attention-HWA without drift compensation are compared with the proposed NSR-HWA + DriftSlice "
           "(validation-selected B). This is an overall-pipeline comparison: the methods differ in both training "
           f"and compensation (see Fig. 3 for matched compensation). FP32-Direct (no HWA, no compensation) is shown "
           f"dashed for reference. Mean ± SD across {F} training repetitions after averaging {R} programming "
           f"realizations each.")
    save_figure(fig, out / "fig2_four_method_retention", pd.concat(src, ignore_index=True), cap, provisional)


def fig3_matched(summary: pd.DataFrame, methods, out: Path, provisional=False, F=5, R=30):
    ieee_style()
    comps = [("none", "(a) No compensation"), ("DS-B1", "(b) DriftSlice, B = 1"),
             ("DS-val", "(c) DriftSlice, validation-selected B"), ("GDC-safe", "(d) Saturation-safe GDC")]
    fig, axes = plt.subplots(2, 2, figsize=(IEEE_DCOL, 4.3), sharex=True, sharey=True)
    src = []
    for ax, (comp, title) in zip(axes.flat, comps):
        for m in (DIRECT,) + tuple(methods):
            x, y, s = _retention_series(summary, m, comp)
            if len(x):
                _band(ax, x, y, s, METHOD_STYLE[m], PAPER_LABEL[m])
                src.append(summary[(summary.method == m) & (summary.compensation == comp)])
        ax.set_title(title, loc="left")
        _time_axis(ax, label=ax in axes[1])
    for ax in axes[:, 0]:
        ax.set_ylabel("Test error (%)")
    h, l = axes[0, 0].get_legend_handles_labels()
    fig.legend(h, l, loc="upper center", ncol=len(l), bbox_to_anchor=(0.5, 1.0), fontsize=6.5)
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    cap = ("Fig. 3. Matched-compensation retention: every training method evaluated under identical correction "
           "conditions on paired PCM programming realizations. (a) none, (b) DriftSlice with a single class (B = 1), "
           "(c) DriftSlice with B selected on validation data (shared by all methods within a repetition), "
           f"(d) saturation-safe per-layer GDC. Mean ± SD across {F} training repetitions (each averaged over {R} "
           f"realizations).")
    save_figure(fig, out / "fig3_matched_compensation", pd.concat(src, ignore_index=True), cap, provisional)


def ablation_differences(abl: pd.DataFrame, split="test", expected_folds: Optional[int] = None,
                         expected_reals: Optional[int] = None) -> pd.DataFrame:
    """E(B,Rtot) - E(1,Rtot), paired per (fold, realization), averaged per fold, then mean/SD across folds.
    Strict: a fold with missing realizations, or a cell with missing folds, yields NaN (never a smaller sample)."""
    a = abl[(abl.split == split) & (abl.compensation == "DriftSlice")]
    base = a[a.bins == 1][["method", "fold", "realization", "R_tot", "error"]].rename(columns={"error": "e1"})
    d = a.merge(base, on=["method", "fold", "realization", "R_tot"], how="left", validate="many_to_one")
    d["diff"] = d.error - d.e1
    per_fold = d.groupby(["method", "R_tot", "bins", "fold"])["diff"].agg(
        diff=_mean_strict, n_real="size").reset_index()
    if expected_reals is not None:
        per_fold.loc[per_fold.n_real != expected_reals, "diff"] = np.nan
    rows = []
    for (m, R, B), z in per_fold.groupby(["method", "R_tot", "bins"]):
        if expected_folds is not None and len(z) != expected_folds:
            mean = sd = lo = hi = p = float("nan")
        else:
            mean, sd, lo, hi, p = t_ci(z["diff"].to_numpy())
        rows.append({"method": m, "R_tot": R, "bins": B, "R_b": R // B, "mean_diff_pp": mean, "sd_across_folds_pp": sd,
                     "ci95_low": lo, "ci95_high": hi, "p_paired_t": p, "n_folds": len(z)})
    return pd.DataFrame(rows)


def fig4_ablation(diff: pd.DataFrame, methods, out: Path, provisional=False, F=5, R=30, K=4):
    ieee_style()
    ramp = {16: "#9ec5f4", 64: "#2a78d6", 256: "#0d3c78"}       # sequential (one hue, light -> dark)
    marks = {16: "o", 64: "s", 256: "^"}

    def panel(ax, m):
        z = diff[diff.method == m]
        for k, R in enumerate(sorted(z.R_tot.unique())):
            q = z[z.R_tot == R].sort_values("bins")
            xs = np.log2(q.bins.to_numpy()) + (k - 1) * 0.06
            ax.errorbar(xs, q.mean_diff_pp, yerr=q.sd_across_folds_pp, color=ramp.get(R, INK), marker=marks.get(R, "o"),
                        ms=3.5, lw=1.0, capsize=2, elinewidth=0.7, markerfacecolor="white", label=f"$R_{{tot}}$ = {R}")
        ax.axhline(0, color=INK2, lw=0.6)
        ax.set_xticks(np.log2([1, 2, 4, 8]))
        ax.set_xticklabels(["1", "2", "4", "8"])
        ax.set_xlabel("Conductance classes B")

    fig, ax = plt.subplots(figsize=(IEEE_COL, 2.3))
    panel(ax, NSR if NSR in methods else methods[0])
    ax.set_ylabel(r"$E(B,R_{tot})-E(1,R_{tot})$ (pp)")
    ax.legend(loc="best", fontsize=6.5)
    fig.tight_layout()
    cap = (f"Fig. 4. Five-year test-error difference between B conductance classes and the single-class control "
           f"(B = 1) for NSR-HWA, at a fixed total reference budget R_tot (R_b = R_tot/B cells per class, K = {K} reads, "
           f"gain applied per conductance branch). Paired programming realizations and checkpoints across all B and "
           f"R_tot; mean ± SD across {F} training repetitions after averaging {R} realizations. Negative values mean "
           f"B > 1 helps. The test split is used only for this report; B is deployed from validation data.")
    save_figure(fig, out / "fig4_conductance_class_ablation", diff[diff.method == (NSR if NSR in methods else methods[0])], cap, provisional)
    fig, axes = plt.subplots(1, len(methods), figsize=(IEEE_DCOL, 2.1), sharey=True)
    for ax, m in zip(np.atleast_1d(axes), methods):
        panel(ax, m)
        ax.set_title(PAPER_LABEL[m], loc="left", fontsize=7)
    np.atleast_1d(axes)[0].set_ylabel(r"$\Delta E$ vs. B = 1 (pp)")
    np.atleast_1d(axes)[0].legend(fontsize=6)
    fig.tight_layout()
    save_figure(fig, out / "figS4_conductance_class_ablation_all_methods", diff, cap.replace("for NSR-HWA", "for every training method"), provisional)


def fig5_distributions(npz_path: Path, method: str, selected_bins: int, out: Path, provisional=False, stem="fig5_distributions",
                       layers=("fc1", "fc2")):
    ieee_style()
    d = np.load(npz_path)
    fig, axes = plt.subplots(2, 2, figsize=(IEEE_DCOL, 3.9))
    src = []
    cols = ["#2a78d6", "#eb6834", "#1baf7a"]
    for r, layer in enumerate(layers):
        ax = axes[r, 0]
        bins = np.linspace(0, 27, 109)
        for (key, lab), c, ls in zip((("G_read_20s_uS", "programmed, read at 20 s"), ("G_read_5y_uS", "drifted, 5 y"),
                                      ("G_corrected_5y_selB_uS", f"DriftSlice-corrected, 5 y (B = {selected_bins})")),
                                     cols, ("-", "--", ":")):
            v = d[f"{layer}__{key}"].astype(np.float32)
            h, e = np.histogram(v, bins=bins, density=True)
            ax.step(e[:-1], h, where="post", color=c, ls=ls, lw=1.0, label=lab)
            src.append(pd.DataFrame({"layer": layer, "panel": "conductance_density", "series": key,
                                     "bin_left_uS": e[:-1], "density": h}))
        ax.set_yscale("log")
        ax.set_ylabel(f"{LAYER_LABEL.get(layer, layer)}: density (1/µS)")
        ax.set_xlabel("Active-device conductance (µS)" if r == 1 else "")
        ax = axes[r, 1]
        for (key, lab), c, ls in zip((("W_20s_norm", "20 s"), ("W_5y_norm", "5 y, no comp."),
                                      ("W_5y_selB_norm", f"5 y, DriftSlice (B = {selected_bins})")), cols, ("-", "--", ":")):
            v = np.sort(d[f"{layer}__{key}"].astype(np.float32))
            q = np.linspace(0, 1, len(v))
            sub = np.linspace(0, len(v) - 1, 2000).astype(int)
            ax.plot(v[sub], q[sub], color=c, ls=ls, lw=1.0, label=lab)
            src.append(pd.DataFrame({"layer": layer, "panel": "weight_cdf", "series": key,
                                     "w_norm": v[sub], "cdf": q[sub]}))
        ax.set_ylabel(f"{LAYER_LABEL.get(layer, layer)}: CDF")
        ax.set_xlabel(r"Effective weight $/\max|W_{target}|$" if r == 1 else "")
    axes[0, 0].legend(fontsize=6, loc="upper right")
    axes[0, 1].legend(fontsize=6, loc="lower right")
    fig.tight_layout()
    cap = (f"Fig. 5. {PAPER_LABEL[method]}: (left) density of active-device PCM conductances (inactive differential "
           "partners excluded) after programming (read at 20 s), after 5 years of drift, and after DriftSlice "
           f"class-gain correction (validation-selected B = {selected_bins}); (right) CDF of effective weights normalized "
           "by the target maximum" + (" of each layer; all 33 conv layers are pooled" if "conv" in layers else "") +
           ". One representative programming realization (fold 0, realization 0).")
    save_figure(fig, out / stem, pd.concat(src, ignore_index=True), cap, provisional)


print("Figure functions loaded.")

In [ ]:
# ============================ TABLES, AUDIT, REPRODUCIBILITY ============================
TABLE1_COMPS = [("none", "None"), ("GDC-safe", "Sat.-safe GDC"), ("DS-B1", "DriftSlice $B{=}1$"),
                ("DS-val", "DriftSlice val.\\ $B$")]


def _fmt(mean, sd, nd=2):
    if mean is None or not np.isfinite(mean):
        return "n/a"
    return f"{mean:.{nd}f} ± {sd:.{nd}f}" if np.isfinite(sd) else f"{mean:.{nd}f}"


def _tex(mean, sd, nd=2):
    if mean is None or not np.isfinite(mean):
        return "n/a"
    return f"${mean:.{nd}f}\\pm{sd:.{nd}f}$" if np.isfinite(sd) else f"${mean:.{nd}f}$"


def make_table1(ret5: pd.DataFrame, models, F: int, R: int, selected_bins: dict, out: Path, tag="", n_test: int = 10000,
                with_original_gdc: bool = True, dataset: str = "MNIST", arch_label: str = "784-250-10 MLP"):
    s = two_stage_summary(ret5, ["method", "compensation"], expected_folds=F, expected_reals=R)
    rows = []
    for m in models:
        for comp, _ in TABLE1_COMPS + ([("GDC", "GDC (orig.)")] if with_original_gdc else []):
            z = s[(s.method == m) & (s.compensation == comp)]
            mean = float(z["mean"].iloc[0]) if len(z) else float("nan")
            sd = float(z["sd"].iloc[0]) if len(z) else float("nan")
            rows.append({"method": m, "label": PAPER_LABEL[m], "compensation": comp, "mean_error_pct": mean,
                         "sd_across_folds_pct": sd, "n_folds": int(z.n_folds.iloc[0]) if len(z) else 0,
                         "realizations_per_fold": int(z.realizations_per_fold_min.iloc[0]) if len(z) else 0,
                         "invalid_records": int(z.invalid_records.iloc[0]) if len(z) else 0,
                         "cell": _fmt(mean, sd)})
    t = pd.DataFrame(rows)
    atomic_write_csv(t, out / f"TableI_five_year_test_error{tag}.csv")
    hdr = " & ".join(["Training method"] + [h for _, h in TABLE1_COMPS]) + r" \\"
    body = []
    for m in models:
        cells = [_tex(*t[(t.method == m) & (t.compensation == c)][["mean_error_pct", "sd_across_folds_pct"]].iloc[0])
                 for c, _ in TABLE1_COMPS]
        body.append(" & ".join([PAPER_LABEL[m].replace("(proposed)", "(prop.)")] + cells) + r" \\")
    bsel = ", ".join(f"{selected_bins[f]}" for f in sorted(selected_bins))
    note = (f"Mean $\\pm$ SD across $F{{=}}{F}$ training repetitions of the per-repetition mean over $R{{=}}{R}$ PCM "
            f"programming realizations; {arch_label}, {dataset} test set ({n_test:,} images), $t=5$\\,y. FP32-Direct: no HWA steps. "
            f"Validation-selected $B$ per repetition: {bsel}. Sat.-safe GDC: per-layer calibration with a "
            f"non-saturating probe; n/a = invalid calibration in at least one realization.")
    tex = "\n".join([
        r"\begin{table}[!t]", r"\renewcommand{\arraystretch}{1.15}", r"\caption{Five-Year Test Error (\%)}",
        r"\label{tab:five_year_error}", r"\centering", r"\footnotesize", r"\setlength{\tabcolsep}{3.5pt}",
        r"\begin{tabular}{lcccc}", r"\hline", hdr, r"\hline", *body, r"\hline", r"\end{tabular}",
        r"\\[2pt]\parbox{\columnwidth}{\scriptsize " + note + "}", r"\end{table}"])
    atomic_write_text(out / f"TableI_five_year_test_error{tag}.tex", tex)
    return t


TABLE2_ROWS = [
    # key, label, unit, kind ('layer' or 'network'), definition
    ("signal_energy_mean_g2", "Weight-signal energy $\\langle g^2\\rangle$", "--", "layer",
     "Mean squared normalized weight, g=|W|/max|W| of the PCM tile weights (as in nsr_penalty)."),
    ("prog_noise_surrogate_mean", "Programming-noise surrogate", "$(G/G_{max})^2$", "layer",
     "Mean sigma_p(g)^2, sigma_p=(0.26348+1.9650g-1.1731g^2)/25 (nsr_penalty)."),
    ("drift_noise_surrogate_mean", "Drift-noise surrogate", "$(G/G_{max})^2$", "layer",
     "Mean g^2 sigma_nu(g)^2 <L^2>, <L^2>=ln^2(5y/20s)/3 (nsr_penalty)."),
    ("read_noise_surrogate_mean", "Read-noise surrogate", "$(G/G_{max})^2$", "layer",
     "Mean g^2 q_s(g)^2 <R>, q_s=min(0.0088/g^0.65,0.2) (nsr_penalty)."),
    ("nsr_layer", "NSR value (per layer, w/o $\\lambda$)", "--", "layer",
     "(sum of variances + sp0*N)/sum g^2; nsr_penalty = lambda*mean over layers."),
    ("G_active_median_uS", "Active conductance, median", "$\\mu$S", "layer",
     "Median target conductance of active devices (inactive differential partners excluded)."),
    ("G_active_mean_uS", "Active conductance, mean", "$\\mu$S", "layer", "Mean target conductance of active devices."),
    ("low_state_occupancy_pct", "Low-state occupancy ($G_T<2.5\\,\\mu$S)", "\\%", "layer",
     "Share of active devices with target conductance below the PRESPECIFIED 2.5 uS (=0.1 G_max)."),
    ("ref_gain_B1", "Reference gain $\\gamma$ (5 y), mean", "--", "network",
     "B=1 class gain from R_tot=64 reference cells, K=4 reads; identical for both methods (shared bank realization)."),
    ("ref_gain_B1_sd", "Reference-gain variability (SD over realizations)", "--", "network",
     "Within-repetition SD of gamma across the 30 programming realizations."),
    ("ref_gain_rel_error_rms_pct", "Reference-estimation error (RMS)", "\\%", "layer",
     "RMS over realizations of (gamma - gamma*)/gamma*, gamma* = sum G_prog / sum G_prog (t/t0)^-nu over the "
     "layer's active main-array devices (noise-free oracle)."),
    ("weight_nrmse_B1_corrected_pct", "Effective-weight NRMSE (B=1)", "\\% of $\\max|W|$", "layer",
     "RMS(W_corrected(5y) - W_target)/max|W_target|."),
    ("residual_correction_error_pct", "Residual correction error", "\\% of $\\max|W|$", "layer",
     "RMS(W_corrected(5y) - W_programmed(20 s, no read noise))/max|W_target| (excludes programming error)."),
    ("test_error_B1_5y_pct", "Five-year test error (B=1)", "\\%", "network", "Test error at 5 y with B=1 DriftSlice."),
]


def make_table2(diag: pd.DataFrame, F: int, R: int, out: Path, tag="", layers=("fc1", "fc2")):
    l0, l1 = layers
    d = diag[diag.method.isin([HWA, NSR])].copy()
    per_fold = []
    for (m, layer, f), z in d.groupby(["method", "layer", "fold"]):
        if len(z) != R:
            continue
        row = {"method": m, "layer": layer, "fold": f}
        for key, *_ in TABLE2_ROWS:
            if key == "ref_gain_B1_sd":
                row[key] = float(z.ref_gain_B1.std(ddof=1)) if R > 1 else float("nan")
            elif key == "ref_gain_rel_error_rms_pct":
                row[key] = float(np.sqrt(np.mean(np.square(z.ref_gain_rel_error_pct))))
            else:
                row[key] = float(z[key].mean())
        row["ref_gain_rel_error_bias_pct"] = float(z.ref_gain_rel_error_pct.mean())
        per_fold.append(row)
    pf = pd.DataFrame(per_fold)
    if pf.empty:
        print("Table II: no (method, layer, fold) group has all realizations; not generated.")
        return pd.DataFrame(), pf
    atomic_write_csv(pf, out / f"TableII_per_fold_values{tag}.csv")
    rows = []
    for key, label, unit, kind, definition in TABLE2_ROWS:
        rec = {"metric": key, "label": label, "unit": unit, "level": kind, "definition": definition}
        for m in (HWA, NSR):
            for layer in layers:
                v = pf[(pf.method == m) & (pf.layer == layer)][key].to_numpy()
                mean, sd = (float(v.mean()), float(v.std(ddof=1)) if len(v) > 1 else float("nan")) if len(v) == F else (float("nan"), float("nan"))
                rec[f"{m}|{layer}|mean"], rec[f"{m}|{layer}|sd"] = mean, sd
        rows.append(rec)
    # paired 5-y error difference NSR - HWA (network)
    net = pf[pf.layer == l0].pivot(index="fold", columns="method", values="test_error_B1_5y_pct")
    diff = (net[NSR] - net[HWA]).to_numpy() if {NSR, HWA} <= set(net.columns) else np.array([])
    mean, sd, lo, hi, p = t_ci(diff) if len(diff) == F else (float("nan"),) * 5
    rows.append({"metric": "delta_test_error_NSR_minus_HWA_B1_5y_pp", "label": "$\\Delta$ five-year error, NSR$-$HWA (B=1)",
                 "unit": "pp", "level": "network",
                 "definition": "Paired (same device seeds) difference, averaged over realizations per repetition.",
                 "diff_mean": mean, "diff_sd": sd, "ci95_low": lo, "ci95_high": hi, "p_paired_t": p})
    t = pd.DataFrame(rows)
    atomic_write_csv(t, out / f"TableII_mechanism_diagnostics{tag}.csv")
    nd = {"signal_energy_mean_g2": 3, "prog_noise_surrogate_mean": 5, "drift_noise_surrogate_mean": 5,
          "read_noise_surrogate_mean": 5, "nsr_layer": 3, "ref_gain_B1": 3, "ref_gain_B1_sd": 4}
    lines = []
    for r in rows[:-1]:
        k = nd.get(r["metric"], 2)
        if r["level"] == "network":
            cells = [r"\multicolumn{2}{c}{" + _tex(r[f"{m}|{l0}|mean"], r[f"{m}|{l0}|sd"], k) + "}" for m in (HWA, NSR)]
        else:
            cells = [_tex(r[f"{m}|{l}|mean"], r[f"{m}|{l}|sd"], k) for m in (HWA, NSR) for l in layers]
        lines.append(" & ".join([r["label"], r["unit"]] + cells) + r" \\")
    dl = rows[-1]
    lines.append(" & ".join([dl["label"], "pp", r"\multicolumn{4}{c}{" + _tex(dl["diff_mean"], dl["diff_sd"]) +
                             (f" (95\\% CI [{dl['ci95_low']:.2f}, {dl['ci95_high']:.2f}])" if np.isfinite(dl["ci95_low"]) else "") + "}"]) + r" \\")
    tex = "\n".join([
        r"\begin{table*}[!t]", r"\renewcommand{\arraystretch}{1.12}",
        r"\caption{Matched $B{=}1$ Mechanism Diagnostics: Standard HWA vs.\ NSR-HWA (5-Year Retention)}",
        r"\label{tab:mechanism}", r"\centering", r"\footnotesize",
        r"\begin{tabular}{llcccc}", r"\hline",
        r" & & \multicolumn{2}{c}{Standard HWA + DS($B{=}1$)} & \multicolumn{2}{c}{NSR-HWA + DS($B{=}1$)} \\",
        rf"Metric & Unit & {l0} & {l1} & {l0} & {l1} \\", r"\hline", *lines, r"\hline", r"\end{tabular}",
        r"\\[2pt]\parbox{\textwidth}{\scriptsize Mean $\pm$ SD across $F{=}" + str(F) + r"$ training repetitions (each "
        r"averaged over $R{=}" + str(R) + r"$ programming realizations where the metric depends on the realization). "
        r"All values are simulated (AIHWKit PCMLikeNoiseModel); surrogate terms use the exact formulas of the NSR "
        r"regularizer. The reference bank does not depend on the network, so $\gamma$ is identical for both methods "
        r"by construction. " + (r"conv = all 33 conv layers pooled (per-weight means over all conv weights; NSR = mean "
                                r"of the per-layer NSR; weight errors normalized per layer). " if "conv" in layers else "") +
        r"Definitions: TableII\_mechanism\_diagnostics.csv.}", r"\end{table*}"])
    atomic_write_text(out / f"TableII_mechanism_diagnostics{tag}.tex", tex)
    return t, pf


def claim_audit(ret5: pd.DataFrame, models, F: int, abl_diff: Optional[pd.DataFrame]) -> pd.DataFrame:
    claims = []
    if NSR in models and HWA in models:
        claims.append(("Overall pipeline: NSR-HWA + DriftSlice(val B) < Standard HWA, no compensation",
                       {"method": NSR, "compensation": "DS-val"}, {"method": HWA, "compensation": "none"}))
        for comp in ("none", "GDC-safe", "DS-B1", "DS-val"):
            claims.append((f"Independent NSR effect (matched '{comp}'): NSR-HWA < Standard HWA",
                           {"method": NSR, "compensation": comp}, {"method": HWA, "compensation": comp}))
    for other in (DAR, ATT):
        if NSR in models and other in models:
            claims.append((f"Overall pipeline: NSR-HWA + DriftSlice(val B) < {PAPER_LABEL[other]}, no compensation",
                           {"method": NSR, "compensation": "DS-val"}, {"method": other, "compensation": "none"}))
            claims.append((f"Matched DriftSlice(val B): NSR-HWA < {PAPER_LABEL[other]}",
                           {"method": NSR, "compensation": "DS-val"}, {"method": other, "compensation": "DS-val"}))
        if HWA in models and other in models:
            claims.append((f"{PAPER_LABEL[other]} < Standard HWA (no compensation)",
                           {"method": other, "compensation": "none"}, {"method": HWA, "compensation": "none"}))
    for m in models:
        claims.append((f"{PAPER_LABEL[m]}: DriftSlice(val B) < DriftSlice(B=1)",
                       {"method": m, "compensation": "DS-val"}, {"method": m, "compensation": "DS-B1"}))
        claims.append((f"{PAPER_LABEL[m]}: DriftSlice(B=1) < no compensation",
                       {"method": m, "compensation": "DS-B1"}, {"method": m, "compensation": "none"}))
        claims.append((f"{PAPER_LABEL[m]}: DriftSlice(val B) < saturation-safe GDC",
                       {"method": m, "compensation": "DS-val"}, {"method": m, "compensation": "GDC-safe"}))
    rows = []
    for text, a, b in claims:
        try:
            d = paired_difference(ret5, a, b)
        except ScientificCheckError as exc:
            rows.append({"claim": text, "verdict": "NOT EVALUABLE", "note": str(exc)})
            continue
        vals = d["diff"].to_numpy()
        mean, sd, lo, hi, p = t_ci(vals) if len(vals) == F else (float("nan"),) * 5
        n_better = int((vals < 0).sum())
        if len(vals) == F and np.isfinite(vals).all() and np.all(vals == 0):
            verdict = "NO DIFFERENCE (identical in every repetition)"
        elif len(vals) == F and np.isfinite(vals).all() and F > 1 and float(np.std(vals)) == 0.0:
            verdict = "DEGENERATE (zero across-repetition variance; no inference)"
        elif not np.isfinite(lo):
            verdict = "NOT EVALUABLE (missing/invalid data)"
        elif hi < 0:
            verdict = "SUPPORTED"
        elif lo > 0:
            verdict = "CONTRADICTED (significantly worse)"
        else:
            verdict = "INCONCLUSIVE (95% CI includes 0)"
        rows.append({"claim": text, "n_folds": len(vals), "A": json.dumps(a), "B": json.dumps(b), "mean_diff_A_minus_B_pp": mean,
                     "sd_across_folds_pp": sd, "ci95_low": lo, "ci95_high": hi, "p_paired_t": p,
                     "folds_where_A_better": f"{n_better}/{len(vals)}", "verdict": verdict})
    if abl_diff is not None and len(abl_diff):
        for r in abl_diff[abl_diff.bins > 1].itertuples():
            lo, hi = r.ci95_low, r.ci95_high
            verdict = ("SUPPORTED" if hi < 0 else "CONTRADICTED (significantly worse)" if lo > 0
                       else "INCONCLUSIVE (95% CI includes 0)") if np.isfinite(lo) else "NOT EVALUABLE"
            rows.append({"claim": f"{PAPER_LABEL[r.method]}: B={r.bins} < B=1 at R_tot={r.R_tot} (5 y, test)",
                         "n_folds": int(r.n_folds),
                         "mean_diff_A_minus_B_pp": r.mean_diff_pp, "sd_across_folds_pp": r.sd_across_folds_pp,
                         "ci95_low": lo, "ci95_high": hi, "p_paired_t": r.p_paired_t, "verdict": verdict})
    out = pd.DataFrame(rows)
    if "p_paired_t" in out and out.p_paired_t.notna().any():
        p = out.p_paired_t.fillna(1.0).to_numpy()
        order = np.argsort(p)
        adj = np.empty_like(p)
        running = 0.0
        for rank, i in enumerate(order):
            running = max(running, min(1.0, (len(p) - rank) * p[i]))
            adj[i] = running
        out["p_holm_all_claims"] = adj
    return out


def reproducibility_record(exp: Experiment, sel_bins: dict) -> dict:
    import aihwkit
    from aihwkit.simulator.rpu_base import cuda as rpu_cuda
    cfg = exp.cfg
    rec = {
        "generated": time.strftime("%Y-%m-%d %H:%M:%S"),
        "runtime": {"python": sys.version, "platform": platform.platform(), "torch": torch.__version__,
                    "torch_cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version(),
                    "gpu": _gpu_name(), "aihwkit": aihwkit.__version__, "aihwkit_cuda_compiled": bool(rpu_cuda.is_compiled()),
                    "in_colab": IN_COLAB, "sessions": exp.manifest.get("sessions", []),
                    "aihwkit_install_route": SETUP_REPORT.get("aihwkit_route"),
                    "aihwkit_wheel": SETUP_REPORT.get("aihwkit_wheel"),
                    "compute_capability": SETUP_REPORT.get("compute_capability"),
                    "tf32_matmul": bool(torch.backends.cuda.matmul.allow_tf32),
                    "float32_matmul_precision": torch.get_float32_matmul_precision()},
        "source_hashes": exp.fp["code_hashes"], "fingerprints": {k: v for k, v in exp.fp.items() if k != "code_hashes"},
        "original_sources": ORIGINAL_SOURCE_PROVENANCE,
        "config": json.loads(json.dumps(asdict(cfg), default=str)),
        "architecture": exp.arch, "dataset": ops(exp).dataset,
        "resnet_spec": asdict(RESNET_SPEC) if exp.arch != "mlp" else None,
        "dataset_sha256_decompressed": exp.data_hashes,
        "statistics": "two-stage: mean over R realizations within fold; mean and SD (ddof=1) across F folds",
        "calibration_modes": COMP_LABEL,
        "reference_bank": {"R_tot_main": cfg.reference_cell_budget, "K_reads": cfg.reference_read_repeats,
                           "gain_clip": [cfg.gain_min, cfg.gain_max], "floor_uS": cfg.reference_floor_us,
                           "ablation_R_tot": list(cfg.ablation_reference_budgets), "ablation_B": list(cfg.ablation_bins)},
        "retention_timestamps_s": [float(t) for t in exact_retention_times(cfg)],
        "pcm_seed_rule": "root = seed + fold*1e6 + realization*1e4; main +10(+101/+202), bank +30, ref20 read +40, "
                         "read +1e5+100*ti, gain read +5e5+101*ti, eval +7e5+31*ti, GDC and saturation-safe-GDC "
                         "calibration reads +8.5e5+31*ti(+1 for fc2); ADDED probe search / saturation diagnostic: 5e7 + root",
        "optimizer": {"fp32": "torch.optim.Adam(lr=%g, wd=%g) + OneCycleLR(max_lr=%g)" % (cfg.digital_lr, cfg.weight_decay, cfg.digital_max_lr),
                      "hwa": "aihwkit.optim.AnalogAdam(lr=%g, wd=0) + regroup_param_groups + OneCycleLR(max_lr=%g) + "
                             "decoupled wd %g on tile weights" % (cfg.lr, cfg.max_lr, cfg.weight_decay),
                      "attention_gate": "torch.optim.Adam(lr=%g) (training-only digital gate)" % cfg.attention_lr},
        "folds": {},
    }
    for fold in range(cfg.train_repeats):
        fdir = exp.train_dir / f"fold{fold}"
        f = {"seed": cfg.seed + fold, "split_seed": cfg.seed + fold, "loader_seed": cfg.seed + fold + 1000,
             "selected_B": sel_bins.get(fold)}
        if (fdir / "digital_best_validation.pt").exists():
            d = torch_load(fdir / "digital_best_validation.pt")
            f["fp32_selected_epoch"] = d["best_epoch"]
            f["fp32_identity"] = d["model_identity"]
        for m in exp.methods:
            p = fdir / f"{m}_best_validation.pt"
            if p.exists():
                q = torch_load(p)
                f[m] = {"selected_epoch": q["selected_best_validation_epoch"], "completed_epochs": q["analog_epochs_completed"],
                        "optimizer_steps": q["optimizer_steps"], "identity": q["model_identity"]}
        sp = fdir / "split.pt"
        if sp.exists():
            f["split_digest"] = torch_load(sp)["digest"]
        f["pcm_realization_roots"] = [cfg.seed + fold * 1_000_000 + r * 10_000 for r in range(cfg.realizations)]
        rec["folds"][fold] = f
    return rec


def load_histories(exp: Experiment):
    dig, hwa = [], []
    for fold in range(exp.cfg.train_repeats):
        fdir = exp.train_dir / f"fold{fold}"
        p = fdir / "digital_best_validation.pt"
        if p.exists():
            dig.append(pd.DataFrame(torch_load(p)["history"]))
        for m in exp.methods:
            q = fdir / f"{m}_best_validation.pt"
            if q.exists():
                hwa.append(pd.DataFrame(torch_load(q)["history"]))
    return (pd.concat(dig, ignore_index=True) if dig else pd.DataFrame(),
            pd.concat(hwa, ignore_index=True) if hwa else pd.DataFrame())


def generate_paper_results(exp: Experiment, allow_provisional: bool = True):
    cfg = exp.cfg
    F, R = cfg.train_repeats, cfg.realizations
    models = exp.models
    ok_train, missing = training_complete(exp)
    ret, ret_meta = collect_shards(exp.ret_dir, exp.fp["retention"], models)
    abl, abl_meta = collect_shards(exp.abl_dir, exp.fp["ablation"], models)
    diag, diag_meta = collect_shards(exp.diag_dir, exp.fp["diagnostics"], models)
    if not any((exp.train_dir / f"fold{f}" / "digital_best_validation.pt").exists() for f in range(F)):
        print("No completed training repetition yet; nothing to report.")
        return None

    def complete_folds(df):   # folds with a shard for EVERY model x realization
        if not len(df):
            return []
        c = df[["fold", "method", "realization"]].drop_duplicates().groupby("fold").size()
        return sorted(int(f) for f, n in c.items() if n == len(models) * R)

    ok_ret, msg = False, "no retention records"
    if len(ret):
        ok_ret, msg, _ = completeness_report(ret, exp)
    abl_folds, diag_folds = complete_folds(abl), complete_folds(diag)
    ok_abl, ok_diag = len(abl_folds) == F, len(diag_folds) == F
    record_check("No missing or duplicated data records", ok_ret and ok_abl and ok_diag,
                 f"retention: {msg}; ablation complete folds {abl_folds}/{F}; diagnostics complete folds "
                 f"{diag_folds}/{F}" + ("" if (ok_ret and ok_abl and ok_diag) else " (INCOMPLETE -> PROVISIONAL outputs)"),
                 "paper", essential=False)
    complete = ok_train and ok_ret and ok_abl and ok_diag
    if not complete and not allow_provisional:
        print("Results incomplete; nothing generated (ALLOW_PROVISIONAL_PAPER_OUTPUTS=False).")
        return None
    out = exp.paper_dir if complete else exp.run_dir / "paper_PROVISIONAL"
    inputs = sorted(str(p.relative_to(exp.run_dir)) + f"|{p.stat().st_size}|{p.stat().st_mtime_ns}"
                    for d in (exp.train_dir, exp.ret_dir, exp.abl_dir, exp.diag_dir) if d.exists()
                    for p in d.rglob("*") if p.is_file() and ".slot" not in p.name)
    signature = stable_hash({"inputs": inputs, "fp": exp.fp, "code": notebook_code_hash(), "complete": complete})
    sig_file = out / "inputs_signature.json"
    if sig_file.exists() and json.loads(sig_file.read_text()).get("signature") == signature:
        print(f"Paper outputs in {out} are up to date with the result files; not regenerated.")
        return out
    if out.exists() and any(out.iterdir()):     # never mix with files of an earlier generation (kept, renamed)
        os.replace(out, out.with_name(out.name + "_superseded_" + time.strftime("%Y%m%d-%H%M%S")))
        old = sorted(out.parent.glob(out.name + "_superseded_*"))
        for d in old[:-2]:                      # keep the two most recent superseded generations only
            shutil.rmtree(d, ignore_errors=True)
    out.mkdir(parents=True, exist_ok=True)
    prov = not complete
    tag = "" if complete else "_PROVISIONAL"
    print(f"Writing {'FINAL' if complete else 'PROVISIONAL'} paper outputs to {out}")
    dig_hist, hwa_hist = load_histories(exp)
    if len(dig_hist) and len(hwa_hist):
        atomic_write_csv(dig_hist, out / "training_history_fp32.csv")
        atomic_write_csv(hwa_hist, out / "training_history_hwa.csv")
        per_fold = hwa_hist.groupby("fold").method.nunique()
        fig_folds = sorted({int(f) for f in dig_hist.fold.unique()} &
                           {int(f) for f, n in per_fold.items() if n == len(exp.methods)})
        if fig_folds:
            fig1_training(dig_hist[dig_hist.fold.isin(fig_folds)], hwa_hist[hwa_hist.fold.isin(fig_folds)],
                          exp.methods, out, prov or len(fig_folds) < F, n_folds=len(fig_folds))
    sel_rows, te_rows = [], []
    for fold in range(F):
        fdir = exp.train_dir / f"fold{fold}"
        dpath = fdir / "digital_best_validation.pt"
        if dpath.exists():
            d = torch_load(dpath)
            sel_rows.append({"fold": fold, "model": "FP32-Digital", "completed_epochs": d["digital_epochs_completed"],
                             "selected_epoch": d["best_epoch"], "best_val_accuracy": d["best_val_accuracy"],
                             "optimizer_steps": d["optimizer_steps"], "checkpoint_identity": d["model_identity"],
                             "selection_split": "validation"})
        direct = fdir / f"{DIRECT}_analog_checkpoint.pt"
        if direct.exists():
            q = torch_load(direct)
            te_rows += [{"fold": fold, **r} for r in q["transfer_errors"]]
            sel_rows.append({"fold": fold, "model": DIRECT, "completed_epochs": 0, "selected_epoch": 0,
                             "best_val_accuracy": float("nan"), "optimizer_steps": q["optimizer_steps"],
                             "checkpoint_identity": q["model_identity"], "selection_split": "none (no HWA)"})
        for m in exp.methods:
            mp = fdir / f"{m}_best_validation.pt"
            if mp.exists():
                q = torch_load(mp)
                sel_rows.append({"fold": fold, "model": m, "completed_epochs": q["analog_epochs_completed"],
                                 "selected_epoch": q["selected_best_validation_epoch"], "best_val_accuracy": q["val_accuracy"],
                                 "optimizer_steps": q["optimizer_steps"], "checkpoint_identity": q["model_identity"],
                                 "selection_split": "validation (mean of 3 noisy reads)"})
    if sel_rows:
        atomic_write_csv(pd.DataFrame(sel_rows), out / "best_validation_selection.csv")
    if te_rows:
        atomic_write_csv(pd.DataFrame(te_rows), out / "weight_bias_transfer_errors.csv")
    pre = [pd.read_csv(p) for p in sorted(exp.train_dir.glob("fold*/pre_pcm_test_accuracy.csv"))]
    if pre:
        atomic_write_csv(pd.concat(pre, ignore_index=True), out / "pre_pcm_test_accuracy.csv")
    sel_bins = {}
    for p in sorted(exp.ret_dir.glob("fold*/selected_bins.json")):
        rec = json.loads(p.read_text())
        sel_bins[int(rec["fold"])] = int(rec["bins"])
    atomic_write_csv(pd.DataFrame([{"fold": f, "selected_B_validation": b} for f, b in sorted(sel_bins.items())]),
                     out / "driftslice_selected_B_per_fold.csv")
    abl_diff = None
    if len(ret):
        # keep only folds whose retention is complete for every model x realization
        counts = ret[["fold", "method", "realization"]].drop_duplicates().groupby("fold").size()
        full_folds = [int(f) for f, c in counts.items() if c == len(models) * R]
        retc = ret[ret.fold.isin(full_folds)]
        F_eff = len(full_folds) if prov else F
        atomic_write_csv(retc, out / "retention_records_test.csv")
        summary = two_stage_summary(retc, ["method", "compensation", "t_index", "t_s"], expected_folds=F_eff, expected_reals=R)
        atomic_write_csv(summary, out / "retention_summary_two_stage.csv")
        if F_eff:
            fig2_main_retention(summary, exp.methods, out, prov, F=F_eff, R=R, dataset=ops(exp).dataset,
                                n_times=len(exact_retention_times(cfg)))
            fig3_matched(summary, exp.methods, out, prov, F=F_eff, R=R)
            ret5 = retc[retc.t_index == retc.t_index.max()]
            t1 = make_table1(ret5, models, F_eff, R, {f: sel_bins[f] for f in full_folds if f in sel_bins}, out, tag,
                             n_test=512 if cfg.fast_mode else ops(exp).n_test_full,
                             with_original_gdc="GDC" in compensations_for(cfg), dataset=ops(exp).dataset,
                             arch_label=ops(exp).label)
            # traceability: recompute every Table I cell from the CSV on disk
            re5 = pd.read_csv(out / "retention_records_test.csv")
            re5 = re5[re5.t_index == re5.t_index.max()]
            re_s = two_stage_summary(re5, ["method", "compensation"], expected_folds=F_eff, expected_reals=R)
            mrg = t1.merge(re_s, on=["method", "compensation"])
            same = bool(np.allclose(mrg.mean_error_pct, mrg["mean"], equal_nan=True) and
                        np.allclose(mrg.sd_across_folds_pct, mrg["sd"], equal_nan=True))
            record_check("All table values traceable to CSV source data", same,
                         f"{len(mrg)} Table I cells recomputed from retention_records_test.csv (re-read from disk)",
                         "paper", essential=False)
            abl_use = sorted(set(abl_folds) & set(full_folds))
            if len(abl) and abl_use:
                ablc = abl[abl.fold.isin(abl_use)]
                F_abl = len(abl_use)
                atomic_write_csv(ablc, out / "reference_ablation_records.csv")
                abl_diff = ablation_differences(ablc, "test", expected_folds=F_abl, expected_reals=R)
                atomic_write_csv(abl_diff, out / "reference_ablation_differences_test.csv")
                atomic_write_csv(ablation_differences(ablc, "val", expected_folds=F_abl, expected_reals=R),
                                 out / "reference_ablation_differences_validation.csv")
                fig4_ablation(abl_diff, exp.methods, out, prov, F=F_abl, R=R, K=cfg.reference_read_repeats)
                # traceability of ablation vs retention (same seeds -> identical 5-y values)
                r5 = ret5.set_index(["fold", "method", "realization", "compensation"]).error
                a = ablc[(ablc.split == "test")]
                checks = []
                for f, m, r in a[["fold", "method", "realization"]].drop_duplicates().itertuples(index=False):
                    z = a[(a.fold == f) & (a.method == m) & (a.realization == r)]
                    pairs = [("none", z[z.compensation == "none"].error.iloc[0])]
                    pairs.append(("DS-B1", z[(z.bins == 1) & (z.R_tot == cfg.reference_cell_budget)].error.iloc[0]))
                    if f in sel_bins:
                        pairs.append(("DS-val", z[(z.bins == sel_bins[f]) & (z.R_tot == cfg.reference_cell_budget)].error.iloc[0]))
                    for comp, v in pairs:
                        if (f, m, r, comp) in r5.index:
                            checks.append(abs(float(r5.loc[(f, m, r, comp)]) - float(v)))
                if checks:
                    record_check("Ablation/diagnostics traceable to retention shards", max(checks) <= 0.05,
                                 f"ablation reproduces {len(checks)} retention 5-y values; max |Δ| = {max(checks):.3f} pp",
                                 "paper", essential=False)
            diag_use = sorted(set(diag_folds) & set(full_folds))
            if len(diag) and diag_use:
                diagc = diag[diag.fold.isin(diag_use)]
                atomic_write_csv(diagc, out / "mechanism_diagnostics_records.csv")
                make_table2(diagc, len(diag_use), R, out, tag, layers=ops(exp).layer_groups)
                dd = diagc[diagc.layer == ops(exp).layer_groups[0]].set_index(["fold", "method", "realization"]).test_error_B1_5y_pct
                r5b = ret5[ret5.compensation == "DS-B1"].set_index(["fold", "method", "realization"]).error
                common = dd.index.intersection(r5b.index)
                if len(common):
                    mx = float((dd.loc[common] - r5b.loc[common]).abs().max())
                    record_check("Ablation/diagnostics traceable to retention shards", mx <= 0.05,
                                 f"diagnostics reproduce {len(common)} retention DS-B1 5-y values; max |Δ| = {mx:.3f} pp",
                                 "paper", essential=False)
                for f in sorted(set(diagc.fold)):
                    for m in (NSR, HWA):
                        npz = exp.diag_dir / f"fold{f}" / f"distribution_states_{m}.npz"
                        if npz.exists() and f == 0:
                            stem = "fig5_distributions" if m == NSR else f"figS5_distributions_{m}"
                            fig5_distributions(npz, m, sel_bins[f], out, prov, stem, layers=ops(exp).layer_groups)
            audit = claim_audit(ret5, models, F_eff, abl_diff)
            if prov and len(audit) and "n_folds" in audit:
                audit.insert(0, "PROVISIONAL", audit["n_folds"].map(
                    lambda n: f"{int(n)}/{F} repetitions" if pd.notna(n) else ""))
            atomic_write_csv(audit, out / "reviewer_claim_audit.csv")
            # GDC validity and original-GDC saturation
            gdc = retc[retc.compensation == "GDC-safe"]
            meta_c = [m for m in ret_meta if int(Path(m["file"]).parts[0][4:]) in full_folds]
            sat = pd.DataFrame([{"file": m["file"], **{f"orig_gdc_sat_{k}": v for k, v in m["meta"]["orig_gdc_ref_saturation_fraction"].items()},
                                 **{f"safe_probe_u_{k}": v["u"] for k, v in m["meta"]["gdc_safe_probe"].items()}}
                                for m in meta_c])
            sat_cols = [c for c in sat.columns if c.startswith("orig_gdc_sat_")]
            if len(sat):
                atomic_write_csv(sat, out / "gdc_saturation_diagnostics.csv")
            if len(gdc) and "gdc_valid" in gdc and len(sat):
                worst = max(sat_cols, key=lambda c: sat[c].mean())
                print(f"Saturation-safe GDC valid in {int(gdc.gdc_valid.sum())}/{len(gdc)} evaluations; original all-ones "
                      f"probe saturates on average {100*sat[sat_cols].values.mean():.1f}% of all ADC outputs at 20 s "
                      f"(worst layer {worst[len('orig_gdc_sat_'):]}: {100*sat[worst].mean():.1f}%)"
                      + ("" if "GDC" in compensations_for(cfg) else "; the original GDC itself is not evaluated") + ".")
    rec = reproducibility_record(exp, sel_bins)
    atomic_write_text(out / "reproducibility_record.json", json.dumps(rec, indent=2, default=str))
    atomic_write_text(sig_file, json.dumps({"signature": signature, "complete": complete,
                                            "generated": time.strftime("%Y-%m-%d %H:%M:%S")}, indent=2))
    return out


print("Tables / audit / reproducibility loaded.")

In [ ]:
# ============================ RUN HELPERS (workload estimates, gates) ============================
SELECTED_METHODS = PRIMARY_METHODS + (EXTRA_METHODS if INCLUDE_ATTENTION_ONLY_AND_GAIN_ONLY else ())


def make_config(arch: str, fast_mode: bool, run_id: str) -> Config:
    """The experiment configuration of an architecture. MLP: the original Config. ResNet-32: the SAME Config with
    the documented RESNET_CONFIG_OVERRIDES (dataset root, underfit/transfer guards, 12 anchors, no original GDC)
    and, in the smoke test only, the optional SMOKE_BATCH_SIZES."""
    if arch == "mlp":
        return Config(fast_mode=fast_mode, methods=SELECTED_METHODS, run_id=run_id,
                      include_original_gdc=RUN_ORIGINAL_SATURATING_GDC)
    cfg = Config(fast_mode=fast_mode, methods=SELECTED_METHODS, run_id=run_id, **RESNET_CONFIG_OVERRIDES)
    if fast_mode and SMOKE_BATCH_SIZES:
        cfg = replace(cfg, batch_size=int(SMOKE_BATCH_SIZES[0]), test_batch_size=int(SMOKE_BATCH_SIZES[1]))
    return cfg


def full_run_id(arch: str = None) -> str:
    return FULL_RUN_ID if (arch or MODEL_ARCHITECTURE) == "mlp" else FULL_RUN_ID_RESNET


def _validate_parallel_flags():
    if PARALLEL_ROLE not in ("all", "coordinator", "worker"):
        raise ValueError(f"PARALLEL_ROLE must be 'all', 'coordinator' or 'worker', not {PARALLEL_ROLE!r}")
    if PARALLEL_ROLE == "worker":
        allowed = set(SELECTED_METHODS) | {DIRECT}
        if not WORKER_MODELS or not set(WORKER_MODELS) <= allowed or len(set(WORKER_MODELS)) != len(WORKER_MODELS):
            raise ValueError(f"WORKER_MODELS must be a non-empty subset of {sorted(allowed)}, got {WORKER_MODELS}")


_validate_parallel_flags()


def role_tag() -> str:
    if PARALLEL_ROLE == "worker":
        return "worker-" + "+".join(m.replace("PCM-", "").replace("-HWA", "") for m in WORKER_MODELS)
    return PARALLEL_ROLE


def open_full_experiment(resume_policy: str = RESUME_POLICY, role: Optional[str] = None):
    """role None = PARALLEL_ROLE. 'all'/'coordinator-full': every model (writes the manifest);
    'coordinator': training restricted to FP32 + FP32-Direct; 'worker': only WORKER_MODELS, manifest read-only."""
    role = role or PARALLEL_ROLE
    run_id = full_run_id()
    cfg = make_config(MODEL_ARCHITECTURE, False, run_id)
    if role == "worker":
        methods = tuple(m for m in SELECTED_METHODS if m in WORKER_MODELS)       # canonical order
        return Experiment(cfg, BASE_DIR, OUTPUT_DIR, "full", run_id, DEVICE, methods, resume_policy,
                          arch=MODEL_ARCHITECTURE, include_direct=DIRECT in WORKER_MODELS, write_manifest=False,
                          role_tag=role_tag())
    methods = () if role == "coordinator" else SELECTED_METHODS
    return Experiment(cfg, BASE_DIR, OUTPUT_DIR, "full", run_id, DEVICE, methods, resume_policy,
                      arch=MODEL_ARCHITECTURE, role_tag="coordinator" if role.startswith("coordinator") else "all")


def coordinator_training_ready(exp: "Experiment") -> Tuple[bool, list]:
    """Workers may only start once the coordinator wrote FP32 + FP32-Direct (+ split) for EVERY fold
    (otherwise several runtimes would train the shared FP32 model concurrently)."""
    missing = []
    for fold in range(exp.cfg.train_repeats):
        fdir = exp.train_dir / f"fold{fold}"
        for name in ("digital_best_validation.pt", "digital_final.pt", f"{DIRECT}_analog_checkpoint.pt", "split.pt"):
            if not (fdir / name).exists():
                missing.append(f"fold{fold}/{name}")
    return not missing, missing


def selected_bins_ready(exp: "Experiment") -> Tuple[bool, list]:
    missing = [f"fold{f}" for f in range(exp.cfg.train_repeats)
               if not (exp.ret_dir / f"fold{f}" / "selected_bins.json").exists()]
    return not missing, missing


def coordinator_post_training(exp: "Experiment") -> bool:
    """After ALL workers finished training (coordinator, full model set): training checks of every fold,
    pre-PCM test table, and the validation-only DriftSlice B selection that all workers then share."""
    ok, missing = training_complete(exp)
    if not ok:
        print(f"Workers have not finished training yet ({len(missing)} checkpoints missing, e.g. {missing[:3]}). "
              "Re-run this notebook when all workers report TRAINING STAGE COMPLETE.")
        return False
    cfg, dev, o = exp.cfg, exp.device, ops(exp)
    for fold in range(cfg.train_repeats):
        fdir = exp.train_dir / f"fold{fold}"
        train, val, test, _ = o.loaders(cfg, exp.base_dir, fold, dev, verify_fast=False)
        digital, _, dpack = o.train_digital(cfg, fold, train, val, dev, fdir, exp.fp["train"])   # loads + verifies
        packs = {DIRECT: o.make_direct(cfg, fold, digital, dpack["model_identity"], val, dev, fdir, exp.fp["train"])}
        for m in exp.methods:
            packs[m] = o.train_hwa(m, cfg, fold, digital, dpack["model_identity"], train, val, dev, fdir, exp.fp["train"])
        verify_training_fold(exp, fold, packs, dpack["model_identity"], digital)
        pre_path = fdir / "pre_pcm_test_accuracy.csv"
        if not pre_path.exists() or set(pd.read_csv(pre_path).method) != {"FP32-Digital", DIRECT, *exp.methods}:
            atomic_write_csv(pre_pcm_test_accuracy(exp, fold, digital, packs, test), pre_path)
        snaps = o.load_snapshots(exp, fold, val)
        o.selected_bins(exp, fold, snaps, val)
        del snaps, digital
    print("COORDINATOR: training verified for all folds; DriftSlice B selected on validation for every fold. "
          "Workers can now run retention / diagnostics.")
    return True


def _remaining_epochs(exp: "Experiment"):
    cfg = exp.cfg
    dig = hwa = 0
    per_method = {m: 0 for m in exp.methods}
    for fold in range(cfg.train_repeats):
        fdir = exp.train_dir / f"fold{fold}"
        if not (fdir / "digital_best_validation.pt").exists():
            s, _ = load_rotating(fdir / "digital_training.pt")
            dig += cfg.digital_epochs - (s["next_epoch"] if s else 0)
        for m in exp.methods:
            if not (fdir / f"{m}_best_validation.pt").exists():
                s, _ = load_rotating(fdir / f"{m}_training.pt")
                left = cfg.epochs - (s["next_epoch"] if s else 0)
                hwa += left
                per_method[m] += left
    return dig, hwa, per_method


def benchmark_training(exp: "Experiment") -> dict:
    """Time ONE real epoch of every stage with the real code on fold-0 data (local disk, discarded)."""
    import uuid
    cfg, dev, o = exp.cfg, exp.device, ops(exp)
    tmp = LOCAL_CACHE / f"benchmark_tmp_{uuid.uuid4().hex[:8]}"      # unique: never resumes an old benchmark
    tmp.mkdir(parents=True)
    state_paths = [tmp / "digital_training.pt"] + [tmp / m / f"{m}_training.pt" for m in exp.methods]
    try:
        train, val, _, _ = o.loaders(cfg, exp.base_dir, 0, dev, verify_fast=False)
        out = {}
        t = time.time()
        digital, _, dpack = o.train_digital(replace(cfg, digital_epochs=1, fast_mode=True), 0, train, val, dev, tmp, "bench")
        out["digital_epoch_s"] = time.time() - t
        for m in exp.methods:
            t = time.time()
            o.train_hwa(m, replace(cfg, epochs=1, fast_mode=True), 0, digital, dpack["model_identity"], train, val, dev, tmp / m, "bench")
            out[m] = time.time() - t
        return out
    finally:
        for p in state_paths:
            for f in sum(_resume_slots(p), []):
                Path(f).unlink(missing_ok=True)
        shutil.rmtree(tmp, ignore_errors=True)


def estimate_and_report_training(exp: "Experiment"):
    dig, hwa, per_method = _remaining_epochs(exp)
    if dig == 0 and hwa == 0:
        print("All training epochs already completed; checkpoints will only be verified.")
        return
    b = benchmark_training(exp)
    per_hwa = float(np.mean([b[m] for m in exp.methods])) if exp.methods else 0.0
    seconds = dig * b["digital_epoch_s"] + hwa * per_hwa
    cfg = exp.cfg
    rows = {"architecture / dataset": f"{ops(exp).label} / {ops(exp).dataset}",
            "training repetitions (folds)": cfg.train_repeats, "FP32 epochs per fold": cfg.digital_epochs,
            "HWA methods": ", ".join(exp.methods), "HWA epochs per method/fold": cfg.epochs,
            "remaining FP32 epochs": dig, "remaining HWA epochs": hwa,
            "measured s/epoch (FP32 | HWA mean)": f"{b['digital_epoch_s']:.1f} | {per_hwa:.1f}",
            "measured s/epoch per method": {m: round(b[m], 1) for m in exp.methods}}
    # measured resumable-state sizes (MB): FP32 ~3.2, HWA/DAR/NSR ~4.8, gate methods ~12.2 (gate + its Adam state)
    every = max(1, int(DRIVE_CHECKPOINT_EVERY_EPOCHS))
    if exp.arch == "mlp":
        mb = dig * 3.2 + sum(n * (12.2 if m in (ATT, ATT_ONLY) else 4.8) for m, n in per_method.items())
    else:   # measured ResNet-32 resumable states (MB): FP32 7.7, HWA/DAR/NSR 11.8, gate methods 125 (3072x3072 gate + Adam)
        mb = dig * 7.7 + sum(n * (125.0 if m in (ATT, ATT_ONLY) else 11.8) for m, n in per_method.items())
    rows["Drive checkpoint write volume"] = (f"{mb / every / 1e3:.1f} GB over the run (two slots per model, overwritten "
                                             f"in place every {every} epoch(s); Drive may keep superseded versions "
                                             f"for up to 30 days)")
    print_workload("FULL TRAINING WORKLOAD", rows, seconds,
                   capacity_report({"drive": BASE_DIR, "local_disk": LOCAL_CACHE}),
                   need_gb=cfg.train_repeats * ((0.02 + len(exp.methods) * 0.05) if exp.arch == "mlp"
                                                else (0.05 + len(exp.methods) * 0.30)))


def estimate_and_report_retention(exp: "Experiment", stage: str):
    cfg, dev, o = exp.cfg, exp.device, ops(exp)
    ok, missing = training_complete(exp)
    if not ok:
        raise ScientificCheckError(f"{stage} needs completed training first; missing {len(missing)} checkpoints, e.g. "
                                   + ", ".join(missing[:4]))
    stage_dir = {"retention": exp.ret_dir, "ablation": exp.abl_dir, "diagnostics": exp.diag_dir}[stage]
    total = cfg.train_repeats * cfg.realizations * len(exp.models)
    have = sum(len(list(stage_dir.glob(f"fold*/{m}/real*.pt"))) for m in exp.models)
    remaining = max(0, total - have)
    if remaining == 0:
        print(f"All {total} {stage} shards exist; they will only be verified.")
        return
    _, val, test, _ = o.loaders(cfg, exp.base_dir, 0, dev, verify_fast=False)
    snaps = o.load_snapshots(exp, 0, val)
    entry = snaps[exp.models[-1]]
    t = time.time()
    if stage == "retention":
        o.retention(exp.models[-1], entry, replace(cfg, retention_points=3), 0, 0, cfg.driftslice_bins, test, dev)
        per_shard = (time.time() - t) / 3 * len(exact_retention_times(cfg))
    elif stage == "ablation":
        o.ablation(exp.models[-1], entry, cfg, 0, 0, val, test, dev)
        per_shard = time.time() - t
    else:
        o.diagnostics(exp.models[-1], entry, cfg, 0, 0, cfg.driftslice_bins, test, dev)
        per_shard = time.time() - t
    rows = {"folds x realizations x models": f"{cfg.train_repeats} x {cfg.realizations} x {len(exp.models)} = {total} shards",
            "existing shards": have, "remaining shards": remaining,
            "timestamps per shard": len(exact_retention_times(cfg)) if stage == "retention" else "1 (5 years)",
            "measured s/shard": f"{per_shard:.1f}"}
    if stage == "retention":
        rows["compensation modes"] = ", ".join(compensations_for(cfg))
    print_workload(f"{stage.upper()} WORKLOAD", rows, per_shard * remaining,
                   capacity_report({"drive": BASE_DIR, "local_disk": LOCAL_CACHE}),
                   need_gb=remaining * (0.0004 if stage != "diagnostics" else 0.0001))


def check_leakage_ledger(scope: str):
    if not SPLIT_LEDGER:
        print(f"(leakage ledger empty for '{scope}': no split was read in this check session)")
        return
    selection = {"FP32 best-epoch selection", "HWA best-epoch selection", "DriftSlice B selection"}
    reporting = {"retention", "pre-PCM reporting", "ablation(report-only)", "diagnostics(report-only)"}
    bad = [(s, sp) for s, sp in SPLIT_LEDGER if (s in selection and sp != "val") or (s in reporting and sp not in ("test", "val"))]
    unknown = sorted({s for s, _ in SPLIT_LEDGER} - selection - reporting)
    used = sorted({(s, sp) for s, sp in SPLIT_LEDGER})
    record_check("No validation/test leakage", not bad and not unknown,
                 f"selection stages read only validation; test read only by reporting stages; ledger: {used}", scope)


def save_checks_report(target_dir: Path, label: str):
    df = checks_dataframe().assign(session=CHECK_SESSION["label"], saved=time.strftime("%Y-%m-%d %H:%M:%S"))
    target_dir.mkdir(parents=True, exist_ok=True)
    atomic_write_csv(df, target_dir / f"scientific_checks_{label}.csv")
    return df


def end_stage_checks(exp: "Experiment", stage: str):
    check_leakage_ledger(stage)
    tag = f"{exp.role_tag}_" if exp.role_tag not in ("", "all") else ""
    return save_checks_report(exp.run_dir / "checks", f"{stage}_{tag}{time.strftime('%Y%m%d-%H%M%S')}")


def merged_stage_checks(exp: "Experiment") -> pd.DataFrame:
    """Latest saved report of every full-run stage + the current session (paper) checks."""
    frames = []
    for stage in ("full_training", "full_retention", "full_ablation", "full_diagnostics"):
        files = sorted((exp.run_dir / "checks").glob(f"scientific_checks_{stage}_*.csv"))
        rows = [pd.read_csv(f).assign(stage=stage, source=f.name) for f in files]   # ALL sessions of the stage
        if not rows:
            continue
        allr = pd.concat(rows, ignore_index=True)
        allr = allr[allr.status != "NOT EXECUTED"]
        merged = []
        for check, g in allr.groupby("check", sort=False):
            fails = g[g.status == "FAIL"]
            merged.append((fails if len(fails) else g).iloc[-1])     # a FAIL in any session stays a FAIL
        if merged:
            frames.append(pd.DataFrame(merged))
    frames.append(checks_dataframe().assign(stage="paper", source="this session"))
    df = pd.concat(frames, ignore_index=True)
    return df[df.status != "NOT EXECUTED"].reset_index(drop=True)


print(f"Run helpers loaded. MODEL_ARCHITECTURE = {MODEL_ARCHITECTURE!r} ({ARCH_OPS[MODEL_ARCHITECTURE].label} on "
      f"{ARCH_OPS[MODEL_ARCHITECTURE].dataset}); full run id {full_run_id()!r}; selected HWA methods:", SELECTED_METHODS)

## 12. Smoke test (run first)

This runs the **entire** pipeline at reduced size: fast mode with 2 folds, 2 FP32 + 2 HWA epochs, 1024/256/512 samples, 2 realizations and 8 timestamps. The smoke run:
* runs the original `preflight()` and independent PCM, DriftSlice and statistics unit tests;
* **simulates two Colab disconnects** (after FP32 epoch 1 and after Drift-Attention HWA epoch 1), resumes, and then requires every checkpoint to be **bit-identical** to a separate uninterrupted run;
* runs retention twice (the second pass must only verify existing shards), then the ablation, diagnostics, figures, tables and all scientific checks;
* writes `results/smoke/SMOKE_PASS_RECORD.json` to Drive only if no check fails. Full stages refuse to start without this record for the **current notebook code and environment**. The bulky smoke outputs stay on the VM disk. When a current PASS exists, the smoke test is skipped automatically (`FORCE_SMOKE_TEST = True` repeats it).

Smoke-test numbers are **not scientific results**: the models are deliberately undertrained.

In [ ]:
def compare_training_runs(a: "Experiment", b: "Experiment") -> Tuple[bool, str]:
    n, bad = 0, []
    for fold in range(a.cfg.train_repeats):
        names = ["digital_final.pt", "digital_best_validation.pt", f"{DIRECT}_analog_checkpoint.pt"] + \
                [f"{m}_{k}.pt" for m in a.methods for k in ("final", "best_validation")]
        for name in names:
            pa, pb = torch_load(a.train_dir / f"fold{fold}" / name), torch_load(b.train_dir / f"fold{fold}" / name)
            n += 1
            if state_dict_hash(pa["model"]) != state_dict_hash(pb["model"]):
                bad.append(f"fold{fold}/{name}")
            ha, hb = pa.get("history"), pb.get("history")
            if ha is not None and [round(h.get("val_accuracy", h.get("val_ce", 0)), 12) for h in ha] != \
                    [round(h.get("val_accuracy", h.get("val_ce", 0)), 12) for h in hb]:
                bad.append(f"fold{fold}/{name}:history")
    return not bad, f"{n} checkpoints compared; differing: {bad or 'none'}"


def run_smoke_test():
    stamp = time.strftime("%Y%m%d-%H%M%S")
    record = smoke_pass_path(OUTPUT_DIR)
    if record.exists():        # a (forced) re-run must never leave an older PASS valid if it fails
        record.parent.mkdir(parents=True, exist_ok=True)
        os.replace(record, record.with_name(f"SMOKE_PASS_RECORD.revoked_{stamp}.json"))
    try:
        return _run_smoke_test_body(stamp)
    except BaseException as exc:
        save_checks_report(OUTPUT_DIR / "smoke", f"smoke_{MODEL_ARCHITECTURE}_{stamp}_FAILED")
        atomic_write_text(OUTPUT_DIR / "smoke" / f"SMOKE_FAIL_{MODEL_ARCHITECTURE}_{stamp}.json",
                          json.dumps({"time": stamp, "arch": MODEL_ARCHITECTURE, "error": repr(exc)[:2000]}, indent=2))
        begin_check_session("session")
        raise


def _run_smoke_test_body(stamp: str):
    begin_check_session("smoke", prefix="smoke:")
    smoke_root = LOCAL_CACHE / "results"            # bulky smoke artefacts stay on the VM disk, not on Drive
    run_id = f"{SMOKE_RUN_ID}_{MODEL_ARCHITECTURE}_{stamp}"
    cfg = make_config(MODEL_ARCHITECTURE, True, run_id)
    exp = Experiment(cfg, BASE_DIR, smoke_root, "smoke", run_id, DEVICE, SELECTED_METHODS, "error", arch=MODEL_ARCHITECTURE)
    t0 = time.time()
    run_all_preflight_checks(exp.cfg, DEVICE, "smoke", arch=MODEL_ARCHITECTURE)
    # --- simulated Colab disconnects during training, then resume ---
    interrupted = []
    _RESUME_LOG.clear()
    hwa_stage = f"hwa:{ATT if ATT in exp.methods else exp.methods[0]}"
    for stage in ("digital", hwa_stage):
        _INTERRUPT_HOOK.update(stage=stage, after_epoch=1)
        try:
            run_training_stage(exp)
        except SimulatedInterruption as exc:
            interrupted.append(str(exc))
            print("*** SIMULATED DISCONNECT:", exc, "***")
    _INTERRUPT_HOOK.update(stage=None, after_epoch=None)
    run_training_stage(exp)
    ref = Experiment(cfg, BASE_DIR, smoke_root, "smoke", run_id + "_uninterrupted_reference", DEVICE, SELECTED_METHODS,
                     "error", arch=MODEL_ARCHITECTURE)
    run_training_stage(ref, verify_fast_loader=False)
    same, msg = compare_training_runs(exp, ref)
    resumed = [(s, f, e) for s, f, e, _ in _RESUME_LOG]
    real_resume = resumed[:2] == [("digital", 0, 1), (hwa_stage, 0, 1)]
    record_check("Checkpoint resumption", same and len(interrupted) == 2 and real_resume,
                 f"{len(interrupted)} simulated disconnects; resumed from slots at {_RESUME_LOG[:2]}; {msg} "
                 "(bit-identical to an uninterrupted run)", "smoke")
    # --- retention twice: second pass must only verify ---
    run_retention_stage(exp)
    n_before = len(list(exp.ret_dir.glob("fold*/*/real*.pt")))
    mtimes = {p: p.stat().st_mtime_ns for p in exp.ret_dir.glob("fold*/*/real*.pt")}
    run_retention_stage(exp)
    untouched = all(p.stat().st_mtime_ns == t for p, t in mtimes.items())
    record_check("Checkpoint resumption", untouched and n_before == cfg.resolved().train_repeats * cfg.resolved().realizations * len(exp.models),
                 f"second retention pass verified {n_before} existing shards without rewriting any", "smoke")
    run_ablation_stage(exp)
    run_diagnostics_stage(exp)
    out = generate_paper_results(exp, allow_provisional=False)
    check_leakage_ledger("smoke")
    df = save_checks_report(OUTPUT_DIR / "smoke", f"smoke_{MODEL_ARCHITECTURE}_{stamp}")
    df = df[df.check.isin(check_names_for(MODEL_ARCHITECTURE))]
    fails = df[df.status == "FAIL"].check.tolist()
    not_run = df[df.status == "NOT EXECUTED"].check.tolist()
    print("\n" + df[["check", "status", "scope"]].to_string(index=False))
    if fails or not_run:
        raise ScientificCheckError(f"SMOKE TEST: FAIL (failed: {fails}; not executed: {not_run})")
    rec = {"time": time.strftime("%Y-%m-%d %H:%M:%S"), "run_dir_local": str(exp.run_dir), "minutes": (time.time() - t0) / 60,
           "notebook_all_code": notebook_code_hash(), "all_scientific_code": exp.fp["code_hashes"]["all_scientific"],
           "env": environment_identity(), "scope": smoke_scope_identity(), "gpu": _gpu_name(),
           "arch": MODEL_ARCHITECTURE, "checks_csv": f"smoke/scientific_checks_smoke_{MODEL_ARCHITECTURE}_{stamp}.csv"}
    atomic_write_text(smoke_pass_path(OUTPUT_DIR), json.dumps(rec, indent=2))
    print(f"\nSMOKE TEST ({MODEL_ARCHITECTURE}): PASS ({rec['minutes']:.1f} min). Outputs on the VM disk (NOT scientific results): {out}")
    begin_check_session("session")      # smoke checks are saved; they must not leak into full-run reports
    return exp


def _current_smoke_pass():
    p = smoke_pass_path(OUTPUT_DIR)
    return p.exists() and smoke_record_is_current(json.loads(p.read_text()))


if PARALLEL_ROLE == "worker":
    if not _current_smoke_pass():
        raise ScientificCheckError(
            "Worker notebook: no current smoke-test PASS for this notebook code + environment + GPU type. Run the "
            "COORDINATOR notebook first (it runs the smoke test). Every parallel notebook must be the same notebook "
            "version on the same Colab runtime version and GPU type.")
    print("Worker: smoke-test PASS of the coordinator is on record for this exact code + environment + GPU.")
elif RUN_SMOKE_TEST and _current_smoke_pass() and not FORCE_SMOKE_TEST:
    print(f"Smoke-test PASS ({MODEL_ARCHITECTURE}) already on record for this exact notebook code + environment "
          f"({json.loads(smoke_pass_path(OUTPUT_DIR).read_text())['time']}); not re-run. "
          "Set FORCE_SMOKE_TEST=True to repeat it.")
elif RUN_SMOKE_TEST:
    SMOKE_EXP = run_smoke_test()
else:
    print("RUN_SMOKE_TEST=False")

## 13. Full experiment (enable one stage at a time)

Each cell below first prints the remaining workload, a micro-benchmark-based time estimate, and the GPU and storage capacity. It then runs **only if its flag is `True`**. All stages resume automatically from Google Drive after a disconnect: just re-run the notebook from the top.

1. `RUN_FULL_TRAINING = True`: 5 folds × (FP32 30 epochs → FP32-Direct + 4 × AnalogAdam HWA 154 epochs).
2. `RUN_FULL_RETENTION = True`: the validation-selected B per fold, then 5 × 5 × 30 shards of 75 timestamps × 5 compensation modes.
3. `RUN_REFERENCE_ABLATION = True` and `RUN_MECHANISM_DIAGNOSTICS = True`: no retraining and no new programming draws.

### 13b. Optional: several Colab runtimes in parallel (coordinator + workers)

The four HWA methods depend only on the shared FP32 checkpoint of each fold, so they can be trained **at the same time** on separate Colab runtimes. Parallel runs produce **bit-identical results** to a single-notebook run; this was verified on the full pipeline, including retention, ablation, diagnostics and Table I. The workers write only their own files on Google Drive. The shared steps (smoke test, FP32 pretraining, choosing B on validation, the paper) are done by **one** coordinator notebook.

Use the 5 prepared notebooks. Their code is identical; only `PARALLEL_ROLE` / `WORKER_MODELS` in Section 0 differ:

| Notebook | `PARALLEL_ROLE` | `WORKER_MODELS` |
|---|---|---|
| `…_P0_coordinator.ipynb` | `"coordinator"` | – |
| `…_P1_worker_HWA.ipynb` | `"worker"` | `("PCM-HWA", "FP32-Direct")` |
| `…_P2_worker_DAR.ipynb` | `"worker"` | `("PCM-DAR-HWA",)` |
| `…_P3_worker_DriftAttention.ipynb` | `"worker"` | `("PCM-DriftAttention-HWA",)` |
| `…_P4_worker_NSR.ipynb` | `"worker"` | `("PCM-NSR-HWA",)` |

**Procedure.** Every notebook keeps all stage flags at `True`. Every runtime must use the **same GPU type** and Colab runtime version 2026.04 (the smoke-test PASS is tied to the GPU type).
1. **P0 coordinator → Run all.** It runs the smoke test, then FP32 pretraining and FP32-Direct for all folds. It ends with *"start the worker notebooks now"*.
2. **P1–P4 → Run all, simultaneously** (one runtime each). Each worker trains its method for 5 folds and runs its ablation shards. Its retention and diagnostics print *"WAITING FOR THE COORDINATOR"*, which is expected. After a disconnect, re-run that worker; it resumes.
3. When **all four** workers have finished training: **P0 → Run all** again. It verifies all training, writes the pre-PCM test table, and selects B on validation for every fold.
4. **P1–P4 → Run all** again, simultaneously: retention and diagnostics shards.
5. **P0 → Run all**: the FINAL paper.

Safety: a worker refuses to start before the coordinator's FP32 checkpoints exist, never runs the smoke test, never writes the shared manifest or the pre-PCM table, and only uses a B selection whose checkpoint identity matches the Standard-HWA model.

In [ ]:
FULL_EXP = None
if RUN_FULL_TRAINING:
    begin_check_session("full_training")
    require_smoke_pass(OUTPUT_DIR)
    FULL_EXP = open_full_experiment()
    if PARALLEL_ROLE == "worker":
        _ready, _missing = coordinator_training_ready(FULL_EXP)
        if not _ready:
            raise ScientificCheckError(
                "The COORDINATOR has not finished FP32 pretraining + FP32-Direct for every fold yet (missing: "
                + ", ".join(_missing[:6]) + "). Wait until the coordinator prints 'start the worker notebooks', "
                "allow ~2 minutes for Google Drive to sync, then re-run this worker.")
    run_all_preflight_checks(FULL_EXP.cfg, DEVICE, "full", arch=FULL_EXP.arch)
    estimate_and_report_training(FULL_EXP)
    run_training_stage(FULL_EXP)
    end_stage_checks(FULL_EXP, "full_training")
    if PARALLEL_ROLE == "coordinator":
        print("\nCOORDINATOR: FP32 pretraining + FP32-Direct finished for all folds -> start the worker notebooks now.")
else:
    print("RUN_FULL_TRAINING=False")

In [ ]:
if RUN_FULL_RETENTION and PARALLEL_ROLE == "coordinator":
    # coordinator: verify all workers' training, write the pre-PCM test table, select B on validation (shared)
    begin_check_session("full_training")
    require_smoke_pass(OUTPUT_DIR)
    COORD_EXP = open_full_experiment(role="coordinator-full")
    if coordinator_post_training(COORD_EXP):
        end_stage_checks(COORD_EXP, "full_training")
elif RUN_FULL_RETENTION:
    begin_check_session("full_retention")
    require_smoke_pass(OUTPUT_DIR)
    FULL_EXP = FULL_EXP or open_full_experiment()
    _bins_ok, _bins_missing = selected_bins_ready(FULL_EXP)
    if PARALLEL_ROLE == "worker" and not _bins_ok:
        print(f"WAITING FOR THE COORDINATOR: DriftSlice B is not selected yet for {_bins_missing}. When ALL workers "
              "finished training, run the coordinator notebook once (RUN_FULL_RETENTION=True), then re-run this worker.")
    else:
        verify_retention_grid(FULL_EXP.cfg, "full")
        verify_pcm_physics(FULL_EXP.cfg, DEVICE, "full")
        verify_driftslice_mechanics(FULL_EXP.cfg, DEVICE, "full")
        estimate_and_report_retention(FULL_EXP, "retention")
        run_retention_stage(FULL_EXP)
        end_stage_checks(FULL_EXP, "full_retention")
else:
    print("RUN_FULL_RETENTION=False")

In [ ]:
if PARALLEL_ROLE == "coordinator" and (RUN_REFERENCE_ABLATION or RUN_MECHANISM_DIAGNOSTICS):
    print("Coordinator: the ablation and diagnostics shards are computed by the worker notebooks.")
else:
    if RUN_REFERENCE_ABLATION:
        begin_check_session("full_ablation")
        require_smoke_pass(OUTPUT_DIR)
        FULL_EXP = FULL_EXP or open_full_experiment()
        estimate_and_report_retention(FULL_EXP, "ablation")
        run_ablation_stage(FULL_EXP)
        end_stage_checks(FULL_EXP, "full_ablation")
    else:
        print("RUN_REFERENCE_ABLATION=False")

    if RUN_MECHANISM_DIAGNOSTICS:
        FULL_EXP = FULL_EXP or open_full_experiment()
        _bins_ok, _bins_missing = selected_bins_ready(FULL_EXP)
        if PARALLEL_ROLE == "worker" and not _bins_ok:
            print(f"WAITING FOR THE COORDINATOR: DriftSlice B not selected yet for {_bins_missing}; diagnostics skipped.")
        else:
            begin_check_session("full_diagnostics")
            require_smoke_pass(OUTPUT_DIR)
            estimate_and_report_retention(FULL_EXP, "diagnostics")
            run_diagnostics_stage(FULL_EXP)
            end_stage_checks(FULL_EXP, "full_diagnostics")
    else:
        print("RUN_MECHANISM_DIAGNOSTICS=False")

## 14. Paper results: Figures 1–5, Tables I–II, source CSVs, reproducibility record

This step uses only result files that already exist on Drive for the current fingerprints. When every stage is complete, outputs go to `results/full/<run_id>/paper/`. Otherwise, if `ALLOW_PROVISIONAL_PAPER_OUTPUTS` is set, they go to `paper_PROVISIONAL/`. Provisional outputs are computed only over repetitions whose data are complete, the figures carry a watermark, and the tables carry a `_PROVISIONAL` suffix.

In [ ]:
PAPER_DIR = None
if GENERATE_PAPER_RESULTS and PARALLEL_ROLE == "worker":
    print("Worker notebook: the paper is generated by the COORDINATOR notebook once every worker has finished.")
elif GENERATE_PAPER_RESULTS:
    full_manifest = OUTPUT_DIR / "full" / full_run_id() / "manifest.json"
    if not full_manifest.exists():
        print("No full run exists yet; nothing to generate. (Smoke-test outputs are on the VM disk and are "
              "NOT scientific results.)")
    else:
        begin_check_session("paper")
        if PARALLEL_ROLE == "coordinator" or FULL_EXP is None or FULL_EXP.models != (DIRECT,) + SELECTED_METHODS:
            FULL_EXP = open_full_experiment(role="coordinator-full" if PARALLEL_ROLE == "coordinator" else "all")
        ok, missing = training_complete(FULL_EXP)
        print("training complete:", ok, "" if ok else f"(missing {len(missing)} checkpoints)")
        PAPER_DIR = generate_paper_results(FULL_EXP, allow_provisional=ALLOW_PROVISIONAL_PAPER_OUTPUTS)
        if PAPER_DIR is not None:
            merged = merged_stage_checks(FULL_EXP)
            atomic_write_csv(merged, PAPER_DIR / "scientific_checks_all_stages.csv")
            for f in sorted(PAPER_DIR.iterdir()):
                print("  ", f.name)
else:
    print("GENERATE_PAPER_RESULTS=False")

## 15. Automated scientific checks (PASS / FAIL / NOT EXECUTED) and reviewer audit

The table lists every check executed in this session and its scope (`runtime`, `smoke`, `fold k`, `full`, `paper`). An essential check that fails stops its stage immediately. The reviewer audit classifies each comparison as **SUPPORTED**, **INCONCLUSIVE** or **CONTRADICTED**, based on the paired, fold-level 95 % t-interval (F = 5, df = 4). Holm-adjusted p-values across all audited claims are also reported. Only SUPPORTED statements should be written as findings, and negative or inconclusive results must be reported as such.

In [ ]:
# Full-paper reviewer audits require stages intentionally excluded here.
print("Retention-only workflow: see the final cell for accuracy CSVs, plots and completeness status.")


## Retention preview
Run the following cell after definitions and the smoke gate.


In [ ]:
# RETENTION PREVIEW: first N programmed-device realizations, across EVERY fold.
# The experiment Config and fingerprints stay unchanged. New shards are ordinary
# full-run shards, so the full retention job can reuse them later.
PREVIEW_REALIZATIONS = 1  # use 3 for a larger preview; leave Config.realizations unchanged
RUN_FULL_TRAINING = False
RUN_FULL_RETENTION = False
RUN_REFERENCE_ABLATION = False
RUN_MECHANISM_DIAGNOSTICS = False
GENERATE_PAPER_RESULTS = False

begin_check_session("retention_preview")
require_smoke_pass(OUTPUT_DIR)
FULL_EXP = open_full_experiment()
_pv_exp = FULL_EXP
_pv_cfg, _pv_dev, _pv_ops = _pv_exp.cfg, _pv_exp.device, ops(_pv_exp)
if isinstance(PREVIEW_REALIZATIONS, bool) or not isinstance(PREVIEW_REALIZATIONS, int):
    raise ValueError("PREVIEW_REALIZATIONS must be an integer")
if not 1 <= PREVIEW_REALIZATIONS <= _pv_cfg.realizations:
    raise ValueError(f"PREVIEW_REALIZATIONS must be within 1..{_pv_cfg.realizations}")
_pv_ready, _pv_missing = training_complete(_pv_exp)
if not _pv_ready:
    raise ScientificCheckError("Preview requires completed training: " + ", ".join(_pv_missing[:6]))
_pv_bins_ready, _pv_bins_missing = selected_bins_ready(_pv_exp)
if not _pv_bins_ready:
    raise ScientificCheckError("Run P0's HWA validation-selection cell first; missing " + ", ".join(_pv_bins_missing))

verify_retention_grid(_pv_cfg, "preview")
_pv_total = _pv_cfg.train_repeats * PREVIEW_REALIZATIONS * len(_pv_exp.models)
_pv_done = _pv_skipped = 0
_pv_start = time.time()
for _pv_fold in range(_pv_cfg.train_repeats):
    _, _pv_val, _pv_test, _ = _pv_ops.loaders(_pv_cfg, _pv_exp.base_dir, _pv_fold, _pv_dev, verify_fast=False)
    _pv_snaps = _pv_ops.load_snapshots(_pv_exp, _pv_fold, _pv_val)
    _pv_bins = fold_selected_bins(_pv_exp, _pv_fold, _pv_snaps, _pv_val)
    for _pv_real in range(PREVIEW_REALIZATIONS):
        for _pv_method in _pv_exp.models:
            _pv_path = _pv_exp.ret_dir / f"fold{_pv_fold}" / _pv_method / f"real{_pv_real:02d}.pt"
            if _shard_ok(_pv_path, _pv_exp.fp["retention"], _pv_snaps[_pv_method]["identity"], {"selected_bins": _pv_bins}):
                _pv_skipped += 1
                continue
            _pv_t0 = time.time()
            _pv_df, _pv_meta = _pv_ops.retention(_pv_method, _pv_snaps[_pv_method], _pv_cfg, _pv_fold, _pv_real,
                                               _pv_bins, _pv_test, _pv_dev)
            if _pv_meta["state_hash_start"] != _pv_meta["state_hash_end"]:
                raise ScientificCheckError("Programmed PCM states changed during preview retention")
            atomic_torch_save({"fp": _pv_exp.fp["retention"], "model_identity": _pv_snaps[_pv_method]["identity"],
                               "selected_bins": _pv_bins, "results": _pv_df, "meta": _pv_meta,
                               "gpu": _gpu_name(), "seconds": time.time() - _pv_t0}, _pv_path)
            _pv_done += 1
        print(f"PREVIEW fold {_pv_fold+1}/{_pv_cfg.train_repeats}, realization {_pv_real+1}/{PREVIEW_REALIZATIONS}: "
              f"{_pv_done} new, {_pv_skipped} reused; ETA {_eta(_pv_done, _pv_total-_pv_skipped, _pv_start)}", flush=True)
    del _pv_snaps
    if _pv_dev.type == "cuda":
        torch.cuda.empty_cache()
end_stage_checks(_pv_exp, "retention_preview")

# RETENTION-ONLY ACCURACY EXPORT
# This cell only reads saved retention/checkpoint files. It defines no scientific
# functions and does not change training, PCM evaluation or the existing smoke gate.
import matplotlib.pyplot as plt

_ret_exp = globals().get('COORD_EXP') if PARALLEL_ROLE == 'coordinator' else globals().get('FULL_EXP')
if _ret_exp is None:
    _ret_exp = open_full_experiment(role='coordinator-full' if PARALLEL_ROLE == 'coordinator' else None)
_ret_cfg = _ret_exp.cfg
_ret_models = list(_ret_exp.models)
_ret_comps = list(compensations_for(_ret_cfg))
_ret_times = np.asarray(exact_retention_times(_ret_cfg), dtype=float)
_ret_keys = ['fold', 'method', 'compensation', 'realization', 't_index']
_ret_expected = pd.MultiIndex.from_product(
    [range(_ret_cfg.train_repeats), _ret_models, _ret_comps, range(PREVIEW_REALIZATIONS), range(len(_ret_times))],
    names=_ret_keys)
_ret_df, _ret_meta = collect_shards(_ret_exp.ret_dir, _ret_exp.fp['retention'], _ret_models)
# Summarize only the requested prefix; retain all other full-run shards on disk.
if len(_ret_df):
    _ret_df = _ret_df[_ret_df['realization'] < PREVIEW_REALIZATIONS].copy()
_ret_meta = [item for item in _ret_meta
             if int(Path(item['file']).stem.removeprefix('real')) < PREVIEW_REALIZATIONS]
_ret_identities = {}
for _ret_item in _ret_meta:
    _ret_parts = Path(_ret_item['file']).parts
    _ret_fold = int(_ret_parts[0].removeprefix('fold'))
    _ret_method = _ret_parts[1]
    _ret_pair = (_ret_fold, _ret_method)
    if _ret_pair not in _ret_identities:
        _ret_checkpoint = _ret_exp.train_dir / f'fold{_ret_fold}' / (
            f'{DIRECT}_analog_checkpoint.pt' if _ret_method == DIRECT else f'{_ret_method}_best_validation.pt')
        _ret_pack = torch_load(_ret_checkpoint)
        verify_pack(_ret_pack, _ret_exp.fp['train'],
                    'fp32_direct' if _ret_method == DIRECT else 'hwa_best_validation', _ret_fold, _ret_checkpoint)
        if _ret_pack.get('method') != _ret_method:
            raise ScientificCheckError(f'{_ret_checkpoint}: method mismatch')
        _ret_identities[_ret_pair] = _ret_pack['model_identity']
    if _ret_item.get('model_identity') != _ret_identities[_ret_pair]:
        raise ScientificCheckError(f"{_ret_item['file']}: retention model identity differs from the checkpoint")

if len(_ret_df):
    if not set(_ret_keys + ['t_s', 'accuracy', 'split']).issubset(_ret_df.columns):
        raise ScientificCheckError('Retention export: missing required columns')
    if _ret_df.duplicated(_ret_keys).any():
        raise ScientificCheckError('Retention export: duplicate records')
    if not _ret_df['split'].eq('test').all():
        raise ScientificCheckError('Retention export requires official test-set results')
    _ret_index = pd.MultiIndex.from_frame(_ret_df[_ret_keys])
    if not _ret_index.isin(_ret_expected).all():
        raise ScientificCheckError('Retention export: unexpected fold, method, compensation, realization or timestamp index')
    if not np.array_equal(_ret_df['t_s'].to_numpy(), _ret_times[_ret_df['t_index'].to_numpy(dtype=int)]):
        raise ScientificCheckError('Retention export: timestamps differ from the configured grid')
    _ret_values = _ret_df['accuracy'].to_numpy(dtype=float)
    if np.isinf(_ret_values).any() or ((_ret_values < 0) | (_ret_values > 100)).any():
        raise ScientificCheckError('Retention export: invalid accuracy outside [0, 100]')
else:
    _ret_df = pd.DataFrame(columns=_ret_keys + ['t_s', 'accuracy', 'split'])

# Explicit expected grid prevents missing folds/realizations/whole curves from
# producing optimistic averages. NaN accuracy remains NaN; it is never filled.
_ret_grid = _ret_df.set_index(_ret_keys).reindex(_ret_expected).reset_index()
_ret_grid['t_s'] = _ret_times[_ret_grid['t_index'].to_numpy(dtype=int)]
_ret_grid['accuracy'] = pd.to_numeric(_ret_grid['accuracy'])
_ret_summary = two_stage_summary(_ret_grid, ['method', 'compensation', 't_index', 't_s'], value='accuracy',
                                expected_folds=_ret_cfg.train_repeats, expected_reals=PREVIEW_REALIZATIONS)
_ret_summary = _ret_summary.rename(columns={
    'mean': 'test_accuracy_mean_pct', 'sd': 'test_accuracy_sd_across_folds_pp',
    'invalid_records': 'missing_or_invalid_records'})
_ret_summary['complete'] = _ret_summary['missing_or_invalid_records'].eq(0)
_ret_summary['t_days'] = _ret_summary['t_s'] / DAY
_ret_summary['t_years'] = _ret_summary['t_s'] / YEAR
_ret_fold_means = fold_means(_ret_grid, ['method', 'compensation', 't_index', 't_s'], value='accuracy')
_ret_fold_means = _ret_fold_means.rename(columns={'accuracy': 'test_accuracy_mean_over_realizations_pct'})
_ret_complete = bool(len(_ret_df) == len(_ret_expected) and _ret_summary['complete'].all())
_ret_out = _ret_exp.run_dir / 'retention_preview' / f'first_{PREVIEW_REALIZATIONS}' / _ret_exp.fp['retention'][:12] / role_tag()
_ret_out.mkdir(parents=True, exist_ok=True)
atomic_write_csv(_ret_df, _ret_out / 'preview_test_accuracy_vs_retention_raw.csv')
atomic_write_csv(_ret_fold_means, _ret_out / 'preview_test_accuracy_vs_retention_fold_means.csv')
atomic_write_csv(_ret_summary, _ret_out / 'preview_test_accuracy_vs_retention_summary.csv')
atomic_write_text(_ret_out / 'retention_preview_status.json', json.dumps({
    'complete': False, 'preview': True, 'preview_complete': _ret_complete, 'records_present': len(_ret_df), 'records_expected': len(_ret_expected),
    'missing_records': len(_ret_expected) - len(_ret_df),
    'invalid_accuracy_records': int(_ret_df['accuracy'].isna().sum()),
    'models': _ret_models, 'folds': _ret_cfg.train_repeats, 'realizations_per_fold': PREVIEW_REALIZATIONS,
    'planned_realizations_per_fold': _ret_cfg.realizations,
    'aggregation': 'mean across realization means per fold; sample SD across fold means',
    'train_fingerprint': _ret_exp.fp['train'], 'retention_fingerprint': _ret_exp.fp['retention'],
    'source_shards': [_ret_item['file'] for _ret_item in _ret_meta],
    'ablation_required': False, 'mechanism_diagnostics_required': False,
}, indent=2))

for _ret_comp in _ret_comps:
    _ret_fig, _ret_ax = plt.subplots(figsize=(8, 4.8))
    _ret_plotted = False
    for _ret_method in _ret_models:
        _ret_curve = _ret_summary[(_ret_summary['method'] == _ret_method) &
                                  (_ret_summary['compensation'] == _ret_comp)].sort_values('t_s')
        if not _ret_curve['test_accuracy_mean_pct'].notna().any():
            continue
        _ret_x = _ret_curve['t_s'].to_numpy(dtype=float)
        _ret_y = _ret_curve['test_accuracy_mean_pct'].to_numpy(dtype=float)
        _ret_sd = _ret_curve['test_accuracy_sd_across_folds_pp'].to_numpy(dtype=float)
        _ret_ax.plot(_ret_x, _ret_y, marker='o', markersize=3, label=_ret_method)
        _ret_ax.fill_between(_ret_x, _ret_y - _ret_sd, _ret_y + _ret_sd, alpha=.15)
        _ret_plotted = True
    _ret_ax.set_xscale('log')
    _ret_ax.set_xticks([20, 3600, DAY, 30*DAY, YEAR, 5*YEAR])
    _ret_ax.set_xticklabels(['20 s', '1 h', '1 d', '1 mo', '1 y', '5 y'])
    _ret_ax.set_xlabel('Retention time (log scale)')
    _ret_ax.set_ylabel('Test accuracy (%)')
    _ret_ax.set_ylim(0, 100)
    _ret_ax.set_title(f"PREVIEW ({PREVIEW_REALIZATIONS}/{_ret_cfg.realizations} realizations per fold): {_ret_comp}" +
                      ('' if _ret_complete else ' [INCOMPLETE]'))
    _ret_ax.grid(True, alpha=.25)
    if _ret_plotted:
        _ret_ax.legend(fontsize='small')
    else:
        _ret_ax.text(.5, .5, 'Waiting for complete retention results', ha='center', transform=_ret_ax.transAxes)
    _ret_fig.tight_layout()
    _ret_fig.savefig(_ret_out / f'preview_accuracy_vs_retention_{_ret_comp}.png', dpi=180)
    _ret_fig.savefig(_ret_out / f'preview_accuracy_vs_retention_{_ret_comp}.pdf')
    plt.close(_ret_fig)

print(('PREVIEW COMPLETE' if _ret_complete else 'PREVIEW INCOMPLETE') +
      f' retention accuracy export: {len(_ret_df)}/{len(_ret_expected)} records')
print('CSV files and PNG/PDF accuracy curves:', _ret_out)
print('Ablation and mechanism diagnostics are not required for these exports.')

print("Preview only: fewer programmed-device samples; full-run completeness is unchanged.")
